# Nutrelle — oitava execução: o teste de registro DFood-TJ (seção 16.6 f)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py --kernel r8`; **não edite à mão**. Lê do zip do DFood-TJ (china.scidb.cn) só os 711 PNG da tabela, por intervalo de bytes; mascara o visor da balança com o OWLv2; recorta pelo prato (regra só de prato e tigela da etapa 0); com as fotos tiradas do espelho anexado (dataset privado `nutrelle-dfoodtj-fotos-teste`, cada uma conferida pelo tamanho, CRC-32 e sha256 da tabela das fotos) e só as que faltarem pela rede; grava as folhas de contato e as previsões cruas da R4c, da H0c-m e da H-cap (So400m, 1/4/8 orientações) e do F5c (1 orientação), no recorte e na foto inteira. **Nada é pontuado aqui:** a conferência a olho e a leitura são da CPU (16.6 g).

**Versão do repositório: sem a tabela do teste** (ela fica fora do git). Para rodar, monte a versão do push com `python3 monta_pacote.py push <pasta> --kernel r8` ou passe `--teste <tabela>` em EXTRA.

Anexe como *Input* as saídas de `nutrelle-deteccao-r5`, `nutrelle-treino-gpu-r5`, `-r6` e `-r7` e o dataset do espelho das fotos. Kaggle: *GPU T4* e *Internet → On* (modelos do Hugging Face e as fotos que faltarem no espelho).

In [ ]:
# Parâmetros
import json, os, subprocess, sys
ORCAMENTO_H = 1.5
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_teste_r8'
EXTRA = []           # ex.: ['--fontes', '<pasta com as saídas das execuções 5 a 7>']; ['--espelho-fotos', '<pasta com as fotos>']
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
for mod, pacote in (('transformers', 'transformers'), ('cv2', 'opencv-python-headless'), ('huggingface_hub', 'huggingface_hub')):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote], check=False)
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
# deteccao_r5.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/deteccao_r5.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;HynIy<Gqih(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSQedqEo0ysxn_N%4%ns$4kyqM%f{Q8hcU>P?Zi%CUFlOyx%q6
<KX*wM+ojI<~Y~xyJd>0y@RxmSTd(@rP|(<Z&AW9kOu<*`m$&O2jikfTq)qU_QHJ33#ExTKx0|5N}g)zl%2yPeM6db1rHNsI-mrdPrT-;d!y(=YrJcz+y*!w
Go?}tH3*U4f$lf5pLAQ&F~+woghcn(c!afk)qCLDb8w3g5a?TRrFTITtwYOmI7kOm^dyqD1h%wpB}ziG?iX+}!<z|ShRx$F9oG}Yu_U<*D&dLB3Ob{-qPIlX
kjKO*m<)%M4M+?iLhwhA39*{Vhe%n+K%W54*|oj}3Hbl}_5;D7PtG^<?QZ0q8UD5L;6}XUhq0QV1+6y6qpCr@)s}DBpi)F+C*RJo?xK~guj%O6aw{daAw%UF
1cjkme4KiN-~90;+?nuL5dYh~hCj1-h&5^GK6@#>lpOn_&(EF*yky2J7UG8!0ujcBaL4lm@|yCWaSY7?1!83Qsg`QdzH5#&A=qTw4C@!&7ZJ8wi~~-0m9PQ(
VaGx~*j!d<dn{Rby9i|DN|z7w`R<d$u!!y=EZys)#m{}<<olU!m89dixB$UdihD?XOLv`w6I#JYs;=n-`0IG7NiR~4Uxy)oGwkhx=(~@~@^BKf41M>hFYMBY
7#)^x3H>nVNl^4vFffg9WA_0vOCwz+o+|IZL@ckrM&GCW_VoR!b2x~;4fhYpF+~&-t{BS}c{QL?-|yUDZ#^ZS&X$mcI{A&paXwn@jNn5u{0h@7FVd7YK6GtN
u_?u^ikF)v>$&Xp<!+np(m`JQ-xb&uA-VU;;P_~3xUU|x*`BP!^VA3<6^gr7fhilp5112e^=FH#xicXTo@Ak~vX1d7;^3ry=b&`B^a5X@e1M6bo{G-)xE+h$
g_NH5S*ZxzkQOX6ZZ~&v1E*g!DEVuYKCEnP1Y(w2^vB77H!w)ga(fK}?`r?zl@X>5ARl>^@OUs9VqC(XbAW&I;j9yAji0MSE#M*<{Tlm^npSw=8W2WsEvElL
3y{EZj2!R1X+pmJ%xzbsC|8LUtHV!U_S(sd@z-{6Z!Gr!^Macl!GW6a_Hef+#vKU|UHGAxi3*{Edm}Ns%5;M2A=VFxnpQo?-GGZd*=-)I(dqHBPSGzq`FhAx
7_yXe9EAhZ`6JZj(c!rc9~{aH19TLCajB?l#lLnXO@yVK`u0<4Lz_H6GRR^lE4Wnx)kMVilUrE3kaBs=7Mo_v6a{Q&zT|nc<SZ~Usu^d~%S&9azTjrvcX+I;
E8klnQ$sMTNBR8yH>JS6DPc?~y|UzvO?WlAq;`TB2!myy9-W{aZuPUMU2o?`>C_GvAtWa|z6!<J{WU9|(Bzu&9Jpb}p=%ivlf0lsI}`G{dS~xqDYF!(IAH^)
V(>WOa58eB3G!B#HRC5XR~w0~JV;FhP=h=p!_+F|XC5ylV@e{xUx1y%K-OSzSW8$zzKLq$L3Roro~U{Z64BUuAL-dIA9_>g)>zvVTY|NkkUffJ0&BK1R$HFF
hRF?Tq0Ds%n}wDF`7ov2&uOR6uV$>QY&&U$aetI7&4{t-_5xXy0R^7;kt}1BmlmvUd~BBp;YX)PeV-_DWxaHL-?}PB%&kprWE4C7ari>d34cvjxuGoHxhBkd
OwfQ(8r;W(U}st=ZYwL#aU}*!h3ssnB2A7IRsx?lJ0BXLTQ07M58Irjizu;mWCe}h=^1q#Qetma-&smnzCx(D_Mg3<Dsf#eXn1-CeJi}v2}0&@+vO5GjL&RA
+Dik=sV%Cj5ObK<eC8t~{@I1;tYWrY=+{{vyUWZW%Y{n1ZgdPH=HfhpEX5j1KLY}2GaCx4{%QW8y?93GbCBe`^B2G>?_h}hZte=NDI5UT1WL8zr3tyyDfDB5
i*~0uS*)H4{}!?m_De-8VTO}h+X`nl<zknB8->nRa>0m7koOPFmG)iQGOR}K4I+T{0}~LYbTyvKb<nkhh_|DT5x8~L)~6W91xf=5uwP9C@dRiYhgp#$Q+RDq
*#U~<1pgIF#)SnGXu#8Ir0QK1ywfI!o3gm=DQ?fl8&X7cJ~CXuZn}NDg5?`kL2ncA>Ae8j^5z#ykKEs*XJD0&vE0kGa!|Sg1FDczah?EJ;q-XJGv~MrIdYV`
Kv@b`?!m#65!>AUiLbSGtZn}ML|=t2YD&__apg^fy^mco-^Ro*wtQ1>x`W$qxO7-<ig!`Yxy5=WIXbytVblGta?5O&AL41O>JBI<e|Gb>(@)l4c`!IrwfVk>
VoChtD#GNCKu|=3Kl<7NIhpvsFgwmaAv6@(4xD;iC-b@vdY7sxLsagY(G?Z1>b3Ba#ew4bh-a#a>VQw3HleG<F0l1$I??FlabFk23Qv7Edf^&?NZy45kM0&_
anSPMGayik^{nHF+@x0d_Fh+JG?Jon$lMCTLH_B6IY$pC<ZH9h>Lx`K=~Heg%;6RKj(V+l2k-I-^A-FlJV&vijo77+oM|BGeeFiwn!piur8is(*m7G{RexXf
Sz!t{?1B~h+XUvplgE1{Cv6*d%O8Swt_j+3HVDQeh*6p<q?btobbn=fE*l0v)8Y26S8kcT;x27HQJCcQhTs0sCR8;(htO=5^NNJjzr(2<0oa|C{SQNPAV|qm
n=ggA4_F5G5H3@e3=wnDI^koCt?2&EoRef|6lH8cb`c=aVF_Rn^e({4m%aB*3OP029Mk8VGkcn&6ouw-;x9<=%WdtLA5~#+Y})~4_O;A&m65oMsii4E$(W*9
HSPuNp2TeNB<QXDEVWAM&HXs6R=wVi{M9T)tiBN#y}3#oFw-gCq)u}4%t+*<_WVA6jowMh_3Q7?|B80rckD+t1$b!x(I@yFjEH6SwQJFxOmPYjF)6Hlti}-D
88FVEfQ6ar`b_QA@q5OxAn)clLjZ{d(6Z+sT3g>WIPlQcw2q3ReqRm|GulYPgV>P9aAWa`?mHbvv5S#K4xaQRJ#6X~2NUGeKS+<AGMODa!z@4&#$F0nkC`30
%qh~~hPodXn@Zmjk_)i}GT9%slw(Zg*pF{+&HilMgp@djeRAja<_Xb^Rt^=m&pxoFI@dy-Yxj-EW1k&|JJrpWxhcDtMw=Rmx=;G)mez4R6BAQL4V{|Qh2~@$
a>y&}Ghkk>NtYr7ws1K^XQj2q?-OF7s{HVy+EvLLgE^pJF(|YL(UC48#Ejg3HrA9D=1lmU;)$+QH%FQ%uX4dtjHhDuk;?@;EZ*uP!u1LP^4Yj6`<6$pJV&03
ty4h2G`^=95D6kM@&ClB$A$A~A~s09sM(ywroya0w59`X-l#lRc|@k9K&FIXXut*4{yPKMyR70p0_h@_X~?%%XAAzaR}_5BZ_L+p4zeyLECsGYSZhO49DC?(
`~j0X%{}B*crSw?N&{WQC*eBRUcSCJTyHTXWJ#W~s?Wh2yvP_20?kkXD8*eblU)h&$JnBO-7tbmY|1vJ%<EQ6;wf^$uW^KEMP)@)^FL?qF;<(aM34fJL+iO>
$3lpw`c+*jZ5-1%`D;3fBCZT({OL3SO5-z9u#tE5ut@p%p99(IC&ESsy3~;&i0fq*5Y*^c&5NqBPPPZh)<|xaAEv5us?9?2@>XM2P4nu;j0y3fNF-wS#-tBE
ZW7*5l%BVMW>qmhx~^kdJ(byJuU>W9XPk^18m2;*)&X59MF*S8cDM++(iZ3|Q-D7}F|=v^Ui6@SG!14Wp`^2J*?b}MptV#%=^U4qfB~I8C)yVpK=DGoWKx{H
`o$Q8=TviM8O{Oe9&!lp>k`5zZ!0uiPF!h2M=^BaxLyyBc^N7{6ph2v&!(t)IGp=(@ySUPw*!?oReurls`W1IPM!6LWNN!0(`vb0m!%I)H&-e5hv^S|4jhxx
)-vbaCduf&wN#NZEC@}G_^~?+ri!4!DzQI8&?#%&hgBZ@6bpDu)a)mu_+)5tKL=PReD)9{i{d?C$qwJm$&}S20G%xWh`<cTs<Oa7t9WugA7y5;$qn)@u+juU
mbZVC6KQ~8k`!>6_0Kl|5x_nEz>wD5OzgVvxnM^hwhY+G%jAr)UD!$}KP>{J(9(y(clHDFVsgrd*|`Om^&s6Ps!J?jeKE|!h!~rrfyox$@ybviAbt=j{^bA9
>$xfrlA=bu)453#raoO>bxG#0%JYsbiYncT7BR*;qIVy`hGyNJs_;PuMWZN4Hmr0Nb_#0rZ3)M0Y9m`?nFLvT0Q@qCk48v*H^GcpTXN|sLIT$NNj%9Nj~*#|
5Ui<|`eBV~6t<)IjBtz=bi6xKYyUv3T-0eC5Ymt<NcIg=jcBobD0NKKHi0j5GWuVa80{g)Worp|u>SF9i93n#w=uM+$^f>8Af{KbiPU>vL#tH|V`UOpo%fAn
$)>cFDE(Qc?X?sd4U&YG2I-VF2v2LtOlm7o#7Rm1$9+?5g_Nz_7+%(aj1o?u+@fe#6o9Qq1sX@beLO6?(Krx7J~)N)kFjkFlQ5%M1@6NPOzn`(PJ^AkFt+8H
VZ$C->w8s;*ZF@W%gPQYg@SA%rUsJ(nKnh}pQLx<cT^Jxoz4}<1P6&d;FDS`EzRYhqYn))Xtx(t!=qH@mq!eoHF%7oF+xW}XBrof4IFv0M%zFVmHRgfh7cB{
=;}MNI86Cb*ETu8xx44Ho3j*SehXt{#F%9>3MF(D=lF;_RA<=4G^mlI85O<tkKj%)Pz$c>+igX`bUSD&n7Hnq4WO|Z`gonNio)aHN69u3Az-fbZ+_&eI#7k-
n@~gMZ9HdI9ZZ8aoEfA#srBzH*)(r?YBlbMI*xY+WY%tLC8;yMg8ZFPd&siqo)a*@JL*^_pyP1;QDyB_K9j9u>U4zOcFj<Zq!OZbqFc6$TZ;53g*0my-GYOv
4=ck@v7^FL5oOVBa8P==*??r|Ol#dD8fp~$3;;~)n5}U}m$0q(hNo|?LbODw%wvsV(?C0JLFpM7fE}wp7<rBZ2`*KAEEG~hlpNTXFBx52JePnM=nAB+Fci~u
BWzf}ONe066^O*$)3CqK3t<?RhCIf?2!392AuzysrHGVdVpTQwn1WmAW^Hs?P{iZuv-HFrh5WGw<CtBeey5paD?lN@g~lGLf6`I+Cj_?H5=4^N5<-Fw3Az73
G=_~%F4WmhZaQ6IB~Z2J83$w-@~y~<wX@t^ad<GGG=E60rFaOy!BxZn2U$53_)t!}ZKatqYnMOG6@bOWONb|hZX4fFpJ<g6XRDeV<!v-n!{(f34U0t9s@1aY
>vD#xR54}N_}NAbaVnG%!Hh&cu8t2KXuY@1WziDb*Di|#O!2%PWKgyM9VTf;g-9w5?#jd<i|C-?Y(;pxp{~+#3Dxf%xdSycufNR9tj9c831G0{vyuy3oE|I>
as-ex49L^977f`9Wk&fT3hYntL~qSLW&WCzb&=_ccxrxt)00IJ7GumEmQt$e`FVu$(P%LwxE#~HE=#B1J2E@df(njBT%}YeHsd6gt!~9Kd2i@`JN91;>n7BX
Z=}ap4b{%_(jiwA%3qpz5n7}mTT#sO)-Wx9_L8F1-PF%&v=~m?$3nBo{MjSvm+DY3LPPz9I1_Gv%!v2m6e6n19HEeQB~UY4=M2Z%z}N9|)aVU4?h^52sCyDX
mWWNu6^1#IAoPE~%Gz6-aL-!F)x+TsM0pJg0cX)4l~#rzZe=kOpHK`#^an-1k7WXt;U)LQC1^`QQ(q?T3G97Yi+mEsIyr4#^2>Jt#KiMSOP1_#DhV``YUF=i
pazF&)pSODI@1zhjL&+@M!{&Eb;H-Rq>+XXOtkVM%q$|o<y4<0^nvO&<xHjDK|bx-6p3NbH{>o94%7{5fCrRtio_Zf5~4pOMTPcnMJ1-F(O05c77C>-%V>pH
fJO%qHKr<t(G<78g#SE?Zwb3jqmEvt3vYN>3yBH^I{?YMXIGDlw%6F?D`5&vJ12&T&Lt67#@@is(FJC{>FyJ#%UwNjVjRF&X|97A(|SqB6b*9VmUk1$ijmo4
ND8ud3KNOzxQMEX1B_CJ5RQSV8~V$HAHEMTIywTXoj(|1kX#?dY4K+7LL%KN`SSSD8I5k(WDG9AD_dqSC80-Bn5jRH#XQi+1gQRYlFAdBZ}=#t>p$R>4$Bma
-rNDM)*;@ym=Ed(cK$o{pYVt=pxGKHNlO5srE9Ai(g(4XV%&mPP{9cV6$I2(55a2+_*kYWfZ(8z8N*~DcTr1%fq>t^SmbtE{`Yhu)<q8=s5Z=4I(WhE#tqv=
p0~$5|M-WZB~GY)+~{%g;0g{Fi;rkvhFsI=(v`NwjMhS^cGp>~*3)M3Z`^JdM2%M;%T+lSps8TS`Sf_}iHV8_sLSVw3vChh>c!bjQzP|s>c9Sf#N~qGBQ&$)
7(S$TeAP{`bMNF`eryWC%l5;nmj*)&0r3~BgOtvymE4rmYxU>7`qmjZ=xeq=sW@ez5cHX{GGxnAvN2ix;_tcZ97@@S4G+UO#&J|Ydlf<sSk>gjg6xZmb}M>X
{QC%SGkwWWT;pT+6e076??BABet_HnXH85tPX1S6Y_@)`sFklm%9!}CUv`+bND?mz(B*`!@X-P}e{`b3QZhg2;UzqzOc;WYCUGbbwQ^%uzh#P?ZX~8hy{k!z
C^9KwWNu}tFOb%c6X+eNZ671gOJDz&-2Hil>a88LwVETm_yoGKi-9S*Cse<B6g;%5$$iUzH0_V0!sS}HLN(8SrfR2k)uZqWxl|rCPwZ{^beZ_TtzUe~KS1Ch
<^nSFxd6Ba_!gKrEOHeDTKWkoLkvJiS2VtZ{KL28(50>=Kc{Ofvn|ZoPVDUuwU_OB-iQGVyyM7WE1pFH<GiJOib{rkthWM94_jkd72kuDPRw~&{8e|#(AR$g
;HuaP_jE{=fPILB5~_>M9=$0TV5hU+Tzu6xCTzRws+RF7rZtH$GLAa<qSRk*7HO;^bfJjE6+@;gCWOw$XCJgdP?mQ_ZhYj2>&Eq<jO}=^braR4-rzJZ!bJkN
VKff|SrU`bBb(asStbw*x{5o@Z+3E`5t_{g&ylUB?3DgMiLW1$cC(P>d<xn8+W2u=H%_GF6B1vIU}87#vCnZY?h#2X!}BH%@THC5B!fo}Wd&z*jGu9<)(=$9
VwR>`46HZW+fcd9$@I!i+x>=}p*N3M`ldf2xrlguryq#N0fV(Fe|jTJ&V@vySDcU40l&a_J7jX1tTB%n<SQhu!Y4=$XAoRUi{Q;DNryKc0!>B3YwXLsYVRVP
K~D&&2Qd}UFs-J<gA}&^w?i(4R4p4;P{+%EWXvE-!XChR7o7B3Du;KJIi37`8}TN+=?1ZivFM|Fp4Rt{uGA#@S|F^j)9o7D&dCK5hU=f8BvDc_bi|{*m2)=G
ZRshoQ9r4|8F5wJo%~!UXnOVgk;sg}rEflZS1Kx^?^sQ<<}%sAsVh|D^t0V~^_Q1nZHd;(8Bd7q7dBz`F-o;lQzJ-O>30=r3qIO9J*+4dp-~IUPvbaOi4vCP
{a%e90B`!*zOAxX#G~RtWVw;1FmV7V!Gr#~Dds$@z+Ul5SwIQElE|aqHMoXvJXPrN%=jfpc13;y3<qUA{n~UOIBAr&Cm72Mw&8m;ivZ<q`mS)TC0KleH}Aob
5JiOr?PCl()n9SMI48tDyy%;al;$)x7?hc{_g|`*5l4u>vOC?J^y4D&nDMR$z=&RkXX@uky;7Nu{!HK_!^0W@wDG0<C?;?U{BIm~xTL8zF#KZ@;)kwP$bn`2
<F1(3c)_tp^NgC_l~|NH*Rq+m1#}tZuW9N@6qH4|APOIDXVC5Iw|kzUQwcM4?0~YjzVfw8Q8oBZGppOj+a+RFl+8h8D-PVdJ4>Jo_*y*DKP)JT`Z^nZ6wqa>
!!wqWaU*#Lpoh~BuadG9;7~t|kFz=9x4%Tltu(x)pl%92crx$2z@@jGl|v=KSVJ~TLDrhdU`}*@TkOz=H0ABWbcr{m0`RKtlj0GSgvZf5;W7FXtJ-2mqh<kQ
`=MpDve3yXVARTEq5n)P3w~d}5aN*&H{Ym#?IjLEV->Y8<hU^)4%MIuQF?{RDAv#WU%n2`0<j#R>G$U<ILF?t5zCpgUfu$E5lW!}tG=M0tg{Yht(N7T)%f;J
DSwPv2K0Zz_vS4)+ipEiTWP{nhz%VUZ7s((>A$+m((Th<lj{jI@!Qv&==Pc9n?v+jvTk?vMemS#XS_h2)16}JVniy{ls+popaIxgAqGk>$6p!x<RY@;3mXS2
pi!FH7rP&6K<bxekRqNHJz>r_?x!qbdD5P*`>pKo6oqmCI9Jaf=*Lvkby)Pi5q7_$X1r^eo!sLNd(HFn_`|yPolFj8XV}4g9uRHtf_pYG>Xp`iXw_YZzENHB
V*OPWqiG1wB6Hj{XFn82+-7}pYMHPD?I0UmDJ0w#<<fBMC{s;locxIIT67eaZpa1Jk5<sZ5$_tiP~ws^EQ4sN6Lna4B}axU%5`dk+o}LCeoR)8onBY8E%-_D
>bXXzno>G;!G1+Ngk{t(@FXUy;rLcuv-r)cy1lDbf{UJOkw<c~v+Km{h=nC>-=j)VN&u=gX><|~)7)GFPcA;eeYc`G0S^T7$^y!B^1yK20f^f6oyE*2<X{Q}
D<w>2jj~E*Ry$chaAI)sW&}p9J>6}0^YciJFMOdZT;&j*FM!sK^Bs{fXM7%2aUGdre3|PNFCbaO<`MHp|6GQzf3=XHZgB0MvuJTfHz`*zo6*_JUNlr=QajMR
pUy!)W6f-{J9F0I=#cm_CK-p7h<X}*!WhaohkcRlyXi*Do=E3A$(hN&w6kjynUJ+<K-HUioBd^7e~nG+<cgwrZ+U2!RuFZy>>wq+S;Vi<*V#$wB$y5Oljo^W
S|u-(gogjIhiXLE$_;^|^Tl4cL=aIRWR(A?hA&F&g9e9Pea3XS`<DwxIBi-|9GhNU5Mt>!Lfv3vH$dg!>e3xO@B+ZTgOx%fpJ^Y;Zz*Q3ETQz{!V)n{mVtV_
G9>j;0TG<-nxQiq(y#r!I6N9zJuQRa!j;7f_cEPUG<f+l_Fv8^it(3p#yW;3G<&sfun_*2MxMqvCA1ngn6gFH$jY+7I=U>n7Pjiy!^{q)Xe|5oq~ACP)hzVO
s(pa0dz861#hx}&sAG%#V{RM>8+OrtlNp@H5IRG^LFpTkIus&;sfu+f4-$6!*Kx&i6!kkTgwy}5gK!qtBneY{$zs?Id7Q*`fJy-A7wiFtF(SmqLOFtZ7&%!O
v>Q{t;UX5=U-+UkiDJH*YOpZ_?OHKn=(Y=KL5q;(bzzn~%nfezh||@~+A~`n8~R^okGp3@frMt?i-ShO_r3nbU-d^{#h9LwQgdY$LzX+dVu(_&HxZ~`)r8v=
#>9Xy_~XCYLj!Prx;4*5zpEmv!6Ne@m?G>)Mk`*km@{&RGLoOJcioWh!WcY{puIsijZ9UdfzJU~E$V3>r5UnZMiZ|#CzY!rCYbVMshqPu$BLJ90l>kGVdjr3
7xk?|N57G*OASxI;qW!0{4w|OHNzlvOc#yLl)Xv>*=+atRqR*0=2zqJ<ggqY$%vsq=Xe`%?k1CJe}=va_Rijww#W2(l&QaNaSns}5jC!J3rGlpETrP)8o&@g
x}M36?fPashC2-tuT9a#OTuA7nE1YT0eBhKZ&%Js%;!X+l1+U(Uqw&7NnuggG*~lnT+7Tpp0|DnfZBw)|0vLfWG7<YW+p<ocM?`BxTBDmM4ue!y<Ub<)|v|w
sj(lA_i=jeZ`DW^*kenlHb6P(y=q3=gh|eFB^Q#JiU*nQgy~GmuSh9aY$n)vONpx}^U9}j1Hkp<9Md^;V^HtxaK1GSAFX?$r(g(@6U_N|4zqs^1A#+y4W0&=
-6p^(yBPTciF=;g!bkV1g*8IO_)?RPKr)aqa(gxqqsv%QrsT!Ffg%gPG~%Krvn~fh)<^~m%5fqgPL%3xv2RFG1kc0W1-6y%kwpsJ>6mxR*CDbH0&jW>K))1U
ut_gb;iO~?$x6^pOBHmA+A@D(qCDXn>OVWVExzbW>L5XYn5DB+RE!kOrX{(EL}`eVL7|NsQ^VGkFuhE;a+1oFfM$83Re$YClf=&(94PFYMwlDc08wk9v5+b-
n5Ze5N8-lf+c337F|+gZu|%R5)dvM6PNEW&+N%?4lI!Kme_*4FmWZw*$Jli!zR&I3!g*~yEv0xN+YFRO$5y*X14+C+FNhO!ZRXyB8yI&f0AOuifM@)n9udS6
Jm1MeA;6+5X9)r;UjpGoCV9P7b|Mt1xmi(<DBt-?(Ua<EDK8GMFEc#T;SrvGH#?9YvpJ^1NrqU-emv&XWD$!Sbh9j}HU3il4#1M$hgF7Afh?*nkq;gZ1jss$
Jd$HzIZm=>^1aVYJ%ZUbg|i+S8B#LAdv4t98H861XzbRkE*4T#*h@>3H`h}>1#4WH2|QAN7#!{0=uSNb^nyT<{9p0le#aRR<V<>Fw!d-U@75_EkQ<a&<#o9@
k?VhbKx+$yrQwiruj2OB>O#LfVXN&8TvWx6w)%}juG}5Qd1;buQEum^Q8>tR!Xg9TT*x_`z)v+1o?buEg<C6)lH}}j-xnMIqE|L0Q7RR-9DeW*x4ctsogAMM
WGjdb^gDE(g<A8Jc#3_8j|jwc7t1zr?*I4FSK+fFL%Ev%6JwqvG4Y2y@nPrl=2gsBDQ?{i1fxYQuSZ<l{-4dgf=ZG96YYa+tao=MLA{2bU-L(E2Ay_}tMyxe
Pr?87@9}9xaSK`|nHTabl&FA%y;#)q>U=hYGQgj91p2_vI%8Rlvs{L|-T`ij{k*tmUZNm`=6rCBF!OE7sSPIGaPF)aA7kPP4ccgkqfkJRQFgED`0If6NBzMn
psk!`=BX5!Rq96*?vM-n7EY3PWGe=gujzzC{iiU`SfT<!x#qdXgGHO*p=|O@4ujHWWn!-p4V0~2uZ)q_u4->OPx*O!!WChF{>!-blBgw-b6~}608=Og?)q`M
=MT}xjuzZvSr{J*Aw)U|R$!~TN<AiyELjchU)|tt)q_o?Gx?H{XtH0k(A05%xBboM*b1BfzBrpkCo5eX<1e%Ljv&hDG`ROiv4B-)=(rK3PA)%I>hB~IUMYq+
iSumOHAuD71o*6WFFFbRF!-iqTQXn5xsBzEm`D)R4H!^c6FuY;bfWhmr^qCU5aTyPnI`*J)0hiwBlUJ>nt1(Zfe10iu6RisT>#d6)qU1CZTS(Hz1nH^Ujups
Ei~ik37{tQEErTEhEZ0#atMNo^d^2qOkQ)nS8A<9fx38E44RdeWy)K9UbZQQTBVJ*-OXLK1A%3-{yRZ`WOoWhK82_pP(|ND_vEofXnS&_{e=agB7sPRqf3Iv
+X=~#VU8f0Ma$JY3LgBR|AQ*s)N^KW>&nrsc4JKiUG=oKx4OOhnFD5^r6A&f;h4@~4~2UVhKt+N-Aos7JvyBr<$C#bu0cKnDw{IAZjlBmrgq>527<hA6#+Y`
)vU!43RwTTAja#WK~H=$RYqUrZo^|)fspr?_3B^jSsD8`zZ1pDre69YR%|U_PPH;Sdp$}g8{|)V(vz&e8IBnNMi`W&f1qR$DGYLthSx%qiQoWcm9ffYiyjgY
ph}sTV95O#RGgZ4veEi|00eQSaWrjG0XBZ(R<(WSr$i=nSuoi$UhG^~HNt3#JTAYYuS6%?WMR45u6sT5{!nhL)xwy)jM=qMSxkdaO$Dxn!XY3ZkDajtp*?k9
7>-kBH`V$V??a^({u9;Fw3n@9KCJ%~7YXHsw$D_K>JccQ0?d=$m}Od4T|x)2YF%^fk+)NCB(n10$H5=6_I7pMuNNCa%TsQjHaSR4PEE?T&nb<#2_FWmC-3|)
S=_*~M?g!d&nobK42^iUHC+3(u*=@fE&Na_Fi;T|JOI}EH?D&0S;-Tip*dmDn<@pY9_60%3b7&x-fUjrZrgOAq$JNU*o0+p2nc|t%{cI)_?fZW@gd7iiUDKM
m7_MyIjV=CigMhbd4&w$aga+ZZ=f>57?^%dkHgH&bop@YEr}?Y@~5zINKLin-jOJ0^gf#Gh)}W)*aE;xZ~=K-95itt$j4Juf^BnMq%(MXhJ+tJH&*_>s4_Dn
zIMi&D>Q@*#0}hay;&II^J!YHn*l=+&OcRT?8sV&lzFr!pR!`yGn-V#j;LyZ$Qa+#^v1k33Of{W(Dclw$sGiUC7`S;d%sCYUEnIp9Ftx)wbl$}*WsN<y=6&J
6K<oc(&t+ha_AoiH6I~bVKvEuljDwpAx>uEDkO?hAwA(9CoJ|m<8O^0-&@uqn)>Un9U0cu{(DU$*im+0yf@^L%(bNe;~F@R4m_dQOc@m0xgGT*AD1|w6Udt5
A_lT!DF$|5Ji79|2z!%>>)%Yxhx2Y1Q4rf?Ut4C?9$3HHzz%tv^IU6V4%*yvU0Xufp8l9Z=I@#Yo?U5?_!N>{`b7+i77pY&3=eca7yLwk7PQfr;AJ5r-08kJ
uf!43XMdbf!(-TM+Do>dvtgA(k)(LZ5=`sLC1eTA6bCJWBH@Xdaq;{55xek-%L?%v<e`spy4)+PplK|;?qimWc0H%m*am-19f%bNbXnm3L%B?$bqPfY);pUp
gQ~CMKst>ywJWbkbNkEbYt8&<LH`X32(xFsK6zED+;+2>)u=wfKuP0U(qk?6AXU17^K_%$x$R>bNL|8PEAzfePJo!@Pr&G|VvF_%;>N4>B`v<T$Ge`-MbC9p
?4(KCTpQHQm6U=kYfKQ2l4vm}Rrl&?4HTtGVtE7NtoxN0FEd6pbRNImm_h3xYL=Dn{lRT<QHE^Taed(CT~J>fMy^EGf5RYz*=T37)7knhkwQ^8s$3WbFap@0
>B8$Xs)vB81kCo8!fOVd&V4<~6|QIKQWkGt@px={ie35f9@HrdxYVwd4++4e_=M2VWm0;f*Pe&cA*KK*gJlW&yuEDOu#!BxKK3781N<ngw1x+z#q#0bnGpO*
R@Wt3{M(PTlky2Qb2?r$<>7y>9aF}pYOsvVEA01oz}s^=C>X+R{YS2h|L*AC@aX#jNJ!{cZlnoaJlaEU#Az9saenQt(ces?qoXY2R-4*6c|@n=BhuC}wG_sN
`z7#Z3wmsPhUdXUM9gRF@kZmQ*ZQ3aPW$f9poHq&VbgrULgl2U1L}v(?vcz#!d@T{@Lxw#0Q(;OGJ6tllDmLEMj^-D&)bR#{$A4Pny`One`hUw8lB5=o(FLu
02#UGrFYWjX4$e>M2Lo>z~X@Ph1QV0G#BRS|A|=|C>8iWgH&-LU@PB*g689CDc!?u^yNjL>V!krvC|x$k|b?y!l1>f%6;UsRxy|=6=J73EKz4f!Z+8F2vY*a
QqVY|Z^<=9_z~bo4+}e36tf3c;0tKlo0r*(#Yo%9UAM0>Z!EpVaHfff9bjkD;AR`Ft@BO*4Am_f(FuuEJ+%xWs_XF?W5zK#3uS2|<V2*3)u_U)NeuvRM|=g&
dFg&Oss3_eWi@5lfTBw1su#*mQCV#Y)=_QkgjmnM8b@TDn#c3f_lL$F;DcQ&e4rOj@^cTo%85}LQEcBLh@Bjkl5O!Qe?E@MzBP^Z04{tw?OE9!spx^p^*J6F
6Xg}}_c6m?NhC&TeDAJ3JnG-zl?7T~9tuW}u?VS7#L7gvmX{?oM7Hqb8EWp6qQWV6kwOlC^ZsOV4wO8#ghAqa3AgO^U(I-PQ6EL>&E4jaMdV+Br;bN<l2vTV
lN((YWKR`_zOF7XFS!kBVt$%0PybN0Vf_XjvlppDiHzhZ^_az`FnRB&JP%Xx=G>q(-CM)k7=D&C9>NpS)?=+f^=t@<EQdJr7$%iIUuQ8%({TVvtN;(D&LPR?
LQp@B8gH^@)416s>^_?OEi;8I&RBG%ScD(!SD#!-K*io#(o%CvQL)u?fqTtvEl>U;^KSCiH98Pxt~pc09?2t&r=jiUxBi|cY^kbDa)UqDm0)4F{(x;Kju3O0
&Ad$RUxe8SW<B0{OyzZRwEyrE+y)2!2+t>{7Z&|-iU6IJRt_?$U^y&rQav5Flv4TF4+da=gp}0q?BU)|w93hnskATm|4~w$g9J5mbr0^zjYOX{6@!qB01~d`
1y~_)R>6HBsNR+RHzw~|Swg;OPOqK%08G5Qr&yeUvHnyy8*U{wg3JQdJqYOe?ujH-EeQX+{eILyC;rkKeFZZXjhNJ~`8so`{$Rzk9kBlxX@4gPkGW>c+|dN$
hrG$eNZKnL+*&FFK%(JXSY>gA<j{lnOLQ2mZ>G7ykla3eS3F8^6Q^(GxGfUitBakaQE(aHFrRc~CwRsdB%=?ON%b+sNtpY<{x{pyPDrBGjlhV|8i0M`I9Ao>
f8ArQ-6dM%UJJrANXGo-PkZE#g#xb7k)tJ{==PDGS2pnM3VF>mj*W4wNnHIjB~+%DV?*<-cGq{$YzMHW2dcz_6y$u}?l3Yuzc(vpaKFLY5((F9vZKXCDr`3j
B|M#=W|<xNZAtsYv*#tm;LjU6>GDnYnriw@J7Nh#dbJ=q;4d|)1f|oFsE=wk>N9c=T@JU+Z*=`-noV6k<J#oRF1&#__QgI<r7QrW^m8*iN%F7n5WEbZ2)JT3
7$a<t+PT+pRwuDiTGblAI4+moMjB1pg)?*!8#CiNt2|1qWA=J;b|IK>x;=}I3D!D8m7Z3%Z@V(ekdZ8>uiIk?_SFLRQYBGgpEMS^AYuC3K|C^2_(xNLw5I{^
HZL%x;C;Faf-UUNz93$?dsEE&|8~d=if*pU7otx<1uLQra?8V8<Dm6qSkBgRZ{gMjAY>#1t;k)t#`wHyVPrzpL6RgBQPkU4T((GdX5IoMXBq+smXte1!r-LY
Zqb(XT!h|dULnrsE?PD>m!k+*wNY&D++@io*uOF{oLtdoi2CE0$S-kd7iQy1H0vMGKX28X|4YLr#+RkC-lkbjd?+OT+&8l7`QdbWk&^fliMa#_oL+O|JMWRO
$-Hno_s-t=QD^^iK*l-<;^ld5vD~wq5R&SJvWj)lP2-@d4RCf1)?JhKFaxuP@%}hUe0|tCW`LN-RBhw(i9~FO|KOdejE?#K=-xqIk@vGaQZJx~3}JveSVDML
;8-=68wf=5%}8u8^^W~P%dnpUed%*vZ`a@BH5MetHzc&2L(ckF_h(h3%u`?;E>7HnJcA<Sl$rWU9wLcqAN^6_Jf=(Q={ReDQ<`#E^ax0ZhWlTreAL$t%D3$s
%7jP$Pf#?^+V-~P{$fC|0<utY6F)A=E5c>O`w_F%7Df%&EYpJUT17VBs}T6z#0)O6eM%wm-_<D&X&GL@H@wO)gV0ehN~#ESL^-m}a89xR7}J>Fh<;$$4ApBK
MM@W@3~o}-a)F%2>sU41iWGJQGaRlW?|SCHO>)1QgRSdZkd?S3*+Z?A;c&_WA`axx2zF0Qnj^-Or|vudSPRY%du)uY3A__zcc281dmq0ED@1ZgIc)E!Y;bn>
5`3(IsKP_&5X~=mUM=CpQpUfD+$r~mc<4qIXErCVApdd%7WQ@LY>LlwL4u#2DVzr%Z;p0P`DnW9eb&!B<4F|~gev?=jKjQ#*jK_Nug7h~wg+Wb%VDkf;|V<9
gC{2q@;t_dwBOiw5=4-BjdZ)=1JFBkocA`PUjlPOONL3RmTd7mMI%D?g~I-*(;<6igdzj9-E+LB!~k#^3@Q#t^V=0A=a9*eFTT27H9<>V3?O&w-=oc1O_nat
@+k*K+Sy<r?qlJu!#bL+F@2VOm|Evj(cW0B<oZiDIxz1L;lGR6Q{^DU=aMf#jo4gpk3Y|od&&9iq1KTg+mptkDCpCFWQy#1n>GEtQF0+@)>M{e>kER-K_Z@B
$Iuhda3bg?OIlo|t9oR{i2~Af45BE!>F8tC7se(?xZU~?yDvz_(8}VWu#1Sb;2BQ~uLYfc=DmOI1PNsJd~NXkLWkQUe6x#W34A3At0X?YL8mN8RH7!hhp~r-
Q$d$>z4n~s{DFuZNyC=6q_F{nl1I@l?@?suNl#op8m@Vba-2-+Zc-v#gNcM&*>s&HHhDtpRl*qho-(q0uKoF9x?bvKOy9<#LxE;_fJUciHvReBS#cY<(F7Eg
I$&8o9p&<X429%AolUlR-V`MgUvd>*;x+dzeErw^e`dMBMd4Hi{fXl4N<+#jRbKK-C_;BpBA$TEe0<{F*)CSo3J&jqx-gSt3gSKw+|6=`ZunHwb>#@`ub)L2
IAB#KS=1w3fss&K<-vq;!eN0MV*KE=H1<ggsQAGC(JvJ;uI1g2o!gh+J`r@%aC@!PH&a5ET4m{-L^k+lQc7a8RgnG$?T#oxC{Mmi)MIoirJkm77ay2H@P!Q+
@qi-(yDe=uWO_PlF4lfe#XW#NjZn&o+MNk}3_npWvqjAc#T#>rPWPvGvG?DPm33f59?=L9H|SXzW^S$g1x)36hI#DJvo$ZbYzj3%<!had1xeoLzn2!khidBk
-e?xKcL4wpR?p(9tefC1SKai%Cb7}rK%j~-Mo8H#MT2Ix+$emgK_dp@1ZMV?#ApL_!{s!|qp4-&WRur}yV7lgv*F8B(7=83y}#=SeffNmskRZg1coe$O!~X{
AVY59lIm`o*|HYmDP4sW<0Qs?OJV$eboni$n%7-^x=PiJ@#*?2MU+=!KWpro(qP$YJP9ZiIjY7QRCMhl)iaRU#6f{U-EVr~9~k>GXgu*GT!f*MT466Wa+@s6
*?>C+=!g5*zy$n3I_|nwOkuow-W<xDLPPzTW9(pcPaCug>imJDZ${h(#3cJdQrp>{9sTMQ3(9$6jh0j}=~+9hd>>Rzq!<J5{Tn{!nhFG>zN+sez^>y>dm2-p
5{GQISg}o8{k-r0?xjee(pa-<c-Fmwck?%9w<2hxaD)Ze=b0>Pd;OTC`;mM;U&n-nK~yuk#nZ3Pf?~@%8)xlie5(y`N;suotaki<iLy3@E(A9xygVFzU{bjV
^i(XV+H%2yW4}HzozP>LZ(SZrOm}iwstBY0mu8nrReYi=rK&BJQ}jTPs=e%B0GGarD7|cBv~E5fQR4YS^b56S2ARIfgRCg8h73{-IC*<FnhfN1^^E##OFPah
;sp!rCr3*st-<$tQ+P*}x&6bVrc%fNLQ_rJOBV!yP<;mycp>zt{k#Ml0&I^co_t&?TLIY(usK-RnG!Ok<-srxB(VpwND2}W$tOJN#HhY!hH9rV_Mq|xa6rWJ
WlLysn*FMV0m%p{!*(aF%Wtn$K6RI5>aXZwCQ2Pl%EGVrGKP0)E`+QhK(Z6%xH{EqajDxTN|f+>SSTum^GFXB&6si5UC!l(az+`F<5|TZu%vvTMW5^wa~!MI
2KAj0kI~h0ZD1(yS6PngiQg)@KO{|A4ybptw(jy$G4j%|SZYC!+4~atVSFg?77cR@lw_6hHy@=Z9pJw366Z6TW+@Jd`h&ZEP?T7q|G;`ib7b5r<8Jk)tt%ma
>n8kgZXVsrH&d21uaN>B=g6;zoEJ=Ao(W1NzR*Fe8rmjd23|eA9!~7eJMmx`;tO%}Y{i<PgN0dpY=GO61GjK|BrptnYDg=91j%kZx@`c~IpalDi^iyAye2f+
rO5!eraJ7&<e-TPjZ*r}!S0w!yi4w*K_$@zlX>mMjO&%~qP|1DVzu4C48%7FkW+(UA?_Qvh<Wq3)g;%>2de)oCQ#K(eRux7e8!8pW)*++kt7#&QW2VXlkq#D
7hVR4(8L>YeC9b2T?f^E950+BKa=Ytx1bs9H=+Y%Oe92!ejW0%dNR?h+wjLx0Kt!rMeBm7TFkoj3_T0rF8&kNZsdkR8nXx)v^2mUFEOg1t5`MY?-ITZeJ#B%
J;W)N7@g$uI=;&gX}~=5=JtWGlp0Noa%!u2He&M%%9lp1Kw|@?=`q@hy+AHmPIjuUJ2?i{J7Bvloh@J~ykv9w^6XeZEnD%S!ML<-I%wf)mQ6`Y70f_&QM~mA
Z7I=zs33S!D8(PzqZIz$5Rp|f_xf0O_vVUl5Y)pES8{1F)Sfs&1wYU_Q8E6`2z3onbj8iHFE?Ur2+>wVZ_AICJ}zs7X&eV34<V-oT?N+R$(yD8-A*W3I=S=I
3hW!H5IPj%2nOu`3uAtjSyf@-8^ap5;tgblvLHFH?DMef)hdH5HU;Rk>pduXAcCC|Pl~6$f{DBXH4eFFMyR8s?xNEie*C#vKO-t!%N|Kbmt3^jGhrJmnZ+a&
Mi-}`b)2o9`2x`V9W>6rKvT5*Q+Ij{r8$ZijYP<5E<UCxWaii7i;PL3A7R^>_m9Zd9?D8x`ybIibj$8Y`3=75&HH}bJfn_}IVEbh@n0NqYms)hPxT{+^PF1F
HZ?JCJee>>D^AWei2HlqQSW==y6`Q3mMqvUi-l!*yiU3qkT_8Qg1piHlXSqiKwlGA;21y}+fVAZ=l?3B3U-qnC5Ls5WyUHuDoOXk!bWGQqNq517Z;O|bcN9=
56?}YTcuT&g9#2Z`>s*=>qAXOmbxSIx2iX1gg4{MHm>&`vCQFq3g=K~<g`?b3iEx?t(uOP#dvSB?d~Dt2Yht-j1|3fGdD1y%b%zhJ`K9`SdzhZDnP0@d+qyb
QkRkb;h18JPeI?k4GLuXMPnIS0FA0o?15$Y=tSP^qi9t#fHCw6%@FnH*T(+iGW)fqRsXsfE2+7e5G9oZ;$=`zC4qf(1=wGhp`@TNyz@Avq<?)U^aVw!2o$6S
yY6v+he}Z-d96qnUwEY0Qw8S9?3&4^`EaRtDvA^`M+oc(H+m+y2}GhT5sElp4E9h%&h;er63U?*00000fm%Po>8<Gz00G%?&ej3|aN>3!vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/deteccao_r5.py')

In [ ]:
# teste_r8.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/teste_r8.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;K8ptOkDsGh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRPfZV2gna?4fW4xgY73xC#LJ*0YPm`2_xo3Pq(D8T}!fQLh?
(e*FO{V2jB|4`(mz6g9uiuj%ZJmfjBf>7`UgqVt0{SSS*W^tSP(rl4HHy4`f3eDTGXeK_(_cyy1iDZy+Z=Aj^b79kBM&0ZhW$39O7jc7^+S6C_pbPXo@h$|5
qkj`Al#2D3gRr*2l}zjVmJmFMOv-#|S3R}odXn2aV@>NtakToexRNN-wKNh+h*`3`e&~FGX;FSn)c@Sy`8qFNM<)KvLj)SZlr3nT0Wg|2Pk*CN;Vj~++HUNP
hyol@+>E!s5~ct#c9<NeAnPd$zQ{4=)ULcMrDpr#*=WJZmp{oBI|}B<qKAuyb_o2C%4<K8NZw}eTx>KJ;K~(~!d&&9-{skX*>inpL`V<P!&-KtmeEffOB@?t
T~KXQ96oT1jkq_h{r`SAbn+nKC|%;zeXWI1i{J(|<$nV8s=g;C#f|3H#{;12YJLWqTQhXO15B@GZJkj*0n+H|G%YzIdko;5$Pn$1!6Kai69J77_n@SjaJbXG
Q4x?O#SdcL>B9DQ_CuR%+`-<TTi+z@?4x>;@51b3E6{y%@i0FCXXxJ>w%er+fuV9#C9NrvdV|t^0aisbSex&m0M8G3z%zJjWyD-QmyHbsLH53E7YJ~ov~^Mm
7^dLIsVSbRy$@6<v8aNRKY!cRNM<lXkwOm83~i7+-EPy5iAH>lYU_?w71lIj?L-4wcMRl84B^`c=g0|`+s>}r`Yg50?^Do;u(9$<oo3^g2Zz*f+wGDc!{V6E
4~0-h(1)>DdK{EeXI5{1k39ZaYQ(f2u?vcVeuo1P@|K>sz{HZ_Dit?O!@^1cbjAeX3dK|}pmxjYB!?%h0GlgeAtJlUvGOA@WkMf$Vai!A0PTc83MX4?pR3oJ
`X4iwcwdOt0FRn(We(5vnHEto%L=eB(k*{Ne{TU|I%1RN=lzuatOD-IM$N2bAivn$n~^<ve<DR}8Ty)=dHm#+Am}i9fFAvuST0DiIB{l#NYmE~Lf1GTJtraZ
x@tC}*APz|B7-+Wg0H_hsp;P|8vS=8j2`aVs!~y#?znXBoZgWODCz?^iLBcYyDRJhFf!kF6Dss^E1qcOUt`Img1rjU-V$%W^hx{6hGPpmPl*t0vUv!=b1U<~
y9r8fv3Mw_o<W7GHqH!nE;3k=e+VgYBpNqz>qPD8yDkFD70i%mfrQQ=6J3G8cx-rPI5Y1|mVvcABm0H)k;PKqo>Ao+x@npLmvKVm76PE?jA<i-#KtCUNLiVI
RrM~vW=(WD{8LyuE(^3r^J~PnsHX;YG@Vtb0rz1*g90XY$mWLwfKGZN;x2t_DHM63k^Ymn8ekM27G|gF<%%~*h>qnzs@RC>k#yT`L06KQj{jUu)n6Vz0&>kX
pYb2W6Tyl?Bf^RmQVWJzpLBJ<JYEm%b$@jBJIjAiWMNd|mgTi$@aKe9c>06UJ$`lE!CoxK@H{-2K|*StKbzb;RW}@9$<NGf*%K8`-n$K1DME|G_OmE#PU+t0
kV-rKYfW!EuUj!p0c&SEn3WyDh%Ae9D4jiCgiW9ULcTL?wLj}r0#8Ped|Nd<#Nu^W)l=e9OOuCh3FDjHh~&2W7y*nqPj;awGPpAzs`A+EJdqam16ZCgin-U<
_y$|5UV+J<SrEU}o_q?N#HnygML1-0@j`|L+fd);^t4(qi!&KgMQp1D?oIX_F%`A12bih-;)mTpneHbnDg?tAtgTftuSal6!YpX!AVv#~iAyvt%KHEQ#@H6^
*$bp`gvl(CQRl@k>;&C&^D3N{{P`t`M!`}6kGx*O*17KedUXH1uf-IQ<~LHf>-Vp^BTWg$D=vO@8@kk7jNv4&7|!lIlBrjqgqF{7u63-W#i&wn$!Dx_@al}m
U}(%AemsEPJKb#V$;<vQ(}r&!ch5>)YNyWcO&VSs!$DYF6Tmc5D<#iV%lEzY)@7<F%+1qFp>Cl{l}gm;Vq~o^m(n2~yBpS!d4i8qdK{Q8og2TaQ!ESj$hgbi
t|+k+)FDlAY>lHMpHsQ`9KIX`II85ngAP<**+%(O+x(dsbq{-dlRqX!vQoGULSz2$xns8nO#MJp19W*c5A#8`BKPGnf5GXH0oeVrg>`mi03zYf41eiMzbmlO
XeWEvITU>{3#=*xZ)Bh;$T3B1<Et}V#@390C2==e5?_=A1-b%e4by$*4h*Ph46|Ns<RkuYzM{6z);C?Xf|)I>@i}kO>Uit#0IZ#hCWqinzRO=BdPX?2rM!NA
M`G<)GKmzsVo!$M)m5|ib6md#n&wsUom73>2dUNdNYqi2>t#4xnjx2x5umnj6s<{_(Ig$OFAswFALN)YzR4+m|6e=2ftHN#A<Hy)VG@0*5MZMD<HYeN_09w~
pj`mBu{MB}qLD894HZ7}nUvdhw5psBbKai1du^rflQbfJ1LEBr(<37a=AG|zkj~ZVlXS6>uwi_!bgkCbPQoE?f;n|&P2WU+i=N+R>U^LxZ4aL7t~#wWtCCiD
A}z?dhK-sg%`Z>}cZ7ha5WRZs;>(G0$%lFAF$fvyfC;2-YDqheU*;oEPigDh<@6muQ;hZ>1PlnKAd5TpC5Y3IvsAI<w;7j=oD}tG8b+$0lT$}ypzEQ{qD5II
5Niy;KS;U<L8`w8TxZeP7qG9qQ4>L)7wLwltBs7{F8OLzvK`C%BioT!A;%Q_Fu;=Ui9T3#Ih?#9kV87o9|)&KQl;L}YVdRrw9m@4$N(4X4bz9tnL`Se!a%ol
+s6k}rH`4}=Tu-dXcjo~X4>kHy}?qgb0k-oTeKqqDiCtv*V3oh!XPg*6VN0tR8R%L0ts2_xBbg&#u&oeX*iZ60I=O@p<4q4J(CUoJx!?Fm73yOm1*k_r-oT<
(CLuPB2wBdG)0Lo8u=f`uMPM0Vr5%IfwAtuuSlda&(!Vl79<VZ(%K)JM}^tdGP_{bT}mQhEl~%kczTe9+Uwe$v5f)H*4#zaFJqi@CVT#4P-#hm%S&py_be9A
5`QKN>$9{K{Iq2w5)kvto7j${HMoAUc}CjLCLzTpzvh6YIPO#opW-mdU;*AB=s2|WUSt^<ioXW>#tyHP={wY&U;58UqMxH4O+Oef&yGu9(?YhfLL8uamH!BQ
k(krTgyn6nB>W+HjGxvg*yc$R@s2#$qd#%CbqA{|h*pgKvd!X*`iGOnYKBk*B5K>|#CD3``3T41H7B<wP`g_pfyM!M5e&En(iAqsud2&NrtPY}7i7a4qxwy-
UnOq%CO2y#z7mYYN~=Z)2uZ9)mONyigBgFsj&1oj6Tc_`l`Pmd08{FMgZ=I8ih_bSir@+LeeH4i1YDC*8>~{sW!kr$tVxS!nN{07_k=u$yzm`6oFds<`E_uB
h++_tB61Y(4tWE$%kVEBe=Zp<9_mcy$d`I=)Rf8jZkWV()G804E}=I=l?yom`!K7c$m`*<){Hl{;Qktzr>(+&;ykQoTDCUMo!Pn_apn|KYKH0xQ2m@4;0c@d
gk_2m>4+bG`GM-bEarFD>xz;1d4!{~QdMhrUwkKZ8{(hiZRfZe$Hy@)DwyuE0+DgzyN4Rssc3R2XoHx?K$+*@`i_IzhGP{}f<jdB5X(M!7wp}jI4jc5T;Q7g
e2HCp+9Y@OYS_IRM>0F^ct}@0(8&eX7G+|{-Tbqz^JV$a_1ZLrhU0IzjIUCm%`9D}1a=)V#x>&UGbyZ|D8aErbh9ihRqFdQK$wzRX>x(^-F82y+@dd{w&=c8
7@nysW~e7|$sLtQ!&@pOC(=)OR}^3*+tcjP^;u1|y~GK4J9|PsIQB^)`xKge6u?=N0xH{p5Q@k4335v|wH9tFgy!*t)~2<^)%u_sMY?H6)}$!wa^5|ZCy(RJ
lvY%))g#jx7y4nLGYs$K(Q~0<u#43e2JNSyjP@}HGL-`Jzr?n3YQ<jveg9HrC)tmRC*3qEU)Fu#7+Y|+1+^3M#2x7J2BdFhnS)AiyUX$Y(&MJ|WM=vfJ-wut
jMl(yc<}>#0g;ydb8Dg}_l?nwwR2QxaNewD`|#*G%a+f>ArazWMNYZ)vzqru6f$X1pGUAXg3dvMjNy2yUd$(jM1%^|O_=J9+5ib^7%STylE@>jrdPDEM!k~7
C^s624Ba>fYr?GrDYi0{^}YTz_R{9H9_ib$+_;k@sPXj;=-+a?s?GcAnS?QZ=77gi$++a1z|tq<1(H7KGa&s(nE0pTc4&0n7~v|fGi^8KI0AO}bLMKIAOC{{
Kcwb^<+V$~#57%6-lsAlpgVf@r{v|U!Lq0(M9d6)VG{7FsTF8Iutx>?@V%BrWnSjVc?gpmwbTjTOq@eX>V);Ao_4&HwGN)PDlpfsdTF!14&hk6E&<e5>0U|K
RvY0sMP~HfjQ%Oj$_K=z07S{4?T#lET1U87_)?Dj*-L}=s7+-CYYZC$1Ko5Qrk5O3VmMj6_=OZ}^|177{e!HXr7al#iDm+@jLIo;l{cBw8y%L-pV;9(6Q8>L
ucmtE{k0)#)u#nqMV<vLoh~XPnriFSd;|4vMpbjW=)-eb9v1hIy!FD_(NsOc0FCZ7H`6P|C$35~N{bBDaQhcGdndTXVg^@b<6BpAmH8!Drq2W6B27QF?Ti0z
39z2ro3L@!ej{*=RiZvHI}{+A)|MCk1-_y;hv~u)+$8bBDOEkae^@E-GGuryKang7W1!S|s?(o`?+DlpyfYXcWS;F!jA$fV81TK3j8oUq$5Nh<fAse5ivnxa
U^qgc1-YmDF8F>0hH_r6wE_liPisQkdow0~rjZG!Ee(LP4dti2ijroFbv&iSQ8(sP0X-7G5!V02F1r)2i)r0Aywrl;)3=f(KmJ#fI+J2I2fJ0aZ&~oyZkA{X
^;6IT$FztLjE4*oUV>Z^uW;YFNZrxpDXcDTEK2v)JfNg@`M5E+nL!6)y#vrz0@}+NJHfjEC8^zW?Z`n8<N#R=GgUfiTj-)+1`>o2YU3^X_PTz4sfMtD1=)6v
B*V=tGAf=52+{J55y#h}EQ6FXr5>ph6r4rp`-c1(_GQ4(_QJn0hLXj3(_5j|pn@P!Nv&-`6+1d%^Ol&Fvny@KnV30}8otJ&QGpR#C1OIt%xX&;+ReBX%<B`o
5=2l{E%v&nd812m(0moihnsz)w&MJZS^OU0tzQcAoQ1#08hR0_+SMh|TJp^C6p3i3dRUg-L)ID@%h2}74>r+g((r2OtsZr-6Y01@WF@`+<qaa1Gikh<3%Evh
G#hgIyc;zItf`mC@M8;L42x-S61PLLnwpY562#&o@xn}zgKuS2SD0%6iNd(N?pzY7p1UBv+ed4w!C!Yk^%xcgef;<sLp@=u4F8~Tu`!WT&hNiimaPxK02j#&
PwErN24bpDIVM^!>vtGBvjs@-g1_5=0uubyM)xkk91nq+c78$1cc))kylfJizJH0SHoQ$^EP}1AtI}9gMPg*p+}y}<k*dyriGf;-@=YyGlIN3)GBEt(-lm+?
)EGJtUv;?4z}D%3c|skobKif6O^;&*@}v;o%;3p3FbcoV-V3Kh8dHWe9kHvCl(WKf#9blKBJw#H8NJ^u8Dr21f??!<qZ}M$I0VD5QX&}f^f7q=|MH5%(Uzoo
M#~|})UHbhTNaKoE%?qldJ5WFpRZXpmFCW*&mHidhGb*(AL6c^8F#n^KKCu|#7&hT9V`q#n&ZSkYU4*@(=XXM04xBq%cjcfAYpkL+#1!7K);V0qNQLOfi+f#
tSO;EZOf7>gH#tWTk>juwvWD1bO{lCets)$wBxzC<&}G;AVB!|U2Bm^GsdfP2~5|M0<|bOw6QJmJ3+V<SIKmyW`Y0DiV;W386me%E;tovq;dQF_J#Vkw@_VD
T`&<10kCzK?uO=`uPN~|+tRa8O2|WFg>a@K!p6)oZ~KBaTwXxiMqwme$_wD<=L<s(BEnzz866k=epf1DHsxOfB@=_jR8J1>3ik4d480rpUT!~Vj4sfTeHWY#
hLpZ-?hXfU(2krJ>XNXsu^UzQm=fSK(mE;t!fsW8l4PMk&Tt5@sSd+?7Hzz>qedXKdT>A){^j2n^43;ch5JH#)nO8@&9;OsG#)g}nO)G@1Hkv+xUGmqwi{3F
l?nqBj$IskBQ2C7eHLdKvqn}-z=PcudilTvlZgTOP<A;N5Dx(0he#&Ksy}x9v=*15WFPuL=#N;6!C1xrJER{sxgwuU3#+Mjc`0;DVm3Xv?2cW}>REqWEd{KX
Lsu1GFJKj!WuFca3gV(~Z78)N5B-28G50M)&*o}Psb(6;4}a^d=SIvQ*4%^tGU;)GR=xjV1~n5;BP$5+Vl8Fk?WA1V>#e;X^%l`c<W_UT!t(R6neh+<wPetf
HX;@y*wo!d@nel}dPXG}$m$?m;u9ZAQohIDQbFw2<9i3G+s+J>TZOIbK`=TPfdc4OIIrX^8G9Ine6MhR_`5b)+VoYiE|Gcbq-5y7!ux<6WDcSfcio+G>I69C
%5bz;PEp_ZVunCPNowEkm&(8ok7NL++#Ox)1B?Ut@#8P0xcbph)ObIq5@c{Z*HD&Y_R3Ax;9xVPj!Rbduzd^J1U0iwLYL@$)b^JQ>+EWMl4`dciny4Uym9R!
!wb1cxOTIwCU^O_IIY40RPWn1=>}!>?HoM!uq)j$mPQw|5t){>J(3b}Kz2JgHyIYtOGi=2f^Trcf;EbdN(6qZDmsnKSL8m010AZ8i^OODc{l@ilWN}LB0Smm
BF%nP5s~$QMANU{qCJ9R%*I=7zoEO8{xbfYpc@0iwbJ<Cd7ZbnF@^hnf)1K+t<~B@FP`2t!}bn8Q@~Zck;p?13fR~(-#%6^vswJ^o}9RBdmW6@OAYj%*m;MI
76>E!q?W}On+5(iT-RoguiSM`S<8tACzrq?HPmj2>R^ffj7II$Ka!qrQxIL2&Z;1y*ZS#!`s;GlZIbB98zEksum`huep1Y^NeUM0ZS8qzFK+L@n$$DMH!W?0
s9iD!QYxibI!)nTSmn48@yD<4Z;yfmwA8oJgouip>pL*-Oeu+d-tI@am_ut`5xjfXlMxzc|I7aMRGef_TCEu5Vg$!L?y81l;IB;?Fe*4;T?X%TEDqq2ZJn#Z
EN*>NU8e{EbNwaD8r<M`4#+?#Ocu4`AYO>R--<Tb3k<HhhmfJcr_k2BaI79)G2+SmcHcx*Bq}t&7y$bw$4v8I2D4p4wkfrnGp2c~M74CuG^x0qQ~~Yl?1`4b
^g3L@ub!*gETy~4&*GS22u*K~&4`Tn)%Fq&3VYLK#<wzK04Xl8()mE4;dNaKL{@B(UZ^oS5ZWROws1JH^7McHwW+KI*~jqa_UKE+?hn*JBRpvrJ8J#xe0ZLY
)2ij1RO@{}Oe`a;9Gh;qEY_sIkknRfypZW6DOyk%k__d{FlYnT6=-x7q;eAlzsQs`xtbL64dG!vS_Xcav@7?=4L4SnA{p_Uck*?mc~X{5yv87)qsJmbXPSnj
Uy|8P)?D(_@q^IN?bdAh;_1-jgRbSX14Lz5ac8w5WUmW<2k(CTI;5b_g1ppDiAGB7NwTi3Zh1&k{o*9d!jtlB@&`@_iIuZ>C4Hwzpb*0HF@<=KrZ|x$%5A3J
$qKDhEMYbtB76J&chI5SRJX-OSiIOm`FJtYS)5*WF6`Ftj~^ijIS4rFr#~lFgs2bNhtI;#RcV#@k_QhEL-5YP2p&cP<8-Y7%^KDAVAi7vMefRV(<A9X&`*O&
*5}K1)};4gJ{TkH*G70y;Ax}yK??rDo%MWSo5856sOg9?%~$#CE=duGz7Ix^w$FD2Z$R2Oo+PJtzwd%R;i5-}OF@CZ{7>bYo;)ts_^D!^VW*z!)e@~S$=R$O
*i(^W)uW3W3{1ZxwMY?UmuFB}LrF?N{tjFP7<)#pFLL6yTIoUMdm^G4T7z9Mf4|;0`>eku*brO>`YyVeC$HME36BGheiv87voY}5mfkrNsJ>@c7HKDmd8a^{
Rl##Wv93IokI-CP^l=+rn$jD5JLpms@J9=|Z_@6)x9k0cgN8$WGRxs5J^T58e$z_63`nJ58NFPP0)}Ly_o=z%0?xbEE*9x`ltnkC)N3{|;RfwnBnFbiB2Gvy
q;MjT3J{3#Le^0ZaTZ8b@^|qbL*``w#@%>SvMk)X?d+hPLK<O6jSM*ql|TUiOS(xK(UwW3r>P2-*_v;qC8f8DC+FL_D8+`rri-l+=&s6`#%#0WgnLt-8=JVN
nTRdb!@G-&^^BRKR20;FZ-4x+lcvqx`u;YYEL0d>@=2;BRY5sS#KRRYPe?YAPBLG`Ei^O<!P6t;^J%fO)-FxQ5+-@Az*Y+vlT7i2EZn7^;Ii$-IXbamw?A6{
8J=I$|K0;O;j5DRhM<lLyC?e1Yh4I;{czIOwy#EQVQ9x7_D;Z&Xql*+2&d!0BXxBg@I0;NQQA<pXk{$IV*nZq!#A92E3>d%a)QrU?dQXl{uniXn_Gm1ry~f_
Y+k2oTlj>rN2;=U+seD<RH*Ds>W6+}0!PUVE%U?1YPNvqY(Q&IDc$i{JCUg4mM;U^S<$lKLMN<w%Kx1mW2pf`Soun<=(Fj9Sxls`HDz7h;o8w+4QsU(H>LY_
4?x>sB6A4HNfz?u_C+5QLA$0mr3g|C#wt;wpV8j&>@O9+Pz2e0l-!SDr)qr=T$|E`%g_^7ev~X?oJF6|6~|&_ekGPGhMnqA<*zJ_W&*PmfNpzi8?})<N4|1x
@ce}g<g;i{ZMmV;oiSq?dF<FjFB0!H45<i$uBoz2JYNVu5`CwD_f*`FAl)#ebP^1ME7m>8x%oaiu!?7--cRKk?y{cR>)z1AG=a_L_0RHdnRNE-Jtk>hRTs}*
166Y_*^ouMC_$WZ_IHVpyGPlf%OeOS4SNJm3OU00D!h+el^?*7N++Yb3nuOTOGeqjqimf=q7^Ti(^avFOma8?(BU7V3KxeNRofe713j`D@3I0`<^_1BNmiFZ
>FaXuoho|^YViD{(u@}t<IGSEVZeZLab6vmjKdZiyd;NHv4<+6i!BcD<1YfHB|k-;IlAbun#7w>hb`LL;DuPp4UU_R)en`_k>4}WbSYA(`%0bvIL?3;_ms9>
wVptdLcGG$>OlRkkxs5hPvszK|J#|rycdO`DPP^mJ^yRfo|q{zE4oSU$Z=zdyZRQr8iaU-Q7$NtZ>XfIXHpL9WQiUOAFb^HI@wN`ra1VU;Yqf@4(Cg07_7#2
QL~#I@>ywN38Us8)L`=-7H(82Qq}-ko3$Mw*KKaW6MEb_zr`X>QQ14N`!HfrC0j%%N7}T$#yi-5Bmwlzg{K4?RgZ<(SZ$ofDix5xfBL!xw<5Dk(SU-pR4jn!
Rk<Vg&jF(Crm91(+riV(oWQW|H)7f1*>(~+lW>N|BRWJ+Nh&B@e#^i9@+dvm4UiJ`mO+LZiYu|;VN?+#8oYKBd1{fkH=d6!5@@rg(|znaeteLfOBr;>1ogmK
9(k$<gC{GSJ1!=AgnysFo+e{^SjKAFcaC~Imh9k2%|RDg(reR0!|x%ExqY1ZT7=^#?Pb}sigU^O#uM3t$wINWlg@s-Td?9+tTSTEpAGmlNCgIK*&iTXdTi4}
gNj6ehy~dm7B<w#O-lJPO~3N5><d(Vi!<N7MD_)ujbgb)YGJs`@f4j|77aYy0{maas4EQq_d~2(&*mNRKl>x)m++>?)3n&c!LM_RXE@F+<tHW_3YqKN+n7%A
hV<S2HM_Gq5CEK1f|hCJT@4{)(@Qss00=&YpYCj*fJu=CAbY1~{ULnWPXBf0>1gY(2IT&ci?!%PbD<(KM{LGJ4l_$=`@B^%a`263aC$xX;=$SQ8z?a;V)hKg
V`(Eqy#b2T`iRqq@`otH`cOrrCg+vNy5HtLIrK3Wh$mb)aUU3#TGtRb{-`pX1~cY-@}LF6m8EANEs1}Q1>?33ws*4Od(LT*K(a>_lby>zX(T|6_Rw~qJz#dA
_qFbN!v6x?kR_@&UAT1jGV5&fgD%ZRl?{CWRO*Sx(#s@_MtMVkp8K)9ATR#C4w(TivSyQtg+Lx0nn3Se!giHJD^<2Px(WIxLC$#g0}%b6k=5Q;k^}Tq*my!;
&z0ET!B<#;EY14F=s_G_2kE~UjDN+61aY<G@2?LXqbb5S!I$_(kk89ty{hJS@BQ-4wky(!jeNFbYQivCtFgChRcM?<G?JXq@sWBR0CmLZt7f$L^dO~X_eu$^
nSU&7pyyWRY0Z)psR&_M^i<dV=e6`hDEVO<A#6^ct`GyTj^_R)jq~47NJ%6Jzgph7Q7|koSi_5JxDws>(6n2(z~A`2Et2G$@-w`Pn{H(yOzpt4Cc_ww&(R)T
6EfBG?Z*|5xabdwn2+%5zQ~Z~TX(9Vjqob)OOpR)pL1D8L{AMw?8~cndtkEHRw_!bI$F3aaRUbnrV(f4ia^nU2Ki5OAH}|T=kV^v>6hK!Mt$%as`x3^JE5UT
iOwPb>h`rg@UQ@K1i&82PHZ5f_{3C5ih72%b=!`pfWsiT%k_!LjF1Gf1lP{e&4{DsAe<e(%)9NxiHV3I8vxz@&>#YbvI23A-=F<7RJw_R2w+8DVNdsYWZzk%
bf@Dz!6Ku5gTjD9>S+!SreHd#_@Z3<glHvG6kPx48B4rpG525CIT!XpRCPH{TK*STb6c13^WN#t67B2sJ_fmvyCabN0X|xocY&qCj+;*hj-^Er_<;(;6hImi
&@kR6tW%tsTqL&q?AKa)8*PC*54VPV;qy}$Eh-=$x}u>o>KI2S@?9}5`OyE5YAkvhlk@F|Yq;Aqi#F&`u49=Hxz)=Ld&8vC#Nyt!1nw;Mq7nPOQ;d1g%p4Jh
IFsdX(}{p-?!!17mW{l>BJLM+hEa67n^VV#LTH)Nb6vGL30Ww}WOZU<i4<|w<{7xo5+_(zZJzDj7`JpD?%(Q!*F5C>!V%jlwzNM~XHC8f(AgL3nAG1&2t(_;
LaZKWG5jik$;?@q4%2W2Y`)D^qhdMPvC?*4UR$E*bwvLX4tPb-S)HVMdlF9EL_bHpQ!DXTJzcEpIVhKmVv3flL8H8)`Y8$}Tn=yHNA`rkQ-4;4;<cg<`n&P-
DNBSKI$C$qyJ-DgqCpOl{&z*klAzDY^kkZurn(mTQ{}b$(j&M#axbTg!3ulh_MQtMdAXYiI`l?fh!&D1K+pDzP*qAnH!`(c0T^<&+9jQ?0O`_*>9Z*BD2#e$
TF#ZvosU^xev*($v+E8MvceTrH!J2n>pLrLqEjumPOnxWxvX)5(t5^D5Gc_jnK!Mh{GW23rqg~Ge6z#PITM=igvt|xsBN^+g{IBRoC(G3?fUT??BxZ@!(r5R
(K)-28{S=}R`wE64<(hd&gMOl4CP_-pe>x`+>{CV1J%lqtF4VURyUV<!jD;6LcA^aqHoQO0y>w59K?rcMGe!n_;pf4-s1s75+{Q5?c8kFg=!Tu;YdOqG5JVO
tBj#_@^Y$3u?`!^Z%g&<kg=d?Wi_oIW&(>$8USh9v|@w{h+O^QCn427Zi#f+)QL%VedUyV`O-{BxDs8XMH4N43C+o-)pS6x9QgVK)tIw%a8CIJzt^BD#YgC7
sKgU(y*??G^a}ARF@;o}-k5xCPBNIxTXPUlbi<;c#ef|9@UOz&FoG)N%<$@PC?jpB5Jo34T2ehw0C9kbB}FAK)vfYglem%bQCz0WHFr}vWpTiR`Zf{-<<g=V
+sx#8C`)v7LFl39Iu~CVqtxxD!_A27(g*-pPO7%|rOYFk)ESZyp;$)!b@Ehk-t%CUJv!atyXy~trWuOfRbp4H91(2;T|x6wfJro*z;NI5A~;p}4oXP{<$Qp}
T^g!{r=PM3Zzd3uJ8|o;01l9WNUMW4nEs^@(*AO_mF+6xb8U<_A|LQLeXbbcgk9kchRWCGMJHVxE!>JRWy1lCD{Y;bz`8IcqL!x#tO{HxP5WQi7*AiO#Mc$X
I11&NZVwu4lPnP<;%gHnkqTm}344e<u)|csg4V?8GL|Q@=qNPOi9fNb0Fb4Zu((})zhTp60OdnxmrmAs^G;KYNSJhMJwm920?_iidz=gVn_YyIJkQU0w0LR*
2f)?*`(pA@FRQE8wF9ImYX`^f09Y9wc8h8)u=$+b!Bvd<#0S`|7w**m>e|0tizId!9@!w6enz3$w8Z*HU*>m%A7CX(vNA(|iLO>yIdb>wI>DDaMT<n}=Nc=8
6h%dQ9!OI7{)8<Q$<8nCOE0bvPfXzdLElGL?aC&8Tyx>=>4qIjNEb9=xl)8jb;}31jK+e;Vv!}xkG9g9LIcbT&H=IJ0-P{_1Jf5uk+Yejm1xI_8G-!;tYAhJ
Es*tVw;Cc0wgfwJ<eol6XO?$7DkjThS{8XstJ@a{{c$+`(<u7xA(ElJuqh>4-b9Y$bjAjm>r?_I-kFe&9kH3xaAnE(zDZfa(_)CIZ!|BJLa~XCy(yS9QH0{e
E6W2kV)=oWqF0<?U_Q1q4`Yn*X=C4XTf`>vZsgYCt&Mi^%Y3Q2@D}rf6v4YdDXE&+Zvv1j|2;Ef78c>-No>BjHOSDeNZhaIf+_gk2)8$@hAy@fo}%R>D*b$_
m$qG&_ps`&sasM*!wB~4!d?lTcw+x|iw?DRvX%8|60TQ(PCI`poU%(?w9nUZXn<&HKZlHnOjKIdY<M6daij8rCIpyiwi=)U{nhfO8376v=SA8F`w_9BQ`;Oj
z-SP~cGz_uvKX0(vQIvmYn^7xcW>$xe!XNj<SPgZlwRX_5Hmz&O903?5U`tvfK3Iy_7k9D?Fu;JPRNR3>=U$15?6qCn;TGkg@6TN|6&-KS39FRIVt429PG0Y
A^xSoj=3iNMoGhORv|*TEVE;7+_qiVtcs6tzD&xBG63<LZ3RPD&1+!U>cf)vve28K9R-xds19OxKukhc!Z3dOL)_x85fWrr?#2PcQzcw=Po@0S#hoVnz`eWe
NyvR^OC6yj#2ljnI50Ow7Xe}pJqwV!#%rbQ;u>RN-Mv2`p)a12u=PJ!^}7OS^E0f<=dm<^wZVM8Ujg*_CrihJ$JIR*p+Q_atI@?fY5TGvD+VXKKr>cn#G@4v
6d*PS@{%nPc&1FnDUBPsYo>x&-i~GUH;(-E$7zS$w4GoE?(8mUbDbv=Z1}bROQ4#!wrb^z;$VAsB{tDMLX?^cdjW)Z5)xZ%T|}a3fVs_k$Rpho8bh1feD}OP
viM~)zlQ|&aAQea(e}+DR$<1`s)Vp;QGF-tZYS*NL8TKZxMQ{N*|RfV@UbTiY<P{)bneO>R=p4D`yH3ij}HD55hBgHNTMssrJp<`QN~i;xsb63Y;t!opAX18
4jePEzvXwf2bLJ&r{By9Aa070mg0(mYon@*^J@4iYjAI8F-X1)%^|SABQBJU@vo0pLatux@GG)^dGuzkT2it>N0dtUzXo49;=OODFSL3GZ{yBhZq4|nrO062
!Ao2-DpA~BhmycS`$BG`ZDKRZh42i++Z_D`aVC7?IOG9(W=oy>0(F&Oy$;2X{Y&wkfX!e(3ahezmpI=1=JG~E*0w^`W{IUm=@{TLCnlH1;!uHEWW(Fo>PK9T
MI-g#+27o2<v0w#H)*pUyobm#QymCzNxa?d7fOZR8KdQAPGYjm?ccrEh<jSMu=Jx2hkx3OjOCh8r2b6As5R*(EO~l{QoC;Wx5Y#j`F)K*Xu#py)^jneGq1LL
)0~<&RQ@CVI{Duzw{M&V9g=TKu(J6P*ZN<vZxt*|ll6lH+Z&Fg4d>81tw{sa29RXM|KXx8KhH&`KQ`4Dm$3KoG=LrGYMsmFhjv4%j%aeRG@(|2Ga?qK!<0xI
Ifx6<pD0l@M7(A`5@aNNYGHU<{ZPx)E^A9OTM%<l7CbR0s>@{(J;`@*C`oz4kat!c!BMwcwGAzo6Zgsyjmf;XA}B+EFlh@bLm4`Ys$B^ml;lGYqFnO*>}=WF
E3EMrZBzMKp1~tnGz{_|Upa_h9P<BG89*(AqpU(o3uT_l(g^e_hX49b4UF=u3+`YaeRwrFn9e#Wmot#gjNB-%exYsC&joXoViYHSei)q*R1mSDtNnKSC&9bu
UQu|CKM%{b6N0WbRx77GpId<w6y)YAZnu+nl3h~(-_{FVR*X}vBI3Ep_40RR^g+q!J46kDJqsB7nVt4q2hvk}wze8ryE}rgp>M!s)N2Qp_iJPB@x~oiUG6sV
SgMCcLN3aTeh{}#V40h2jL{fvZ^itypH^H7cnhJm>$n3(s<Q%i^=1hXIY*_YV42#E__LF0&a?<M%LKOpI7KD63p74t!#a`{6drjs0Rqs_s^8Y@`7caM{aw6`
hhwe<H`tOVGW>3tT~mR62$T@MwsdL*jxjpY&WvM_dE3n3{UGvW3xR}0;Ue|qU1fA;hlSWm&Cm%TFHV_9>}Tie{urTLQ~nSna@xl(mvg7zhB2#Qh?9YBIi%bb
ujv!)*-MIiSaSe+<`{~81QsM3S*f*c@9y!Sl_xHeia>tfhs*rk-Q2T+)tZY%2FJB@XUnG3K;+}^Tzb@RFs8D761fryG7;0>&Ol`aL7WVYGkn7O(rC4wlxmP!
+nX4>4#<n@Sb)IK?8+b)NsZWyfnxEZc<an^P&xP=dOApSyZT-cGm`SjW*s?wK8V#H<4n3XnZsgtYXU+AC7hfVCN%t-ra*}z8!wiu=$Y?3T%PKZ*`)dm`Xr8<
xMrsxE^n7TGq`<z>>}7HtrfeP8*so2zm1)z_`vf&Sc0Un+N+3+mur-iS(rUnU-@L$j6x`l@ytri&!gZh4~N%7iXTe8N0nsHxp+WB6YrFwbOK;2;U15z*wJr(
q<Ci>HVt_j#{HXb#>a~b)gbx@xQI|)RfJP4CN>pW6xfsH;lV}cGu}9zQk4m0>3e02fColIJ?c&wVYIg5;gO7ZuH~ZV`W9w!d&QA^SNNrHqG%Wyp@DK^s9L;(
SIo|55FR&zcMM|OS%o?(c#CTxW*{eg`5if5EBzKr<aXG+l{B@q2N3Z5$^6qj$VGzjwSb4$4It|tv{DFPpxru!bcEz^=5;hbB%M_b-(ohJ&*pYmgeuPYL4d`R
+^GZ$UNEOZcF`S0M+aAdbOBN);?0m!yI(wA`dv@>M#L?*Wz&pp!yY@>{){nk=b30=0ID@|&TmrPo=KJgodTFc4dL3`zy6BGoD{J^B>}ywt=EdgGrFs#^(MiH
hG<Nx5!E#~1H~7}7KDYBN}w$9dMIrJFYprfH_il*l038*k}^|w&pq_Hnp9WR=t36=L}9c}n8jHVTY)9aPnn6LO7Dve_Fs8;Ij?v~C{aLKcc(1AtX93`F`?7F
cUCUM^9Nj|%t&iC(tvOG<k>~(FC$R7N)q(z2%(zOljC`#&9+$m+e?b}Wg^1Mta*c|-3&s97#ShtMllFz^0+8DRs=?8bY9qeh%BGyW}*upW=H=1y*I!R!fL_@
MX;@BicqQLiO8L%UfaSET<F46&dwKKBR9T>FD)lWuARBbCiok!FY@qOx!<(%;}=r=r|KvyZ=ZZdSdxv(ghJEf7XN|VJJJ?PhTHF4h<&D+{Wu2B7Te(}aGgf!
mBh;KFDrfB1=PwzTaAo13ZrRoXg_Wys;~0&70`yP+V=w3W0@g38;ymTYf{5UD|f)-<x)!r4UPNnq?J3bK}YzjvmATl!qEx{drE;&yj7!hjG=(*uA4Rry(-FW
@B@XQdh(j$HZDM7%>ED|7wk&i)tOPA#Oc8-kImGbb>gFrwFEa!X(pcv&p10Mpp+>G4a(k>CbFbM!&V}EXLk&fc_|Qc3i#Mp%~!faGU7<V47~G8v6q@rR>)Ou
Z#xbniwSUdtZ_U(&vZCY_V$FO%qMi$Mp2Nn3qDyN_5Lca$wS2@#f~A-$XD!TlRy~fF&-bt_&k&zgM+0e=u*vO6*q-!BrE-SQBcGKjsTp<-msgt>99*s6{tcu
DR*-=A+koRU>dVt$w!%BliAB;(F@EHg<nF|iL~Mf0B};i*6M7JyV40sZC*$>BxW#XB1kk>R~#axd1cICzGCm|b^vh6ipHKa`py4-^$-OWT}d+Xh29vxe6w4K
Dhi06Xwg`j*=vP=QOb!b%5iV>#6ymo`6*E^vqAn8@#3%RSJLqDN3qE&_U6mA?WM!-?R5h=^YX)cAd7t5_*=iV*41Sx#`fm=bPKOA0I0H;PwU22R5h~8=6v+E
&J8s<Gox#k=pvF+>OGoK|7VhVCcM9ogX6T#;D4Fa=jk!8Yb%~}0OQ(Iq{1g#^iysOe#vBgPD(#d0EL;x%)E&=0fXw6cHRy@Toz64z0DfvKU0iih+mCt-Ie2l
ws~VbyU>+Ysi=wZtTmNcpk>Ll^X{@p>5IpcG={R~ZIbICLaT8{qG87d=))$$)pqo9_!RkA?Zk4B=%iwhIl8Pn7g&m}dRLl4II(AK&KxINQhb9vV<J~Odgy<9
S2&Q7M$WXFoc!R3!~>{G`_f%y<FI2f;+5`j&x8D>IODycWS_Rzu_glssFs6byMT~__ta-EovB8QRFAfuiymaC7#p1wR|l@`3b9b*!_-K%Lr(f)UWVBzjR%E+
>{(yYKeKS7h(3%MW#nrl+F|xsDSVk<rXiT5L8!0wd9)GTeuE?)B(y}o4U3FMjl70QfQDbbA85xU7mZ6H{^Is%`OCT#j#Gv6<xPU>*Vl^gaUyeW)K=tmY2<W)
U2LY?{oiT=58BKhllqdxJV#-@jl5>$a+j1jlfFi{M6lv$B9WbY)YdVr!chfvZL@NcMbL}7E20WT+K5+RQop9dSmtR56x@|gx_eft5Vt+lN<w6HHj9kb!_D;c
NNdZDIJa9aJ0+)kr`fYw$TR{WH#S9M{<+~nf|$`*H-HSD`A2njkpQHY@wO84eZt_++gN(t(b{<4eERiKd1NSf=lf(EHhLD0<t4sWW^L#Ty%GJ)aX~g81~G*?
rgCKg2h4%%M9fX++HaQ~1d9&~feQtTuo5m7_^wilb322vkd`t%E$Mp@7kWv!hb_B|Ql5nh6%yo~o>UQnb5B<OV6n<9ktj5`Yb>W35FKE+2Y9B>B@kJjyq2Kl
KeIUjTS66Z8%X2|k9wicv(YP1LlOEo%zjC=#zV{KN%1K@?RufyVN-@V`Qk($p{75HgGJRiaqU0^;F!7w7S7r!l#6?Ej*cqa@Rm(ViA0`*<6107+}xmGy@v9_
67dJ50iaaJhbZfADcq!DeNyGhfYP<;@F1$?7_s7~xR%S83r``}M`btRpL<oVTH+jE{ArV}nAG^`E<*My$gD}aq3E1Df-T?m6)V|-Njk}^oe)-H&e`vU#F#hw
Z${h%5iJt2U!#T<)|hsHn^BOG1R$W)Q9*pZ(4_(NlL;m!`0zAVS~!u7z{LOhO_4pR&ssjps`e3sA<ztQWhAx|UjUY;e7F5bIU00>UV3hW2O<5Q3jKJ>xgWv*
lvRxG7~@0F3pZ>cbyGoLl0Z^_gp-AafZ`IV2wURlFM5IQRTe>kFv5#UN}xBm;sO??5mehkvVO9H(a2WdRiu-IMdGgq_W<zI8!S~G`QyB-vqkM)C5wynzh^@E
6aAg?(p1uMUUJbFqVLf`&#(1g7R>QIC#3_xR0tp><L5jPRdaIF7r^_VzCe$nf`#k*JMfT8=IFgbbk*7ycmE!XB3l+4G7mWh2nW%jm?03$jTuJ3xoWRM87D+0
jbD+NL0aTOSY6wr(M|MHmV>{L`28}+lb-&$wa6kP5-3W{e%96^JePhdYdOOKX!iADmMyK&=&)!+X@mLW`zd;dF*8P-b4)q=m?=~?5e;m8Zj+8ad#I{qi!+7K
E)PC?`LFQuAb(0xYi?&^)u8Gy-P^1w(YQt(V~rzQGqt{>y=BiH+Lx&=8}JPIU2%_*5p)?RusXnpQ_s{*vWwLQ^wt0IcVX>^(B-^sY|qQ$9FsytonhHLPYgJ}
n8jkb*O^;>pV;a3og{n#Yb#d{ki?K>@whPFty|nE3u`HR6C7KqR7%mddP@rx5SSlQ;f4T8ITQjOH#-m{bOKsJ$0qU!egK0<Vj0rKt*)y1@WHV+uNTWzlQ=PH
W<<D=A^06whUU}VTL`~O=pa|15`HSHX0nJ?byAxepL?x_0fqs8R2--rj2Qo`IY>W0)n!l0ODLGDudXNT>FE3DmWmk)I{lAhF?16y%{CoWBeS<hzpyvSw|q3%
N&4U0mLl1f0a-;(<R(=U@C*fnj$^za#oL_QFc7>7ZSWmZ8{dgB210#>lyTuLZ^~kTUaBr1@RZ`1!rm&tS@~yTRT;FVEREs9@-Cu0rzi9?zXM@ywRqPkGhu_w
NP9=PBE1E7mw-QKaf5$NTIFDXS$)Y+Dx%i(Z&^ubG{F2JfoCt-oXu{^CL3=W2e#<rbpv*{ISVr~v^p3d+%*=Bt9fxD+!$w3<1YiY{|C;MH;q}A?;v9bCqSOv
`;fO=W?(_;Y*7`rDrCK|=3}uAScBkD)<TFi9*@7Y@8d}sMI$jOYS5+muyjJrZgPxBqRPG4enotF){zg+FY)lSF;n5mt=(}UeJo?i*d{HluOGBd5ai~qS~zuW
s^M@j3t}HK0Ov$zgDOCYbV%o@!~(1TSikgIG<Koq|M*JK5eDz#F5)|$sI2`pkR`?gv9$lf6!T!LQcw3^$eTYs@8Q;-8ViQ5l3Ai?&b46b#~76xS#VvVYcM?e
k}7m0L+Cp}2PjF{<`$E~4s>1(^6M(|Jb&|b9<<&_-0$mW67R1l!K2!6$nL4Qg@0T8_4#?;Y*kMfY@SKJXbTEGSY&dkbl=!I=k2`QMM1^;N@(Q?TsQ?`RUpAQ
Tey!N5|<g@8ujIy->?I&Li|Cdvj7zOv;~PuA@zQZh`Rj}9nv92)#LsidG#?2JtKMGY>K;Nb4xqSty{UoKtA`j0X}R+sI$Zo5|0EV!0q*84uFZqYG;j2Wj*Cj
NyQo@j;eoOB69*<`}QGPTAMUCiN4bY8`%0B1Q=tg$3azwnq2boO!jFSzAzDifN<G&Zo%P`DOgN^+_yP(+3I0My5TlO^s}ib>g{VOI>KN_lwAvzm)vWG)ZNHk
=?w!5IM$Brsg~mpuKFv4F5XxFN<a7|Di40wB(8ToUzBG6`=>4hM$d1IPr8jCq;QS6J+bQ^nB3$vQEWGa!!A+O0xzB+(20X`k|v(lCCDQdMa?-guGf8>%~O|X
{DnBhAZ7J~d)4Ab3<ja#+jIi~6I-5YH(Rul=|#bcSt6+d!(u%gAR0eJCQ>Jx%py`p%G_8vZB~c&7*1*wf7&9Hb+Ou08yf%sJ!AzrDc)na00HQ9u!92ty|X;v
vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/teste_r8.py')

In [ ]:
# treino_gpu.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/treino_gpu.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<F5Vh|6KqPh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRyx75<tfLmlr9v8=rA^vK9U_wKSxGTYDxj-jWATV6)EKC(Hm
YNGRX;bmU+;6s%e2My;AdQ+%8O$p?ZCbjDzN$>!xfgl28Q=I#TvJTAmr^77~D@Zu{uA8j}=Ah_;`+(Wh7Pf7bceFsb$RGZ$;$m)?GKB?9<~?!B1vArY#MT#@
mIAho^Z=^{$g_MwVIU6R+-vlNl8?BI_hHwb%$T%+eNqIhigtVv!`j{?O7Q+YhT=3|SaqgO8$j!W`>k|m9zhNv0uDp5Xr3Q^6;J_^66184ImAO>Ppp50eUOUl
VD&>9$J+8n9FytF5)CM~Sc#tdYxtF`hWZz|<d_JaC{_Y4R^}Ij(EC{*VE_;CXsX}VhcYbuwO8~%Y#jlw@lc?7#bpik-7eRYpnBU+;&;cT?6a+Jgm6*e!rBs;
mKwl`A@b7S_1st7`0v2=*3SJ@q!Ng+Dz436NEB|wU%m8E96_)<P-l^1)8Z?<H?6y_C|Pe<q{q@wqXQX!dJ#C~>+^DJ`dry)O_26bgb{w+1LNuP?4__ckTBNH
8ZM%F+=GRyv@U9Xriv?F0cc*hfgBhv0A&Xfo0dg?gCwhr_Nv2Cvv2xm=Kg~d9_%egGaM2gHrbpRp<p_^7`FByodDM#@>~_1{YUjb^lk%zA9{jw^h9RP*#)bp
rqu7<)>u@JT`MoDkSEXn+zcToA|eb4jJ7QgSU)TBk`U1mEvwZ1d1Mo_5dBC793jB&#7;t7sK70M5}=QTo5t;EWX_fVIw*rY2fC=vCunV^cn(e~faHkK9yb>I
-ID}rWTL!B3<fDBL}uT+Am2>PTAH~+P5RN;$qv?)0KjM&1k&KG3`=wOe3?b46*LwXfw_$!Vv!Z7;tV|2_ZS5IwTQz68?1dAI~=u$OR7y;MGV+<P82%<so7rv
-GxjvyNMxuznP@-EK#>sRosEsNu)Cb1xW%JhNWLu8%%20`kE<9yU7d8@{j{|D6Hq}(Sxywk?hubJq9N4gaTO3$1JX%$W^`NwyMy(s4w1|3h|Jjie6d4YB~Q2
of6stzib(J%>po^qmT#yz3z>W;)88Gf{nX}+&Ym+aeZ+^Y?!J%A~&m$b6|@`AExp#Y%ozq@OA2TY%RD~p1ZZHQ6C>a|BD$+pI7iK5y#?tG9efCY4~UAm^w?C
OGEcuiTXKYQwMR3oZyhdL5qLcA%K2niCI?4&)(T|pJhn6IO}y!0v>kldb($B;WkYdtN}v~tEAWpqbx=NNUYM=>y_!`HMYg#tL7FsfKJGq9TkJH1e5+gtwYkp
jB!kU=|OcOdtkrqqD4H-7p5(EjZpXT8}cI4Pql@8Z@^u1d^XQV7C?Yc!3#W>99B+#I1A2|Xb||1>LcZ+tyg?kNlqg$-4WLW2R>3T$;d_T8HR&?CVMY*nq^RB
9yS6dF*U;T0wwZ%7OvJeg)K_p9ilWw$RyGtU91@qI@yf$P2civ_bRp1ys7pRn>B@~7yShI4Q&?SUCX<DVwkStv)|rJBr?eS4bd=He@kXdP`5)ax7&I_Fw-iB
2e{{U_hvt`P{yBka#iD3@BJIeaP8g`>8gEpxSl*+5bB%-u<>JkfGN!F=ZW12Echb0!#=a69`qoE5I=rHr8KO}`TOXY;*%}e=b47B*N0mlSXAZZFH{e=cd~}X
iAW=wv$>OC1EHtIqXxG*;?FbQ8uLSHZ;!tRXa3b*KAmyB_R_J>J`~jFp}$TN>ziKC;Dl$!ayk{@&-oWm%JoTAuOz3Pd2m+YksbbT%oa6Qt0XCJy1V3&OtYd#
n-R0w1kM{WRJ!S|LL8KftvrA8E$E<sjjMV<N8Ye>GKMM{)7{QoFeL?t87cO<sYGTba<<wZH2IMl%p<p7^ai(hQxey)wuGTd+DS&SU(2vYV&;<h@3yi&$^(jo
;K<E9ZE!>yioi(FNzhsYy|9ScmyQA12T6`Vgi0h1?o}T~@JT$k6_+;-$TFHtOK;i(>@pWz+*2z=Pv-1}+-R~rcK}LyVPSP1`rYkS<!}%dGRF(65<E>n#^0xr
q^$WO91}M%uGWupdWcuD=m!o;D;6l04;1!#w53s;#yv{!X$#7G*gm@eKLjrZfi@sviNY>kKGQ0EX&`b=1WwLGQ)zG))!P9xm_%oQvBJSy>``!3V2GQ}De~ns
>O4~Hfw?uXr4mzU2!d_H$*NQuyVAhh*jJkj0G-n~5MhQ!9M}0jgfs6Vdy1Fv9byO@<t1j3)H6pwaxaCC-+MS}6S)8KURSZC9{Brq3oH)=l!?6c9ajp-G4GX|
0n}J+0yAZGnkM_(Be&M%(6`i^Qw*6UGUT8Ev*ql(Sc50Bm}apCPfb?E%rNgPc%?b9TI7udl~+O~%ltQ%_)RQFW~x|>u6p>FUb|Vo%&@V<XC&Bk5(^*O<p!9o
qg5)q<YbL5g7jb|is7$th$%LF77QguoV#U4x6&pgXD2+F&Oo&VBSI{<%^onm8j}cCc=|Iyx<oqfOi;=4@uvlCw;x$K7~al{Xmy`bHemR5Zq+182Y+qt>x8N3
hI^hgk>=YA!uWIEw|O!wlAV9)FT4<$r5sn%&Otwt%o;FK)qqWrZOAtpB3}LlfuJPdw!m@vdq__VP+9MZ#F?|`OPt~83=+tk%|4(+sU#GkWk5}1>!)<m$8(7m
f$}tPq0E}CCs5=H4d^<7@J?nWIu;>5l<cIL9RSg{{a%}Fk6ouKs-X=!_wcHhwADSa&aq4X>t;b#i5cWt!J9jm@2i^DWFqS1D!^uuY8zQ=auvHTVaPa$S}yh(
W~AXBDLQRItIayu6FIn7ms#emS9@jOc|r;zm5Z>%@mxWSEMW?l!}T@V2!{|ExV@e3D4aG9&8u<&+@ak|(x<Q98xR`7d(jF<7g&&H2BtrAs+$0{{C)DABifSa
e9`SVYzg2Y#6=oSii#YFZ;41DB%MeyUE7!ccJsM*i4A<2W6Vf(G<4z}Ob3YeCxbf9w69tv`LN2y@Xf!+bqbne{i=+baOMq`Kmt8N5hzH1;{6kk;EN%a;g1{|
a#~RSl<bc{%c58GEEAA+XQ9)m5?<Szwmdd~8DLG_LaaUq$$80SkQtsl0e`Dy&z&}n6>Ma50P$(0J)}7%Y3QekKn`OGx-gYaM3?VKhen{h{trnQsv?<*FdwIH
=rVa3X$xf?@O}n5GfjS=RS%(k%WS+C6~5(1yC|$AD6~oG0A%*J&VegTF4}$y0b^x`?W%8cKX@~pZFEPD?nL~z`)oXf;wty&=<xbXj?=>&tCrubyxEcIC|Bfd
GbJ`<1|FPmT5j4MAT1=&X3{V66OPf{o*T=Lce=l=SBlrN7ixzX2J+vdCNah@XbESIoa6>Ds*I(b<x6_upyX~1PL^%D&YK%=wkgWCu%&y~x7|t!%j^px5u>9~
y>O-syynWAaY5O_$||3v8EOolx2|!vkk3?$`!CuuTpX)m=b<UCJ|-T-5srw`9v)p?g;BAHh5T&B3rxc|v{qA;ZuLLhrK$Z6w$9s6vrk0|WX$o&072Al=X>Da
GGq};Um65tEY-3s%>r<bM*}XB1(tt;TRus1xopDv%u7EC<Ag-2OW$DcMusaLv#KWADM3H?=<l&n{lG*!i$CA|jizgxTClj%NFLhk#OzJFa5#c?-er9<$sy<Q
9Yq5ILxl5Yx|hZBpi)^APD_XJ<bh^G>O6l=f4F0|FD_+8ZZE~n^;s3R!EEe@+d~9izHKy)e>#ctiGDaB%W{ofzX$<js)=sKHxWH^R|L=NP=B7>ZQDNF?W<8G
r8(s`J)@pTR%T<(7BKj9$l>48|FCogishAT1IfT`vc0!W3>={Kn%;IHd^3DBmIBCSx$G|U_6Utdti117^}(S8thuiCx!pP|kRsaS*RS)$<zuZZJut|?VBW+%
QIdPB(xQe<7MmDjF&*KJKc2kc=2=XBOiKhl8nKrWIx)6J7z%B|xX)XRuG)suL$07DQ{=TOdMOrCd0Kt0!0`eh`~c9_*hTM=!1nZ$GDI6izhvPUjtTFX*FBMX
;$TEw_-E_yIQ3xOYzFYvEEa2_fqkdXc{Ws<x*GJb)nP?z&mnWBsflUhdF=IDEA<}ZCmZv9p%zK_TYc`~?10~fOEI(Td1HsoO+^;?<cedIlv2Z>qJ6kY<75tY
-D^1|3e@IY*P#-3sC7=BkIZ5f*9Oq0Q|M-V4M$AzG$ms5xm^k7Y->xq-f@|g?GN4Rq)pQe=a5(xRoU+4f`V@ggOG(1@mxPc<wIMLvcao<^B1$WI%9l@k6D=z
9F6$i@5p;Oq<n|)UOt#9;r18Tj?VcO>DylzFkOREj(>qpH1D^h5TM#aR!*eBfd+UXR`vixVnMtV=_O~;WBRXsD{B2Moq~5kINFY0JKQ&`&iQ^M0%7F3gZ!{H
i?a4~n8whYWu30uB`Kse<#eSb8oMvFHepYL>X+nMK8Bc9qBr{QOyN?-pijR8FfPxoW?c^@rp@=?r*xawmtdOD{;1&de~grK)H~t!1ex1Wx9MnhuN7$B_?9~o
;{O8GnWj~^Lk1sy<2SHsAB-beICd5G69E@)DF$o%d&se(Txu>$&x+1-=nV+4!NJ>k$cbl74M&Q)+~}puMt<12z#o>gFR|+`{-!~+)@QC^)|D%<hcO~w*MM0j
n$&*R^YNLM<wr74Ekaze=0Ir9+|g%d93|!!7wf*^v_So9HLT~(#UX8k!@QPU0esizO%K|tcIj=L`$)ocD@aEZ7}pOziD>^o&)H(WHDavsrZnJsqK)<aYdDLb
^T|d6YDPS<T$V#x=RE14t0Tz0>}mI^8QGz{L%it;OT;|s;YE*hELqlT&{X@JKT)GCWJJCTl)}L0bw@HEhz(q=4s|}5+R<H&69xDlwsUrGFQ-qU<_fEHtk|By
;>a2|Sca5Xi^JR4x9E`fF9)Q<-(q5o%kw?0vChr@)@oV@@tX!K0Cu`hepgYIVY{XJW)YD3Cd%E1fj)zUey;G1k+&Nsr0y<s&Naf`q@O6>voN_{?z9R;0>S9-
gx;cB854h`LiShazuqdqp9&-sqv9qCUjGMbj@d8=!H0$lCD&isZoil5y+Nq4*i)OBWIGOkHF_9C4E_7acvBIucj><piC{4w=di2377fG<Pfd>C@p|aFh~x9G
kvFn0!Zywbv9g%hs`kIr8&cGnxGV27o78N<XY)J79n*vl!1wv!I;<l0a1bWx)lYp8B3IfNeKeD<${SusmDTRoir`*>@k|=JWjX$SvV5Pt^yj%u6g^}f8%RMM
Cnxy!55=^^c^6q#ft3x!?pK5(JUK^G?c!=HW#c~H6-+en(j#L=E-mQWz7Kn`1;ed9DCVWVk8`{5adm;|&`eA*0dS+F5lG^Dp!Ua!_);#EbKtZg^igJxyJ)ns
JdUrZLDTUe%~xAQz}75FG~7_`(lsOE>n<oXE^F8wQ4jRqLLQ<<p_GDJ4>PseHtYE$BaS=`7f$5>klzC^Hh%t6<vJUjsWXpdXtDa=ROH>AdC9#BPQWK@fOVlp
ePdBK6T`FG45*d^ZvZ%KughM|fwQ7O&qRL6`CXat^r69~!*}4Sd1X|LcVRt_z>oAlFG_ehp6z^g(T$H6sL{2%#Ahafcv3dAMG5x0%J^19ZZ;w6>KTv&e!1gE
SIbB<X<8K%_(-`2>=4*wnQ<ZuG2phGPb9Jy3}?poZTYwxp_MwGx;EJ2fFpbr)CSym-SZ&A^cAo1)WTPTB(&2j!(hR6d#`ze6*#5(Nj3rDf)OjoV6oau^0Tez
YTH=qQihR~G`T!GV<g<JkDZvooDHe-hh7H_d1~$llJ&HYQW?c861&{TUq~=MM&rV699OWMa08K{0oq9SM2<pv=$j-?ojE4!_3=WTlJ?<GTbJcx6vBPRNP(2(
y4U<{Vgr6NHGE^H^5-yP8-D5iH5qz^t}YkL!}cYWYA2pC0YA<g8Gq<fLv&1@9tN@TMd83NPvQfk9X;88CPHcc$sg(U?8-S3r~wjOjMkGyl1D8=l0N}h<d9oT
oQQ1!&^h{k*w~=b-jb<zkPDQtxpi^zxKmzOcv|_sk&!d>JtuRbIxI@o!c6)lz41tg%PI<kCwj2fH^%-#={m`d+h66+ibp&bi?6l+VHlWd5&6{RPrwW_6C#y2
)mxe$^q4wYje==qNbG0R5mEKnb_B8Q3xazRCGnFBt|_+brGhER=FplOxX3yxAL>4x{I33UMTYajo5pN2%8zEz)81GTxvaBEDz;V>88VEPuE8VxV@Dctb!y{v
LHp;qL5$*Q$gG|;;2Bsx(Q^bR50d;PTn-E|k1B8l!Dp9>^8_t6BHO(y&X6<r@-9U5AQG9nPng00>;Zd6NcB)62)WO=V+y!E<S%J#p%!0!!Uxt?0{)aeJj@aE
G1{!yt%=li-H=kSc1E6W!()nE^EdX$^*wUcaxUzvHU=g%^ydcJwGZ%dbAowO{DMm6WT0_Dfg2UxMGc6iln`;A_-V4w2GsH2vOyFjSf_bJ_#}wjX0;$Y#~>V$
#M#2{bVV)*l1WsVExh7Fb!tN3Is?O~cfen?+Ref4+kD!XtZNTi5Yhi!z-of5%uPURNnO2zi!g`KYsenvD0TGz#RfqIUJmuY9Z$<xsSktuzF)SIvrwxNp@oLJ
G8TgJZQz{h$>NNx%HwWW_M_A{C{R#TqM^+Ge!NIoIw;B|W5>qz(E6?hEmFMBh8q}oQpTubf~9mSnAOv^xuaToER)-$LjHckY?Aapvdl5avq~OuR$2r4@K)mP
p8tQ~>PiC@9lI`o9+y~*&KM0mj-9Wkni1lHtQ0xT<ly#=qT>8&DT$IwnmSzTjpml-PbpuGF|HmW0qOKqE@(1A^eC%t%pe&d)a4x*QhV1*#(jy|1rZoS!|iNE
MEq|70bi?BNTS<5)9A@XJ(FwB{{HlfcDis1I{PG6j|Th^J<sch?0#wy=EF|WTqxun6pYy1yvtw~SYoFkL*wM#!rwnmGrQwxTM{1>=m+0WfVwp`ftT7hg-^rt
lg49X!@#V07Y7<`+f>D-^V}X)Z(Xhn-0eD?gqsjpS4qcR(3^!idcV@ld;$Bdzw!xcmoL5L_9D!kfT=n1HEF{<@$Pz1l`|}@az$@JyytjRvhU}*+aue|X+XLo
tCbQ3xHxHg7LBw!@)+v<sI?s!tBZUa@f#ieS!hwZa!~=zx^kdMzQ6;z;x*RlnZ!P>FWLx~AXGW$WE|5I#p<l9eK)@WLt6?XG7pp&Z81xt+(geeM{shbRxW30
#od#;bQ(*>QC8-z)#MTtGN^;3+uH549Bno;xy>1}7d4dxY?Em|3aE5lFE%&9=;1D>`G35N-#cac&9(1Q#ypRCHJi@K`jvC1>U|2=ba@0!qdWL`L7_KZCdvoa
^Bbk4CyUY5v%)uNEyJPOUsow7i;Bk=UGr)GFwrLYmiR|~*Z{CF%c?=c;K6_)fdaA<HEEtNZQf*DAzJK6Qon~CCVDw4JMDXd(?xeTYuB`s&8Z@OnVCiQ^U5&n
OJ#V3NUgO#YWQ@pbM|jK@{chmOYh<-Pb4Q&jF61;Du`$4MTlBjzXno!vs2*A&8m(u`+&7Mj<0h%6LP6-%hZv7rN(x&ZF**y{W9&QZZx4bhFv5~R>!-D3fUuU
q8h~mY@GM0<b_?(N=>FKoUq|wP&>Gf60rqMCOuFppqru?Q-IsRRydLiK&0|pGpJX6L=!ogQxJf>qG`xMYDSG5v*I+XIoHq7<67`ot$3n}M-^Oj@LZ6p0O%~p
hNF=~ep)FZ+(j1s+PhYas)^LV3*4MidU1Uk6Ze0+OOcJh(|;R=Q6Tyzh*8a&hZaOFPRbI)sH+H*G>S*dKmrE4Ye-8jbo!hN4#;k~wPz}^N4IsNsv-sBciCK^
x-Sc7Jz2x$gQ`klfID_IA?$Zp0}XHqNC9eToIa-DTeVs=gxEFU0($VU>P%3R+c`(>Ygb-<v4s5Jtg>rI?zDFMp8Fm>{&+%8rbeXGR|oHv9mNkK79b_yRy22n
B`JlpBXveQk>~uzHu1ko!&`6iiyF(Yp$`(TnTl0dNpAjfl(nT0!qZkK<_NUB6IpZH;0oG415PyQW#ltTI=Xfj=7jtQ12S(vq>dk*_>$}sjh4bqw97+M!w5_B
Ai%C~W(`k$eq-<nG>(04lN`@BPle$d=K^`r5`#SQf9iwBqzy)ju@i21|8kE3vH}<he_~_XleojjZ7wiDQv#YuOQD&e?$44$#d|`RePuxPqrhLOY~!}%dIRd=
^z{;3cfK9Ym68!EFta}#{GyZ5&QkvB7C{-fQ+^*DNEjh@p4O!R(u*^)G$??!nOxq5-N-?KuE5rsQL0xRiD!l0jDlKI+x2pEt>G#Mlto<S9mca!LYoRV%xp);
fu_G_yY$0=w3G2GXw-q1FL^QDRrcH4t*ZAjRpOaJOewoU{JE8D_S=FePN||s&YX2B-)kY_N6OeOUMT_4)<iP$8cvLdObzh8N<^#OE{2$c=dl=ovb?C5S0RH0
hWO{hEq8@?`7&u`+#N7CkpVDdA<aTqXo%Ci19jxfja+RxH{(1b?XM;aZeI(-@<~XQ$R_dmW>6b+po}v#ohtw+iZllpcI!rb1z=GXb0we~83b{)@5k`n=zon!
aMFAdY`2c@?FyEPe{v6!r7veW93Y{G<638ZL}T!>(2ua6Gvue^wo%1Cvo<#cevfJl^}#Kw`nN~!DI6#g%B=)@k_&O1vTH~V^V1)-FBLw3+roW1#y^7J0&nJI
xO%bLmmO)53A*-)rR``~M&!Ni{_wnCfLv3rIli_9bh1;avMju0UXF>%sz@3n&=~mpgZ{aGkl(Nn3M_X$`dqza<}j<Lic0yX?digw!|VHk=&wn=y91vqbOVqI
9yb$ok8~=gS5==eSH<ZI*?-i@B}Gt8M|H86+?+7YJw=~26|^8<qUkM8`^u(7nAbMAi)#^-rTMBF(W|LMQEMSO`&Fm0?)MhEnEFZ{G&_>E9?b36{Na6J-g*{a
k+CYA=d=(v_0d_#2E0un24+pJ3Nntz{WP8|w`Tyt7`H;HMrNQ+G&o~S-c6I+dg}Z@|JPRO#!!WcS#|W63xp#x?SA_W|J)$BfH(!fz?O+UEYj}ZW8R%e4Sv<i
{rD0@Q$A^!yqD8{yLwwH>Oe0wGfaJ-W}uSE0wgnR;8^2rnzNWXiK-rZRv<iYr^x;v-g1TTr%tgEjv$|FJO5m2LcI#_cc^Bu`%!!-mvQU3aS+fDH9}GP1k4V3
UuFbNnt|#;CXe)=qo5-HqTql$szm8_$Mt{=aMcEdZf&eQ`XvEGbfpGBpR0X{#RcNLv=F5^&}^?}2`eHKtFC<kQ7fsp14Z9A#Aox5gS~?kUEZa$^btB+Y;=r@
;&VA`N7(YxR}TBm*iJFrr^a#IT4T{zRK-Ph5Z0ogln}ct#E)IRfbfm39uG7~v6a)1&?uPjMfZAM9;tJi5!9=)ripHf9{!8QscL03>UH;2Ff&VM$`|{W8*Fd!
x{QBN=U9p@McTlcbPAP2X0F6^Og}!F<QX?S1-CgLzy7_4f18JinPk5kO5BgkUmYWVjVR(2D<M;3M)_%bBr1c)y2~gY3?z(qx$|X$)vW1B<qY%z=dcG~K>=)~
MjXi}yKF{o)xI&sV(Ez;kgfu=UmsXQ8QPD(i-~fy`~5T~5G#I_US-$#`}p9Vxc3hYv`rc#>|SA^#xRQ=v}Id$@T$yGveNR4hq68r!D(_c<qOY<kq*T$y0n)3
)`bHaSdYb)Ny(eSmmvcP3{N}WJ)46kyZG{ZQnImNRFIm~|KfJ5!PZtx1ke+dq8k(27Dn*%Ft_UPP0wjbEzDmO;sv2ubD^bFSyb{;zJgaO<6Zz*dITH!c&S+9
TAad)C1jadV-Ba#W=i!8Y=HGfl2u!@qExvok3gUzzP^VvQqs4B_<Vndyj+mrX}=$aQddL2t2WbJ@;zw&jv5Q-jd1>5VjiB_%cmU%0sCN43@P@3Ad@=s@nsZ=
`bo9~ne<DnAYup?7i==iAFEh5MMT1z1d}tsp+zO8@IvxbjRH{z=)m^mEcjY~-d*gBsL!r>ii5xvlu^Sow3|Q_;alF*GUNsOD9;Se*$xG3vzGaJ>%Ou?pvG^i
&|RK%oT8M1BZrZSNwq1f36+tu_gO-~{?8c`t(jkKwR6m~k;&K<_JMK&!A9Fvy#s2UvaFmFsPZKmQ_4C8-AOhZhA|^`Vc$?o{mm^^aheu(5NZo@3ST!K%&*BB
B}mro$x62tZh579xLBH_*!U>uF*_V%B)&AHAPR3NpW{@}i<dZtAg|gCY{YwD9%R=keuRH7uU+QeI<JzXP;;6A)DVl1klJzD?B~yb&*P-}==<_w&J+hv_w%r0
T_02XVKW0RUaH6y&NTV%^totMIOLC0(wh)vKnKnzkDxEl^J*?~a08YlHPt1FN$s_#xt?YlT`DU`v}rJxRp4ml0JCNodWv6&2>|KLIP0bjAN4xEAqOMbL&88t
zvro~Cr$xV2bsLqEOui<hpzH{eE4J&cYqfPh7M|+3-t0i(aqU}Nv>IGWBJFt>98Nz0<vFfhNKVC<KU15%8O8~a=yH!vC6%a5Q*g*8_di;@OupnhIsr{82*1|
MKpJ~DN^WAWH;m^BHhs#$|%^$vR!MPB40d@47wXu&8mQ%>-dOzH`c27tURt6BF+D?Lw0;+UvKH`6@iy2+Z~x~PZWKRd{^6WInDzEH<p6BRbeD!laV{7jo`-q
U7k$*YOAY?Txb}s=0(Ui8X`0j50b`ow@R}_Zq$c3j`Fk|M*4R~j783!acp;TS<h{0md0r{=g3hfS66~U@Emen-D7huI*SO?1GBRwNHXg4z)~jMb7sRlcuf^x
tYuf-#z<D!EY#{*s+qvA3SsVX0Dj3f5VF2zvcT=wLHL3Da}eqTgT~5O0k-YLFY!X(3W(=FA}a`j&*Z(Py3^y@Ew>Q_I~h|r7fkNij<QuZ?H^GcCC?mr;&4D<
;w|gmdPH$$2x2764klJ)8#zEl+M*kp_#}dLdyO1^V)XYwr$i<x_R&vM9OG1(vwTp(*KinmpdIEs96EM^Xm6k(WyvIE>eD1-N}zBHn~8G&J?2Ob8m*fzUe)AN
RG#}H?R_htO0WU&u3h7@oY^2giVdyAK9*n0<Ua}~fhf5x#qd`4-Uz8%WLGwPSl9mcr>Z9;eO^=%o5=`!`0bp*`U7IB+Abn2CAdBlA3t+t$MG9)`P+G;25BMO
-rQMj*?gDh1^}Y9sxxc#an>(YF_u=i!jplNe+PDLLzaC2Hej4_wlkWDWF)uymZANe$BU=Ph;D_}CXU4q%FIw;D<u2=G@wOJgs~R^@hrzHz|^zIdjjNQrZAcv
EMCcUzzZQSkCd^4UXm3~JX#|1HX7!X*;v<_cvt~rWl4Xg>btV&PUm6xW|RB6xM4*KI9;BE(pDPbqy&GwUp%_8T8RRy*^W+g7rcv|tKd|5k#;x@Ge{=7h8AA?
0)b~v?jsH=heyo-!L)WP&v&J1T*wV=$9&4;Yyk4^lE|ugMQ*T?G>{_*mH7fhl}B`W3aXm^ogPFSDs$<BQj<gy*;PCDzA+WB>yzf+(-6ko^`;gW7SiykZ;ms7
sJ`9EGZ$?%isJ!2%k0=1suAy#h$Nt#GdYjZT?k5ZGs=44jI7VE#x6`q$^sctrgFk=;ZqXr6U@Lla`^MB8E|33tuZwU3K#E<A86EhkD9Ic1?$zxeIGyFMzP#A
(Fg7g+tLLV8mzYK_@BCfMrXfOA7lU!VLdX}au}yhxxlb6a_p_yTEBNU3(LL8gc|9v{gXRNgfUG~wM2?{q5WeX(Ge0^$aNCRsP39N2)wrD&F`z!cyRXC!qD5a
H#nye>K?(IBZH?;0>zbehuh%<In>;YPPRcMKN;s&JL`<ZFyMxckoytD=&Uc0>f&HDF2j{`hk-$^pwLi+r5=zZfFw~W9y^d#waPqvbC>@%aJ$Gj??piOXDfz#
tza#-yqOmJc9Fpt8!Jg1r=z>7#fuFsbr$%mWp=4jDP@-lJw2uiR?!{LWw3?U2rmZV#snXs5-xD4{kKoZ0IwZY-moDvP7E_3gpl=e4Yexn+tpI4q#RCISqTKO
-gfu0*lP<RFn=>rc$fqr@NsQPbFIcE`#qWGG?!qxHz(m+7}S%0H*}A;0T~6hV9G>C7^-_ZVRc=aOg2EJ1bTMtB$O8;<pc(C;(Q;?1H905Umj#!2zPF)WtVsr
c|R>wr>+)g`OGw+kVhXkf;Vp7Y`7(l$j)~`?^LgpIX&ME13Mn&J4(^6=xkG{Wvul;JHEmwjWWKY(zjw;1wXXQFI7qgEso`3$T>JY2sQz1ERPukWBMbw+)IGZ
Q`qJXub6Ou90qwes$L8^ygr;@Ow7XvN`pln2}Nv#0z-abjzZLY$sgf7VcZ6RV+f|yp-9R6h^yoS0s!rq)@1hgHybC>i%T{>@UK-9O&aWq-(tt{sJWfTQFrM6
llrzd?yFu?);BteOu~iNl;(~#=CF~Sk(I}as!OL|hcs~JNnEcK=0MKxs}@M~5R+Ciraen1jW0hV0j#cI+X2JpnH9rEos)B%j*4z>xADxuyIcSdBR1D<I!$Wo
!|RQTW~j!LIoDR_yffX4>qZC#+%++8?cj=)C2TR*&zi&Q-PrYCcRxHmgHAJ$oJIzf3-$1OUSV{+Q)ESHbEs0thTNOeQPbrz$uMFkiEUuvO<6lpPU{;upKIcc
2p@9f1K*YcH}wWm{<f~`-{TNBOeU@NNg_L$hZNJo1J@+ZL#8j4jdi4OArVI85}P>o`FnPmtLN^N(OTt@0?91o&IAo5HfSLH3Yg^He99D<&iK`DG26qX{`b?d
+6PQij%-i8>Oe1l5BQ~gMJK3fJ^@_FjvyM+zw(*hqb)TSppY(H&HcN6U&h+Y6+j#%_wVWIk!#r>b95PujGkT`RE7bTb9$~!fAIXT>4Qu9R`xm!X9Rxu<x{K_
EyjTyN_vZcEgJM*CC(b_piEG_F<!rsw3ff<i96MGnVWi%&YZWU(I-e1Mf_<H(vGO0ESA1DuyQ%{is4Vmqbl!;HU;`sYUJ&r->w_4+pp{OG^tBY!Gaw}g6IrG
lg9dIajnziQN<UQ8H4-LmQlTGI;jf*If(Uw{132yno*D<3hbJ-8l?CIAj~=5TRVQIOUiAxkkGs@LJ^c*KCy{`Hrwo9@d~A>QBG%B2>@F2zS6dgNz~d`FrRYM
tyU$!>yAlyZ9d;SI}eig^zkFrHU$*il11J}N0)(In~uv~ZKri{J*q|b>VD(X>ARR{9J3G6`s!ElCf~3P^DGHAE5T{E$Qx=%ED7nY47Bw!jKxmZPZrf1Al;<l
A|s+Qc;IXkl?|<L9Y$zZ5ifrs*s)VDxbmOOX%~t{%qti2X%DTh#QyBg>JoIKss3hX$0%Q6PN)RlM4u98;0t5h#hz_U{EmRq(Fw!Q!JpY0f<oj8H7}&)Ys5cI
H@@ME(XWr}^4DGhb0RLz$EEbeoHn%**hqnRPTQp_aLQ9yd*NCrQ!E=|0EPq!ACo+7m`r?Wo|N?-i#DRVgb?vqh08rjqs;bKaqp#AUk_hQX>yzd%QMtu3TPV;
%M#9J!kV}p>~!u8xUl1U8re+om6m7ubU2I=x`@$vH`1YWzaa;Qofknu;gy<)%@avKgLhPml}Jv78|f{F?rF8upesVd{en}loO9k99F#@d`%|zHv)k4&vC@$H
n4?3AZAh(oM3h?`%whg>zzU(x5`q~{&Rd}P(HuGA7*;r#3wis<!<JkUHnQ6zw$)@BnK=t#rc`u{TF&{RXv(r5cfMhuanr&J#K%ucJ(tu%OPJ3YiBQh`M}idg
hvk^eeP(})y=zFI!Gny;8`>zurwm+j=9M07t!Si><(wW|J;cz#p?m+L$w@SF{LIO#d=4ZQBaLSbJG9D9{)`_ZuEZGcs@(z^mQE{m`T7a;vX*?s=X!5MWnpnd
y#m2kdJx4f*iOGgpoBKY92Jk?8fFI<NL)c}sYx~07n$pF$$P82^TDK?<U*)<%q2a65_ahh2;eca2Tp%cs5j~6w`PPhwKABYXyLF*CSzsMjyr*h+Sro7r$nfm
q%(*Rda?+ZkuMs1iF~Ul>##Lt-dgkTrlb8iEO&~#<&PgHw;63)yLvCm?+696pbFa6S8un>Ws!q$n&v{aGC%-wm17yr^H|yYn~G*x1?l7nH6t*1i;w$pmi&s&
JgalL0#t{~y1J6Z$OaA)qRd+URkiRetJXyUZPq`;8nRx-&8q}+Z<>Xwn|(Vb%i7i`Phx#;n(6pC0QiApk7~HyvK#gmz8iq2{S_V=YY31K?Nv#deB6xat7z*a
j3oz@azZG<l*5d2q{Us~`%X14j>S;z6;Zm)&qqMC#j{%ArBNbvi97z9SyPGw+Cl&W$F`|>%A9RvAArxsXFs4}6ozhuPF&HRqfaIqDUXN=K>-)TeXJ|!c40f8
$Eak#K0TtNm*bJw)CMk9ORZqJa`gq_uR8gBZ#pYq@q#=cSBa`#XT&APEoaLD+8*_SL$1pLtF&a7SMf8FYIM+_2<}WCYkpe0P8w5Y*3CYUd+&_?{syQEIS^B}
LrKT@*vpt6TWxF16QoB0)CUx-Exw7BdZ8xh842Vm0w2?SGU#&KAV^V6D0=BH^91`~Ca=7$46=_QqKuEQ&t*c*>zy2&tSTF0m%XPmST$8~3Aj#oFHWcwqrH)$
&RfZa$<c)*@u7k(Mt9@9UEq+5=dhh5m<5~)ZT{AobAvt=rh|Y>1rCM5*``uvt<!BIC+>y75-pgx9N}YrBiQ6w(=>ERGkOPb`*kJC{c1e#5*Q{cpF+t+y{8Qk
l2GH!j{@svo|3OG2Bm?#x-B=Z8<vy{S#f$X#G)(AADxbXbAIT_On!_Z^bX(EH0L91kwwg16{?Z7NG;4=^m|ty;vT3r9_5Mk0$D6&kn!5H;QYVD5~0)dJHn%f
Y#S5%ntb2a<s*XO*ve3ATcjkBp{gO>P11~iJ#Nz<!wSHW1l_K-A5bhUy8rGu13(&V&QpThL3ML@_{@$!;HSyzX@ghN*1lHOhQW?r5oFr6<r9I6JVX#}`k71L
%s_qUz3Vb#HnMb5UR5y40$(~`cp?;%@hN@TlWD9UY{-o0NRkqGRbaP%VfjApzea{-YRN|hM)Iu3vw4L!56m2~CrkLr4h<+Ta^^hR2nEVl;D{2PxW44WL)hO$
tG$)x@paZ8vg&XbG&JY=Hf+!%1%kr94tjv2?0;zwacjZAncoIEk<5+dW($%@(~gl(t4@B7HVU#U3<j~v4v&P*T|Y7<rI>?ydM@-1DiALjp2)Z|h!Cl?dZ#qU
o>nv5{F@w#(PeH5SOcW<C)={H?w=K3`dJo3@hdrxb4)GX&&IkKIq*(zOySI*f*cCHeGRMeew(&>)|=n%iE<M)m+k=z<($b-8{su<!o#B_{}@vt1K5O3TZ$F^
v=jMD$*Oz&cHzg|s*^!L;?N6*1v|dXWZxnEbyzK0LaY#3d<y)$HM)-M>&!GQbFd8M&~kV*;up%=TZxluc%ROwwrAkK7Yp4dnHUa+s{g1gu?$XnuPQ6zBdseq
K9_#l-Y8~}75@7!Bm@LUa~?aAZs`?M4E*8Gd{P4VQ=Qsei7o)uoqfSnL9p8|`s2C$Dh${=3l!C`g<Y3+Q#k+E4i?5obf4M@M=1q8Fyx&>VAo1KYq8onQx{YA
Rh_DnM)}LM`rC{O;&OWkY|}S{0HF&co2afUjNOoncAJ~jt|~{CDZ$D0@1qKDW={eiDu-$N(tS?TQ0kkK&1T}Hdu_A)bf;GAR&oFR8l~$?SrZrrTbZ!%#6~?m
5x78>7Ypl3!+wq&iwhdFcmYO>NpDXnaj}HEj$9;B+H=$^s9VSXA!3F+vf_w|UQavJCf+K1E3`RbN6nokTX`&GeZBXZRwuY|6T|r@5eTa$%ncqL*7Mx?!!59n
lJ8W|c?b;IpMqBnWg)E?5Jv0Hp}vMcLXY^Yd9p`?-W63=6?MRL0GizvfT>s9`J1QLP9H?1CDn8ipFy&g-x}o`32$iTs&jj1RBOPc$8b<ajv*Q^JbX6&P^e}@
QGiKAVE`I!>U&%)BKK@?m%wMve-tI6e>|9!VL@SRlw4noBjC<b3LMFlITY7aeDZpr?Y>yYyf}|<S{B6EMUJYsY|)Nzy9pwiv;-e-pbP2eapkt=U!YJN$+zh-
oVN6iGVh-ZIqrFQV&gBLO6UM3gJVg3deVAvU=5EOuxUOzY~cZ_2>`5wT7w-e@*;@L<1-FUTz<nThLH{U<o?9thF>X|N?5UNU8E~NF84(-*|Iysyaj%c%k|+r
5TwvLY&0z0aj!+_i*IjG6so4*yN%_Bt&nM1MQz^~kT{7QP;#4JDu4y(i9O!xZ#QXJKn>ZcdpYKN2-39oh;@~YU7AV2)1{e7Ly2@hxH-MQQ%XmAQd`u0Zor{w
p%BQ+;@Asm^N9Z74&7JDey(^V$6BOFjL`w{99h)bLtDv2Km?ed7{@-S-XJ;}<;&h?Y)!!dyRC2#j}ZT!6v{sI#~piZBE;_DSt!)ou>&H^n=$WYmP!^op7Ji}
TII$O)+dtO@fzE#KXyVxN-Cz7+>sw}HA6y;Q9uZTf8PICzLRmQ_%I~+jVF2+XIpc`N$fy4k9tT6wd261Gcpf;#hxng!_-2)PqS<B&M!{?Q*U_xW57_2IU2Uj
j&RGy0cifnfx=mBU#TwAu&wwgxeHXq-O^;^6pL))a-2SJWS09cJu=mlLQj44^@wPtIP_#pJdVw7r2pH_>to4BX%tyhrvXusHar910X5fzaBc(^I?qJ;3%5!f
giBaJYv^H^H<Ip`tG<41bnl@Vxgx%~CFY+eg;g$pj;wUCx-VNNwa!$6;H7b*p#Lpakl+jQFvffE#e5#hg5}(wo&#BUBAi1-T{hHF4?SMFNR^Bg*jE!P_MVVx
^1uz-q(8-g>IZ3ev4yYfmV;3iE^S4ZiRX<Y+XRcze-5YWvXF8D47aJ|;fZDU{&5*it7&WXA~|{TeP!g=<B5ReUJ2E<SkDTBzhCFfSr9M|9vPbPW!A9G;4VlQ
^oyUPp=Ch$N4Jc|$YzdnA1s{<z57s4SWK@rkd^C{?O}cTbzRaf@PGuxfT&MH!C!Gyt1ws9fu97tLziF8Xc9HL$wG#qwcEQ?QYn0$<b-&oR+1b_X$9Gce=bN?
IbxmooDdd2zl{<P@P1yum{OhX!I?p%JRxa3e#zS@fR9v#=0wbhx~c_96*~(|?s>wh^_G@uJj~mu7!0vkDhqFlNbCIIhZI4uZIvII90WobR+LusJyeI-9&ID0
R>$QE(q1YK%-}n8Xu|ycoAk?dU*Fu=Qo69cd$I@FM&33#a@&I)6bb}N^iMr6JRH+zU~1nj86c&O&CJyLXU;B_A9EKx)$Tv=rC2$@uGbiveht`I7kMIAv;G1U
57?)!Zz9icy^E3i$5aZ7mnE%p+sdW^-XT#HTjZqs7KcqSu|05(x)ie(&c?_xs9YaYw0tr{tN>h?fsNyQIq@ik^rJL95|PO$A`E%y6kwGiO$S`8pQ107V5zrh
lia9Qb$&^C5mj;IR(!vwZDA|o({sq7U@M#<<Bkx$y`?F{h`fN+6bkoh7bj=!2P);TFU4A_bUq&f3%9*0_B?ZQgUCY3uM36H<sr0cg6!`!wx%#4pLW8irx}Mj
!pnxLM!Tj|m<62G(Yi=E^s#)|X@t1-^SzSG-K3f12`*5FmQfdr#Cs!rc7;`!S>9VtqYn94PK)XNJjjwM>z6fJcZ{)i=eq??HvKCD&r%bwd68J8tGQwosG?U&
ZStpQ$2{sXH(D-=*fzP1K|5Il)B6bsA^*A_;?hbvW}Nf{kOz>Ls}<&+G5$@4EMbQ@oNsWXKF15v&Y^L><WIZ98mPaZj7*>l3nR}?TUPTRMjF}`2`3mAnb7yy
{}a$PPU=i4nk!5MWM6Z1-}f#_X5QnG>5YOZm<SIQqJbmK2F>5^SXf228CiayRPM8wc@{l=&!7|A21-3_jS9r*>}S^Gve&p%v5@c?+*K5(^S(z>VPCoI)DGa<
afnQj^$&5iqW$)-u1_H2^G#zq^#mfh3_y5%9aUIkt`RM|;HT|9X9&V4DcqWk(Y2sayNPb0tSnOGvd{@FC?bEPp{{ZH+m@HDY@K<O2=i-!^X=M28m^;O<5oCZ
15?a-{T3y}ducPO#;b0%!sY6y1(D7bk6Uwnl=Z(k7i#gdj#O7h4%k{LzDoE;(rphi1X^0Dm6>5Pe^1lz!$6wDo?PZtLz<1gS(*>LHH#8K5{Kz861OGdtfP`n
Nk1Ca=hsJ3pB2dHmoz4n%x!$NxnSE53BEGS-Rpx6C`o&UcQI7UMd_!+92(+;_W7}y3f<$NyP)894i?udc+`l++vP}_Z(q>0HULT$3Xs@P_&Clj&XPjA7<vD#
mP`TzXdD*gj6SuWXABo%$etiW!VBv&%wADL0z{69noAL^zfEo0DTgxrq1~|9^ksRgEc|RO?m#!=z>&|9clqzI_YxH9+Sz`32o761+eoWgEdi{QM}`<}FgMD<
<=s<1F$)&-^VzrBm3E+rmlhz9&^ksnLrLz`a#YTcr89iwyoi1orcGDvRyafYPaBKNS6vUd0BJ^(UB0wNaEnK@p6MTUt)fEO_0+hh$!V~c%`$Z9F*3RkPw3en
v1+xn5~Qz%@+@F92YFB@QuE@8o9%)r)|XyW(SGVTvv<F$+*DmlI_(?dbCA;txY!`GvcW5uA?@iR-lH~Q#TIHiXyYM33e;VSxb<$>MwbW8Z+_fm0vx05*sAoV
0yK*}VBIf5VxawJu{3ZvNp+ZI0`CaXg)2!WU;399CZ&(SC-1Mdk<()zBF#bgis`&aAAd~sze!?$m9%R(aT0xyrZ)$EdI$sSRz7WvO*P}a$WQOF_P;Q?(drfZ
%nGcONhvwrQT%qaeh&J$_a4)m4)01w&XOJUkN6?9v)soH=T!q19m-OXc^|=i@r?XhH-{n&R*>D~7uY#NlK6|8`RylDUL?<^y5+6%xJ?A{y>m+2E8tT4(b9Fm
g$qRb5>cY-<p5NQl05r{k@Y2mQ7mY+=Z}Nar$gS)h|d+WQ4-GpKGwQR>g!Uv=v*>Rd9v=hx(C@u&aSuVf}l(akUp*`DH!f>DWM-M8bs1wFTfpq&j}PdN#(;_
uP?!{R12Ja?$xdYk9T$U+am|8Y{G#jJQi0Fzr|zjjr!Yx(~^r&i~JBNe=^g$$V$%-12so>M(U}$Ln$h=F_gGXL~8&m#|O51Tp0*bGAaYY+v-52!1`8o#yb83
n?7{AT5gP<KS}*<9Eu{N%EFJOVYM?LO^@}ff<)0{&kZzrKHYPb>*bE{!1;SW*6Xez&UZqPz^9_tm$Yv<{1*x)CqjR$J2cAk3yQGkCE9of>?lBT$9~pHRzi)}
)=aAk@r_16ZQT}!(_cPDAAdEL4FtyGcT+k(HG%jFq3GW2%6l-LR_inoEXnCtq^TocPm!MH=!l{uv?QA$t?YY04(uw-qME%rA(L~Wd(ti=E$Uc1((jR=@7duv
h$4g>-Tn!QthI(qjJZN7u<7$UW^D}9BQ5!*N9)>_%^*fw+dQ()UUFrg#+aJLo;lPD+~_JISd{XV6$czahj~Aeldgk>Q?Cjwtp(|dRCgER*H8s$?ql{qIu4d4
NIeV?B~a<vmHxLLw@?LS#g)bhp>=HrMQ_;b&y73LI!c$Nncr=IHJeY-RLLO&O?*+r_Qb@1-1|!V2{kvm`ux9mOz@@<6|D8U>;Hh~sz@@P6(RI!1O`_8|7!TP
F&@z{m~o^4o=ecK-K-aC9`LZy+g|AIV!QTRuMJJeF<BOT;^gWMsJm?v->)@crIP*~lV%^0Nn&O0oMRHgSu{rG0B<Xm$Q<nLO8{@bPqMI~e`yT6(8eI1$=PG6
{=inw5NC&QnYew<W2a+`wAXr<iNFg91{03~o%6GGLCx#V^biwL9-CKcB!ie+z;3zvb&WquisMeB?~h@+y*l0*#0LD(A(nX$vdJyCJ^3RLuH7Gfetu)QSs-GM
nS9@zttbGEP6OHJ(_t0o2|e0`8Qz~Q>49Nn=4fF7rj<$BoEPo{(JJ8q|0BnO-P`l}c%egmy{?}6fun~Q_NLly8zBtJnH2D1QS8`4{VjgY`fh`|8v&z%$P?OL
KqNQWVYoYBx&S~1=xouM<n}{AJfbj^UDHF54=A+$h<piP0dQe>5bwCWEk8%4=nn+vU;N_C#>c)4M73iRLi=5x?*m392+2tnS)UGa0|PJH$Kn!LD33kB{IZ$R
*k0V0MYq7%P0QyR(<{#dtvm@RlZmcpW6QpHr9pVh7W{egGT9wV<6GP!gV_S4DwFAVv6)ykFOzQf(F5I8VL<GN{SpJeedZEQN8YqJ#x|=4zXh|7;v`rg%R!BH
#zLVU<|^!#KcIJypcwl+X;9H?=fl2q*A8iNSHXwQQ)^?T+s)<OXa%}#Y-W58C+OOC%%`c3!q8WwK|!^!dE`jJE*B16QDF16cCvu&p;T-V*!m;DQSj!-OAi4g
XfdY5FwX?xMvH0=M+T`t1S6D;4T<-Wr(D}uv(`q!`f{zTz<W^@5-)TyuA(AMmxK02q$dx*yVJG|qVf{r5VE?^0d*1k|3**Gs{6%|7Z5sp2+9dtSVbA5t(Pj%
>bzZ)rthVrc-Ipce3@hPuBWDR{$b@Bv=|?l{Mf9eApWjP1B3Sx<1qh}z}rzed@Tto^jB+9FsfJqj-b$lG5Uj{^coaHgU15fLdsRTn3NxR=6}8M;VG2P;oi<)
ISEt1G;c~pVF8l0d)qIL8FXL#4hL5I^#k&$4KsUBAA2)2els$*yfaf{=k!Z~Fhf^gqSQjKU(+kwxi!x8co9&+H=y)UsK1QceL<RD5!$UXv&RfiQ5_=2{t%at
P}%)z&1B6fHGNoM^q;sEOE|>h$-S#`w^%k(h?`7L$Eu)V4__!s$P3rSk9`mpe*{?_#=W7AmT1>Yj9E<^F_}Z|dtDFI-(6E~Z5nZVuTh9Vz?po{QxAA6Nl>~g
v3<MmS=dA_vUm|KG&G^iV5a=nyH&!*vc~Y>q^p6c65(veD4YS+oE&3rnO)=+pKiE*g=CVZj!heIq@<gxA&S)O0mQLX<=cT8siFNm&NwUlxPavZ=mW8VNMaRT
IlUW_SMGHt;5d-;bZu6}LA0Lcg&-Ps4<-m%tqEqU0G-BQ&aZCq5J}X_C=d5!#L*}GCt#YlGRAG{&udqD#gjgHDG8T_^Y1gZ3E{)<;D;BqFII<zAIUc{rto_^
Y7CgF2?(VG^X4i~OO3-L&VFc;%CfKvBUB$2i;ii_k-m^9od+xB33LDSojDgFmyk;8o+WWUWnf3qmk_Z+WU13l0eEWuZvrQ^zm=H}t8`bs!T4^EbOt-M#0Ml_
%PGsy621bBt^9Z^x(46pU8WOcoY{K>PjFK`V%wO2vlk_aJs8W<J0H+@zT(??9!{_hM9F2!h{;hG!U!<?d~$4ynkT9_u*0qvt-L_8&ExfJx|Qj%H*ac{<Ir<K
at4H!O(iNN%uwt{)qE(6wL5N~i*AWXR81TO--5LsaJ_gBZ(*vuiv7w68hllaGO`_J*)yI<IQW4}Ziv)Iz9QsW(oBp_Y<!{azQ+FqVvgpMM%jTtiwfK^0>A02
X1`623DuTgDOQ(ot{2GI!+A0dMMggLgoR^&B73=5x=L|0+hzzV>{`wSdl4Z}sa|=F_$i#dSo7SOgvi)L_3?-5uZ6lN>ba3Rj{jxFe)JC(wP2CU+ogV)w1<jK
zLTuT!|l|+)y`elF8ZU9tY+C8P=k)@bOF0+AuS!>>Xnk}Z%9z!&x%kgdXSOa6;fQS!#nZ@MxE8Qi$Wk%*j<IoKQo>o+UI*#AXB8*au03_T^?Rg8rwyix^U7>
U%m2fK9)0$*-lKtb2!kscr|n+mtdGA3a2*c*K3jU<JSyw5{V%F`p~jV{vLH1`zLAdaW&u)Y!X}U?;c^+!MFNOjdP=~O&zbY>{LH8H#YEX?iQKwBZB%4Fuj5H
Sw&`2gFASuS`@H5|M<dr>S^1V@3DyH@Ss&mq}kU1^ZE)GC&N}GIv`8xuZWO<XTAcsc<l5Uz`T3VtLynoy1MX56$=VRuAoVuK=yuOl2j7Sd>cHosDnbO(x`3e
abRcmr=BC<R!RE=n~O#X3VU+3QjZzJzsV_=K)!X6Xd467@uJ(Iv6HW@g60R>tw`LNl}TYVks`-JGhueMt7HGl{$4#mDXFR`9rI$KZkAClF;KSi+`ezh8udwJ
9y$FZi&eNBd9dA|bKM<Y^I;{D(`l0WYyDpr!W(#@MY>u$ouI5N+sxzw#I7#nssAA5<j9?gct~0@m1SBrch|+zS+~ecAVGfgXr#Na*cPdfQU|S>&JxiSxS$ld
!5F*<jDjtLwZ;4Fyo~vau=9Fc9j@~yIGqwd{#-*JAl_n<RmNBNurG_s0X!%i>x94!mb4XROS#s!8-BX82pw3^>=^Oy-zgL+rQ8*Qn_J6|D_qY|BEEd;%8$1h
l#64?&NoR%e#%h30p--&_YTpNo~ik!D&RAS*LrVQXW^E95=j2+k{I>^*DkZErPCJ=zCPB1Hn2(89Ulkz!G8X+U?8{l+s8*%8g9;6w&X6O$(wAX=3ZVhQ&3D(
Ft>oSI08g^u3S-Rd=UO;OY^D7%<2VM*g+}dQ56(r1QZ+Snf9cS39*&km*Srgq<K_7?)0{P+#Ht0LqS&QDb*0cu5Hv-da9Q0Kg~7l$GxdmFKM{2kKN32gEAd3
c1Wjs-uH<>UeJ91N0xLc^h(od8b^vFGMhriVG%Z5#ymXVKGGlL{19pUcdsjN`<;t;WKU;a=9gdMF)D)jK4+E*PXqI`&}FGw__6Sk5eC5XWM9v8oc5Mw{h)a<
LC$FV2)(t1i764%qa0)ase*=7d|3aI@~pKvrW$i%3r(`$`2B1R7X_Cg#^+DpL|0Zrb*4{hIa_Rk^qSl+15}2v>SL33Ih;3NMN*TN2P0{{%A%~e7}b*~09Bwq
Oe#{7T3Pgy{pG4AHSp6ihax=vdk&K{bYmLw;F`>O3<IT-(6-Hxcd}_EZ(OMXW3I~XB~l&O0>Fe-wdRl%T<#Z^0=8|G>UYV50>0WI%edWCf`)k7ox?dCUsZ&Z
%MXfVp3ncIU;9u2Z2`+;C6%Kd6iih$4H*J%|DD@^jkRu<oOpZts?LcnC3g%}z0|WlJCb^KS602zGL>K%k@Q(tDK}Xz#xqS6w&73Iw3l&`8<y%S(m-0ua_8$~
mX>gGII3&u+*{GW<&#<mR{-*%G!!~&gBErH(w_Bi&<+vHN<4lCu8(1ENGlH9%#`E3!U_@9P8?%c($DK=4*#FUUN~!(g4OpvI3Dz8mO92(a7kkg%78<6^>Z>T
rMHFnDQAFrMC_P{orKGy=u;y#skPUpu}*Cd+Hq;R8{9!yW<KKC&~ky!f_j;~DqoD&UnN+q#oBXWbv6OAd|$su_E|<%Y3#KgG8)QYq>wQd-ToSBnsNh1_$@p*
0{{lDRO=e(MhvWXVF|b7WAH*3O$|P8HMDd*G)@viigl>3buV}Ab-%^X5~cbL4pVIAyv{&~a(H<(m7VoCX)l;FwI!5`)a$>@kYn|FF*wv)_L8uF?ZT@E$v6ub
B-lj4Re+0d2^Sh6n@K(VGObwQ%U(BE?n$6KS4`)%I}^+4kZvRZt_1zW;$z0<Otl-lWgvU%M2a~v$Gv69Fz1Jzzv4882U{LA01j09(+;Pi_mz)bN5zu`SVP*>
!2m{O(}HlAy&mwE58-d8zv7k9vl-<oU@f_&xHGDrUi4v_?aZFue^76}N5I;a-cCy6aYq0HH-PamhK8Z)?zL6sXVuRqCF1Y%6BGJ`_3FY?9%sC*h|yJb0(PC3
u&?{{8c`mGta25s`@T+T*L0x<9{38u2!c%^2`b5kBJMZn^O0jbB{IW<+y7Yr=G_ARunD)fxlTy)PQN2C0jdp)D#{z%Gjldmpc~evM~h>5nBk+lnfli;m!fC6
sGEk()5@+6rx!F?tPlTyBn*nhSVC+iFHrVgyUpeAJuI2?(Lv0D;lKXvk}7)#-6nbbVi%xk?r}>=KB&u({eUKS9qIoQ%#U723f3<AZIEyE8r!=o(Wbflia`PN
Rz>z@a%{bVtMaB3yv|<F5Hbm0s>trbDb$<WrgY6wn}w(^^tC@crVgr6#rh~4Znn;;kUA|of&iu6;5B<G$_H776zzIJiIL!u$;+Y%mcJdDIw)aQiPb`Huan)5
qrRHf_ZexC%{;Xfl>C(Gq+289xs4)38A-UlB7vFbVhK?<K=QF9su`%ZlA3xSL00={9qV*+e$gpxURPru#e9)$iIL9dx)Z;W;TZYW_gXYOhma-6m&>Tt40o4}
eU<mzm4$K1zim3%E9WZ|m(IzEP6{>7?B49!HT&bzUwu{4Qj<?4*_i}qo$rKe^tt>>YVy;l)9o}{&cYjlzRHXVjozxh6KO|LR6fHHR|;+Gz(kKNwK$Eq(eM|3
EGC<dirZVZ?7cpr+}_RO`04%eADdR<x)d$O`VHbOFDDe-f3GGa3CsGjSmcn}Y`<zEe`909y+-)EwZ7eDza-xkr`G#;0+g(aE`b#Giv1VjDA%ImV%5U%c!IIO
b+p+Z;RSz$%*m$r-UOSpe(rzNl8aDjEFa(HQ#J;RVWxl9>!qD@x49kRo#qZvGIaPHNLYZFS1~Xr%0@O930wWhZ-^yq`u|;d><UCqbQf=Tbl;`9JBXT(TD9%&
?Qc4Qu0BT(1~PcIYwc`EU&&d^pJaVbcBk^MAFFEoX-1FY8tM4uNYJb3R{ugCi;^2(_OyQq#q81cY~rdh%+X3#m_5@Fcn=0Dv+>s!-hM{Ar<cLaa3z0ELqHPq
o|JUz{;MeCB$zWdt7<QqHZ{tqC7>_b=E5=GnLmT}jaPI#rn=>_7NCUJIMQyhJ%$pS!V}}N#^EyLOJNO&=+;D!Wr(5lV{z}&u%HPLvta>mi~1?UeaAO@t)|&|
h<oh#ACm+TuI@z|s6{&3HOBT|RP;KlsXLd^Ve7w+HufeH+pK=wzxmg>N*!E^<&cn~zL0J(nFd;Sl!reBl#NYBD4?+?`AxFfOU(-=xgA+ob<lhY*I33e{;p0d
&a&eocNj1YZ>xqJVOlqu&OWF!eK&oue~`y((?K=T6H5kQp~K^Q(wj9gRd0a3sI#Tn3xmXJACNy4p}qivfn#%d4U4`mpfF{_)K5nCW88)6snr#<3?C5H{a>fw
fiBN!ct97jA}**vz*(l(vg4WPIc5rgp@rVFPQ&i|-mx~Se>8+XH!QTws=KuN^?MQ$5y6{Aug}WVL;sNaVU{(Z-sbu?*zb>IQa>G0lgte~9e>I&>^YQNJdw}Q
BI1jvTj)tO49kk6qGC_^y1y|^UrDMhe5UO1AVYWdhf*TubBSwpX9Z!8cP`f?U<acYCDf41MH@zP92)%1Fo30$4(Zjh`(!J<vit8e__GkbEkw7}wAtoQoU%_~
JMBv<(uLWL)6=XeY_M%`n72mqZOT=P?v<)ewZIC|H-)%upavZ`5#5U_ndasw80bvSk{f}P@p&)ew1@C8mCE1>?vHs3-aTeW{EM~s(vPqtI-Q^)X=((Mpz6A3
EPWxeaF#PkSZReg=H&6k3!eB{4e!%d@?lt@>Xi6-CS+=GFKtW3V?v2?GlEnk#52evywI`XD(2Sjd23~S#TRUmgUV9IEzq3`cvo-6GWnT%aByQok@A;q_flJ)
59+y~bPq!kwiiZFh?7B$$Q~u{iVXB+szQC0d{XAgwu8FJDRxUNT=-k9azRtnlONnoxQSZBEH3(G`CMlrW%Mf*u&Q6uEFGaK>%}JnR#$q%(?FgAoy4F*)}m!&
S;&*aBP0uwQv+Ep=6LcSs$u;24~dV|1=(XwqlTLH&lgrA?9W0cWIh$`57{NP)EBW9;M&BO2|h|i^~Wz>wp-uf`ZBgs11|KUv32(LLmCbLx_yDM11$TKQc>wq
IE<{^MKA}9w4-(-bXZX0dnGvF7zYX)vn0x8eGbkPHIE>KoC)<|$*~@(u?Np(<!Rp~t73L%!QtaC{OGwP?!dbUHH_8jQIbiHbUu;Oat^dZ?gUZH)_k68Liy5Z
fcTal^?7y>OT9765R70c?9|U5=e#CWHBnX>JXPuph+2q6hW^lb;HM|$!!`V|%9=K8KD)^@D@}fzi5sq^K`k~p0>pPFLhcz)EcqE(6KU2*Q)RbjG)}Z}MfS^4
Ey4v8ry)6s{VKz&*x_IlhEKu#AaJ>uk7i(QDtc6+e<Y%DcwGkE7d>3XLQVUE<cIEkLI)EzS<pDjksLc(65z2qMPQ%SpOXfqRqHHUcs751<{$8W3C60C7^(RZ
NkGfL8O9r$u^V>mA_*r(Kzg`shV?(adB{hqAFEoz2sN|-Bp@1A)6inJfoD+p*6_mQ7J;S|O#m@p#|b+6lf}}%&R6d`1c)I-oafZo*r+>cj};0uEeB;=o_a=3
O?L}3;C0FC$Q#?*l!N{UBqE81H{peHg3`4XQRZlD?Ndj=!y*Lf=bLvv!gip<=HRNg^cWP7GH>)I!U_+<3zLNpf-T~6ahjRF>=<veaHzq(N0He|Btsydy5iIn
J`SR2pS*t%!!Y_+$F=y*141S;w?ck2O#(3C)Ef?4avZ~HOtXb}{hkME<U!R)4<(oPRmi7affZ^f*uLfT^~*MMt>6uAEPOD4&1#_+z95+AtZo+D^``nmiqhkJ
hFlxC<#}X8=IsEQ45nfBpZPB)I*vU*{=)_Rn#1oA%hd$go^}h5_Tnu;b)I4oh-5Va?Kb!`C1h^UNwgYw33Q}Tw3dCyjfw89m<|32mD!+45ilc<4s>p_UX~U>
2CmhxD{B^mtwX&2gtFipqusI$xm-v&62=?x<&S64a0y%@qQx{kGLu67r7BnFpb4ZI4~~7hjA@pFcCu{~Zh8InmS<Vj2mN)*Fp!o~G!)&3ff_vt`(M%Fz*Zwc
|G@7cU5SiaX%*x)JzZZa-(z+g-d?E=41R8}ETKW%hf7K82j>#}B6Xfh55<>48c*06^(XV6j#KT)Pg_rEqA*@R-5kZ>Y}V1<WbyZUSuXPEj)Hgs*?aL_w{PWf
Ed6mV$u92JL_Ae+J(1>)P#2w@Dn>5Ff(IdAJU$Jhl1-<AEo%H1VhtbB$rlw`x(y?Z+nc;bW!Sjio&kTEpe7RU6eDr8s~y0Pw>?v!dEKMWX<J&)lFs7!rXPE4
qTnqnA@?A%z&kq3$#B9(0msK1TJ&Jk@JZ<CDHVVYTRhI;{#c75YFi7bQahoxT-&F`zi6-{!8BRZkS~LYUD4tK#t>)Y9?D2NyQdg@{f5_l-G9pn$3F(f43u-J
j*l)9ET(~Z>0wX{ZT@OZC?9d7X!WFx%gxEutt15=+@X6lu8qh<(e&{p8aAgSWy8Lqm!utY!%yQ@4Gbb;=sdxKc8c5+Q*DhW4saiDp(8}A2^9=BzO{KCXk*HY
DTlg6cU_(zwl^~JG(a2Q?;<5^%|R`#Z&;sD1UECFn7iZ=kINUcI2zQ-QpL9*u4Ecdsbw%5h6@ELX?6VVRtJhjoq?peb&GpTxS*OofNK%>BUs_}pIk)uI&kkC
b7-(F6anBMofhZV&c|hbzK)5Y2et7Cb~gLX%EJ2keVcp6lZ}gx3i;hty*Ox?T=@n}|IO~RUXq)kU0r?p3^d((A_5ZXpyVkKu}Hq-8(Snrc^yObi()a|u)>&i
Zg2Q+v~KkDD~9Ikp(#h(SjNfnz^Xp}fW};vo{4tNhe@1BKzPsyFClo>_BtqiG#W*==hal+tV&@$tuUKS`>rID3I*JMiC)nz>jyO`^$&1?<}-|B$JjwD!Bu&X
IPU1`MCT29imcb=e*UMF`P3}055wJev8`9jS4t`how;@OB`Nn;bt~3EP&_lh!ZaZ8(sfB(`8<h=Cl*3iCuIjYV|Du^xb^!`#NZrDdmfquCgPMi?6l6zmvUOf
QNK%WY7l%24{GQn0QWvUFdxNsy_q0%i{-9UzOF%j6ha`OsdEar>*1D%ajMts(TVhM{`=3c=U;=Sq~Wylh<;c*#eDR0SH^|V5`*!Ev}h`$y{x>psy-Dh>mf)y
dhS2(ze|<M2C{AD25w0j=<BMBDlfY-t}sgZp~mppz5@N5IpNI_!i~E`(YT#bYuNW-LhsQF`6HgHx4rKO{qaEhbsUb_uQ>bDWb|OHu1_vD2NrQo+7%TSZfrZQ
Gpd~8d|dcN-Jn#cN-`{T)p3`cQH2H<st6O~YukTh)bH2j!A}YUVmrhwHI$t?xjUSBn>pFdiyf7k^CSOyZWzGUR?+=dW2-Y+x`UWj4~UL3NR_7!PAyI%h-z32
z&HdOu+jm{SsSqSpo9BQM(Nr|EZ{CnZEbgy@RJ(lM^AdRWt0>9=~P6p?~mt2YfHse4|TEzPvU0$6G9Od<ymHN7+tAMZ&EE~qQ_pimL=wIzc$6rI2_?m+<zH>
ScX~8m~=Q?wY5;aV}8jgm>z6FCGK1>uqEId4A8-R+bP=@l#6a1k9wv>Ob;aq^6av*Sk60H>Fr<?=4t$}Str7D&(k`mBi5%vGdGl_u<voVGk0P4B_!zcJkbIv
*n3+r0RV%KpB)4xR%YzMPcG}lnq>IMGB@kwz&EQ*<lWLg-di;??FMB2w+&3YA^Cx`2+>H&n)|HhV=GZu;@xl7?pZCO=HBGEt3ee_FAlTwLwb7x4P`Kf9nVcx
u*ThC#*_d#F8z25oF!c#_~r#9oaCy@jectf0{&;|t|@06cyRB^e<v{U^U(kOf6h3=r0dtoqnJremQ%TLt<;hm<i{nJ7~io}0?Rr&o_U4mEnTZCBR&#YRr13y
hWB4N0q~Jpm6Y%otO6nPD<wDZyfx^<Y0%oB_9+rM<gRX-ZNos>2Fu_L>K<)9%6WGz8<H(_zVG#Sj3W7Rw%mUL9V1)HC-@C066Sc4RU}%3csW-OsL~;rg<!kD
v;_T8Ib<PpgJwE|g*>l{I=HggQEyvnElkKz?3B22%{J~1Z_#xcIOsQirW{5p$SRRp5B%VibI<VZw~WF8pf6!K+u&S-J2b!EHnjWu0y3`<<z%%BF7;ouY}zHY
QtVV(naeiGenW)d8xu5@Ama@b#`wrr(^saayLi<D2UPUyu}o%+VxOn)FSe}mrDz}dDcLF*0J>lw+EV4>vN*F5chv(_AA)zD;u5w8407N*qXY8v-ql?|prk=4
^J<3V^DcreFrm!G37+TZ6|+Bu`lQnD&LeJB<_jUGh3WT6N9j0b|E9Mm7Oyt<pZj1!8};u6h|Bly%FC2X+KyF&$8~4EsKra6J!dBbdMd&v6#AGuWuY!Kw{N0K
<<KfsG&p?2HuwRq#;jwe**C&*I4qP{h`BR0_QtvX8!BHU>ue{1eKRccI%4)R2zOq%c%rJ1Naq6w7{^FX92w0@LhZ>(ML0NlslJA|h)ITXN!;n1m)wCbh5>kC
G!dG=2WKe|w#x<d6X|Uk%cI2YKNy>G;G@SUpkB=$gWdRtCmEv2_<gw0&~oK4R2}ykfVc5avhgU^uYjwM5RnU%pVMxfG`D9j-^k}1^=et)VxL=Qg=v}w;p5pT
bf{RA^GrsT>i^r#9=hW#C&TH1Y_Jj+XNz2`WLDG*dp(>^9G)V3Uq?fJB6UNOk3k<HvMYyW5<9~j2Gui1!Z9zr&wzjpU7Itvsmk5oV>Fu4?&!_^@>A^@6-2U|
vG`!>EwR{hp;aQ};kGhpchwsB`Ml+-!V}5!hULp7^Fc5oTk^$Qn2RIRp#0<>UGy8LeDgSYC}pk}{E$g-8nCs6Z$ve-w5+QEFq4IO58UC>$(7aGhLdf_9;(1?
tXs4EewrG^WY)i1(_y7FfYMsp?we;%Fb$9#fPP858$>~i60BE-`?IM6ng9jWv~?!h0le0EnY>hB$F{^PaBYSe{jHF_x=A$CvjA9_AZyd$|FB~^04PkiG55=Q
&uWy7e{j?f>Ge_s=XP|UfP7zN@N*QuWnKnuD&`#)_)Ab=E?SC-^UdViU=*2m1IAA18<9ywbe{(eaPFghe)+eTNtSrp-h^-f#BTNdG((hgUqMi4TJOmhjc4!@
5sO--eac1~);!T&0_AC}qJXoSnY24o`YAC+I{-fP!Gdko7wT(|uEWd93YO55xh_PA+^4mPI<l)G3n&UZdM%=FX$3m!CsR1Y9Of)*Jh@&-_-x5q_<|Bekt4>m
hymzn!|e~qSyk2UV1loXiN^HHeEj6ttC{Od(0AT_V>9sAn<O8N@x7wW8N@iX+~VxWbDfjEqd}2v4TqO>Jv;<9g}y!UzW5#y-JzwysCH4!2Kjbhxu%)Z4jYQf
jxX9rP%xa9F$)m*A)JaZcY(<6y}y~eqR_Vj+hBu7#q)^C(%aTJ;}Qzgwdx&4YGgd^vU=Gzf~BdixFOrSt6GSXXX02VIepG7;>6}-j%yXgSe5PCAw(M7DFhX8
Q=$k+P14A$Q+?3KhK%OTdQ)Hp?F+k}V25rkWM0%ECPT3A<_2R|CH&gmu1zhTU?3BZe%&F63?LH?afADLWCN?qSU#d0KlRDUme4BZ#6-le(??Jki?WOJlux|(
RevrF=(;OuxsH_hL!;XAFyc8l7-YfB<(~GJ4(-)|_16p~w{dLHk=>f*b4)$mQ+9`Jg#^@SW2JKMRt+lehTyT^w}t&kTLf;%uEjTAryxZ_72fs{!OAS;OEkbC
qqRWu3~V%rr_K9T!lB@;B61C#QwCEajbDwVzHy8rb*s&kv6?9tC;&tG)vEiXE19*`1hrj^*KdK@j>diik~f+}(DcrJ>eUwGp|8uMq*mM`s!H+&;cRd%C_~#z
!thYacKxn(bkpK4i!9cVV>dYYeBV3xc^rr1)o7OQ<h!9{>vkcD>s}&V0v8+6Jap(-muZjVA7NCCJOg0xH+wsk<S76Y#$9b%>QCi);@81Z9KgvUT!WHVozuyl
SF|}7Q#qfq7z|=RD<w(L-7COUcQG$@WFj4g4JQaj2^_0!`WflP*WxGzdi8^5a=V)8867sB=@S{Nti&jn@sl2+?vajWb&GZTK9A6fHgZ~`9dbzMD5O1OIriw6
Ey0w8v!~z=mEQCkVu!IZMMHQ=OJ?J@>qovt_uO{Y=&V0=SrpbpK2%h*4w1GHYp92e#XyTCxdjI6PVnCX?*fhMay{*DM%h)ga4DM=G+}_XN?QksK*HA6-7jJs
bXh;(A@5uuB&V+gNTYtO@aV9T9M0rLQ$as;b!F{HpHcj#<4NMRS28V^B4CKu0KD3jQ<YM`c}|~_pJxDsZRS1VaOrjpT%GMkg>MR*;hDA53S+<dC-v3XMY~N$
XGqCzYsZx|sVB7|B~Hc4jpd~X@^1e|rv}-w(DjQ2qqlbVT4c250=o)HzSG=($^ZD|j-`e1U#N~bvub|`tdHIq-n)gvF)AwiTHE}&#>|GzPe5p1)GlBBHvxg3
U^Oe4sM1uuqC5bHaM5KHV^y?u_oB1c^=f<JO776}s%N<6#stQMcN$Iyw2C$^7P_TFZ4bRt@hLb2weKhq7KbK(1`ohQ0VuCvpWwq)&x@>yo}7B}^rueWl^2Jb
Cvw+_)lK)t+}|K#$k~_ab4XFR88U4B2N@Ro-a!Z>6#qo!R><H^fm0?7L;Q;{bWH7-mx9~CDm|1l1F}t}mcmWUY*P;6pEQb%j&<UU8rCAgEc_VeDOt7xCmU!3
j{VXN)<&5DLNiC_d=q}O%nRneE6Se1LFhpqm*39_{logt9F%c8!9N)}o_@2-bH;O2b&QGPJ$t|ZI0SYpQ<5(%NyY>}u>GnEcE7MwZSRyf^{=+FH^>#G{95Y?
x7Be*Ge%F4;%Sl_*E?*nBWu<@ZatSRYoCay)PMkZcIoS2Uez17<*LTSd$ifESkYRFO9K1Q?$nQ^NKB!s$(FEwQ2Mp{rQ*ptG}BRu4P?kf+{Xp$+-*@_&$$uN
XuN_EJ@AsA&!v9(C&a$rR5zVY8}`f^KyFL;je;cQBLBMIV_1od1pNUwSgP9wQ%5QXXAPstGpxVYf@#KD;g?%oAtfesQNVV5<E<(?s7Q*|CUA=f7Yn={Z+tr*
H^uBIvU%E8wKOx1Qx)4Bv2~vS5Wn#ka7~j9VFN&PYQ#T1P#=G{U8FZXiq3l6x4<>o^vCO`5W6W@R_Ar;bESlh7cl@>s2n~D?LAtL>26<HRHR44`#x=i!n1~c
zvYSi9aOS{HuVAb8#;^V_ZKY`*PLj67(noJK9Nux6#AR~RT5qOx{06<?nH*+J7-W1<nIM*FOdt2`@zwgl8w)%2B1BV-~DB*f8i*Bqw1XIa>1O?@M&LM3ZA?i
<8L{5Zr)~gR)mKzhDW#f74|V3_9StOIod1-h(mV_!sgY0ZCPd!#kPmI6G4-O%p@jJ;0$LDoeQHTkvdoCI}EXr{3Mj!z~mEcFsxz&f%mkhQ<ikk6{b{zCpSnU
KRv<*GHf~jIFPabOkjcBgI5^0!IL55U3jk7ktu7oT@X3>;<Y1T6F}18pSbK+G>-81B&}}c>r`GSgVfX~T2LL%(3Mv`o@l8(%b(JV8ZTP`SxM9$(*t7?oX9$n
i7B1S0!TPzaR4H7()cthvLUqYzTGOW)%V;A+a&fXUCVOnF@y8o4N0*XqqW%WLf)n%UsuLW*z{|y?5RA)n@Ix_4R}On<7^w6#@D3($WD;(ycMR^2Xd2w{F)O3
!Rq%Ii?GQJ6h&IyZzh;NW2mPSAfd_;Gy1co{qK#Yq)oV%bsdxzZu&frfaPu_#JtiGme$pmZbPe9jl7->{Ciz;r;@&MroUnR92^P51=5W_^gkGdsd2)!oGiS?
WBf7vR~L*a_ucDv_|syA84rS{FLBJQi@@aU-jE>Z>#}b8;1xa|Iwl24EH(O+kHHbQfO~Yy1~wKjt6i#ck+xddt{)|xPu3y8zjh;jXHi*f>%jp*@D6g!^`?SN
AoPEqiFcbb+#v4V*U||H`b36$?~q2OGlXA___QF=RM?V<+LAV)f#E+dvgxtRlWzPNpF+Kvunjyv?K}<Ji8(5n)ZH1d>1SGaUO17DLqr$o0CO0m9Hf3yTBe&+
PQIfo^$hSHc_$|<zNgLH0-XbgO!y?uR}Dfo`{dJjZS%S9B)WH(0%Xmq`7y6Dc8v<ZZl*tKiaTbR;x@RnBET!gPVYq1ZL{Qf`o0Y2A#Etf0J^6|t4-*?J3>S)
8sF_JlX-2VOuRy|Rj8y@N~6Qbvexo1xH~v4d-0gKY5Nim@RQKrCRAUYu$!7x*ulS97WuliZ)aGa{@wwM7*G3Pp2%K=$Ugzp+OWvkp^@RM!iMcdmV49pFBGqV
&B#8Wwr}>+R@ywpjiynF>_UL*grcp`F6C3kI(hEX`Ox9F$9d+h@7~A2^x5w?dHT>(W9~IuJFF~K(aGFQ%(KS%r2UOjE){C)MrEi8lZNECfr)ktYvu``_wK%k
zw%x01t56{eEf|jj^p3~OAdj4%0d8l-|w(rNMq5cyQ6AM9zTV-zn5-W^9viTx#W=ved7;DrLtvjE(tgNIvPYsq!UO!mDHvU-&xG_&(i({%?bac2tKd7DKoGG
THLr9f<gP+{NS@ReEJ5Nv(5AglG>*$TD{s8%;*OYObSBFA=+$jNrIudarwySjg@9DK-}gS+EoH}ice;)dwV-LMc4Ehovfvh;ufRh%mM>iGM*|q;0`fL%~r!?
L|pT_?S{N}%Pa3}t)R5So=R8>#>la{6D**!F}r?|zFx9s1Kz7VioS3h|A#_^8#}gfbLurp!kEdl68SB(3pqFfHXPTN18l+DG!?7j8=6`G7(!peJ>=8Ca7M(T
0G^=b`4CJ-3?C&bEv~RFvG*YL$69nyNQi;SZ;2UXRdxVKf|FAWpcP)3@|xH*GV-5Ci1Z_ReWU{yST{;O_I!U;&7M~~uJ2bS;0(EqHy;)5l|iQi)-W_h`d&yy
T5IAR=;)}_>0Jyu3hGQZCYpNKQL*-;mu=J=Fx1ppKFynLXhrA+UX_ME?>sLfUcWs5St0L1X?;<801}Y%q9ij1f|e|dSN|hUuo9glHPF=Hfcj+<;1ea_Zu31K
s4(yKM59~BmLp3Oz^KVjcVO^0mwIy7n5zV;S=l@31I6P?isirNj-ezgz^0J7Wb^1ZlZiPENIl)U(MA7ZKmGw1Cc$|3u%=irxz@UE3gXgEdZuTXutTNMF$^3t
=pg{)SmTUtpNQ|_?0+1${4y&QTm4+T7;8ZNXIDw#?OHim0|QuL1n$g2f^;$vG3P$w0)^m-C+&KMGI`G|09$vSzKKb*4vCs;*}M_^`QX}}1z2V#&d`4J$zM|^
*_DZW1cF57RH|2MFfnaa&|-^K*dM}h=dqP!F*`h*tj!{~qRu0Acfe$AC9ZWS?{4Z5w$@NrKFC==syv=LpKS7rd1?xjr7~S8X<91Ynxvcp=1mIRM#v#%mA`kZ
g_0!0SaYF@4AcAK4h4canGI%Lqo$c^?jD%kC~t0*$&GBI5p_`)rM1;GciZk=%d#OLA1piG4mAQaD*UsnH@0sQs43d}MxmAAHtI{?!kG$~o9?BGt%{ee0R4f<
31Qe361nhu>yJnGQ9YjfOj^=!9;2Dyx22d*TCFLYiFJYbwkqw%Vrpyu#1(l8hkKgqE_B4$-j@NVDofoO=FYn@_hgN5JZw}7(cfJD!GfscHY4e~p?Os#c2DvO
^%TOc8E%cAs<a7uka{NO;otDwNhCv#TM;;O-%Y@x9^F6t#ofK#Hq&xrphvH%q5-YAA<%g&@4E$PN2_!f>XAzTswajV(eLG}_(m-|9OOvjH7SDy;0dHTD=3Gc
m?zTS&MB|I#qC46PXEVsjEM!APH6|xWy;V|j?>3#QqqjyguCS9uHjCDbe{VM)UOJLz0H-hZJsA@v<@O`)EiN<`*BJN&59~K0P+$?gsgG3@Q?wgnGY@s#w&pA
5hDIJgc(h|nY$1K77ap;!wBqCvnov91c`+9^0C~@dVgs#!1h)kb35B%$UPIe(3YsWJbx`XdFBJ1ol2nr4KpNRY^o~ZRDNTUSLxnU87zPkSy@CuC@<b8Ut<)n
X6!Px!jT@V@2J*+^O5UtO!C>Y$Ls7k1Ad>0km7Yb=`CPZ*m7aNM5}^1d!;*5*vGH&q|@J5!xo0`IG81{9TZyV(wb)~b@WY1t`RlDX$^2gU1eHS?iE>tYG(85
RriC@5l&Ie4^@V}<{J|<y-;sLDkEA|s5|?%bAlF+T4$obU?g{;=j+qiS=GXL&yq{ex+=1d`kmtGqHka|s>W=;HyGf?s-a$A3j@l58jY3gFmK&qLNX)NvdoyA
#Oz<j)mVC+02J7HdfWj^m>aIICqSHhL~sHxrY@n%%A3}ra79dQqHw{TEUCOBFDr#ga>JbuWiB|H|HfQNLMj^SdNX0J%8eY6?gEZEwbfX^Dm4Ag1CR`hGwa?B
Z#35U(55U>+?@ud3-j<j7<ziyB7H#23*4ym6D-5`Dyw_#{0eUGGa7TEpdLr8!_BWHy`k90x%Vd$vfh4JwITh8aopvR^<k&-`7*OvSSDzubZmpisSFk03(~`9
R8UgvgtL~?netCh-`iob{OFlrjlyFVjF0+c2P~j9VK1p7=xcK!$5c=EUWs+iQZStO@IRv%$YS3$qPWtbG#_3kF;cP+Ic2%<VNMG$QeSzHR%oOWJ8Ie2qO+!#
bElua9Vv)aiV;~0A<riV0bS({R7d?dn=?0_lm4oF;WdnUHj7HYn~yvRz!D%Gf7ef0xLI@bRa^S#Nzx;a@2v&FFsr*oHa{}tytMqwRJZ&d=ilC?SeaXD2OEQ#
t5&ape-F(72HSI^a~rur)kVyM+AgD-I**^J$#VyN&))CP`VN0QZ4hYxfDG4y`0^XPwJ^Cqes}5rzS)rqI3T%eNQ=?|99YdjxE3KgZ+9=T0V1T#G#X)Ojc6IB
@G_+c>WH`bV%pIuPuLv*ygl-ux_VsEbN|_XTMn+~s)=Ths0JKhM8<L0WpAc>u-}#p0tA><NY5R!6_UB=uiN73F-TxjQ1+Zek`Z+@+#$3)*nx9tKxrhQlAaIa
L4kYVobF6&HWck~-*ZaQ{*M2m#RQRGDaiL`2)6$%hSsu4GOio-u6-STfz!76<f5Q;YpLsOCH_Vggx@c;<=*Ldp0!?K4T@iStOXc_GaznGrA}>W0wa8Nr@x5m
WzZ&whKW?xa-`tBd-5X2f21V2+F<%8`t$7b9Nt{<pR1w}5!?pfng6|Hj}sm)neCSW4s1dHZN28-06;qQ=vloH7ckGYnq7m$1ORtVeO>fZLWnSg-+P#nxU>5H
V9OoSnHOn9cL-Zb+o&_PP`dW%lEy{E#W;x>p9+93r(=!5slqg0B1fzv8&kR!_1gk(m^Zb8zLR#I6@uuq+3H@5@HslS*S$Y2CK|asp%55!jmw4vu<EBKly;Q?
pukuK=l}7F1LRX?(e_)ku(!V(x}QnQ@S&&q<eX`!d?FWOLhU_6A)%aIf{)O_wes{FJ+70#Qw5X^O)eCauN=qE=U&?EE7J&r;!)KBI-;b(hhM3OuK^Zbo|LVu
hn}T!PVf@4HC!FqkG<Z*5j?FWi<?J~<KAcO-O4WR-@N;qzMYX_0$=R9Sq4!v!;u&X?UoJ7cpMqB5U5>$6z%)&N45MUMOm!FFOKAgyVg4MJKfKHjWX=VGRJrW
gkzXXglcC>>Y1}mF#Tde_H(z0wDMn1007jX49J-@bnQ_52%;iJ`K0o&o$>K!N@A9jyKDBPsm>#d6cGTVdyP?4=_%lmHdEAm1VjF|e?8<w;#ixZ<<eeo!ZLBL
>v=gQP-~lc82|*hei3YLK3-}}LFu`EUj>bcQ<nFUsn)m#yvll{83@~AR-==8>x1`W^c;;<iiPxr!(1_jc*n&hXFalq_t^QV*v^I=0IdjnZv-<(ouWXo@N{;<
l7Cxw{W+Bi`yt7*2?F_nbfZoG-!FDkoNLs$FCaenQH%}+x`{vI%dJ`gWAJZ6{7bn`tKN#P3L|-Bc(;y)WR)1YAcgxTkeBFsTtz(LQ`_4%ItH%)>12j!L6;5k
rO1^=GMpR4IodRgbr=>pH~zlcgaQ(cSfi#jf9%ZT_|MgTW@Z)rt->Gw&cgO~7BgfhZp$$&81t?FISiGd!&&$%+;C&?LjhbGYzU~@$^OiJ7{lo-9;(jf=x*x3
z7-VPd#~FsQj%<>`BxJG=4v+ulQ-b5LQ|F+TZ>~zC_?3G=~A0rF;Ek)x4La{d>l7AzLPAkGC#W3@P%^-avpT0P}5{U2j&wid^vF{noOG;j(usa3^gDT9asSq
P{J<s-^~^*ZAYWNmN9Vb(q0`?;aan6N_o{jHcP=vF-Zd#aNZMl?~*v54mmBz{Vi8Ce3f?Es=49U(|@a3p3GCs)NLC`MXvV*FzSb(`M(>hJP6;<%R$p7pGmUX
eQCei8tEy1fQD<AO?8Nu!6`bpx!i`UGr^w0(CB`wK%6#+-R6F#eHmm^FJJE2DQ6Ifb}7$`m6lMcVwCz%7KXITUF(qQCo5N0Dtkgwk^tbnvwYn6*YMw}2F_(B
xX0>Zc~xn{kFE7sk%DZOUr85Tubdv2oOHe?x0GU&;Ayx&Mc2CVx*BZT!)ujLGYc@|d3JPjyjKbA!kb^6_sxR!S|A}sB30Ha%mu^tvPALC&;qqQI|Yv}u=d33
gT{O0QUbY>9E;Q;IShE&nkqq61ex`N0eS2n_dXWWHx&Ua1jUQUEg~eUfhPA5GD5jmT>XB8Q5@Zg8^d9?Sh$q$Km!Lz$LsJhV}EMMW3YnMTUcRr#q3&VprN|n
BBzB0-U96=<%h79u&#1hMm(=^)i*s2d9`4cCRCe?W7}(cRHKv+>m?1kfGp~ZFh&z0qJLyb#Pc$H2caE4enK)pzbk1Gjuvz))aZg)F#s!6I~-feNFgxxJBUX4
UUExX%no@Yx4{V)Dx^6eC>ktwIpfV&M9zm>gb&)k%@`4IIwsu}2OTK6)j3W{P$!#=wV710hEQ{O#9j5AAepqpN?dHs5nu+;CIECJDT2yip|sfLB<ct1{2J`o
S5<vg_kI^!CXoz4Z-y6K${R3!khFEi=dhe!toxl6m7XaYolU`q*QfeD{NZ+rO=67Mc$@N>XOIz{ei73{Mz?K$^CpXIo%c-sipbyuNo=d6mkKQ;>1VMTh#bg1
%599xB<|;DpP2&a5c0ATJzyz!F3t{B;J}0Aj!saxo-pgvTkF=^bXK+l9kMkJ6(Is!Gg$VRZl{4kOnWSBdV#QZ{(=TNTh8OzII~LwHhfZvCu=sVagS%D&BpS$
q4&xfC2PVrCG0gMqT3b;fKs8)D%SlHl?-BUO<?c>i^JyzfG66)l|3qo+LhLn#F!-RU8(P|knr5D*{|y)N<B6m#;6=NluoiTa(7tF3ENyAY@2&8#=?6{LUUTv
weQV!LcFvw{0xXBOJzUM#<J3uBOSCq=-Y>J?l7Q7h$|jV4@lsmH(+G%9-V?ZZ3Wq#IRJyK0hBj18z9b?PnIOUYE%7_VUn<ED0bnh_VCfQtrL~tIh|Yj+@BKu
XMA#MSr9K!4}4l>lYv8C;U(K*s}}hkH$KV`m)j<fnlu(;I<}#p_L9_SHX%xm3pid0h#D?0$_N9KZbk~$#R+RVY9(c7UqU7ZFVoO!gn_z@0J%)2d8AgMALh%j
9Atf%V-KSXUYo}e^<t{JYrbu}&?XpDiCH#<#|2P>N9qcS67ckzb6ye)9?}qDLvtIJw&7>KwuW!VqDJVLA%vtP0)SAA!6p9lTtx)w-S})4WxElkw*&_WiZ}U(
{?a_4Vmt|=ReMqdmN^3Zct9`@vu3t#m~%7XC7K;^o&g7CQ0zxEju5X9ZSKJMjlu5He(r&PYQ{vQ*fAizK4z82bL`P?kyl~TthM)H9N^Z&cr-<|HJ%&^-GZFr
%HG{pu@__(Q-8-&k5>@oEb5gix%ui!tey7Y@8((M^`XdEDW47QLA5e#)gZ~Fe_~)8{4!c-MhsQ)w2rJ}H`gY!;*>WqO7}`fJ+cD8bPo*B$8ZpqtKggEl*GtC
Bbe%N3VjWVK!B7rsy4i3>Ve&vsRFN#m8jaY3T(71$%BaYxew&>jMvdR{?ykpedpvCezm53IK`5Sj0mA#C;16`Dx5;Kn1C2H^`v|+q4%W!*SB&_NY5;CeU%mB
UH9B&?r#-Gs(N3nH9XI(3A<ou8^(d&%VNC#LUGsv(8g)+UMt&s;%aSqYA*wCHXQZuxV}qbZ*ew)|3q5r;ik8vQ9?9SooX)V(p)CHeYR>-q<EiY3BKLa7)Vh<
Z`k_k^&g!ci~d!=ymIefBcGL)PeTKygV{mA`Vzc9XyZ3s?bl&{>qfHmtN2Ivtgj`M-SAt=?n*m5aFm_4BBt}y!{tEViCR4jXk9K>(ow|Q4h7{xZ%B|$W$CTq
=>cF}(`gqAB927Ai;Ta0G}5P+0jX};_bRr5O$~0=LSFcbDVTy6YOX11SJ_IPcu}ouxb-I9?8@yk&Od_>){30Q>$_MN$CeC|O*0G<A#3bi-xz$N&e|EnEMMJ<
hn2ckaqdwBnDq2lE*NTFdDF&408ZYp8;CoEn2p8T$JV2M{+hbSQ;yEe?%0Z2WNY(&O-gNf2!}NF(>Bf=cQ#dSW_fA2gmNjtq)(PFMrcaNUcpv$&)+7!%AV6k
+c9amd<6J)cT(mV76ngk;)F%zbpXR{szVSLzgIfXSSkDorv_VN{GPVS$61)p5bm9r=)=?Sq_x}cap9lEEV8-zHsVQyLNy54P4FOG#^#q`Mljgx=BWY;8rV#9
Gb!lXqWWwtItlhXVL=N>)=OA-H4JsXFmPV8EsRV`*3p#&!LTYe70V*q0*B{(P4@0#t|K9$TSX^0^*+2%C&MYfPw3?N%`XD$^h}<Q76pvXsj20Wn*_v1;}7W!
TSeM@$LWxExs<8&rA1N}m_9!QKk_edr{G+zkN*%B1XdT@#~gD2ypb)|kQ{lYYiWBM>7<MnHM>-wL;bPKF;b1<9yXrXg`!Zgws~cOD+Z5?jjWq(ATl44QsY$B
9nQlE-#i0OevJATgZX?04d;lAlKaqh2oN5%^}Jui?lWjP@mhb3<7VfHtcX-*5_>@b%L5A>YNNS3*3`iO=<<k`xgO_R>rQ)qgqYH)E4TV^*p5z%pH{QeKJdA_
yY;Rm>2D6^f(-wSDi9pE!v-zj__2XQUeKQgkCBO&3O2lm1VHn4e@s1|BD6~52(-~Xg^6Yy;UGNDEiur~Ye(!JRtRLMcr!wM4e<QwT-$|{xt03vq*1b6{XCxD
@{1QqiBP$8)7N}=R5=zAy|U%Y>-@Ac`A7?q3heM(WcT4fUhRY9!kA{3XR4W6j5!#4+-|A9qR`FmcQrZ1extZ1(M(bB)X54e-fLKr0@Q~-i%pRS`82~X8Z@I-
GA}gLexcq+6QoeX1;A0zK{l7mTSQ4_q$YIRun|Wa6%>bYQ2Um5x8g%Z9C-7N%Z><?6VAjo6^8dLGceOHbo0y`)Hl;&R;4<zuXdW44-h31A|?51V3d}LY4i1x
KOGOKXM;R8D&dL08%7`Z*y29CIXRSX^wkfdM|>4orvz48ic=@6@Pwh$JoW;8{*;#-HVK9fI$Arz<A+U-8ZCjw*5lRnv=}!iEIDCpUv65x&1J_w7JXjjQUy_I
UHu$L-@i<p_HgLfEV*Y!{hyC?OqIVve&cjaoDTsi2CD*#nWoE~KoB1((5EFO_t(5UpuGa#0S`3oyVs=aIy9bE)&2>9;eL*rBiu+afG>^q7V%JFE2bEni-*bh
02c(xkX+dWv1i)Tgfr$0kmHuA74j+>36PZU#|Gm73T-&R$1;E1a8J!Nvh6l`)u6FNJ%Rwwi0vPAw!`74Rd<o-V!(RZZRGGx|9&z5w>SUeyRG^&cUX%&H4pQ`
YM!0)F(DzD{(CNhjwRI4wn-&{(F0B{fSI{{1;mO6C~*2wG7LgnnnDWx!&O0EeNLGrQc#Z66VZl}&EOzyOX?B$^vPJe`jE}pylI6u@(p32l?m|+CO?k(Xw}l9
l?*^X!tbAMsTO2dK`N!q`%90Hf1zBz3Op^dt102L>O={?*d$$O3<tPv<{x0=@K%mkoXc%ZG_lMvK8*vI>~kUXb)dt2p<`>5PTVuPjiW_~Hp$N<z|JPQEV4s(
1yStC*jq?z_$U-CD^^3?inkms8b{lWIHV^h&*<T?WzY?#^@y;GR>D1KnOme0-{7@ZE++BWGlb#ZTFF<nIXd&##NL)N5)wIodB_M!KF|j*607Hfs}iFl01dv+
h44z7VzBv!Hg@<T7`r>GF(b_ntKwjus_{)oDz@z+*R-U#m*!=Gxz*ZAHceEr30m-Us_$z&z>T+1&b(&1cAJk@_T+#YKAktU==dYS99+?>ji6GTX;afc?u^4P
7HCC#rWb<4_3OV^ly#B*|J`2BJayNqe~OmQ?Wk1jYj6=gHs<(m)N{hx9Os!KvWXc{QlM=x*{Xdie_ZYm%nr<l0F<DC4EH+LAXns`$F`eKWw56ZuV-aAN+10g
C16(llXJvM<&>kWL85?n>G%lTmca64^n6B=8;|o!DP3h6+Xt(1Yh&=VOhB5C_N7Vq^{WGDK%fEH-9XQ*YN2^_`9yZIg<?6?EG#uL)cMne!#I@E_0A_%H%4FP
&@hpR{+-Gw-ZN#qe(^ta##6m-#>FE7%(sj?z9vP>tC8U{rMlJc57Gj!%uDK<=SnZx>Sk^Oa5wO(>FxG+L5uMANG%HFelCC&R>>bTv2JB?_{mJrM6rHpy`i0>
hcfSSPwQ7}?2G0lBYycV?vMWfQ{GHg6S4E8$Ot-2m|U~76i&MfhYf6B8(@l4f7?Z-;Sxm90Tb9VwAdgh!e99VHRysmET&th9>sqEOQfEtu2E9g+aS{a8)fAB
S3k(b^U<pGzF8qadC`C16z{6}NIEpcE%Upvlm0Dddg|DrL;nAEH`;;BD*P#P08;c=&Dd$$E{cQ`5sY$C1K(<R<laT-`qcU6>=j8G+s>+UJQUxQ%^B#0$TCd5
40$36XfIHCQwd36b}Bb)MzrQv13BMaj|8Qdka;v($!QNA08m!_896ok-tC=<Xx=-@MMfe$7dvt{it{awQIP6S;YjDiDFlXAc7tOopoq@=Lm+vml~)mB#?Eh_
BxHI;IEQCL0I^GlZGC6qHXeS`OMBj}pbz?K`gXWr>lKC5DJ<r=+v&F_e8M_jpWu=w;H$)f*;15><}6hn_!c8726`~Fqsz7Ju*KU+>a3A-_%A*VGPlP(FBtlx
u#rJ*uLP*=Jy<jdefR%+K(L;XQ)Y6{_v|wVCi@*e4WL_-j#xrUc2uzmAtegA=rg!ccALYuU9jmEJ214Sp!eb>8*3)t64QwJe9D`%1d)2R+zUf-WOalxu@WiM
=WJ$nH>RXlwpzrY^!i^V-3a{uwxDWHjJo4+D4n=_y7SQw38dbleh<Q2D0A&1>jl9F_zq3laDaTK^JDf-r>&i)tFWK0)9k0gkLfD)cOZzqvwoOLl@Zs@kDrVc
Ke3>TirVc-=haB8VSHP!OiaC6TJm<dcCa^VC?@Bp5BU*X^U6r4hCTpCmFYAtBwC@dn$_g<B;YbmWyv0BD0NP>e#d~2R8RoM`dA|Jm3_pfV3_*Fx#hP33lG>y
Jxo#mrsOS22`n^=w9wG1&{Oi@%`N{G0J}vNx_2#J1WL1<!AtLRU#9}7qf=4+WvnNG#N=s5#`q+l<^nR-&Z?_1kFB9vmy}Ps*J#zXCs3yCgz>RxvMM?k`rOp6
Sfk_!L3=0aoo=k;VN7)QEkItUkv~vm_Q&vIk*{iMNIX~}bo|hM?WYsV=_N0>CH44Py#_kraOL^5&z-e|VL(9?PD`?kIpi~WIKi~sGmUd<cw?5gNQ+34DFe#p
2#L#$n*J`BvH=W45bJawF^x1zEQb@n*zuc$yOEW9XGmt~cV4G%{-G;c3rQZZFn+`4n#kUs50H2Z`m<ZMOpYhXfh<@RhWkZ`|B9W-vmD5irz>`UOl?XeAT($-
z5a*c-ZX1<WoSc9ZRNcOq@ih<1LrPtsLgX;azKYozeFa)Jpr|@HK}yc=D?dXD>2;Ia`xhMG$NZ#u7b#68SgE0h>f{{pEZu5We{2Ca>M#~^2sYl%Sc6#B=xF6
o0`WVq6$xI#iHiJz*09p(gLp%_T%aBsnb#(kek=Ko;haHp1(N$zfycRS~<Z1rbq=|yl+$Fp+FKwPqU_H75Szr!2ppJ2~jN{ZyRRzOt-~eH6dNLz|<;_d7=7Q
__}1mX3EtR0CTA6<Be!C=KwV?CeC!$wU7=6F4WDH5j>+J1gKQ4Q>l{q1HxEc*K~PL*yYAccVxk|q{S(fD;rIW5xtu|6mzhga#BNTQ0W`>U+x^6E0)rT2<L#E
YZF{a?=wR;=~1Qu5DuJMG}mdHA4pAF)p-sP>i<n+S#LH;sp76=_r1dYXqp{RQePC^TkaG1h;TIeeNID<Ao%&&gh8D}lN^`MZn(u<jJ<-7r_t<sWAe&^qxIFA
@sT-a_nYK+tC)J0bWkh7-ZjrDK$}7$8lAfdh9X977?dlgykP8ulCGb3$4QR&T@b2n>jB0r!$tv2zmtTM)ZAq8{a9}d0v(TN9$*7P0|U-T%`Y%5Qk$9i;fcPm
8KeTTwbjOcgMLU>FLd{Bw}KAfthFr>o}FKqI3#8~8e!|woBO9GsNGw*;XW6qipSxYI&I_P9$9s_23ioiw2?oq=agoLQ*`@V6)T=sNXa<CF!*RRyF&2VJ8R!Y
!kGvNK7sPM<IcH7L`LL(cu6YB8^^D%=(kFv+VedW9_5ROI17KshwSGoJ#}0A_&JK^@qR8^R{~!`_~TKC8yofE5)9qwN_p_x#IpYioSnRxOV98|XBd!3q+HCN
Gk2qjkq(9>ixqS97Cge`5TOUdLSo$OCa&X9h*3A%vNG<o$oD9{Lp9shCpH2UfcDBV<(%3l-=88Uv_7+IB!S!RTC*781!e+@4J#;uNE)mv1PbNfm-olOK6(h&
+NpnES7*M~Q+P*O2+2|mje<`_S^%5xl^|||>!X>ZGaHyd`8@uxSio3ek9tW<Y}?ln=LGCfY>7(A)#Mg^0FtvIei>VCyrDO23~%zQK}~z$5Hq4%PnG1#Qr?(h
2zKB7CG=40P+LnGsV!dyLT1zFxZl=WZ*<;!PcfE28{H@bmRlrDDej#to#c(|>Ls*jZ!y`!9g=$z5AtQEPBl72oBCa~&&sNby{G|W*X`!0+$K<AP}M@zIa;wp
I9sMkWO7m3rXb!W=6rbx4t<2BJX+V)gQd(vjUoy#pEQt8dNnM-cZRm)l6goK915ox86?R6i#`y*(@$y4UJ|s8;xn)X*Oof31SQff<Q$bd?84Nytx1dU`OLZ<
F96s@z=!XO7h(cm!1c6m6v+qZt{+fd`&g_H)(h$H)W6iOe>7194x7Z5C@=i?dZSoG6JPxiW!X0(2oAS<D@p-S;&}LCd*jICMh$K$=f|Rwwv2cblhqbiH~1f!
f^adBK0Bs^HYb6Od7n$)OI`LKSDF$Lz!0W`%is~^fXP%HA50GhOvSVlcXx*Qv<t?nr!D|`L88+s*pV(P&bzhbts|wu{d)y4s?Fv8OBJWRT~K<(74y{`Wk;X2
FzIj(985^)$~2z+9G;>QA?NNPKwO>FJZ4?YK&Wb{=H&USr@tSVE50DHN4u;j;m$%CJ7*{LFf!k#JwI|ok0shb1AvF_#F!23SbXn#mA&lF9l_ZN=f>!y+RRsL
_<336zkyJ%Zf2PSE<|pIr9_5BV92$40KDwC6uo9grkzPHQH6OaF3!(I^Y{Vnhk$o+=U4<gbMm}BhfO+ytln&Kh10=k8%DU0{c#=@<!Q(L;kb9dITBd@8!w^f
twV4e3m-9XwgNB>rAd(_Zax+#3zoxTd3kc-Y|em@Gt0lJ2P>^T-1J@t-;Ps%8W|FADK1TQyOeSY{Mw=%)=`FoX&mf9Xn`<B$!YffqDjRUqZ@Y%7eP!;-nOdW
AG3n3H;9R10<G(V7X@W-3-bY9E%VDQDAlaUPhC2AtdeG`?ZgMAtGg5%eoBoJ-^bGH+ha<%Nd1N7rO6NmY{^n<!zDJ~0B_1?^l-=;tn&gZsGo1V;vB#{D+^uL
3&Hm1mQBjZn?RMJ(%9sZE7UZrpt?G66H!w@X=LG9FYmxilCbINv8pJ}hW^q_V$b=r3Gp{?&6w@)sDDS7A$Xq2Z69HVUt~pjB)JhJvj?9#FboVKRBT?1qO6en
6hyPTQ*K<DkN?)Q&e}|c-z%9q4=W2k^J&bD)($A16^JDqU3v5aG>%a6Rn%#80`uw@Ztx@-aL72g#$i;5{hWV}w~DWP(GW6^i$aIF@Kd{Dx35-D4-DT^z}Sfc
z3kVQ(W(jNx`D)P2{?@Y7)yfaT!8f3g_4Ep?Fccr7YvFHLtxSb(PT(i(e5u@@<~U|j!?O=x!pC<Pl`slj|a4aE!y3_PO!VavLC)p<4VUgrOQc-*r0i-p`L4r
>w6d64kcpWK*5VOQh^9aV;JpgiN_davHZm6lWg7$qc^FgyKxB~iNi-<amUsDJgE<Qd_4lVA^TA4x_u*#VYA_~ik5k}7WWIF+xaX@b1q`dJu7^V;yemCH$R$#
x;9N~-{ssWeT;&?g64PM3#h#a)Acale>Yw{Jhi#8(>R7O*>3-ekdBXQ^(95hIJ(?H`Qg|K8cO{)roUzF`EZ3jXn46`eH}AX|3gn9R4e$Z4!&ZB3QkD2kEQ%Q
&oWtDU~a=vNN;q&%|7gUV~-8SMFdWFvEs>@gK7Xm-dBdYfZtf<a`tz)ITW_*1M4P&LOTYM#BWKId%YmA<Fn`+bMcxwwO6LDjPHYTrLE0ob!nlgzQs>}O<2YT
U=1F4dv4oux3nwwkrQtJ>*S6k-E&u`<8M#bIH7l)ymFj%B-&%eSrrPfbG5$9+K#Vq?3Fih;wXbYY#MW`3@6{s@s+6X>~BKi<@AmJERI$8%`rUzIs7>7Y^z*e
fdOs~kzMD^*BM>DQr3n{RKI0O+1ANL$rL!a3rIPAOkg@K{U11*xk%Ohi0sD;p$D7cF~=PT+=7YhOVapl3pE{I#1Se9c6&3xMdh?Mzr7(uZJGr(^7kHmg|Im7
Ww&2(T=`(>`jHzmbeX#G2v#^CBOku1AcUuRDur<;zDeUjUA_2CjUVr*QlCbXD05Jt5{G_7aFr2wexNGpChdeU<oBrH!D7C0h3EpGU$QAAi5rv9_kxThaV3F|
XWoQZhF2GqH2U@;g)W*FyLB-Yma+2jf`}_}XISv`&kyn`bL9um^;FQrT0fT6Bbq3dcn67~C-w3NRc5vgkCL`Nzfx~fk3E+aB<Zy34I1MfIwh*3f`ZD=od|Zg
vcVE`Pt8_lzuGPr>6Q6Nuj6h5<Si@xwFSkc$q3|*DF&PNYrhJrJsGOnmWqvXTymTY^@GqV>-In@ZgWE$%604N@+4hW&95E0Td5aoKdaC=!$I#wEW$#2NOQ=d
JR3Z4Ev0-Dks61nq^r;2OaNf-Qi*ES)IJ_TBy(8-s5NyJ+l0hB_EuX(32*ZqJHgau9TLTJhTy1TV-5%IgAro}0-)Z5(}Dq1>YZT=5EMH%m|$wnH9_D0AIoq0
+hrEvQz6q5wM9nDvKMki+b^W*BSRW{!S%<!h`N=a0xHp+0un^9o>LItn&Q>M#*#8t)A+S1Nb`Qv!R#Db+xJ<5HFJFP&MzG!r5IYX0=N=_bpHlC1ti`<{9{w2
Gp781<F*z$wNZ^3EXU@%xMoGu{)V`Z<GhM>uCfA(S=MCBIs@nZj5QO}oS_Gu_dP%@Phkma9t0^>zB({V2%RgCS3C$UPGGUv`QZS1qP(<{3{$7Gw<c;_)LCo*
C8lyJHxp2vq$NuDXyp&dj3Hc);KI;?;Wqd<Sh@(RK18AF_o8eg0$DIErU3;}wLDznMC1%N9dj{wy30n1TbZHWezrHwp|t3FNL?Fp#S?JVR9Ms+<fojcXQUin
4;O=`xXW2M3=}7x!m5<v1aLQOw<B9UOsv>fJD_YrWrJQcr&T+Y|JeE+GU51*?Uv$dsH)A4)b}s#A<>v9%5D8L>Mrt5dSAf+I>(_A`)8|B)!$u0R+%)ho}q=V
i2Fq;MfxhaRqef*V5D{s=8`@9A0oB<Gg2&1xjXeN>3x6#(Pmrv&vV6Ko3xUiAEU`f3!*<5XJnBfH*f>{l5SUE-tA_x>Rh#sdT%J`rCPb1ckjC{%04jD17&T}
mzhCA+k+{3<)MscKEdeHj~ei>P(m--Tdd+?0fu}m+3?wz?Xo<&jMw?3i|Zlq9{Bh;xi6GrbCYTO`CJs<Y3!D@4ZBL1Bl-IuqQ`ZxUc=0V3b0ONI$M0-@FQyi
#^4R6%&bfw+P8D-P~5u*lvZi{tB!rs$0sykWay<VKnsiJQHhO9d#J#DqNUbQARK^dQdiU~aYBci+PL38T1P)-h4KCG*FpkWb^nsZQYp|SUD5rKL~1M?<Bq76
v6l{3()!s1Y&t(|!u3c1S=k!tFf;ZHGZwd4H3%jT#9&A;O9FjTw~Qxm;)CFA9a6%xtcw-Q4&hN~K0(gihxo~FCIW!8y%D-y8E5rs)mEfd?9(LKz193Z)52J(
5ZIv>F*y(=5X`ArH+_x5uwC7p`x$jR%s+8T!9%NgUaN@uf1QTU>V@~d>24yQ_;&K2H|5tbul7h;FfY|uM_gZEIP)&xXB#*sAE#7cP@TPd@?wB9E!86+m2b+>
6SD(_8JeF)(2m#<Z#}l!2L_{{t0=50v09xcIg7L33aAqc(Pj!z7;prD-T$w`rgnlg-1hP?_zLJMl$mFW*@3~WzE+rI4;$pPKJ6L<bZ23XLat`g{ruvIbJ5t4
I7Bprgc+oz`c`Vv+Gm^C<CXSdGfswgq*{`{fFmi#MlOS<h}h%$wWx1dyt|?Z9z#JkRXBKtOES%+8Nq%&J<tT%<})c~l@-%u*5U2~j(4{U$Bf(-3-gx?rkdfc
2DM@+V>q{vs{BZImO0krAQ$S2>nV~_D$!lxS&}$R)I2Ic){oCkeY(sF*E*Xrm*PxEXVxpUpL4E4l0mhw+D|5+Bo^4As2ONd3XoH6!Of8+vL3JxtAyf9?$22(
yiBdBJV87i5!v<~vLZ}%;>9}>HckX~){Cx>dK5X-&X5!1UqJR~%Qe}s?>P)xh^?+`h{zu;GZ<RUN;sCpoel+wS_);@0zsAjum4xORIKO0tZehc9^8NB;dFhm
yroU>P-jUxy|MYiO&oLBf|$j9@A}SHzE8;k({*%1#inSxpKYMU%8xGz_Q>g;i)=wWp^D>~oXKj=M|zdEk$*2pg#9+ZnCpO=qkW#{SSWXXx!SQkY`OB)U8m1w
;>QDv|D|l3f3Kf~^}|I;?D#Wwoq}GalLY9)#4mQMOjPP16=1}+ydn)!NqlidNBp;9AL8mNhJ-M9fMtG`qtu5lPct+@s4A6G<*B&FuQ}cl-!)&NIW+5H@k=pk
J}L)z&G;Rkgk~*0G2v2>>x29>V>HW&FqgvyrOH9n?Rkxo`NW~+;eUAs3p;z{P}^Uv0xnj8OGCpz?N_;xO?r$E8U<JWgeW2C8x`p0*=fPLcQC?0*`A>XkgNil
y4+z^kaNbGCKPU*G&O{_kLbtmfQ$U-c?mU{s3YiDMQ4{0yM9*N^Z=#|iyYiAn*8^p7t{hJQWZX030q4vKy1;w%#S$J80HDP?P&^4BVJ}nI9d7tCF-}=;j+g}
R(Yo?{G@Ta5J)*hnQ7GiP9jlEfp1d;MkPIao4RE<{5iKh2N0-JYIHxr$6sNhYt*wu@s}%HoZ2@HeZ;ksdjgT7pA7T$8AFdRi|ECtwvs);Mm}sLQPd4nsS<^i
0M_OvZ{)xth4~|eKEzbnFk)y-a$BmZ5W%uz#)7&D@f9AjzuPNVr|;~JdCtzmJE(fc!6I~u0;K?VGB&!%ZS+bZnb!)RvyzwABg;uF#MVJ@h-?>&e$CHzZ=J{C
mmVR`+zho;#jWpLNE>ST;+=c)8T5bCa5VWn$(kf~>sbY{0aAyU=Nm{Cu9Rq=aZNtFwYm_18qL?ib3GIvD=ia!`b!{}ik|UjmWW<ko|-aH<+X?&KJ0RXofn!p
U>VGk_~Q|XS-;q>mK2-V3F_p0Ki`8q;^iM}G(CG1JEC<jBd?9@1D`zQ?db$<(hLfRg(=aysRP@EN8qyYs%nV?9DMWAH#@j#k9tt0Muy1w?~h1f4D#AObTcYw
{{$-dmqI>!ACRIND*H>jBP<%W+S*^p06KWTo-S$SH7@Tf>Jc6f<<x0D2R^Rsq(FO;fe(Q8PbJN)rKw&TA*K|;;=sL}oy<S$QEj9^iU!syMXnkQ8Kn4Y#38}V
`187vYHEf;-0i=-n=*N&OEo}*@kcs5`!LZST&ggiHl53RhC!VjNA!izg+>Pd@>y2H@*~?65(X1hO>Ou#k<m}lE)F5>h3P!*!9d+ZRMz!@QZ;t4I!CvFGEroo
jZhB(8)DB5wp*ZuX|U^5)+fe7x(mx5UDAxuZ8)7WTA2jG#kuSECbM+}lCcxKOlUtb#06*}^dxT>IE&Wt1W;6JHDk~KAY6b~FXnI7`rZorjI4yBk6pJBOpl7%
Y%A7hp8Weno{ptC*n2RATJf<>?8I7H*+|W2iAEHaaOslgqHiQt0wRHPKsbTaA6OZ1123c&xM5Me8<mudT?TZMWF`u|sz=e-j?vlicI+-B#pJfw+ZDBUQ!}HP
4vZ7?GjUWY#bY>8FaI*kNI5Eh%<OTD+HeW&`{@d^g$*21&G{tE^nmvIPkE(QH4#2<%>C-UbpiDZ&K!>1iP8WNa+}L4-SyDy&L(l0uTKaJAkozY(xL(6UKnhm
9$Ms5FPPHQVne;p%L-)4q$-Z2brZ4kQ(SEJC7ZFdfMmY=y@c<za)P_Hh2})i8C4@okiVkPWfl>YuUP?Dq*}>_vG;hcNgaqE2jDlPmh!&-?u2kl57l5rc`&`T
w5c22b?V~2`lID;r5yZ=kDSx4p6fM6Y<8%G*?sWhCXDPtwtQ=+YuS~fb%Ddv?ts65LMh{U+!g-Y4ht1TyjVPg8lj&O-Yk*P&O!_>@3Nqrww^|Ca(@|lHbUy{
uztdak07iPaQ1_vFIXdUc62LQ)K_7G#S1(^80lQXJq?M37bj~$xU?S|XhRd|BJig^Y&zh<I=|hy!W9KF(qI+^)FHe=&4`*$KsIX}9VP89m{&McKEYEE*U{9H
3W2EIn7m2o#L*~*eGD7#@akn^sYPXLJ_LSLJ5w%EngIA18-g2$PwV5fKV_Xfe1U6wQB6?Z%PO*8zDN@EQLSy%jc?`j`SWH>QT=kbz<rY5QV#O(*7Lx;oYPsK
-a-<L;8in#vN6sCMIcyI<Y06{y^J>BPrTlXTbhNLhlYTEt}v;9iKtlQ6V}79TaWYHMA0>6C(DsS(c)k%vJ(qb+>fga8#+-jx)eU6(K?4;O9)t138smbll@^0
9u_5(@#H$Sav`+iQ@LFX{xOaUuURbSr>SE)&S+*&%-kkk5hV*yGGK`P|Ht*02|vaW(9gzmt1O^vG1#t>d<}QE>i<kuD>{Gvc;PmsG5iJ>eYle!l~y65(1qiz
)$gL9%-jB$Z)1|V@?Ld^iE&jtiBk$&l_Eb>SIU8aQn!oJFDqi{t{GEElNkn4NgV^hIwTALGC!G>S{Xb!X=N_`P@3<fqC2FGE4QZdPBv=!UX=08x-CQKb9C<x
svQRM*mz-aM|d6~bDNompLHpQ!0M~8=*rf?;m~sD=$4;X{0z-4sJ<mDY9fb7dtLi{)3)i+Zo>!3X0kv0A!zo~6_U}(G;oCS^9&QA<(1H^q_HB>Es34eJvrOD
;6Xv*T=<4XKP3nIpKg(eB#xsW3GwfQiYZiPQ^idJmSw2__Qgf7#q>LxL53yk8c*$Nq>A*hHz(oLezU?${1}>)<<puyThDB{gDEPNsJV{QlC^rM_EAtCn}k@r
0TiKVksP!q;}TMG!q#xh#Yv(!MdJ*z3``i#T9l->ty42ZgKJ$NT!Orhe9?-oUif|MB=hX<x!DT+Umy$!vs^nC79Jp{g~C}EhmA>1-apqO_D6O7^+4)9{xs5O
WEI7)b+xi#MjXmiX@fwl@Q(6q^k}VmZHMjcyrwtaVkW@NJ`pGbZ{DXkfMwAuHIN2x_PF#GsKtKinL^Me-&l6}0ZS7&v?gW^=~aZOK@0LC;ml|S=z1`~%q*&G
c=6cM^+d2Pty=Dm(65U@gYMnhST99YugJU6OVp_0K?D<(CGFU#g<}4R{umW_cc$VN6fLXjrgFcB>llS3hH^X{&UD6>pki;6RK7J4X)V+Z==i;S%WVLlD69S2
1cYwiHZb{O%G#hme=Ve!QX&p8|A_&0zuilHM5QLWY`nSJN)k@)u^d-^X?=ovqJ(glk-_L+5TJ$EBdU%n@0t%ORD`6+iEPRFz4Vdq=N)bAEj>37M<u;?`H`?s
lK_-#N-*Qu1K-HcM^XrZQo+zQK(0>z0%r9?2|Ibg=2gw?2&RF&#xu@IahgY2(*9CXX;Q{vs3g^vc<vE=^@cH2ZcvZj_ZYDL&}hxt-VShJOQ4{&uQ9<<gZ8w8
DVwzjz%~anRjp5e^DK{i75^F4r1IGk=Go(^tMbHLoP?RLa{TrU8zj7!OA0y&D%860fZ&J#T#3{lwB6lInp_+O?US<Peae%+%!V1Bh0tezYahq(YM$F9@=w_X
IF@?GXa6O{tkMx8Uzn4!(4#&R{6*H%T!Dw`?Xhnm_OyjCRN?HTQSK$rv>Jf))ag9L!A^0T<HPLV<@;i^0!cpi_OFrBBVj6*RpgI;daUW)0swyEB1FNS%k)%b
S91r9dt#@V?5!1rJSA8iKCNPY)SO`*oS-piO1N7CKRC_1djBATx2P!AkWcyG2zDFIT7<Vii~Yp6QlyYjd_C+~kIWMijlvd}D5}m4sgs)_1C8*<D=Icjd&MtX
w*qy3D3suiV(ow)G1OW^Hv6<~HIJq1kgyV;U>b=oudJ$Al;9}tV%Ctu?X)=0|3%%&&XH8ivO3pX8i6s1-5)eRGZO+y{3|jY@!9j4>2xiLKiH7lfgGCxKuT!@
9D{#q6-F0JR$rbEx9-%-*9K2~1Xd#dDW(1^Np+BDKw%l}oJ?&_pw%dBtBSLdvzH3x_MmxKLBq_-O+L=ul^brCw_U!FlQOU$Y^)>eMe&H=2qNYWKA7R|#<0w}
JqQUxYrf;8iv)Ij{F_7hgOe=%DHADcT1EU2t15kp+%7?JoH`V6e)ov7Lol9JJi?SWK{5+1aSa(8l3J|CSERtZ9iCg>H+8B;pA%CS1*oMiq<ejKE^mS?aFS1~
Vd_%~c8%b!g8`tm;NPddWc=IXQTgF3(KA3k{G8}B!xFcr+5pc^Z>7(N*H?3hffOJIns&Oo%BO$z)UQV+r@FqsN?h3bd1gKry6gT#QEnwBFQfx@#&^)|nH~dQ
v6tsWC)XbI+0UL-UAz{p0GJLeiNrWSt=%pAd<#KNgTS0MTQ3d5%>grT)zv4nM`{*mQ!lCjNRO4IT^B;ll05_;+NUJ-j`G+Z>(x&HO-a6wpkRjMLglrXRx{p2
Ix0)372#z|h2%fH#HdBqs=||PiRyWIvU4$O&aPGo`zPq2HgS9<zjMjB`)!@mwzN!?O+V<NSFOG_{`D56LlBk~H59>i){iH?bNBvBSEVBjBvy`p7-B`7Quuk{
0lu?`H{vQKfdc!`L1C5{PK(+<^cbfdF8mHB)k-?3Yb;+j)oY~mntY2T36e44%1jaaJFGS{p3ryhXvVlC^US8m{{IQ2OVi5O>tTxq4qosXe?GXSKoEKM_oo4Q
PQm!=7iYS)<W$41NftsUcJ)LuiSd6_MK6p+94kJu(z>9K!>5A;4P7gyb)};|u8|>%X=^(9PO+rEKsvP@D(K%V-2)QBCD3xpvbJS?S4enqdkZ1<5Uwi!tkg``
wx^mCvCQKZpy4+*Cf$hk)Yq`}>B?!_(XH=HIW=imB=UG6i*5`Cn-&!r4bK=zZ|QgARL)&LT+nFLIcFJR<M`{ZmLHc&4J1y4l|mAfm1|Kp$VqV|iTz;j?g>_W
{riN#zoZbQvrg;qg$d7e^32OptaniSaaJ}51%Kqo6HT=9oCkx1)+V`>xzJ6R@x@LpMNJ9kH|H_itxD6aj6}LG*S(*pRSNv$VEt<sI)1$YLQo8irPdtGPNlg+
<s^2!L8}GtN_#4Zg`R0Aq@)~UGS~{`f&ZYJIyHH~WJ#J*?Ozd!uSSedsUkiQYv9bjUs2SJYU;j)PO-2<i_FHydtlRo?NjANc~rCBERp!za0!kT0%k8F%wDZl
%FnCTd^HxLJY>)G_%2_?HEL=?E(g0|si$_S8t4P-SAK=7h22$_!KXyEVS=ontPY&UcGV#*bx<qLJ0BwqipBzcg^&q7yl_|NAC1mXUG0-%ZgYEB?Y*nG%MqL0
Cuw6DRIhHF9oO+j=GgB@SxY0W`|Ne_eg7D(W}d6#9>b_9&b_P&e)eUjAg+dxNUVZB+m6V)>}_ke`XA+&@4}>k4~{{Yb1ZVr+?bj=-l)kAvTEX+Il2v%)-jm!
RnFBHWBkBv{{@j1Dpw&Pn)9Er<T|Fw3D<vVs7&f+^+!M%{`51NxKSxl0;ojMw5!=s_O5_~;O?3{f<BoZnEWeUg~|g5PkG2z%d7Qh2CvSvqhcEk>Upy0;CV>I
N;{vrP)dI;8-rGk6Dl<k=hvW4pP<SM%|XXBoWOJHO-){H!BX>dh5s6Nv*%#PSh^{WtFO8cVWs%iltUWhE-0>Prhw+D^~H_STw@}2jZ2yfJ@?w@(^!hsLhpLa
t~KDTNcmm-38sa>lPn3nC8V?SJ74)$O5@4aLv~%HFMD`;%`gwaeQFQt_Au{907l2Ej%bF-o3q8?K}&}41AhX=eSVz$bgr@yrKjvo`0|FFP`{=iTzRvwSp&JB
BSz_L(YvE_aNN3ISg3y)_XZ*2&*XY6D<>)|ILGqX$pOof*%WEOlU~tdgdPoy{pw+5^CZfYHv{y%O@5OYdTV#oeHzoOPQ<wl{IIF(<NjaXNanp-OFHz8A}b8Y
jfqQl`oeIy;XkcY)=n&o;R5XLN_i||u$r?+R+3M`a#v{0a|73=-lyq>WRpNp&PO6V`)NUpzdlBt4ZGemSu<N&RD(-Apt-R6gIbuM><|QLld`gK8Zedt$!l!X
uf`USOZvii2G(*WG3)_UB1XG(L*nOth*dkY&XkDt|C5N1N&gFoC!SK&jZOK?O;?zsI^RJ3nU4$?ac8Fo`N(+Jw`dI{)QqQZ0}JYLG=AmEEztl1Vb&v+LwO+h
NIdt-6qHH=tDRC4_j?{dADge$-1+f?_vfy%m>=Pg(a}u}Ee5e%-XN8PE1=M3mD1rUqb{fJIPk#1IX!}vV>VNVjw1>3`$?nzW>|o|P5L(_jvX6RU6RQ8;Xq%s
lbMKK17G0`P)W0c0PCz7<!Bb}fPE6BLGvuT-mP((;_Mp^^&=sl`ZaBn5&MYZAYeOY=ceak=!=f~E#W#};Hb)1-Gofu@XfWrw>t)Q$JqoIa2qcsn>6jLEd~qX
=Au-w-djxJ&DMPrxi+t+vlR!<NQ`of@W>_e*gL*hbEoszNz5BAL<?JF8<nh~kQ{%$43AiM;>@2QI!(l9)AY_SjCaL1@GfM?Xo}E<#KMCMkGr~@vei<g?DIlb
0ppu6Bon;3Ajx|C%LkylJhz@8zsk0v1m)nSA-yf$sXRW_0p*1_)-&b${2g%#(jG;+SLCtbER@n++yo1X!%yF??p6n#!l3=HDI>R624f<*CYFizZ$+!<Y-v0x
k!s0S&w#rD(Z_6LKjz;eK(}yrMM5oTKAly2UP4lM`<4NsDf|k@ev{-9FD?09h%6?aei}EiH|>~f?rJA{qF{$;Hwpj%^e5nG1{1D%WK0AqL{Ihs!LFus0!fb?
VjO|R5j??`cxpW!>s46HyRP0>zVl)Emcb`8gNRZ30e>=wN0D9#FW?Bp1)vy@yW`bxJR8_S-EU<RcPc|s;$VZ<c-~^hwlF-r2eU|79C=*Y@5OrTia`O?nU6La
!Q0n_;GXLgw<pebCd|<?UtEDy1vCght0h0&M_wCn+V+)}tC5>8fUhwoyDH^32Y35)hNp=l#P3noHok=ztY$X%P+(m+-6IeY{YD8jYP++fw49nep%#W_vAJ^q
W1NDxJ(x;2S;2S;t71*X$B*-z=Z4cEYCG-gKobRbbiJO3Czz3TD^+t1;qon&ky75KM;KY00Xnw&ga{)vV0vONmo0?zC(w>5aZA=|GUTmyp&W-2#CpL~P1wA|
I`U7ftx4{GU14?*Rp(s6%%qHC2GX2vA{Rh*3Rt3Uv9el|%8T8>V&)v`yH;gFN(2y<q;5-+PSJ^0_6JQoCl?;QJ49LGdWFXn9(2aBq!y1<cdU)HkC37hy){n*
&g5Zvic@Zzy3(IZ$N!VrwJh|>za_2kJw2`dz{^sdm-Pgsx6q;9-1}T$)$-}kM6ghjR=W!RGbMdz3K%^;F%3r%v}55>fsEK?CVX#DUTZV;sqp?|CS(CkmhC;}
xQ8m80x8KTJEq)VB8_uB02_wUgo$*Xae6=sMH0FOsT7j0@TU@Nxo8_{)l#6F47CO@4wHxViW_HNzxbApL73=R1eaPXa^!#j#ob&Lf`c9k`<1_J?>vxVC@45}
+|H0qVd!vX?bGa=KwMe7{edMIt-RH&m1f#DZ6b40KC7fX);h|`<vAtnCVle?sI_R?Ya2>WJdRRpEQ=ToC@wm-*7UNUpA%M|PI{oqBxIqjxLX0Q!X9Gf$*mlE
#?2a8^!d9*G_CjN>+$AG=s~G{#powOro>)_cn&%t?!RepeiWI)%+f$ic+)7+awmL*)}+nUwcIs5>*uyub=zn0_+tt=@M>n@%rFkg9*^Kmjh$#{+lNK<qLzHV
?UQ7(&;7B^h@4^6kj)HnZt)OJLjv0>m~J&>(9IwJ>=|g~T^D>(YT?juf3s|G`z*`taFciqw@-ql%Ci&+P<6805`o))r3S>otgD43TS?mS`&TTA;e_||KYLrB
a520Wpw?Mx4ng+<`tFgPWN~<7Q#y}SklNBBUv{Xe8<&mzWk7ceN5L2!e8b>mr~}xQ<UKaOkNUgC_6)*BU^a9wQt=7>QGt2>==OMr2w5*ZFC8|C%Ty;}M?$ty
2E^qZQnXf)8Iyh=<aq6~FieyrMI3w?T!Ui2pfX$fy%!UX(wR|wFfApOo*MVlSF{%A13xR2y;V=)f1!Sby3a(TVN%nLf?h{zxJ(?0`L&H|f0O>qVFR3S|8x=`
3(Fr(V^bA#Np$PtwHN$SaJc)~Yls%%xGdl(5IF+LhD&qo()mjeNNQfye<ow3ycq}dw5>52`Rgre=eQ!{KWcDI<X5sqJDfhWI7{;sqL6did-)N-GBD1rnw@X_
O(Yadf7{Uw`prAQOfw`-6`x`BjIsQ!M?ZCW2d#Egy4=FdX``p*hpONxL2i8&0o%JP0>WD7?SIg|#s9N$-B_8^s6&455|B>kl%sdhy9E4tTVFf<lI_2}l?*AJ
AHVrE-DMNI?SuzLv3&6@FN`$;4wYX1Hd4<+LA#7264a-w`>}rKcC0Fk$%>m#s8rD(`KcF%dw!4zk^v3})Kfh{Ac$Pg<J@B(UkG79tpB$D5&A^6y_3uzdy8!>
W$aaUo{#lEc4HC8`f06NB`!9=6s`Z1uinG-Z$La*gWIxy`^wUL=VjCsV^AVK4xiu)U_WJ47T);{-9vZh{3DE&xd_BqcEY1_pPu%Mj7}_{7Ow!rJeb;%FJA|C
q_W1K7q8QJb{G-Elh}1_s{vhVPMDnzX}N~BUseW{x>Ay{8!7mMxRO`+evEndy1bg7b^A6jo!u4<-HNuXTAl(@b+#ON)=?A%1JKuGhsrBW^H$sXr8nMr8)%%%
1xP1~mU=+T&%z?Hq5UYip9FHW+76R$5D&>AC8dB~k*)Kp0k((x2CpiA6m5NKt0NvnWiRmWu4G{SwN7_!1$Dc(6|?I_VDS7o#m`FwXN8u}3`#@AEj3&v!;jm$
m*f}8Wp$|&axQH~0$nqc>LpUREkVk%o%&)e2A1b*J*$EKy^P&&CZxwjhus<i`p+SFyjkTYFjj5hAB6C`F-4^6!h4lFiJx(gU!L-h4nM3=t??6DoZ-?Bf@qvS
6~>Oxa-KY;9?B#LPhk%I`iuQ(qAdTsd@YlJs25H>YF#VwIjVkE+kSX9Z%TW&n*Yw3999NwHKVk*)qGdVB&XoG)BM)4krpS}HG$BOvc-k?SHQ29NJJ+pSb-qE
_R})02vwjr{SUQM3KWlpG)<dLD8U}q!roc!2vr2VS3(|EG^LXV7aL00y22lAXaS*+`YuHpQJUx{?>#Q0bL<C$?QF*?T{1;fKixkZE|raDbbdbxGK>XQ-lO^0
_MWe`4;xQH0Y=4PEO^`DkV;{<$#7|V!LwPtQc@An)IEQO7!29-j0od6BY!17sRoyE=q%Qa97x1yBEa~1<0n10vD(-N5*?kP*0xRX`_4zVJbClo7L4Juq)$wO
1u|xaqgK-@r6YZ``jZS^@2OJ_IhB>eJ(s~S@&f+)st}GtQn#o#`bqHp^`{!785Wg19F?hpKkl7dDh>hkf^-&}u~AfNarv86{pl>JSMh=Jjckd3d7!KmuA=qx
>>J<mG>0u#WR{3yr?SQX)Hyky9PF!#e=|q|joa)l_?mOs&#SF~%nfal%_-#VXdt}`mQ;?<8i!VV8|Ntme>Okg!^&U}Do_k(v^B=MGfG3X5<ru7D_56xar4br
SO^IKKvTpZ4a8H@Je>E3q=DgCp&TjbfYGKRK)k`yWELQPWsUVx)i0c^rhP~NJ@R+AzX+~9#7lX4<Xg`<TsX?*4eDK(+k>n=1=7!(bE?}(=DF*lD#=w)m!FDH
noi}3SqVHj`m59+il*@;i`1n`6+#=<HA8ojH+jone|RJUx4G-J0<xD88jMB?4i<i71(?CyTxdbK92KbtD(NMTN5ewS$WnQ%y6eaRY4ajING5HE%gpOjG{_xP
dnypum}cMuD-bk9dN!1U$)Ryf^lnZt)))|{G6QK@qr%R~b8NLZZ@P**R1k0X``ZT$lSE>EhttW2iU6#U1A8CS!^AZ4EMDqs(DINT(&QB-uZmsPo`d{&-9XWh
u|$~DwX`CCL%?8tBWucLImrgyNC+74?Cb{{PMQ)jQnn}E(Xk20E<&$cDe&b<b3A)Dn6N8x`TBx++JaH-xfK7sN)-0Hj<ofle=d1O%#2vBU34C{OB?u0JKUCJ
UvolQ>jH$7`CF8UH}KnSO{qAuGpxVS)rrpWf%icnX9-&P^sqkzo<fo(Y%mSAJg`KfAY4u$>sQT_F2Oek!$nC%w^{R0KT(-`S-OoK`L9sYvHnF#=b3lsA5-#v
uiav{t;D@nv~WSx1~qdF(C1CH1lf!UIS+`-gg)Lqcuo{LAF4`nu?jZS+WBFL|7jPa<Rx=}`M)1Pz;Ej&GYsX|R%_Qu`<9XN=yBQavPN%m*jUqoLa;hzGU?d>
l#A@N&iSOrg;68*Ug8TjpA{$d=I2!@`kO(&>w`h;Um1MrGVSFlS&U@V%`ZfU@2pWLBv}>ho;dd6o-x9y%eIhsYO^8uEg>uL0+)r6#qd}-mag@<A9m5@;91Be
F^|XfE91V^8>%ImsIDo8UDAyyLjo`*m{icC8uIpcPAdD{K~TgFbn^>~<SS!<TEM1I8oa9j?ec9MUFV9wG)j~=itaTy1^xDD%UI(Q&G}OF7=PgxBP@`{Co+|f
dE1cc9aNqwyqdY3G?snL>ot6Y`-<jUxgjk*dD?H97$n>x?&nO2sSS1qDDsnlbB!1*0F4R$Xd#PGld=cAA4qn7K&o?)M@GHC;myb*(H<Jg2dO@5eSQu7@i1~H
$grf5NUlfN>5yh%Nsx{M-KZeTa2=x*Sg3b=@t~l>Q|CoPc?w%9$FbMIf&qBkp1xMn7_(kxi+I)<h$>?B;=zBD9Q*S;2wYp8XQ`N&HYrPdO$zgDS994(x64r%
4`p@;Tg_o0m(I`5&IG+C%}VML6jvd@xR~m8Oy6foBgb<b8nt1Pq?K|!HR6yI1I!Pbb#~qCvB=3g<$MhF`u8~62CKZN5kq)9Sk-9}Wy*nkue&a8`sVS)iFBSO
W%wWf0#>reI5L|@ES-jaT1JcfEe_U?5;Bz2FK->$f@Pd-dx9Yv!%Z$fz?*N#cqV!O(9KsRa*BoZUBiJ<=~Wy)6IN7mZiMD8#Fg8UqCqr1-)5988~>sK@`Dm<
PR@jD4JH0#o8z3UoPoX&h;f=Trk{kx_YZ$4y+k(*&5)laS%UIXx%P=Sgv+uEvCG*>wpK4p<G8aA$KTK1rTp1e2$lD$I-X`K2qK@f_q3@H)WB=UEw08qvO^Ku
ZNAG7#A+T}YCb0u;O&S{i>$a3^5h@tiiq7l0Ukf^Yt+o;;USF~oFkHAzc&zj!1>PV2d*b9)MU7A<~hG8YNEwOMUt-e3N}w_iDxks^s}8Sv+2QN;v<pPRfi<8
r>De%(cm)MKSBJ+2XMGP$mi+Qm)cbF)20l1O)){<F7s=*=+0k0ONyTO5(Kn>q!x**U*I%>l&g4n3HXO8E(?Fdcocow^Qp(>a7X*xus~iXA@PR-dSLnvixuEw
Z@fW`uFI?L9+)}44`bAUT?cpzr?|(vXeK9`Nlz11pWipT;P;HDdIWw?G&yEum?>cs9atHafbmvQ97wHn$`|*mZU+#q9Y?h0Ict%9t>5z(Ci@MM1GgEzovl-A
b(*eh`fb)pf>P0-G3MUHcQ^7J)K98HuLa>NRLa>-Hv@jRH%bDBQyLXnzzfc9H?oX(A3}woLRUgJOAFN*t=kU>T#U9;EI&-UAqow!ZZ52UO`$vZVxq<@6ouJ5
&PvMD;;YZ$g?lz-=>Q=@doZ^g20}8<4Cq?x^qJ3r{fbpdMi38=#*|4tN*S&uu_8mwQ3Bcwm-6Nq=D%syN=GgiG^`(ZeUFW`r)I<OPK<;aD8K-=kbPwrOEa!(
(5xbQ!H{~<t1rJq41w7eCM{xqJhA{cyN}Q|Vd%Pga}X3Fv2Yg+-E-L(_JM>=xsn)a|6d4atvf;)GngBPmS53&p6S}XO%nNn(2Hy%V^Ba|B01k^V5L3_GWcPx
R5XVNuRsUt7s!)XhwNQH&7!5D3C8ky6a#5Szp@8i1z}>gg-XmYIUCg!9BzsTbbEZ^&~h@sUchP^P;oA!@bC(yeX|~I-Ko^j8S&Akq)Ye%JF10!CYY!_`}Z)4
CVud5rm8$b)%0VLc?$KVC6Q|h@U@(Cz~@r;F0tdFV^DFrZSH@$+St(@R*;^|wbCn>X>3f)U*07DD7wmZ*Er-|YgWGYSo}V^?SFpJ<o>VkCXSX-Jt?A*=Kvg2
vv*OQ&%_~n>rXu>sFe}#U?<h-1MIolyC(=oieyAIuPHlKtm&94)?|xPIasN3nx#SIHsB-yz=nN!q42xfc7LCJ-2rR(9p+-b!$3w*t4QSPPp>vg&u|@M+k3o4
?1wY^r&iz1#Ot$Pp^43MIPWSCz<~TB&OFE<^Sw6$1J&RUc@+2llD&vIzr`|!U7=dGrB=fa&wgAXuaRHsLXmxJ;9%Wcm^p$)lUkT1SDFcUp6@Y+{9k!ZrvtrT
>F>QCL;wpKUM&AhAWHAEWWUP;Zsfrtz5<Iv<DA#xP2CqZuUJ0_IT^sZ*PtBP4wk_^MCjJ)4&QLyCNwa1SoOyLT|B=4r>JdiE9=yNF1x(Lu;!6*Y^&G3pSa9A
ZL(K?BHDuKvGPl81iG+FmFw8oO=7Hny&1D&>#25xCau)P{4_i-U29T|)0|{JE-EYDD;<@B>rUYUsm4mNjliBq27CF$4a|Q%uIZ)LSX#~z>6W2r+G6G+L7)7J
;X`@$BcK${n<FZk5V{G@3W+Qt&ViLrd6K1VxTG|#jU_`?EfBpZZ9W)KF$lTZlFJh%j3pptv$eAH6-J++Ep#Ea)Tqt&7j(kPylsP;^T^<su<h0Bz|RZd307o6
8nk6nPv)jx07KoXCyl=GUR70vFI}n4R)h-=fR(yLx9Nx>_urY8FqMq+P&JsEb5|Fo^;=*CbiWOKh^w;-N(Hr^m5Xj;J0>*gmt50{a}w(}d*PF>?*EyA3@y5A
S3GnsJjOynMUXvvTwvWM+iCYUib6|?7P;TG9|sfmfpAh<IMrejcWfSlH4h2FltDaG`gk+A*5x}io#MBkDVbwRV8ig5qhV29_3Wjvh2yRhUUtWt?!d6W5aGjZ
Sa(}UHRp~9*RNyRd^4ZBXwMheUXtRCsPq9$J<BmFwxH!_p9#DXiRG4d!uoNOak>$zX_*$P@lePW2haQ4Fa0Vnat5}JK0m?@gP&X<`j}wOG5i&Eu_IH1bq1#h
|Moc3lQRBmt|)NEl9d$irvN!U1q73nI4TZKhmv7n*u|wXS$%;!1HG)*2%wW07Zx?-I^$M>gwZo?$%&mI%twsBDG=@vQqTL%6C6LKNh-9IlzUqa+c{$aZU83;
?Bp=)h2L=v5`P{7_Kq6o)JS3YB|&v52UCKYGX8%03Zj2YJn8Of`;XuqidglBK)$8#AXE#YHm_w<2uSZJp}0YR&QiIT?iTaQtVoj5Fp>B{>=<VtttJ0%^Eokb
p+31=Z#+Z1XdYw{*e;YyqKdX`f2*m$VJ4B8R!vUsEB8zr9{`q(lF%720#v((tW`MSf9NMAOd3jwdk2EE`@lMR<FS%oONlxb^Rc4WQLL~uB@w^AG96I|X-|{a
(EmLb_<!y#I~@Xp_I{u}m$RJbBW?A@LcSswN=XS0l0S9B%xaX*zGrntLw6z2N=ROa3j@k-S-SS1P(WR79oFFRmJFtn$^R9mUF1H0@X}Ba@^j8+wgks%dW?E<
WKuWmDm_|`y+0>!PCNJ`%3ysJzJ~@XvR|d`L{7$(RtoJcCxqQQRi-ZZlj1Kn15&=6s;?MHSn{wP!g<mLy$1OTLN?vd{iDQI74Fk;nU@#k@cuH?8dPNoMywQ2
&C<Hgam)gp^bs|_4%|azttd%X(oVDww;cnZ6S|Jc-GxCc5&=x3_bj>P;RZP%N8B?k(^R2+Khc%BsiO9}{(txtgfM)xnJZAgSRj>r6glB(jQ%X@@0V~`RufWh
X(-&(zr!M~hl{5$h^#bXw;|hj7{UqZA(Fk(qJ)mN+(`4qduK<J2CmaWQnCqMBfGD+-Pws~LKy^tEC<fmASm0(DhAf{OE2EWB_Kh!?>h7hj71x&qS0o3VVq;K
X)T>046i>1L2q=Qlx^*lrP`5zh{@1ULygQY^Zb02(-e3+#y~mb&XWUz+i5+&m`%^#<*!D1o|+_`W}X~m1o>|u_(eKQmz_Tt)v1E}D{-%QP1PIdYKG6TO<qe9
vGVpyYEl_KwvX)@qmh1S)M+h(+c0X~WSwuFg3YLyim<WLrT`f*Sh!C=B!vHtxO^(ds=h83Qbe{MT)}I%>_WVI)&1QfL(6Exu)-WQ)YUMi-nV?`NioSJNzk4d
hSBI4-ITLzD+flBh+ElR?&(a*_e`=}?$0Wh99LeO*L?X&;u4tPRAlGx_#*>j03<dXaG}FEve;4Ul4W5IONn>RRKxhG#RMSJ4JW<e;etS5?@DIP01`;pr9CH5
y~K^Re!yzH{h*~sD^UNzH6qag=pcvr{(xnAMTV2w$+W?!@FyCcr`C0QwT7{|B2ao7lGb$_PTBC;qS6P2)JRw{Svd67s;3wONc9{6fntANrS|1y*XnW_hDyzH
*-Ws>%W%{?WL}_^*rssddIhTvzQ%cE_-M|xZcbKNZ)BK&(mC@JQt&o|bPeYf%DJ}nXaA@m<NPJ-7nXk;PQ^3ss8O^aXEe{K1@IR&D7U~@f(bp*4~!;-YZyIF
XUbP~2*vwKw>ETJ5IQT`O(%0_h5yhu??DmQF#N<Qx3COY=~{z;zaKd^I4(A1rgHaj(_!p7#&baHmucC`D-PbDJXLZ^ZwZ!8Bnn4S74XTxvP!a2J%ZtZk@SF;
g&_v#Q<%v#Pgn_0uZ=y2J*tpLx$O$iKz=d_4(3l``V{oCaru1qH^uoiAP(sf|0y?qHMYy&$%r~`uW%g@RU^>L2v$<n$HB<Xvb#fFZKft7JX<zMgWJH}5FLiO
VzMP63tOn}M68QlE!nb#!Ap8H_sn->pZAaQc`V*&sWl@W&{AGf^Z%|avF;8?#7s{qn)r*s921V^#C96kn<|8L>nRfqKgwBzhG!1ZsZ6ppDB@6sddS;LY@L(`
Lu;kmTRSa}(gxL|@uR%XaH>O~Cdy!NO<prgfYqgi9;`P|T{VCPY$I57Y4T5p3QM{Vi;|7&mfdVCi9QRlwkb)y&NXQ@!=Bv6d}%KH5m)p7>PSSO?a;$K6G@Z-
-Q8hle)p$-<3rCMfAoU9cdjc}<{FQ7xeb_j-7ysTY(i2|wFUWydRqSM7Y?o?Q0bw}#&p5vBJJ|(fuk#98|4YT`x~hI#a9l#y=x4RAv}DH`sZwilYTpN;f}4O
HY!sm?K0<9#a}0MJH5xcGXMJl(21ks&0u<<UU-wHJBIame0V9X?IBGaMppjZ`Xylq_Eco$x(`}I8cF5vX9~{lsw212w$N_q9A80HFH55yVnp5NLo5eG1_s&D
dkz+b>?-X%vfbR!k0rXw>&?4`4@cXmp1KJz<{Ipa1WZj-Frf5U?Ob64{DHp~qlK#ZSZ0V#6Cv>MDJa#HOEWxml=3aIsEh?p`gi3)t%{l(_^TX!6(L@gbRb;D
Vsm`rI}RQKK+9lc*0wg$Z-K)?6W#PKM1NVL#MXC=x8$!SD&x2xt>2>%)FSOFrrH3DsMBwOH@q|}^XrhRI`%sY4^#ailDeEH*!O7a{0r78by&(mus(TRSBc$-
Q$%=D>_3L88UyRE`DV{8ivxiYpaS_U)=yeD%~srt<2Z(<Z4291SDIZ**TTju7_sG?{_Ajp3=yGU67htnrjnk!i5!>Sg~C+4_Wrh_)MPJ%EtNG#S8c5W&wTC~
=a^NW*X@D&IkeA0hO51paar1uMGEv%n?W-HQo+e4tXXU%_4q;Dc%MTR-yQQ#w%)y<uwZnP1y}umL{E;bHW2|c4&mW$u;2{rwc3w{ENZMB&tJ+kI`LicM)V0?
x<26m-LgjE@f$ztQbgbOVFdvvR|#MzP9n+Rnak2-#wiN>ykd<4-bZm9`ZF;d!%Dy{`#gzm(VTn$|IOk^G-au2IyESJ0EAliAzq5^cTBgN8*jYywcUIA9wrmM
ov-~G$^x9Dc$cHe0;WmY5?6dy4V^rK7#*Alkp+-x`|E(mC)DIOF=434Q$O<i-!gH08SD}RE{sMNFE%l=No($(Ova863;HG5i(#qKFhGTbe`8LdB~99<z+Yr8
oe9GSoHzv>z|1LcB|Q-T(z%2_VTkh&!0;W?TIJxZA;4e9>+-}`CR{CJq>ix!Phd++(<p~#XIwwX;yLg?t$;%jbgg%ix9y%Q^k~qmO*B@J>GcGBf~Xxj6|gMg
W$tf#%p5D)VFgw+`3oZ4Vx^~!>nx<u%uk_#ko+}GfYa>RY*4|c;v5DZiLFY-KtS}*WS>J*SC_}gSjRv4w4SIOXJ0@Iq)}YW6qc4G6rbRzgcl3f3Goz<(Mq|#
x}n4<L9@GSmgKVh(LX{SX58Z{MWQr(f>f7d?KA&%w;u`zd|eRXD&Zi%1i?a|pq@KRkHQQAi&a-%)xg578j<N@+E_nMWP_3&MiH=kkCW-F3q;`Mumi96J!tcP
;o3Y11dDzBR))Or!z0!xt~_Sws^c7=#r<mW!P-pS^xScXMdrXF>qi-z%Ojs<7F6k>-Q6}1tf@lUi@V2jR25Ci)UhX~I5JrG$a61QYeO7X3uf(<6R5BG=F#RR
GFGA#bGxzOG9M;sd;U@alQd3R&nYAO3$;D>(v4jjic!=^n6l7Xn1N$$l_V5y>ZdmzQPkhx=h9rmiA5-6;mXEi03F@M^n9<dB+bMiVf!Zv7}LH6Z{?frt;gd+
WG8F(pm`^iw@`N7M>`oPnBwd|)SqKIt_2BcR&Ry~*8cuK_En8E8L+f}C!Cp#b4!ON8O=A6bkVoLhTx6kZtl|-VW>o})*F1$-$>*xt9c1PfZ|M(>KBnZ_tX!O
8pVBU8-}<zdFuCM!#JeJt81SQCD9BpcV*gxBc+6M4eB2M+L1+D{BHC95;Xi&g<~W2sl*UKa6at_gj=t8FBQB=E2XgfX<WB>`4l?A)rjp*gbn18m0!zuy*sFb
2zz5^otUVi>744bZYe%Bf6YWK0e-eC5S7}+XwQ|NVOvtTdgd|g!NBQs`*Mb>eL?mV$#?rCJFXPT(vt)-qbDrFRmd^LlDmDfr&iH?O;O``zFvBwyt_~Va(P}K
Mq3SErBt5tPMk9arnr-}op>tLDxRTiGtVz)dkDGLfVVGr`vbQw-^X29<ZQ6!MDjsk1!#1y{CT0Xub&D`FaJE(-9JB{8i4n+)t|+?l-i<gL^-*&_JVfH%y`~o
tQY#$B{VcG6##VjQ?8Ic&{?=uX?y2B=%XBBQZTQE0W)ztMqO!g&F23$)-UjaN0?SJ<SwGoV$4#desRuy)giP1g3i)7X&!-(;#<k7d+izikfGu3K>55AOBg=0
xM}C<1h_zN0n2X2uBp;so!wgfg#YYDA2{7URSzeE?q##5G$#8F9Jnn~u9Gj=IgUl__n1$6TJpPnuC<<M)d>v{CS1vc;r0&2U>500Nxe?a1PXP+`wOx&7Ad%x
hY#5eEU{qDwWQMSL9XKUSGTpOEYvxn{e0qA9njMe=uQthqX-f$NxLI7`K=_A@xzOk;qsEwg8YqG*IS8vKn}65T!GE5*KCCPb`%?VlJ+1BNp|RhfS{Y!G5hBK
l&AI!AKr2fCX~FqyFB9w;5HJqEv%G!=r;KbctMerGUQ(y6g)5c^0$8WbD0q>Y6W{!cJ(Xfex8QPf-at=1>13Jskh0)eFo8hUdLjFAl$=t&bH?l>c-*;N+uU)
??Z7(U;9o^9>Vcy`y=^ZhO4W;8f8TD-qvCmG6WKlsh$QQI$0Ym)#kHTvK(t&t_~vs(|*i_o9<5Cv%HfB;zBfxpIoTVI?_5NF?3jtsw8`awP?|M<%W85ZAH1l
_D|aT^fK}VO6x)^j~XDAj`j5`Lcmxh=HF7jb2WrLUm=|kE{!)DJghAn9h5u&fy~B`*3xZhdNOyv!ap*-6b=X5iZ<F~2rM6;xPk`Vs*vGK9%wb3=Z5-8AZwz-
<M@)xJr?i&YVk)1uG(_W&&8Bm0BGBeL9?egw`&vyfVks4wpLKW)*JG&+A7s^R2>M)!bfjbdPE-H3?y!udPMFwfQ<dTP#2s!Wl=b~#a1!ld81gVMv}3VAgsRT
A*Ymf!QDn5nyMS^?Bl#}-}o_Nn~dLri-OLnY!F|4-YhV$U?l8KPIDz95H%Z@yUP%68cbB4v#RD_f-<X?nG5_M#hmzMV>IiUD=q4|r4WABGXBK%y4R(8v_Vy3
0wuH8iW@t=d_y@9RW_SS+*IVMM#v(erT$06$m<MGRNu<;9?_QHIDds(6}3hXrSjZ7xLfQ;2<j?32lq9Fo(kd=1xrnS6T?uZss@;5a9o}IG<KCIiaSB;)SC@)
hFYzsm<0KSq+4yCnNWU74N}6;+DdmOM|35Xx#AAGj{HcGM~%f1hKN;*SzKuh-~&b0!5k6NM9fOq5+snv&8T@uXp$1l@{C?_O15}JM$DhI*C){fFF7z{NAz7n
@36?B`#XE>rdE3Am{(q*u=djVRuLqP_rPJzPB=3{N{$1wWgV-!h`eKLPs|g(H3l{>@AoOd5)%Y?KZ>wlM+ys&lAqG?=?nG_&X#wUouDUP022G}8eQ2VQp{qF
zT{Jb1R&{TtX1`wF24+v=l5&!Je&mLp{OP%T2UZ9I-X{_ja>2R-!o=&ua@%pe#rvVKdgANt12!gq_A?wAnK?D0D#zMWLq(-=_8k5Kw@gKoc&hrGRVe>r}^ne
2gZAn4Oe_C+{79-D5H@}YiEcwfv7queK}_pvZa9_x*YlsYRxaAl0x~s#A3n$5JGf4j+2<r5#Ql{!ihbr-&ruUMA}UAkt`@iU(?tF*dPByVjFcOeW)-((;p$>
hcRRhW!WbH0~oC7Eq?!EplfoM{t<ILrK|9H!C0YNn^@1{Bz`KKO;UFC7|@#kVI^%~*Q(#)%guwn%;q1z$sMyDa*;q2$ZB{(92R-3{$8KFfd8Km0Y9W6(k&6s
`rhq>W;!G)w=mu&8S(@C61_jbS}Z7G|E!Y*p&(kmCHw;<94pSxMtWM3(wSA4R3j^7PU>Ge=JZz1Kv}U@aZ@C&|E+`+5q||B86nfEje4DY@)W<B(Pi3M#+~ch
LuWSWd!<i*S2qrkmNh1%&k&zTo9+hvAO2^h;$qk(6xW?YiGd!3QEx86MtNIVM^KdRg32j|2(mw`zY!6dyk`#8_j(!oIrI~vUL$%z$~(lS$j<&nV;=0ado-&Y
_j+%M#N3ohL`JtBqdz0s-Ol<gB|TRw>!|!-wU4TGSw904iQe`jsY|5sQLwmBm<e!L7u@rsJb#1WOfcFd@eG>lU(@yu*>^DsUARZq4^pZ)4@=U7Bva-d*2?Oc
9Xok_S;F3n7?3d~G~xseApla%pYJ!|=LlS42<TYoMWQ|!edgWkApS+!A(@qTS(Wl9d|L=Wmz)d=|MsgJJkE8u{dZfs5eL9@NYRlHT^;aop;~d>F9n2df9mtQ
fBW+G#7KScTpO5$%TiP|>!;Kkp0m8?`El;l^XT-_Y^+#{;gS@ri50l_mtcqiea54IR?j`bz*1aQ*pwSP9x@4Z2WGd5(JDX(rt~y2rJfHAkTVSrt|<u1Qfu)b
+#&z*Q4&)Oqy70v46KyABWW_v!OAUY2+z|uwM%GcJ9V3Xg%jn*&kQyffvA=0U9RI+vV#D14yE=Iy-d8b9Yqg}_qePqM7WLjtPbAgWR4yLlT6(p)uUp}|Gwhc
{4K>@FTR%e)$Uic;=ECD{4&NfdyB~GHwSZMq?hNNm(-!)6_D;r09pnGn9#~vzU0gE&N4m_H+QJ`;DMouA<XPvj@8W!M2`2TwT(Er7M}z=cF>d394-E&qdjBc
>!nN-3=F&V)Nz+ESOviwGU(17mg?mO<HTQ<C9y%h@r)l(Ob0j~4;oafmn~gz7ccL#gx03Z-kAmV0L=X-gk*DU`d?_Py;18n$YVC7WI!s`%j<xb(})(WE7m@$
6LzS660H*~>Xh1Yl}EALW;rW9l&hTfz4z@*AaQq%tu>~*wL%=sxPXe&H|W;2*kq-9qbj!TyCu)JjoBIGHsz-}!ZLs?SN#clENSsA`DtO8{Cm7?zve}ZfOpz!
&CrD_pqmv;%fG)@Af8WrP&a5co`FjGSp3{YO^?ef!Mk%4_hZo3{9}(R$?J~hw7Sd>kCef%*ooo<pZek5ngar6S6f>n*<T~PvUZj6&g2`bQXftd)c~m!7u^-_
yNHXOTZD?fT{BwiOH=pD+tw9(isqJ)bSeiA(k4us_qI|aclLV^y*YZ~=LLGzB}P_C+nU#eCdyzE4qr@3!H58c0&8b41?Z=LRJdKbINhfGg<sL&U>Pw1J1ixn
B_~*(zL|FCEi(r>10c)?$otAZA7wJG!frusC<ya0a)r-2&P<stEpYt~9cH$`DpOMf%qq7U6K=Q9Lc<Sg?2o}VU(d{rZ8#EiKXTQCFVSSTw2Mn!CWwK1eK3EP
0|(xU;*w=g|1uU@&>gOQ10hs=s-r}hD7^*(@7u8OMFrE((;E6cR0OjetLK5o*S@a!1~YZhGt)90qV#WZa0Chcy6GLma~2EmF`tV01C3InRY4AfGg`!(t!`9@
Z22>f)_~tQaYj6m==cq_n1|iaL~uif#0m%uT}SXQ(2d>k!jMpFn_dlV5Z-bMeB)e<@p%hw<{Rv?x_<-O@9%3Kt0%%$2K{^;=iXXcPGxh*@|Yqfr@jx;{^hGG
h@gj-*E0-3uTHokA1c!%!Y-_|d!VKmf#CB4NJSC-IXrPp(YK_O8*h{7qBz?W06<pjOjhXZsjr1ihq2Ej%*m%oCZe;L`{?#vbGx-)F``>8^U12=j~ffgt*dz0
wKsmUT2^;Wwu8M~g>N0mW7>}*$>ed^R=1&`F-2b|wCJ+v23#Mp7j#gFG|oXwv?yuackA_r-nwVXd|($GI)o+oE;wFHI0BvWTzdkW^}^|;;xU~^dwnVrIRzg+
Fg_~Jd^y-*m;mA4hn%W9S}i^2XgEU`b;%d;d*~G38WGPfQs)ydvn08Xsd>O_as8YZKRIe58nvk>QB!@9=eXXwYS`a8jJANho5945;_4998l=Oe!}=V`Huui;
-O?6X|J`#LyZ)V!4{*?wCgoINAC=JC4`RV8r|ga+LX{gn98GsW8wf|D9i`jA{rc%{g;8WbOy}u2URe3FJQK7&Nzo!^IG??Mm8u?NZE0S*&ics0sCM4FcX(|m
iE}k1f!0iTqUzIMeowq%o@Kzi{Sl0`uztaQwf(nwv(wzuc4sNf&PltoB}l9|oOM9BU2MwDY&uBGEVzT`<rSNpZP$u1yp3Xh=ye1i!T8bF1(n2Z9f~pqeXSLS
=9*!eFnc&A4M-KuwPZo4c!;x2;HM_u{opW9BxrMppG!BRQeD~Eee7+M*|Qdr2&(wuLYE)kvm!VT64mcPR>HU8?9OZn>!hjKMmhjsbh83}W23c9mH6!VNg4P?
7uAQoHBWQ~xxrq?eM%`hS1HqU()%A&wHrjNfCGo9EL<+eN!N9;BYf_VyhPNR{QQuS*#%aFM3-76Rm{O;hNYkeVQsQ;V4GBfGWSFP)N$Nc)D?bt2cA;zjv_|-
-k$KLW?gxQTCwqE#KX>sk<ZfRlSui2Z{uTXG~pxeby+-vM65s^gL4_p0W&w){#a`zw|m_<AS(a|HrDO!!?oxLnl_Ls6yqJXitivij!Ssvyki2dF2;)f06f@b
d>AL0c2Lr?=U@;k4=uF=OoP&dJPzS4_K4R7fa?{Y-vpeh$KT}hA2{Lffc@_zZivFu%lLm)JBZW{Ypv1YzbXiIyV9JdoInhC)1b@Bv%?7P?T%X!SAU$CODZo*
g=m$yxxHi}{Jnxb?M8{M_d7XOO$TiJI6o2r-$Tl=XMVvGnB!#})r#P~3n?qLY*r8N+ZLVwaf-h38;bG$Yor!YfQDnfzVk3!1SQ<mIAr*<l&Ux^%dm)GR>8Ai
mveaUl!7*rW^r1Xt6VVd5`&dePX?{Xip_k>DWQKgqZA0PWl6d#B`yZiV0mgS42lHFeD*iUrqJhN{X+J>G)@y^Q`^mIYOq!Eb4gH|Jd}jlgM_(3%rbn7M_y8H
pXPcZ$9=Zx)ubf$jnpsjXH4zZO8)n(mWatP=I~G;|9#$YH7KHCWnhfUu;y$GsD{H;vv%U8m`aROEA`^fx&v*<;Cs{bJ#+>V>pxM`z@ps>Imfwj2s^7lkotbc
Kfn`|5@MtbC*x0+)8~$i$*{_~b5AN#gZKMxYQh<c_e7)+h$>_M_H>+MVfc}7R4VSDs~34?X1?#sm;sT;rZS*DMTlH!a6ohz{o56;uV!vv0b1ge5vbXSTTti?
U4-T=^c6=OXHzGhg(H#4x)1i`>)(F<)@s&`J=QDxS=i*=Q=#u39Hqcj;R@Z%=<Zc8#WDH69e_Guu@A}(^)+qgBjk+ai}ej;6!RVU9#cd8(G;FDS8PzD&olT*
C{>R`tJ_kvr2;LfZ>kDe5~dXd-n1s}rf+r-)U09m-9y4j!6s9U(jWZ-37azhx-E|sVzw<-7+px%3}_)!=}|uMFsHcs8zP*D>V!+w8k*T<V*<}otjLj&X{})~
4t2Aw(_Lp=RV1>;oQSOD1gW`whXOgkDsDIBzC!I6bDZyhhHkTZNE0-eolwiMc1zOq9M7{`v^R(3z02_*wA?=N4j}w@n*>JjFi9LM5MoVWF7wWz5WIxL51!sm
kh!!S;9`T*Hw`xoEs8e6rsQWN^z<i>ENF1!7)fNY@!utO{C|_*)Qxdglhf+RxQb;Tzi=2D!cOmr;{u-0F1pNnY$<ew2jUY2mW1CrBigs~JXu3HBva~-P`2Ui
+LMo2JCRNY=PV6It-@X2qZqj|tS%Pt=6iEx`MVKZKz)9gg&XA&6XsO)JY!4JxymRc+@Qyd<l6)|GRwJ3{+wm-KsvU!>oV$6nOwwqs$>HB9)p*^h{Dy1ra{1B
(WoW+)yseuZt>(!px(e@6+&F>A#zN=5Hj!pqfpK-g9a$izTDd2;GXk}Q#7?n(4l5TtWb6?z1p!O{+L9k6+m~$#2HI+&mkr9tiLpKE6!V5jq`{s#TMe#?)01D
W|P+JCEjAeF6*bs{f8{~{?u??w%?m=XLIhs_-;_@dYr$t))Dc_YBo*eHh_iE2&qWrg26CGm3c{`Rd)mhzxWv?z>u4F%sTSj!E;`jh{28G(AwSyhl*D#Y<nN#
D#M2D;ua?#J9v0>^D?;U$kRyqns7t~RDX!UBC7&s#~Mfo!Pg!d+Q8;tSEOEO_Uz!}r8YF)b92;{$Fy2O6+06PGGAO)56E9*fo-PZlaq9I4UN1u5Z2g6#`{m$
pBG!#VypanjFiB@@<d&#m-gj<vC_kq3L!r2`8Z4<jBv}hcdpoaei<{1&Lxkh{GXl)R3Wh;Y>`ZR{K_<*ffs$XqEp_ypnxV_?QfuZuQh27|7XRIXY>gSK#a)!
YYm#%YR>{csh=za(SS^`PupQOys&|FkMjsW+)-@~_4+s$9xWg9PIDzj`l#%uC=4FSqFiV+zQHO~X*ozf?*?Q%MX50@&H)puTRBj99%Ni%pi*1n88J>E>4D~u
8GA_^hS7WK>=n@mWJ3wU8}OF(<y`M9WvH94)I{&Gja0D!DuTt48@Yk4p`Eg}Vj~s;efvcFb|<=ASHcSRc-1ShxtvNYhKIScQ(QE4dYC23o2uLc;%g~N9WNWl
xLn>L&=nV(kB`~+h0T31Jpd$-590=G&ohp#%SN20A6L6!^#%Ti>Qwb7YeeW-LP;WRBkd%>MC*t84^e{*Iu;>e(62fxB1WNcaA8o^f55myf|lPkFD=LDBl@PE
C35Q8^TnQ{5O1!&*qR>u`!s#(dj*T6)~D;e=2e>q+9ZA%%7W2{veRs2{TB+^-TFvW05rW`sEW&#Q^Kop*mq0ltvMuFJkl$?ltPq*9VSun)=WOA%}G75v=cdx
aWO!+ZiOn^n6CC69F%^L0P@fR2Pc{yE`&pswA)XgyzfZJyJaz~-X4Nl!iu=68r2SVCx%1c>L|;)-FK)x<LigqYNUX%(Q+TknHZs<`n4OmV!Js<t)OUG3o+t~
LJ`G_`h{7F0A-<^U~yYZl47t*{PfT6Q~BGG6+f1uF>df-RD|jh8j`EK@^j!4iVp|K*P5akhAw=kTVUa`a6E;NiQ*;_=E_&DwU52U6G|;OJHn-OGz#=;GF3O1
oG_k2T7`=>pEsqx43k;nzHxS9Z>aY#jpR-u61GAX%U#7^(g&t21>{R(QL(CY*Mror{j|2u*o$90qyUpJ3Sr?%{1&UE|4m8ptFka>WLvge2+_Q-vj$>|l%(3N
_nE|GE4aW+TWXf_;NZGZKCac@GWPvVW$LjL;v%GLaQr*hvWvVinBvqhkn#625ES+{scE`5=f{u=i^6RUiu5gzD$g(O3gME1Fv6=ooMuxuV!nl0FCsu6^FfN7
Uy0bk)edg17~RS(?XM$v;z}*9gcWcA+(b!OJa|m`UQ`#F$nLJ}af3XLIgk|mRjY7^L&&|}9G^sw<JSA<tHcx(cFPbdQ!%^HX@9UHn;c}AL@XA997gvWX2MC+
PI*hK^BYJVtSdO&iw`u0v+bt}QDp~eg9PiCLJvquQ44uL9Bfn;(7N0?5gHynT&S!0uy&Wnz@_fx)2y8XHIDc(D!L0kyLY^msG1~95|H(r$eg_FQqWs45gAT!
;zH3Lz^C@y8VqNc7V^c@;{{>I$G0LnN2?zKK_N>W>+;cNpq9#3L`#KlKNvoZT>FU+CS|&5U5u@L%86X)&1M|`=qO*R2)yO#p!GYCOYk(a3HEFKy-v)`J^Nwh
-G^=*XuEb_cS6Uv{=6h8nFM7(0a8$&PRquOl|TrRo;yW5QS2U_e17Cbbm7;5+@)ZJSceM059c8xsg^+%<4t21gR+RpziP9L6$I+0`J<z>-h;GMe?{skA307i
mzT-J{Et8#GgVff*^eMthK}MDf&p%ONn$M5%t?pC6cDSViQW`KkqQj?qE&czl+W+n)8JfFJmrw7;j@*2%kw~!?_XPi(#5YB1$(3j-{>)CIH#;$oMiL!)IQLY
wWVPw^KLey!B2lwv{Y6S`4W77%WHQaSgpS7o2}o;QTx0_JI&F697Z{VRorGue;h_FN)|6@hBp2A4-NX-XAzS6jTV{(k|8EBUv+m)Z>MKY7fNJkxRo|7Ds=|T
7^Z~4iI(arY89GUj->8R*NV9NIZGD8weNUYRfPuQY!In<5x{WGQMm&lshk`))lr(Z!w=uv!O+LFu5&>t$KbqfhIXxIh~|i8lbzKCWw}~Kyw@o25LNeSXEgw>
sGV~(AJWP6urY=xlWEP45<la+UaPo`q<VnKi%us0;Pu#xJtOmkDX>U->8{o6f!oqpRAu1S4GIF8`TUEZHzm}e@>`WH+2;zEoGjt4Iyydjj#kV@bd?~_$<JTe
j6jy_wsO`PTJhEAk$8b%UO>Tpc4}R~i#bd>Q}-cXIhdZ05}us$qOW0&*0J-a-u>GxEuSiaz|H~|3yxK&km7dj)HQ;30s``kAy%)Ou%#LolV7D){oRnmR{~Xf
tjKd1dapZYAOfhCEqA}W(lVJZFZYZtS!-jatsZY9r~f0XZ1$F%R15c!=}h-5!h*untR#~L12Z|Z_4VvVW?%B_Ay0bpf#P8Q!;ef5MWW^q?H$jFRd9tluX#-~
l7}I=dz<ooSvDX@UPk}hV<1Cl>O9|DChD;i(5p#VaNR_v^@9nc2T>CQU$)ck<h6pT7}#!n78lM%%11s=K~=3wHhxOj(C2oD=w&4k<y7P2nk3a*k3ZYFbVZHC
yEBI?8I5;)+@@nvcXc>eIyW9Q9h&-?;alGG1#ZT(D4#!tQjmt%2X%c$lOABvMRc6LfW(Q_z}oz}U*G+{{Hh`TkNdsPNF;=lRE6Z>L`jWM=L_?j1X`WFNaD5J
d6#ADQ+wO!ScTO#+7P1h4Lx-j1VU+b|D=;zR!9|%x06CT1GBkk=0KMWbx;p3ZAG$T@C>3=qp)}Xhtq7&j>GG-Io*+QV+2=qo`$`3nKsOW_5ac_2)zOu99J)6
xIg>Ggx^ZuPX6&qbmT}wq@>GTJ$$N=&meS^$UU+;Fs<X9iPh@}-Qd|%4i(OzKSlMADwvfV8Ie&uI9`d%6fM$oym<+vkz~rafU@4DKHa8=?jj2WqO`$a2oO^D
cf2OgK-oZ#7)w>6Gpho=lO48yBPD|f?h+7H%8n@FZj-OI6D&38_?6A8<?j9jVDjTrBqv9uA!Ng&J<siQxO4+{4K=+Di~Jzu)pj(vjuX5{c_3YP0j}c2cJ4}U
vZ@R%rrL1S@5oqSrvQb)jn|5m{s~d_)!Fn@B(TqN+0J)-vpE&co{Y+&KO?Sn>30S49M4%slofKcrNfmKkw`$kl%onE4O^Kva0;ci+jw}Al(>Mof^c^yxgbon
BP$7HVJJlisI}p#E~P76*$GAY1J-lt;uVCB4ZJHq(kvWSy~8ni|I|@1{*M0Uvk(h?$@*~kMC)Z1_V?VG@6>RC8R^fK9>r?{Hcs~2kxR+KCd<TbxMrZ5U0;lH
9oo<?!=0TD@CI(v>2_6UA%y^}XdRZ%ofya&)wZPE<av;@p}*8`h5z#NIGNR6M4bQJh9^SaPDTz~Jl#G!%AN@W<Cjq%u;N<MqNk}+9TFZYcx!x+QF3hUen4UK
o{nTsw_QdG52E+j9bXb)J2rB%{v_n%#bK29COuG3gI|Jj)k>k2trOV}ZVyM~e2gH1(vN=YjBG?v*$3x;Zq0&H^{)2}nj58sQCRB8QsmbzFlXCd39%@?T@-FN
7^b!g+|X2(IfXuW{CUGNfdy-ZgI+ux`p#FZ*yaUshAnILjG*o-nw*qnL~ABJxINbainxS|+o|Nuwo`v>DsEe30=Zp2P!?LHQCoujB8Rq^Ki7xze}=bUS~pxa
UOc40zEu0=;x+yLB^?}W73h!v@1k}iYb1ok&|U?^&NBJJ@#ea_L<^Ocp<}%!RP9=7C8vD``GV(77Yxyu7(RlPk%X$~{(;kTxK)aIM*Zn~H423_$%0FcCZW<#
ETBO=3k(aPJ@+4Oe(hH8rtuEFI5RPa`8x{(hA<f24c<C`-XNcJ?P}5C;rIJVwH({FJf|+-jd*qYlSYMU-!tNOFgRY5l-01YHhO;#$-q)O#@~#T#M~|g$MbN^
0<ER>{B-8yL@B=s8f^w6kda)%P{^EDi(pLk{a`0owfSHfn<W}VMjB<GU%_EpL|SiizL6@r$CeR+<y#9;e`l2x+eC=I+%^W=kNxF8t;Hwg^<T%HrMd@Xf(!iv
H^gmp3q6&}N~n5G!I6I<UVSZ^x;ImRIQ{V?zN}c9gB0MBWU3y<Gw@?|%id`iD|G@05+K7yS;tzVJ-25D7x*Inxa(a?KbMI7@r#C<1Ma1cpAEj@(0hz#OR>Ea
NC4#d7VTum((u^6v5BeSk&W>`(VVoiz)hk9bPD(UjfjPJO8hP{u8|GexvQrR7&ooxFQG&U0)562{Jx#S?e#qaV}G1~r4YiV9pV$!ip!?!y#6fa*AIjWm>t}o
EcWSz_5$2??-K_{{2HzF1)0>>3Ci;2V;h!uU1Nj|zY5+*BK8!@9`0V0T=jopCay_L5-vN#Ku(UM!6_B(6FK%IKSG5yb>umN?2)29yTOS+`01l11t{_Hi>Ir?
h6<5waPAB_{8AhMZDT$qvIqS|<4nSGzWv)98QB$UacDvN%lee)(O#GL%1r6-taO=Gu?Rp+7nq{~qevy0EuWawoaxI`OmO@3czx0s4X*L<`jka-VO-AwZ5>o)
(jn9%oB3H4PSn-)i%)f!FJ1y7Ak-g6(xfKP^guR+SmjW}f$4)cU~N?{z<eR5VoHgu<4VmP=u#1TTVawzo6;+hp+Jpu+ZO#GXEcPlDTID#vSvk=e-Rsq*^Go!
uxj91Z{MSGvLrMz=Y{p+sG^X_3YcRmT^Of3+bz{?-U|L8opApAA~ilrkj^^&imD8wwu!!YSG3*5WFcKDvg)O-2M;T6cAp7>3((NAH?k0QbL3;4EXHipe!){P
m2|rBCSq2^)Au8s>YLFmLN;O8gp%fk8y2-t0z{&-d*pszUcp%2ZvgJlGH-*wf9f-TM<1oV4$D17O~fAjzMy?B<SAE7TwXPCBu56ic$cyX)1l&JDq8CMJ@SwX
21PUO{@)8}omGneCFr?sbOx?G3Hy(M6R)tLYJaGUN^$6cO{*2_NivbUNl^$tAtT+P2^}Pq-iTW&6M$<98hxNmq_>z;6TEQw^?hzk?Z$Z_^<4h`>`Rz7{2c9w
BugteEB?9g5TD!Ae>{NvE3lP$R>2a(-@jh3$|VUis(QPFH%XOR=zXJg+v}9p*qK(>X%BcbAomVt7Z*$h`qGLm#HShgEOg=jfX3W3-oQ&;Cn)?agJ}vek6?Yn
aj7_jES}UZiCTBSCIQRK$O*vSFWtNLWG00)CE#CsJXw0kvT-?s5w$mZa?)O8Tr@+OfcQe37|mhv%E${1pMEa3lE~PmXGR&qp*23J>=8DuZiIWo{aLH`X|Oyy
PMM0FaZ&Rm05*73(7y~UP6xfRfKZ;NS~RsQw}GcGn>;}snrvvm4lP|dAN;;lty%r9$P8UIH##DvT=+egTl89d)3M68p(;5CpeZC$@DN?(VO;!0@omvMAq!(v
XN9i*G+G-B07bs)PJ8ET;BWxw-4e&0Q;i1Vw%wr|3yr?O6+3Q}2$~$$^r3kOib0ppG>~DfWSn+^Zh*T*BKkd!X`lOGMUapL|L88Ws=-0cW?>CMrGN^>zj{`6
0nPQ|FH=%%&!5tO+LUUt3FT)(bdDq+ht%eOriPEd{5;HJ*o@Mexcj=vP>5EH2M7<+#<L>|9kWnN0R(KbUceU_ly)=p7ALEr919lBOLFq8fPf6mCQiiFBL?Q*
MYw5m7VQ~_G1tx2`L(ye?+hdKBXzj5{)jQdgB9$BJVSTrfU^85qc6<xf}I8QCI&ch){THvzGx9qTql@Y&oAy*?u2+YJw#0@JB@idL8cE#s0TIn4Ot4N<O$(4
BBGhi%U{c45=(ObC(|OTQnUIkRwxTRF>wV7R(U3B0+JW$pty)jpv-Y`%si0el$OZmE?59a%7$DbUO+l?wPO8Jb<V_8;+O`HIVITeKK7p8GlWQ#RH}RwZt}MZ
GK2fOSI;lTrWww|tEOhgiFu1tVP$5{RS|WBK;ma;u4$_6HUWwJ$CE@5rwc%{+o<sX3ZLJvpX(>SQWVV5s)beEpp<6v<^42J%OOVq-kmqVW^4iGdFz|-QGRdR
!Od$9m`{LnX^r6WBMh_2N*s&t($4nlJtgStn~`37;`5c-A!rPwy~w?_*Ft<V!GW&Ug^GQ1Af~j1ZZIIjGG)w~Q_@b=##mBm97@nnECnNT4cN#L(b4^mgzMO*
mKSaN&m{uA|2EAeXQ^&yN~8Eyp>4x9#AzP}a7w8y1~DQu*PnG{9!n#+P_v7H>-w8qHA#A`t>%DKVN&+)+op!XL)Jh7_g1UGh=0tV60jYU!k(v4`=9GGFVnzN
MTw7KD6I<Z9a{3oJ%b_<r}wZcf`)##1OGrRI%c>TK%}&7gX6UvYB<g8xx@h}-sD>~@D0!GPv78o3u2py?U%r1<XOHa#A<*krBH;vpt6CgE69OO3j?+i>t0=$
KQB&~!W+Jkm{NV+uewtN7i4#CVcE7e6F`!)>4~6C<YKxDJmBv@JfuWUV|-L1+m@k3xKwCt3}+!8Sf1@=^h`<_F}atx;}CE+%H7!oRVg^`9Cs1Yt}z`ZiTx7C
|BI)e5m{xl={e|VBPc3`jo7e#FbAus^LL1CA7Wppi4U_)uC95y!ClO}jGE;G9B}%|zscSiw~lWtJ$5$~0)8puZ=}2YWRuD*Cg!(Uot*4(9sdU8Z1+=O=Ppx(
bQ5RCZI)XGh}n)V6cNdBPdbPF5X$=Afoit%6E)N2L3rk?NV%mY(0NdWtcR}Yh5Su550jb`mcy`CMm@fAZG0;|R}~2%=lqv|S4xQ@Z*Np&yQ}})V4<6-B6Q=F
OT^+EAZwuvh~$MH6Y;672aI}MF-~H;l_!*q9@ZGOqha>Y&Way9XWdj9%Q=r^RfXo`7W^n~L(#Jl<w<`FN(Q%u#cDyQMQO9#MlH5oCrz4tFisr(i5oo4aYj4t
hsmwQALj8=l?2M_4lpAqN;0yiCUjj5LKBEs&|qSCbm{Hg$3|M!a#K#548z%eEWS|Si1eP9gV-YyTpibecTl0kg4z1G72lB<Ydr&B0o;=wSl5I2%|Q<watsgC
s9(4#9r;i@p0mvC8zd2E;c11{)x(kXXk==QR~8m$`&l|i##!rFKC;|xxiM0tM9bA}0D_p5cjPWx&>{rHq=5{4Ra~t(UIdjCw;vs6V6cvyz>uzP=`zorv^m<%
NS<mCQp=30e`Ln{bf7;NP+UW#pIE-o8cy0MzmoxbJ+Xp8;{XS1phpedMF&#$TYey1f3|_(aXviunX1_4KZolWAp&D?SlDWcCt5$-*2fRR5sKVamV8xfZ#Kv6
wvz{8a~`1m-KvOiaCe`lr`zB+4d0-2uSE2mr1xE#hBwE-4hZR|7GZp71F+JLp^2!VDHZDA<h+X!MYa4nbOz{<ICjUOd#V|f&*F#Q9@fvy8(yOp=(>m^>rZ1A
R#GtB<&ep6*Ln^$js$Lnm-AIMfSK2TTnbyNnWfFD905aiWQu+TB6gjBWRSOJAx18i6+SNvZGT<JxvSE^Kop&=F=pikrMhi3C*f!imo<CYsYMiN+IPhc*ew0@
XS{|070p#T;CytM)`FVbQC~;6G^9=pjzWdbzmbMEjOXpVY1?xw{WBa8IG;;?wEr<tUBKf()TQYcYXN0&cQj&IeE`_!L5^5Cf^t<L;mgTk_w#169Fv>p)BC)v
4)$9bDr;BA#4r*WMuh2`C1D)(XCD%RZ|i}z^qwf43L3}~rL@`2GX0C@c9>sy%stJqX!%dVb=n;ikwO7U#doF+eO<j0ic<-(LqwirryDOI0gSx!yt45U0JmMp
@<nJK@&g^MP%h4i)bxn`r6<}3LQO6;akWOSxnIkk!^wPqFI_Ev7!{{ynLHf(d7hZx4)(X3c;OX77lQ*HO1gb*OVrnJE+V`434h&^YdTGvV7oV+y>ywhE#ClC
`eKk{X-Xw-GI{#Hg0*azhh<CqjE)Q+%kijPa|(lFMzIqBNN`{(9j;{BAL;3jOQ4XoZ2q4rLP5LsJepr6;VEY6dld#sP$x~T`g43*JK#n!LL#DeCQsYVlFtJ$
L$wE4wP38{L^aA@rQzk<0VQ748-XnyZXI)aNUISA*60${o;VZ~?(9u5>4kl>Oz_oi*xn1L^%u>B;Y<29Q1U@ZkO1MBf0UMc^`z{AI-YDq!Tn*H4Sb|gC$>&N
DIJ)VV;5ZU?gwu`;@!g?Uqbkz7GhIMXN4F*TOGt|Mr;v*n5ExCd{)v#;IQ8&e1XC>Et`Hcb;x(Wt;({l2#84-9?;wD$k*gBLg>`RO2t?+EJGT3@hbJpuWSz0
o0X4f+3D~<Uz<1D!8`#0IxzTNl=;yMsFhWO!_DOEbz9J|Or{dWofY=Ph%CRDj@gLdhh$zRUs4pU-e^1y&K%fS()OW67j8LGB1?O*(>JQ@NLQU{kd6TThkH~w
c3(FZq%o&o@9rT`|0Nno3&1h|zn!8M@2aEgRpLHs3me*1%W`~M>B=}=0G8dzt9aiBo)RtQ^IG@j&)P?+Q8C6Q*oOYIJ!sBY@)lK7q!;MY>lq0wKQ{j8l24<&
-~+bnu>Dm=XvM|J5Tn&G*0>d!^B2)!H;T5SaYNl4;x?vzKTM?|8TiWvk4!1QF|Snh;U%Vv%6?MJULQMe1^?YEDFub^{ieA#(}Bhb6?%zM6YpYyx#VAEpiq4^
GswOeJz~*Aw2>ZY9}{^66;p6$^q}7TWu7@YgTVzHjOAsce4uA<e>WhvXlN43KxvDptjvmOES}PvTGAGs_#IiW7qjF~0>t~(xP9%J$S+R1Bu5NV+qQmrjj10o
&N!CIGtbhY=^s#<R5}Q`MNyGXu%~ri#@)CmWG6qB<j|25qldMr!(fi^TKO#rpVSLljl>*ljBN|^h`>zbLtxo{z8W3}@SEl}Fag&&2q;)=GfC;%*z^RXCLDWh
i(})E-=u8D(=jjk>J4H-zHN)MHxH6uE-J{hx86v%y-<V}>B`0~u7bs%i7#fW4^<pjL0lT`oG?|50k(?I?AOvkdqTsb<0rFA04#TgM%&ya-J&Y-&WK=zV1Z4H
w)W#4nWBoXa!w1mf&O^pRBIQWIVtXiT+L<26SdmU>zav4=W)pT&jxTe{b-u1kxahOmC$YFLg)&h|D+__`U(~2+2L@nhWHkx*2#6IJyh3#1s*+<iHo@MDX3X4
N}%+s)x{Dm5&HwX2nVCI34kamb%p`qd3s+zfRHZQB|ZFXU`apXH+<o`#nbtB#VDJexI>~uI8T4|m;`CrUpH`=2dmLwOypOw3Zv~_e!eb(0b(~mQ4q1@)P^GU
v$ls_7`2QStVY0zRe*^2CTI&9c%-0w@|}yqsD8EaPZQ~;qX-llLlDDea6S~#3LPHcZeAj9&%QOU_n<%H?dj0QoJw5LGZh*}ggZzYZ6n~UdQ6zddJ{6B{&NQP
f+1brsz7k<T4zAGoFdxzoTVy!Lt|~p+HsuE;lR6&Po!{u0uz#TL%M;g#EH(J&Q_5|(DKH;^zuqiK|Hx3{Ix)epX`6LWjcvZF2+-LBGrVTy7IdxH7SwlAAdV$
e%YhT0pzthy8r0HX5=q7QhSU4&+@a6%HuRWK7b$Bw7)j1q^Vr?-6Bz)+55)2_&|=?<8N4;eA0G$ABj(v6PG^VAA2V#N$K!Su#ozmQhbl|1N{%t#ZY{U9*!Lq
fjSLYWYZZgOdv74S<G!TS|1}$+s!soIvj&y0RXk|@YsZXES(et&N&M>{a`9BL!OZN8?MA*K9krS$Yd3%t;#sl<6_8WQ?}TtO*!#Fn91j&sC1FI3JP*^xP-$N
pZkyY@U1tAN6&N4`MRba)I2?MC5mqhz~7R%f`H8z8=^A};Yh6>Oq%(Q0%vE<J6(=3Ys`}4X@za}v@m3QjJ_PwL$tL(gmF=M@pN~uW!-?c(c^pFVHXOE7Aj=9
Tid7TEH!r-b@(?H?y<o4N9Wqv8wO=+n%0H0-Jf>L<iJ;JLhrONG!&@Db+X3Ciwo=&{pnChH_5XLhgqN}_d9GD2E?JNcnQzVSfNnEeM~L&d%gJbywF}axEQ0o
JHaCX&AQLXyFu<IySZ_UxQ<+KR-tl3GbXI%6C<1@M*_`>4aHw32W1`oxJQJ!4<%D5045@)Bt7}VtIVYur*P5lCjr;nqO0kR4n7dd6hcDwd2AKzUo6rJy+9%i
%W{_1l^3;NIsQQZ*JZAFU-tHBi7V7YaKR@jVHenRUAy3aYd_d7PZq_!w55rl4uU%^J3JG;++drki(;GIQ$h)_R0r(!z(kb@0C=^q9MjICfi&Yy($V2~l#lrF
RRxNozOy`E|1562CtDRxGQ^JB4*}Dr8jhAa5dpstN3?L|KNY9jNNCW;Bm`0oXP{oE_TI5jSNIthZBru6FzlH`mWevP3_I;&=PB&u;TwwyGXZi&V`>;y9YEGC
DvUVOhlj9p9KeB=B(3$_qo8XVA@8zTya31gXQYd-Ts6RXQv=p-^>k36{PRuS-L4q;-YjwVLeP5hd1YAxgjm&;8$!3Iwv99!j!r0~ugxfL6Z_Wz>B2iL|0{11
kQF|cE}ApI118D06)XNw$H8a7x@OK~6CdW0p5@$S6d5#ywSESoPwbs?nPI`2lS=vKJCOB|b2HBLtDP*d(9MH|6&X*+D|^h-<!?&mzox?tb||xNg~`KdA=FS|
lNVzu_^2CReS}S~(2BQeVxRPAXC?upmtY);*uz_OgGX^uFRjM)apgxh7Z9hd_7ja|vtVx0h64CLNk6>TmV7|~#GefFLLuaCsS`|xR;c)&nPjHKR-9`Lv5W-A
jQhd35{Asf^7|~Ur0YH#HhAEv)hv$awV57UDtY3%$5Wn~*w{lsrAXJ)irIEj&p3^W=ePd(U;;Ij#M;`bctjE3s6#0RY9%vIfO4IlwiL5_T0>t?zQ?!xH>io0
u~shQN9axL0StUPhy?JA8p$1ngmC+<<-!s3Zy4WAyD;~c?`a27PDirva<6ki8s6vgr&?4fVNxB+qpL#QZ05QAt}x6jtTbx|;g-8Y*OrGl1h#>xFXHs`M-Q6W
5#)TDgQKTQ$V<?$$~NQN&U`$?2>JHTWwIAB<B-_=6*wu#@4IAmop?{I;BPzvhlVs6$3{>y@F^sGK_W&y|3n)#YmSIAOB5nwZ)~mf|3{Ao+^tj8yx}XS`X++8
)n_*V`w}THBS#HM%Yi)p<m2WXyMADU{u<*JgTV|?NJ>B0qtnL3!ZQMv(R}STHggHVr&}C=lQDxCf+qN9{*XXj6SStj8;ket%*&Ng&>&#pj3cJ%FRD0U-K;-o
=3}MOMr`87tdbCns;d(zEuIY%alhD&?}7g}hAxkkP%0^|Sfjb9o4-%&LJsTB=4gU-os8=XNN(;|1=A9C*7yrw@nu<*#=fUJ;|w*15xSTQozlxIhCDhM%OpO>
6%)nBxCkH`|8-anQDq*JoqA@xyMPTzD^=p7>}bhJD=&p-ql@oQ<u@Ub=4`_g2E-R|@o0OKT1+}%I3fHq_a}bu09JyT2KhfzH&%wbS=`gnp;=kV0TwOz_x7#?
c;V-3+xm!E6ML)Qs4UG)%WtUQ=iLX-6hwp#Q$Bq}Xj5HK_oIbZ4ZIgILiuQ8)T#I_Mf46Kp3`zh{&v845I~MB7n(2^UEj4duVuQEC)S>9g=>puLejAa7ft_J
Q!G`<Hny`^sF;57qe7;Dg39tfsnI3N<#9fUD&MQL(UZ*hc>FXh{Z~UMyg6jt(>U?yNEqlrFX{qFcNqJXw1JKjZPm>S2XnFQJFJ(JAe>5#8=+CLh76Ql?Y%Ym
3*Jh_KZkLRG0Hw4Mj{d>X?xnazTjb1Qq^;h136fejT|w-Ug|5(;|KXv2AV2Yx`a)lLV#RAt$l^wo0JBFeoZ3PVZn7`H-HeCBxZY7X3kL`HU1HH<znb;10RvT
q)n`iJOX#2=$hw&d-aEHLNn2-B!YYg@^5BV!0+51l;q(>my_%9XK*&S?Ykmy9x>#fSt0_N@X+(}2!ZhgS^mGa4C-(nET7GEl`*fU@YfyG!X(5ASuP$~sD-lr
eFo59@t1iITfu-@`IkRg{<8&`j@PkJi-V6}9srFUKjVy|g}T@cfWm|;-tR0mYTpe6?@Lh8b5928Z-`B;MPq7gEdybTeHPTd7E{O)ebJ@+(sA5{{KbR5vk&>T
6q#SnF>fT`=FNVVo9SR9BcWM<=O6}U=A(|-NLvOlb@W4J^#_CD2urVHSbA-mL3J*M?5gp3@U$Ic#=(VsgAeA!jUIhChJX#uj6^_^fQqS^1EFICC(vU2>_lI>
rqN{Z%DRh1Zf<T$RCj>KY(l4J9@l+4=U87UF<&+?py3f>`Dml|*A*Aq6P2HzDo^@F11>~BuTvS;vnNKQWSjURLU(XT#z;U<qNTy?OpTtn|I_RJyGh!;47#zq
VO+LNp-c{Z76OPiLm;Z0semhEAn(IIOc7vsR@S<?`}}CX&MZ0QL;sib(KH91<S&j6qN^>33-nK~(jVyZAA^?5wL``%XCh|N*IDi`pR7ooR79%!b-C8?*AkeT
ToLd2jy!@VUd!N}HnEaTn;#N$z~Yq@bnr@fvG}vX%vrkg+G-wh#@)Fj?&XCF9Nl`ePMl2o<G+X%pJv_eNGut9`$xvlmuB4dNvX+-Ivm3R<b3H()-7}>7Zl}z
A~^Z%sC@pZjgac`7>6Q%Hi=FJjjyX$jT?_BvL2G?-U6XRFld@SS!d^nfZf`uQu0cm{UpUvK7KpJv|L<#1pM`h*%W`tT26Mvzts$ZRc}5os(Y<&N;y+zbUnI|
|GUU1x@2kwnrN@cVq-xAs%XZ?4~DZB^-jKvmLUC>FLidr0@Yf|gNv`Kj|>K(a>)|@iNfcN(vh5Omprr=k5?%}()BRpk~zDY`z$RvEMM7PRm$oBJqmiE3q>kS
dRx1Lgb_H~d0pl%M6#oFxN6mz>@oKrj{Hy;?>`tD7Q^PkCK*;?1yezEw^J&}D_XHQKEXb_X>QjD)G1^mf^Wp!C4$zs)v3<pyW@5sleOc2ofF{2Vp}3Gcmd74
D*nEERh-z^?REV1G86kndk!Ki5+`gR>bAZ~a0X~m^<C#^P*Mh)55#MI3WqkQ-wb_m*+0`_$Q2~OR`LoOA0s1HliDj`g(x%v*%+k5jPxO})w4>+Kz^=7VE)wM
sygS`7H6}_#oOdRvyYM&NYXr-@=dn?9b$yrFTcWc)zAeqei=jy5NYt2GmdGYhaN6tc<=&F0InBx_kBLVV-@qL<3cc;pCcUZvj3PBKaCf1t&D1<X|6mm?qc2u
;gk~F;=Y%-8Y&2_tYg2u=7q2<$J-2O_?M&+TOn0@A#NI=ZG%#Wjks>{&%ptHauqbUUSr}~%E#B7GkdHDbi#a;mD@5Fn&hTqXiBQd7+1;Vnq%?_bn=3IV}T=f
JNK9`il-K31A3&=Fb!;4?5;ae7S&1(*p@%*Nr$AYGRDqPIl6t+FmJj@T%4=NKs7<&PHd*pV|sPln0zWCCx1u-I1)^qnHtpbskNL4F<-jz<2M(;S?r8)Zh7EE
lsJ-p#!IxBHAtF)l}Y|Ysy&#*7%_}(r8GPhPs8n-as!$UfcORt5l_QTM88yZF|u^M;Il@Rr@LOsc@%G>eKGf(;p524Q#)U(aHu(sotOa6k8agZrDfV1a&bh@
2nB0l_WaTwoMP=UF+RW(ra34Wm;^i0=+_F;25kaKNQ!A4vt#q|`!1l7WN>AH)%oCGZhX;QDQa?np>%*DzrQ}7B_ZKM>;GngwX~^4z8#|X56FwA(|kNqaC^-b
gqCcGBUe5ipZcg?`Xz$<6Lf!pA8}RT-K9}-hg^{?n8Ck=3%vsKo(x07j~=}B&D2gZR+ruK-6%uDvI^RgMs@7}DhE8jH=@)!BLT|-3tE}Fxb%*^{Kg0iO;sgF
Rwez&>~b!5F`!dI_MG$cT9u6al=KNwFM8DxMcN(VMjW4rw(pyD_m!BDuQ~GGS-<1_D;o7;aIzEFbGW))=>_@jnP8T-d!3W;;}!YzRnbCCz0`K}_dWKWXMX-5
^|Q}NC)LW>$-W+61U-1wr_?T31^JpfFro6j7WC;%!+7ta8T#ZXy}}19>ok21qVa*km=LYYH9lD5*ym3V#guvVn~lmVIO~3x++ILxRM=YetZ<Zc^MY!vP)SRU
CjYWyOQ3zjx8VJhc^h3f)P1#|Y{Sy<I#~=T7Wi+AK<HD~C79{CV<iTBs?rnpO93(xP&%rl^v3ba8H(ky!XzJ4z<Ck>Se+{Gwgv;IvViyGFfax42_C>E3m}-c
!IL2w1aJBh#+IyvFfE+y$-oi04IPBFPCZ#t-Z`wTt18SskvUi9<O+n=JPg&d{)M(twV?OJ(kz5zKKMo+*!b?3UKw7Hw9#YY*kOT38!rcTtP1cb6auVGX&6Wg
zr6!N9FKhvDZ=&QaB8azykA<rahr)uFh|kIWK+JyZw_(ZF+iYMtYUB=Z{{`O7%vW9Wx&JB%-!mK!2ZDeC30`V2Bf#4WUSi)q48L01OSLy=V-O9T2tgs+R9Ij
qvT7|H8R_lm0~yPW44CsFcPn&ch8MMxYtpx4E4}45yfD=d91ip$BQ7!Jal4_GM-N?fn>b!PbR+hTEM9saEVwN374GDuP^Vi4Cs7}Vsth<p$4<e+06el7bXkT
0|Qi#7#(lAOFeun6<*PeigwmcF@4{pfPB+>6Iyd;wWJP`Kq>sgQc@{V0h3!PLG?@_LZ$axOmAkilvhPjfl(L+*M_NAbI4lI;WJkPI**-LdE+<Ly;1Jxu!Q|S
iJmy!BWFM021PE+!=3aGyx;o@FO2Bj4P~_)!#SxQ+!7Z3S7W4yg-8?-c?j*@n|E3yV+H$l`vD%7CKun6|G@9S^&tQgb;%I`mkgQxOx%8Q(6WgaTDcaM2d-nB
S;LL5bbH}O?1FT4Y_sKiogjgFVWbKt3DcT#q+F8(!wQ&~b!3~&0|7n9yCR#_G!uSw8jA3(oD197`o~JJp}B~okE8Di?+TvESyrt4bOxxFDg8D$#7=x_t&r=-
0}~)u7fdY53wJk42nfjoN<EWmAdiylX`LNpJJzXJZq8;GHm}C_tQI|(HT^m}J(E_@AgXGpMB=jr6H;Y(=P3IOJtkFc`2p1QGS|wdk@>~6Swn%SHCT;01Svus
&JzTnyt%{e>QA96T>}-mTL(Qk1%Aol!kcyYA-;a!=46vyM~>~e^xr97Sq8{Oixyfd-muRUs*H!SMUozth|x$fFV7EuW@;OR`vqRv5wAItoj=VV0($H!S7+p}
LBKDHTV1~WJfd-?p&aluP$ZR=-5<Q!_CC*IjqaN9Z2i9X|4cIw<z#_mFeH#8G=d(`FRSvCzZ$6n#<sk~{tw(d+m*z;#$YEE;iT;X2mD~$$rh88mCl>5YTmzi
i+M<pBTllwN=(Kbv&J9L3JY_#By6P8XSOqU3O7&k!E)LN3BA83QlKJx$u~|KRiDfk2YNgk=fvKPfzt!df|*-AjS2U6>lp+Bj6oOE*4U|dnR1lnd(`>)D+2Mh
NXORk<P>3sd!#vO!+yX-ah5t^f!$}jhsd4c%mce}wY{_eCWn9t(Mj(oyswpX*#Sw4ttU)$P=TQHhRkIYx0B+k@=`feV`84R#sD~sLqzj15KOrgNEXh9e9@cS
pFuRWD#n`$ctZmF4diZc{DfH|Z%bq)KxhB+M`-;JuN1%jtfB8kkEe6Ejp|(G%33)WHQ7k4w2+|3u8dEhRxKOPnd$<4Xo`XhkXW%O;sX6}<ySdCAJK5~HCW#&
mBEaKs|Un5Wb#72%TOFJCD`Pm_r2wcUjLW|;A7H`C!F(DK$P9jWLF+UQh88(HOO@vwDy%&cazg6o4G>g-S))ksrk!Zfcv<$=He~0tdAFlvoR};HQtAsi1cW;
AUfJ^a-s%3lF%OrRNkc#_VBd7^re~LJyt~4y->UquN8<Z%?-j9!?rClb2oS|>i}$qtInwB15f%FH+JkWoV`34Y%J-=bT`rSYfeRzDenr^Gegt;thfaD)fP4s
8%DeBbY-JPq|2BN3%7UR^;XL07a$T0s)sQ71tE^;z4gI@W@FCV)|}#_XFE-@2UU?qKTe*I3QaM&wXcsuuCxQqUPK@eJCvE6SiqLF?OmgIQoR_w4reH+eep{E
qIf{SF4)UQvfr;PS3!}3gK=Og*vXb7LI6MQPSM*hrx22hb831_(uZ^JyZ$DWNMA*$dBGlEu+rsyAh<;1#fO{|8otyhioJhN!Eo_Zw`Ir#7yegARN0+f3T7ss
c)kn8U_%R|FC1K;+)5nj@7Q3_DX^+UVxmC^#B|I?A{`oQT>(3xV2>yEyNc9~kyv}bnCnyhbu~I*466CQb~ap&6S$I^UHc>r(7(m5Ez+^cJx7_OLb5esd{hyB
>4g*S2<w0&II!bP;dh+QCC!O2?3_rK0S3uv2447(T?JszXUe&Cb^R}IPh~IIUiX{EbD-!m@B!7>dhmh>T*0wxNp&X+V93uOf!2ce|5<)0e!n+PJ@KPAWwT>E
M%+!@Xu!x?%oa1DnH`K|QliNyoCz(O`M3NWWbA!bl*k+^4k$6}VigcM5~oIew9MUo<BGJ|qo2ea>pAVuUlN3%0>eJ&=*Zy0$*;-U-}Lxw8UiJfk0{a1c|c%v
zI9!UBrslPD4+)~_hSUzC`O@fJ?ai`oOhkllsNrq^R!2Kr|DcT$Wzc%xuv=t`iw`(5ufHB5_8caJYdraW}!RbEMH$u;d%o)3lY1Ug1~SYH@M49x+acg0dUxQ
bz^$ocRBTKK!-GrMF)F^%}WfM6rf|lB9-XYCC>C3oQPSjyZP|*+{8nsr<|UL)5J1$*8!ofTz>MJf8@H)wzq{ljU4uk$MQh6#H^i6ufJWIP&Y?xK|PI1u!l;t
wcvK|xu|$XJX|a&W-7uIc=N%^4Lff{a^E+quuCB#7AUEdTMf&_4Xpsfhs|N@5S?ogM*R$?7OlO}WuE4wM$p+h?-v;BYq!Wxr)Pc#<6ydz5J129Yv0r2tw~sp
VCCP7=!H-l)EY@?IjlAuPX?xhi~CQu6o3cB^xd8D`GB`HEN5AhTN)?Sf{f|zgRK0J5tCPs7$$<TWKso6{?M7!I!$aQav%n8PE!|Q7jtO;j4a3#Ulp~kAwg&`
?l)5T31pzLQHlAvUrcO@L}x?gN10kU6HUieHl9YOXQ%C;hW4dAV1k=63LLLLD*|Go+rIs*!HKk3pDNAdi*SdCm6q6nPj~?0j_f597XQfjV6~_$4rH3l=OB89
5fG9I6sTNrw_>2XjIZZPIENu;N{;RqEiNzP$3QjSxm+g)@2#pIREL#XtHjf){|*7UBaEy;D}cQb7b+Eb#JoUNS5t<03&<3K-r4bqQ3&GdoX*T7qf4{9g~(M4
U~{KTNRYqNH#|5gWhU3f29BTT2BZoJ-0Hehg!gg9j|2`pC7Wo9R{CGbB~4b*6M2#~1YY7K!Ps?5D=mMi|7V=l<%#QvmD)jWmrEArx_ldW*%;{g5zxlAIdL>O
Uv|#E6b8_4e{WwaqKSJfrtKP?8S1Du%4e9C?ISVy{5{JeDH4}QLVnDT+L{_}kC4#Ij(KfTxiw2>hW)TRg%z6#@S^s7C8}6!M{ixwA8PL#_8ZH)sb>&Ajq9qE
F>>(PakIhg91rgVJ164UPr=!<zla3FOIsqBJ%4=y`8yOU02*X`57KL8G2CZuyjD}TuF@wjuUGH?#B8cID3pGRUZK^%b6HUQdYtP)u5;X*AZ2sopAkx%@PR`p
0#Z#UJb#OX<BtjdhZuMp5wwFyywPLG5o-Y3$*0jKI}fNrX(PTxY)>$@-#KAmR<O+1og!*$VBarkw~a@7*)4eR6W0D9n938%<Bh((Nqp-MNDz2|^lZ76L0mr*
w1MYZZu3l%ypmeaaUZr{2W1WwYqk`O9F*`N_iIa=(ud&oAq%HtM}G^FyL4mZmM_m)rkN<OD9i^3OGeQtM2dg^WgP;<D+1=^uht^+L}%ei%*$#{yFiPSVA6$y
hO%>P!tj!d+nD=p_u$U<O`;YzJT)5YAAuF&21LkNdw6kATFW#${vy#%%g5To*Q6z;P_tn%o0K~-Xe=&*ChO&ZI|?G@KG@pbL8z6vf!8pkJL&NRzDs2?J~H4I
;#ekQR$VX?S9ZDz#pLBX<je3RjfG&LTnl(MiaSkdrxF=e779Y~VnD_)N@VmzUJ%%mo%B2ZK57-Dhw#`^xz_21mL&yThT8%C-(4vVX8aVu9rZIi?v{d$L&~|Q
+od0JpO=>1&8{z9xE?$(#D;%<N$&a%WcDaDo0Xzl3bk4QA)TO?Q~BrAj6q{|p}oMcXD3Kj{(`E~`_j{=()@<;L?u)9gWoVzv(i!Xpk?+8P%9=><L00L_x;Xc
Q+VpG%#d3GePpukaAgsHd&A(Vuy`C)gt#=aTi0-x1fc@scnGcm4B>>wp(lDV{AyX%W24g{*+0trhKWRYr8$;OnG{4~8!2v$$y1Iv3a?`|XiZp_B`ZA6k(czp
=Wmw_F$~GJHr1uPJK1Q%{!<WUnvK}7A#}Tkb)9@McU!Le@tPW6=t27<{3|coekY_wmM*5{Og!6|Ceu7KNJtbo8X5Q`e)mcmQQCTV=P87JMtdpx7`OWo5#HMB
h<f4y8|BRNJY*WT(Ga$p=7*~eK<JG#!Rky^8$lC*<PLfCx;N-td0p55`K+3?cHHpQM2E|}Si)@QoEu|CMh=q^2ym#QF??d6T>YQ5{4kdNP2B~C_R|6;`OPx7
jBWqRMci3>8YAfrfJc?I6giKq5|4V3VP7EzBV^CVaQJ(})|aYPZ)$~qMa^a~@T0KwT3Pj61m|HR>U_sqoG8u=R0kah&@m@VC>FNk^OKd`kDH8CP&b?C(OzHK
s2;#gjB?Y7d8F2PsDEYt!GDTz8qO|nj>@y_YRd12<UGled*g|ypMb3IimdDx?xx4ytm(4ag}eABv1@?47Y9JeYNI$0KX*VnTLen=CT%!_3;JwvwEQ@kB>rrC
#G*Oz+uz<N5=VAkSVDBb-AZ%7V2Lo*ocBw9Reh<%`D$BJXkSk6IJ@+fLyLysNt63w6kVo#_5}kT@c2<3z<IvUbppwmV(2;6TUUz;DOa^=iklDfTPB{(kZ{_<
(kWG>rWqWZbgUJ03G0p6>sB}l*(Uwiq|LLdkmV~%6Rc~tGMN@lF?-M<x*pM2tu2tP<N)uKw?Y^p)=tO9zX|??OQKi5H_%j81^Prum7@24t?Op?i^q+VLC1ZE
)OI6=Mm+(3eTmghjMVDc!y8ucK0ns`7LD$2v6sv3KW+8EvTlav9$zM4Etp2PoBEjc!Kd(yk}}~MKK4}t{5`A6(P#D+SXprs?_^<1(?rw%Rp%0%@8aKB<gu79
<-Rh?t%wd51syy>ETIvu6=l9%5WF|tE88Z<#+Kepb)Wxw+8r-XI{|#Ib<57^Vxu!Eo2J1^Lngvdvyg_})6C@~xuy0<Xyl-!$6)a7SP+Uq7(P8ot&>1vns{&%
?BdJ?qxZB)S3`AC&sT@E%UUM0s9~P=2WnO`(gO3nUJ1M8Ifb6MM*G-FL+(>f;k9($b($lru=l5_sMs|qWl%)#{13iIPawYHt!PuM^Y^A@ok0)ejkN6-)8a_(
+{RL4Z+UIXO2v`i<pVYXivOdh{||qm;fKgfnSIU_cAZ&Q#<UJdM~|4-p`%seAOt!=P5`2F@}gh-(|Vz%@Cp6CkR2(lvt#&vDgHn&1d;odaSpj^2I{5LfT-r|
S{7Am2lH!|t>qn=vwaQI+@l{+nx}l#v7soPWI}N0i=r7fCDr`bCp0=V3-73zTTjuvE96BKj+I)68?nH}Ojvp?UWY_w30I-fg<RAQ%6=IJFc6{&JSTBfF9E5j
J5Jo<eiPEIDt}obu)toOetzeVA`)jD*1d8H6-j^nE(Pk%5v{k*&ar4ShdrP0sSxzJE2Jlk16sZ`njlh4&sbBYRH3A`#$LaC6WJ^oSx>V>j2Z)l`D50L3MK8T
=qD-2Z)ARUkfGb1P>*NW4scQNW1;c?aH*Np5s!%D3)xg5W}H?DBq*#Q(NT-|FaMhyAhjg=$D6r;2iv3;D-@?6$!!HjiFs-5#pBHM8RJ%&@~K(;2ch;}tkR!o
R8qHVd*CH#T;cwx&bzW|@79OZu}aXh?3R1z9p-SGV$>@qLllmFKP$3bZ}*qo=f2q1iIgEy6qGUi*Ok4RI<ui^fp=IWwWg3e_InV*Qubt1f!(lcL@NN?sE(~~
Hk5nem!2b*gGZgobq0w`%%wc6(PmGyv8rPyt?`B;l;*p1fPnP^59cUOI}VJrMFhEzb!P`c`l_y4aM_?-M^X_}fNZhcLR}y$ZN*-TL~rlN4Z?TWc;=Q>)ID*p
A4*42)A$!X>F)+@g?rFj)|~*>FjWa2kWoG@#UzR=UroVOz)w)<<{^86Ef#GpYM$tPH7D14mOV;`9qK$K@=7QiQYK_qTgyHWBV#c~8AMxr8cCz?NGXk@$cZxk
|1<x`V~(y`F~)0v)w_D$iwjg~sTX~VO-%L8@$^?B<TBwStyUx);8yeY5bY6pFxWw8bBxaZc&~23xQ5PXCfDwQm&jAR4o~O<wVY1|NT(CVV4<Rtq)6(0a-^M&
03pFr{<Vn3n8Cwc_O*auHWehazqrVF822Xfy3ZcJ4pC+&(<;6L`GvTe2t0SQ+6aUPhI*Pi6^wW$b~>p{<CYya09g1_LePMUUSZ2$!1beXIRFdfRi_Z|rdPG4
BY|~0BH{X0CJ#6_7?1%kRd2PQXW@?pMei;QU?(I&j<h#m$Mi!h^|c)cqIlQ4?Vxk~SFC6jPxc)6uo?JY)ZR+42(#Ckwld4E3Ic*_!uEx_AYC9+qK3NVX5(fK
l<N|igkw$GAP_$!n=cs84oX5Y?UsMPpXg|Skz|niXG*88Q+OxIh>4f<6<jvz2-Rn8v7{bpeiBn8pf1k8%>gb1BHyG)TD-giAMwEUe*rIBxcE$k3z4?6<T>p#
?#~pwnztR^gVFOKi|elS1d)kSt$3jL!D$OB_#2y{-em*fx(4~xKXG^=z9R|F)M;PZ)#|s4Viv)R35%~gpvjBi)iQ&~@uZ4sh<sZEfcMKISFK$R+6e|n8Nk1;
Cjk&*TvuGQXIdW+TdKP?i#gTzYiE85W)-BZ_^<UUg7wqB|HxAvQKCDXsRw4gSt$f>f7zWkI-$8)t5~eU&rJicCjh7~$|&=fg>|``j%DL@M5eOi;Fr4>OBagy
IYtKjVez^@lKo_S{=j7zZKM{Uerm#Nh%AhL?91NtZ}5)V^+gkG!yF?q&4LFn&W`dDy8A*DHalUv);9gxQ>Ht~WucLb=gPns+93(nd$=KJRk^oU^$Uimtqup!
jD<mv?9M#HaBy!GgW+!A14!Z6p6<|a*Ex-SKYQYwl<2_8nc05j_4P-epc&;16VOM>2D<$HvnVwbA6bs&O$Fe?JcA~VMA;3vkYu~gt|sLKm?zXb2+yUqJVAQg
2u*g~?X-|#XK`suf*=j$%t+H8hgRHY2}9+dUokaV8WE{^!@_IA2Jk|o^8Jrh&8buD@J9*Ufs;ObKX5<)MicF4z95&^<W7Kt<bA^#pie^4;R5ob@8pYpWDs2r
S%JYdYZ7rK7tCWBv_BUVQR-=}Bhq^3D0@v6HFx8!K2C!_l73B<T((hQ%OHkt*3<w-DMg4v_G#CW)bC><3I%J6A5a30FIsCkdZA5X_$^^TW64dnh2SD_H{(VA
BtZS&oS_+_i4V2aDswj|!Gnk|lCb;V(@bhLy~K)r-S0EYDelN?P`^-8U3<jjb2Q(&!%EizSDEu{jcFfrs5xJ4OvtRm(T>PNuR<!L2`{@H{a5A12@psJHXsx(
^rZvQA@o7~ci<IL3eN7RW54C44J=M14SO<ta3r8fJdEb4nk$Fo7&qo*mNJ(-Oaui(CI<_lS@voLM=J<Gl8Hskx?c&+P_@3Rd*%%UTN$+)fh3^rG~>NET3`*6
>}r0u$`~Lqs_td*w*J}NU=~TD_Pp3NRouyySn$e!9UX;fx&ATq4a57wHQacRJ7Xn&ahD+5!tdCRq4w`(jr}!F^@Hc}{<n6Q;7jAT(BLjgs^VORYNB^Y%OG15
OKL8Y`|)AnD?*CgvGU=o9=Rh`N$K~?Gpq1+tlxCRY8$=)vC471a2)ag$B&Zm_BtS*`fm2PIoSt+b_o8*paRc!E;RuXZ=}g1COeoH%D=4qazZM%V;Y>6&p*}9
BtP{GV>6@u=;cp%{gWFnpdb_#JxS__G>h!_xX$6<T<|#YZI6E8>*H%?A=sN_GbsA@U2tAg<HbFf-*?X~_Cwh{o7FqFn=ubMpWvtFQ+n>9B|E+xMpSEp#`}SF
U}GtPUs3!s4VE{fGWwJ%i{W&bHm>AKvL4oQzv}qSV=cdbk2jCl7stR249@T+R*tu*6V8fw<FfxOr-^YvHE<jwqnso+h5j1{*P!Q~4T0+uHLhrc-WJj1{buZL
G#y=w(gZGR2KhEp9+7E3`MJSV--Od>O*{|<iCna`Iqh6;j<Ez#v>SK$%x!KB(5N5XU7uY+rf2!?HyVl(=F<P>TZFnlsxNidXE^3iS=I(Z&~g#^zrjIjv0tA!
_H>FCvCy?XOp1M|$-naG{WKwDa{8tXONIbCd{u>V4D*xRTNdnP{`&D<^2Lwi_@fkaZCEORQgIGK3wj)ODz%MH8yj$xFk&9b#Hq5qER`hm@bMbQ3eqWNV1vU5
QTp<E23@4Lu1{B>#2vW)7kOaJXN14oOtJ5WNpR=NpYl_)8blRL$0<Vp-hzUxJym*G!IBufg3B2~_Lj@w%+i|*#{^pb>t%3TWeTFg*T-6K+QtZt!gY3H8v3l*
N_s8;D@~fqJ-Zy>C(ZTgInRRE+7y}1+sSD4O{Yo?t-G=Be@H7aw~>tyPf23mfeQuB{WV4IQb3^&Fpi)0jy*?J=Aa*J_pihN9VH;KHgoMBxKr2@HIxF$&%Ree
Q@dkTC3J_O2kOKv&K^P3^>4V|nN<xwZem?b!`BD>#^4ykk6Ombz2$w?X~9t5jgCM(GiB`R;ah4M&}4`QoEDmOB{$X7KAa2gny38sQ9DOyF*3>i@p#_=uu36?
n5_)+kfNh!nVrrM)R`S|=eTBU>~`cPiXbW-8-y!_yH(q;OemNK1B|^LP()Uzk}hzX-Rd#$XzyI=iiTc)2(J6b%ZV+5%xOChW-11wO$Tstpxp&kQxzu_qROyJ
A&qdRdOa2icMk9FkAB7u5S$dwlazA?pu0N(bN&Y)X@1RgUFGMsH3LFF2tM7}5`FL~GoJ!g6Bp<0Df4vc^=dPlEen)5V!i*m8Oy?ix>jv#LW)jVnc|91EQ4`X
i0TGLp6qGWOe{g$d)-a2E*Ss|-W~axBu_~AqD#74T%%xVooL~TQfK(Wy#W^!zsUu`3?%_tbPm`5<Vq4Sr+D6J!(7|8FpRl4rs`+l@i~+_jI`Xg)Y#gFF_<sK
aD5C=Eu1eKjt;@2XVxKrhXW)3@rj_51&qw5EIjzy`mX93{PwaPdW}=KFoeD?kn^F7#X4}<y?b9Lg3{5`YSX!^l&UlbOUiXy_T7})#m}`bbZb}Ze{2-Cx_xCx
CSr0wzdt&DLI>iz55mVEdP%KJ)r@m~%SHy}sMt;*mKsfJt5pO!GRNF?9v(IYZOBh|c4dc33@dZL6oS4dDg!tDHkd|uPZj%fqwfd(chgDi(u^p+qgbvK5$~yj
ZoqNDe=xBv5+hXW-7BbZRWrK98)_tx_liwn&!9L%sf8)OAszvHE*4%rSJhJq^tQH7<)eR|MuP=5bkLIXfC1LvpQE~N#)(y&!LLi;58=jgc}2qMD3oh21psNH
bj^3Ew-~lPimu{JX6+n7sW3(e;DaL%cXq2-9`10w=~knd7d9@}DSv%iQ#nCAt`fYrqgq#3e_KOxI(xs;)3!2b&nK4HX3@^eCW3rIfQV8c6sMmsQE{JwdvC<k
Vq3GLnd&41=2bqCwO@`yC>_)tO8;s!Lwp-tJtK6FmLtn3L|t%!!aKKDEjIs`qCz+Pz6h~-f^7=5KWK5CeLPeBr!O8*iv{9xiyybjQ)OT9M5d;`DsFf@h}%R6
Kn}xvX^*=>RjaX5MhM1Q&Z6Pn8xT0e*D}IVOe2<gnb*-j%!m6HojY=WG3*z}*4~La`<ZtG?FY5LhH}^TYFxmNp_;*)-a@nPhX$<yo1<Bdxf;)^ovdd93hL#E
Y_W!Q(e+Pd^Wc`6yCLFzdOV>UR`Tgje$Zy;+AFYgMa#+=Yjd|QewOa1dr)Z#OwY%SoK>nlt-<Hoj@B<lkufTO&srR=^^b|<2MNhEvL7ax2U)u|o08=o$+3RJ
=>?Z)C^SAG#`}vi!MheP0@X&T)Z};sr2$a@YR^L6ovkjV@HXNZi=Y=aGA!qpIM}+yy>y9v6o)fIc;_=F_3d2Foee8qBZ-c+@77!^W?9tMK*c^&`79TRp1-j{
B+0RiLb`GAWr*aiQ2mPEB2c#nbK<+tiLu#KNEr4v_aF4QfCtqaHW#ICOlXcxv}Q&6*yQtJK>)nYaLv4Lru?%t<#d!AM|#lvFY-a-p}P^b3}<sX6`i65LHJ%!
?Dum#9=Nj%ha7wQ^KM1CZO017ZYb6EbPVMY1PGPx!CQTyA>K>pRs2Zf!Z^bh0El?njc_qB)6D*Iqu4{VWxk|MWbN{Q-0i=ZC_yH<`xgY9wDVa_Ym-sr94BXg
(eBK`z_%&o>E4jkcA|Mf56yS+B5s0L8||r|L@WW~^rdrKi}mbrl9-1nXlfp{2Vdm``WZjicZ+iUHM0YJzqR>xQZKn#lQ&S1MK;JRRZd=U+H`0VhnEcCxoE>m
uJAm7rPPQXLFveP_P~4XWFd^8sZ}PCRxgY)I<fr!$l^B^%_wRP{91lU*aR=9SMxCde?&r8P)U4|cOM?7K-@TP?C4B>a0vJfngTwM-}z(dWVoZc1l+-bkgdoi
hG)v7WfLM<?THU3VBrd=CK)>Y8_yB*6{RyD@%Ye@Jp5gI6|=K`l?5T@EkKY=CVMt|5^j;>)bOdo$`Sej<(H!JE+r4Utn{81lspy@(EYb)ml&Os78Le;5>~^0
M0ldgM+o9z@_@-ZGcC%~Ev+VJHAI15dZeCaz17TSz06}y8z(Z4EDW2Q!<c~L)q0DDce0<zRRJ6Ur*?wwLRH{W3WnBvnFx7J=N8b&;l*Hnw{J=_?LJ}89--Ek
&wnG!xpK!YA7xcd9)(w*`Jl5NPLAf>w}Ia&GVY^57MDM1APOKywk!w!!FJ{NhDJQ<XD`p8qg`715xCA=df+!w$M+Chi*drH_8>OXT#8j0hOhdjdQX?+W>HZU
9f}Ez=w2B3x|$SYFNTigy^$rTq-ImMBsn#X_6mH&>mYD2_RueBR3bA}_|N@WF*Y(ta*^e##H3sKD0oHBJwOtx+$IKTOp3I-d72linnjj`Mx^n%kj}P^>Yo4o
W8X>7$MZ!?5|Lxbb~_wAuFKw+o<i(JA`%1mEf_Sl$po%?eXx^rhuuP|(M`>wC-(zVX~8e{cfnA~T)+w&M8~8u(h34CNDFa0F`|qcHc{SVCJ#qw^)q-RgcKnQ
P;=;)jF(W>igIx1Zha7OK&D{vlipspS4VqLXxUsm;II!@nt!f1L8idJ^rc8dVi3;u6kZ%Nb2Wr$$NbFSo%H^~@4BHs(4=wzbWN)gu@LcS^0|tBaOneWJuFLt
JBxHT3fAH$LJQ!9|HOjdM+Dj!LQ4DlI$3!4>{*nZ%1xy?yQXTlG3oJNCvrKTWbc27hH1)AQ*hA`8VQ~s(BNrP&~SB^@~(R}`oe-K+klU_2UrbTXvw$t%b09%
7a|Jdn<pXvE-q}B`yJt$g<Oe>gNKnG^0<~l?Zh9;_by<+Ulw&nmnOMLnKjLuJ{w>LCGurekfQ*ayUDwaZL=ZPsie)eZ#dn9rYgZ$v(xn}kM_h#CjxfQ4iI%C
uAMtK)52TIe%cm1F&UTy%%k$_e^>GThkfTv`S;xXy#~6S(lB4SJw>$21H$85d{v#pbh`yd#+YY84!faYevTi%=Y-=!uA+jnqJmXqVcnOkhKlgKW1H79lDMkW
%%FGb4y$&e@6p<?GF^_$`@7}CcAn2)?AuL!F4cPi+6eD+1BjeB$UsFLDXwdz1>1N#K3okMTm580boALg<{sXShiR5d;JWgTD}K@=P7TmpE$V0X<(I%qPaaLg
3Yp)Wki{ow9e1j(exHuTiRj(A#r)O9CWIMEr5N&&_po+D_dT>C7oEZO?pB4rCWCwc$#jvwJ0I--Oj^Fm&XzuzdM4dv(!W;#!vK)nCwiHZ#u`<U$@V$%!cxRB
bIJx6L1+3yNRLrW%BJ<kIOoKcRsAHD3!fsvUQJR!F-6ahCdN!^mGcpjY!Ufx@iFdMrD|=A1j>hJTqTZuTd;z%>8!+Wg}be4oBJS(1V$w-XDl5io{wqr%WkBh
r9N~XDI6l7`kqes51?&~L=2gc9UM+xxK(@s{vziNnxze1bPyl)XD-(aths|9^wLe6aW6TNzM3d*I~SXRt|n2i@a&77Ept`{I4oqb%atK`k7U0#DHJi;dDD3C
E`JNs$+dA6P(-SILj?=jr))oCEc8k}C(N5@sXHYUJ=_Js&2^tNshZm>Q5&Lf6@|~Lp`PpZVBkK3f#o15&3vsDcTFSpLT6DPP=k&Yyngj9``3smo&pDM>%;=O
#q(WCuVI7-W7myt_uT(Gl%s(|aC~f|hemL8Rxddi8O$x&j=+yl8IKGaihQ#&C|=!C^VFl0<e1o~6OhN2qOl#bL%~wP0FkuoMtizIxK-?RM~2c?%D{*Zgs$J%
iKN;$XuWFxpCUMA;XwJ%<gC-@^A1}LZbTlI3>p!^xM;V{+$}3;D|wEj^UN}E(!wnd&EyysCC5?+1OPb%O;TA|;@{Q&GiJP2mMy6J3=x3`i$suQ9bC<2;dC~$
x6jvtiS#d)e8yI|Rs_fXfjkQXM@HytVy9_!9`{_9IR%E|Isl4elxL7NlPw8RA(&stTtsJmpcx*tOG@H8Z6{#)iD!0nc9PV%R}~PWZO0a_G>|wfGeK<x(_2k`
*){{oo!aLf=`|(bv6^<v;!c~~2=r4;InWYa%SUp|jnyZ5PrkO*BpsT8{`=InnDr@E5s--IQ@V>iLIV<@6=b10OGyva_2pRDQZ<5eHT6xdxG(Jqy7useh_Qr>
7k#b|Wau1gH~SStSgLvWqv!^oe<|@5<=tH;wIQs-0NX#va2;H?KGl2PR@X~a`706J=j@A_+kN8UT9W!cYq8R&HQN~Hu<X13wQ+kIbO6@;OjF{;BbumHHU++#
aPKFLewE1AZx1!>Z!<;o#VGpGO9n8Bv+#{Ha$#Jxtb5%0k*}yeraTI^qHcZ=@1e76P$#y1oHb~TLX7x&kN6_9J0F&2LN8Oe>A?A5FaYpJyw5@;EM8i<YKGlS
VY)NO(3={;3<8N<C;h26<}XVeMt|P&){SbU(lPHePsReROTS;eQ$q_C7k{;jLq?8d9q#OzyZH=&o)6M1Sc5dseo4(3tXl8Td}a(|lnCpLFa=35cY+pdModzJ
qT;)e%8V2T`sZH8CUpVi{2fzRgEyarbRWCmryb|;VEWB7#%)<owWci=i290}xbPq(`Y+hmxekJ<?F#By<MruU+R?k+Q17}Hv_Gs}H{r)V5p?mp%5gl(pGMG3
7I!k!GV16RAJ#WjG%Yi3NXrzjN&+WuQ+qPhoeJ<;3MUAIRgq7<OrjWfN9%1)M#lbwN^<05-c`0g+8;@p)umV9L(yOutTzGql*c<7*<5sU3|UJ8K;~Y>eh?|L
*5+PdBiynC;EiL^ikE@E>nazY2+4W-I-RAssc~pr+L?H3<Q%P>*z#Y>bBHh{z@ykrHAe$Ll}o{`!5ruS_Y?-#K;&c;XkA5H(Vya~zMn}j3g-Tm#sBCyOh0IX
&z1miwaAU$hQnvXp(qNe(L-j*$4cxlRohy@nWlXLwp9scjO1hlUm5@+BT{C>tYCeKPLR?v$<KP=<x<O-Fcku?;pvU7BkCoA&%K+^;L4ZLSKV{HhyLy9H#0Pk
ZO~OJ%wK1-Chs-2a0-PidLtWMl;lJ~v8~7IEyRrU%PtZgeoFKVN2G<DemdUpu+(~*+FMk?muq|{yKMeA0}H=a|0a>JF(o2EwX*S35k=C?=4fok?BRgqxYIu{
b{)=o_EB5B%9ZXMpTN}k_Ya420B5hov0WvU+}}Nd`5!=nJeXi8C~Tk+^T^p>Pz=FvMSA_R&M&yCrsBpCc0Z`@6<sDPjkiQX9QQ%Mn|27JjshfVYwxgmJ;vf|
87>TG*tB(B@6nL1N#A;41aGOR5vdA|sK_i7e;C6DM#cyxg=s_%niXep1cx3Q!9AH?<Io;6{uEdbm%1o9eB7FOq>)v)x&>l8U|f{oO{tw$f+%DT!qN92@%Qbt
)lU7TGxPgxO}Rig1M@3@K<DSki-6;00O%$lRJ@$_T6f7awI!%h4U<Nn1{;2#<q!zvi=65}ZlZz!cXyxclf?_6&}!6k$oIY$=0I?okoVX6D>|eiLmin8F4}{#
v^masj@HH_!yKLZRJA-fS-z66J={ZX17b9nhD*_SF4Iv#^`mnQ!O;kHKc9vXeIWg}djy>wKFi)XoSDKlj9f%a=y-t>GYALWEE;3(Uk!Tc7fbrHh$mg!dzeDK
kehW9QRKR`8KxuZ-vVp`-dzH8Lk=j-(eEwu;z5NTOkSR@Yg?o)4xP1ddHy^AJgh?dsW#R9cLIdco#q-?7Sq8S2-lq;zn^cLLw_?MV)K_S`~PxA_7pjDpFS*>
nS02Z{CZrS%6#ywI6;}3wDPbd;V%Ct6voHOkLvUOb0b89EOdgwFG0|~C=_e)0UDjq&#uHq1P{tzX-!tSO**Y82J=`qbmv$cJa@evv>Ce4e9Um)E*gQn3^CDU
yl6vl4>K28{AELL<?E|<L~WiAg3eF_`$BbGggoyZTtjL??M@Ed7igBD_%bs^tMd(h<?mJxCpTb$7w914J!r#UDyr0h6Uu(T@255!VgUdXKL-0E<G93iwBa(w
8V5+P`zz~(O>ZWQ3v{=4Hi&%W#^TT1Gx{eO{c>_SpA`YAtAASEv-|H>uIZ_O>kX>%F(@~I@z@i5gH3Es-7c0t`nF~lAaClW;ged7<j_C>LZb?<$K`h}AQ-*b
6<|y;X+Dnd)Cf@P5wGyDlodVm*SEM%3Vg_O7;Dg@K=W%~3c;Z~Pr4iGZ`}<uNN%r1=&bV{LvEYLD|t2QzS$D*v<jcrSGj>KDfHl9@}zD&IZHJGC5$&iC7$!R
6a%)x9^k!-$;&VAJJ%5xmVtd$ZgE8Qoq6pm<(<c_?@e}p73e%VY`4ig>c;AXi0v(+3|{U!na8)}ZU&hccPg^4sfzKetOw$9(~;d8PpLPy+zwWt@k>BY4S(3r
$BQx$N=s1VVb1Dc#xDW4swU}&$=U}8Ox$g#aDp{gkiUnT=8n!d2|NI(FfYxLJ~r;dX?sGxS9WOjq!1Wdawt(}9d;ryQ?{o%<L4y-_RbnF)ikzBvZH>HrO{kz
=v~mu)=KR#h*M@HeZYw}7eNMMYfNpzhvj_+W&zETt@gV?=j9nN88YABZ^|2CyD=8{B0@=@ll;TSZf`LY5U()KcJG$t+&Ih?a|A~W<K?rha}0i;Mz`TM-l+!o
Fia-#!l4CE!G+LMI+ryvd8U>R`>wATOxqaTi9%9;KM&I_MukP`aE<@@gfvC8;jW52e5+Ik5RqVhWcih6V189RvStnf=Xj+W{S1J=@vZUw{q1iY<b-9I-xDdd
{^Uy8)Xe{u&#u7gCg?)8jXBPX+_Yvsq4;7I5&m4QRmpcb0$1uuf&TEh;!2v{(o03oFp_dypJV9vns+-nsebm0W}?a%$5A{N^-{CJR@F0WnD0X`kr}~+5=X$1
$;RUtI4Q^hCr?6t{}^*1YM*d;;(j#M>+Z4ih2_{|-(Dqbk1yCs-cN`yk=eRKa77Aik2ChXy{YPW8ehx2XRLo^XH9pky}96UYJ<BK_tJXrqPrvOeoH>0K9YIQ
_0ig8Lb;~aD2xPIUfU1}T)UYu#X6a>GQnFeri0kZPrE#ry!_iif2!OH!H>Xu4PzSc$U;6$<`=u0b;!_mbl)V{YgDJyhB&USE`YoHG0jj=L$l!n83lkwUGo?R
8P7=@q;+RRAeW+;XURkit>+3aSr>dXUS^|X`0&=zdQiw^Uv{Z~?-$CTaH!0uQ93d;a*H!ve<^rNoYfb1K$yQSlJ{D@ucO!HkHS3j^$ng48YrQ;AIeK;MRvuM
te648(tHYZylw5i2+HEmQkFy!a+C?FzuV8zI|U*&l0LCu+3?JCH}JGVl9H?tWyV<ZxeNmadQbGEp`$&(M<exZk0|s8I~lmKJE)HsCHgS`I%d$X^ap2#AT__D
T~lwODJ$Oq4lMWd=vVCQxeHKg$PhBX(pUO4qH6tM+2#{X4vtBNPIYju?FGU#^3$@YX)cY!cD-z^ycA5Qif1hg;%5;;ieDim4?Z>9r)qcFv;(@n*r2>YCd^-Q
_=y1w#MPzsGc(_1nX#x!*Nf`Lq@n+Gl@#09PzhQ=;k-}Rud^p~odIKoaMI-@wP+xGupSLLSN(W#Q_Ry67ne+sb5L82V)ae5nH+HRq1Q&(OX4Z9a(?9+Rwp+3
c0sFt>Gf*2c6=sN0CZeAlr9FiG?2Ql=JAI4u{;(Mt2;Qyn3yl5yj2Z)G`o*2h)Q<``fF&t+EYnZ2Jk`4HlhP5g@jonncF0Iaz_H}kOJE@9h@NwipE27@kb`e
I=eFALHNQmvv`G6(jRe(pqMW^;Hh`5E&mAamL6Ly9dt_f69EDzC-knr5kTk6h3StHgY#OV|6234YBUm$xCy}_o$dgF#R~|BPJ#Ut<$(Ct0pp3~#p}q9O!+^B
V;XHzZ&hc3tJQdxn0_&rRlU`<PMHJHC@fodVu(_``2v3sg97?*6#(C_Br1p-|6W6mJFyq{bJ$f(4x~aUhl!UMefC7>@$#{yS3_|VtGgf$Rg$Z)S(X_3Qw#;)
duSkr#V?fd15jj`(`|kV;F9)9*IzeA==$oBY43PCfqoujPAZj*&iCInZ_#)aTTn7odoJ-Atp$#SSpe{z2mj}Q94`@T^FwqwQ+>to(vCWS=|WeJ5HabQ@fCTc
9Xtv`g6me<)`4U$>ft}nB*PIa>pabXQm6|sK1^#K*RZ@>371ILNKF_rI^qMHrC|GvR457^6>#wqvoAt_heUw27*PN&={Hwg<-abGV%*@}-{=0^34F%zjcwtq
(gsPGze2=QztnQ|6NimV9}#j??$<-UO)B|*o~*0T+Xcp4elevU+dh|Z7KV!7Cvd+CgABWrt3XA*%J^F15xETLQ}qs4u3S$@R<`TDeXC6BsWVJ=d@X3pK<xE}
9(AU%9(FF+!v8M>q0u+K0IupM-{az)@TF`8B^C;`Os32@TYu|^<7cDkea~})bR1OM=}@hTaP}rBAv8lZk|8@>*gKF*J6*-<Gu+of;aXQprWWuQ=MQZet0^B;
F^9BzFi85ZM&wfeHfH&-_UyMcQ4-aTg}Ewkc--EP5IBN>cy369XDTh-f6xW|nG3@EkMhel=yBsMW=p6Udm?wrSx#r5jW7?Z_n^MlS3@d_YV;-2gW`;CwP;z9
S3?eoa;!7TCvTszK{$Yx11~U?pzZ5_)HIl-p?oP8_5-sIS<HfZ&1-|?JnZB_pZ#a89@*SOqJF^GV2MUI=Ic2mg&&4gRYA!r(*ekS+$gN_66ak<IP{p%_bl~*
FpgEM9Gr=?v`Gqk<ro(K|KI%Ttj2=f44jZ0P83xAbv64S)NX0#nIHAn{2L&lfg+cd*#<Kwn3rGsZx<L7=74mk(DRV@Olw|Y%L*dTG0(g~a)j+w*YWMJkt-jW
N|Pfz_8jjw%yq{tH+xd&cS!ycF)5gobU~=-eGCDhS_R#dM}%6`qjBx9#`7l<+_UC5FXJhj$PJMcTtYeMQAm1xg2P@!tP(Ay!LrB6vSy^O4*v;m;sw#GTT7V2
Vla+gKjyk@#4!XR1xERWB0a=U(&rs#C8?!ww(5V<%=OB)D3~$Vcz->*%ywXq^*?v-j{yCDl_uKo6%ok4l3~ry#Z-I@>4#q-s6B|6Imq@SnkbFmp<3?c3sY1h
12R+J5`ri45lynp+XDTUM*PeRVGbj17*z-U?Ulv#h)L2@$dp(S2WfTrH!rqIhea$e$>rNxQo4P=KlP9U0~b%o(#IVWuMK<T41p_|EeBa?X%1P4I}8=CXK);>
>BmW0c?g1OzuCxYY+LrH8O&6_fHL5L9(JEUy{4Sn?7J`5lnp1T_>EK?(h{v$iyuQ`9UldjU?%ezY|1_9q|{Z8H(Qb(4M{z>w{hSb`&k2@_H1lya=5<{OBOuW
m6IV<oB8yv7xBeMW^rl2mhO+ki)8M6-8e)C4|?ZaDf+z}L@<Dmwt|A)j3wfTLhJlcc~Yl25%I)pHjL}_PQCk^6i=U?&>_|4uug9&=AvEgg293GI=Ov9LjM!@
j%sjn9J?dmR^xe@*S0{goJEMmvrpHl7)XWl57_p#{I;W%15wIwBE6!?P%!g0s#sActMZV9_~s6CAbxsK#44(;Bz7L|QNytGSTiLH_cIAfw{6U)c@)5HZLOe8
HgqbJ{QJE~IqIT4j1Tl!Ekv7i1osNhyCo=$>vW14igbfJ-sWUV6TsqLx=Z!~7A9(1!##`SCBYJ1J-PQWt*N?jYzhq9EaxCO3}@#^?dow}0Sfyp4Y_EnQXXJ0
hOFsf<#>e_9Ydazn{^UI+hbBKouCDpsaZBhZoVh;scS*l`KESbF!=*z6?l*`0|ipt;=o}6f8%^r8q&7+;eAJJU|r-^$m(%shbdfPtYrR=z(DQQapXk0(M&8D
hlC*)POw`|8!8pKh&PL@$4X~gD1W~jf__OA;v%eOAp}yzN9q@SD_KL8M!Knt9URu;{?X<lvO{?zV^)GyBol&2bq>>msSmDdKHP|?aRvtarm+x1sF?Sqv`gtC
WH)_V9y>-9N^^S|c(A}eianDc3_!wD5zaUsuMD<m;10H2IS~frxB?Y0Zy3d6yfx44^l5u~oA0EGvxW%{dF+<F626$<`-6{g0;>vG;Tw!Q`ZPFA5Gv}S8o9<#
6HEn-F2D3`X&#D2zEFcV6Ggb~LG)=OFF7Q|W30lf1S}MAWOQJJcVD9XekG?f)Ps&WJI|IZh)9vH;hNk`Tl-&Y$pDOQR^5#Zx*WHYhX0;ts<XJP>0Oy;3O5K)
w-7ZTCcm}19=+<+kLonFnQ^>cW<MqXe89Do9rGKj8$=~Saf1?|29DlwAf~(^d;oKqynfb-UOH6sCw*NbY827$U?yY*_j>^y(w;VW57ppW3bvbXbn_1L?j{tZ
Q;27UadHDiGC~i~AsW$2W9T2z{jiKQw*S;wv&C-csU`_9yG3y6c4tqcGlvcVo`G)VzuFi@Xe6dk<pHe$|H=$J`r<(K+T-i{M-Wjg@zEsJ3Xk90*AYTUvTMSX
%9>Uq5dAnD4{1(dj_mM~Xj88UTEWI0M^z%uj(OinoQ@+z1?{P@97a|>Qu4iFqTh?f1vcK<mj`>;n6JOORRFm_NHsR99xmtDms>tN8Sr1qCxPby>~D^6+m%jz
pHqa@YSfLKKIH4-OH?Umb_c~4CUlDO$lm!5^ZFSEI_dv4OM0a2cP^N<v?z`q1s_`_N<JOO?TZF%tsf_!@sAmkb{aY#KFCF7LwIhnLcl020{R3zYVUYb&-d?b
E4nfc&UW=98d<)8{5hRS80Smu`lQkEt@P~7IRMh5!MO^uAEgkn%Z)c;bX9oh?c9R|DO0voWIt=ml84~VFxx^ahoQ@23g?(r0QSzKz`2iWs1SVDgf*`_C5Zd>
=4QeG#lpS6#7dS`kL$r6Ozg{coTa)(f6}qs;C~1MbT{q<ArDthystdh7Zxe!^0-V$dmfBDnAEO(0gGyrv&NGleLO9J7w0>R0j`nS==y_a;6?T^82~I?>3BbH
IqyQ~icY9JK0v|ZWoysQ1@e+ZEVS`|xu8y?L2^M}qPdiAU}#lR<8Lcv!H*;s_*mDjj)Z$kBl09>UjbZQu4lre5+`y#ZeuGHMyL@^Q>?~3P+Kub{BQRT9NOdG
hJcd)`AKgfT$rBfn%K$p+$YhPY<TiIX9!t6)uE0NIVB0qak*!1b5Us>RHyfi;Yj;5Q7Oy6OeV|t<+X<@N&Xn>On0zIW#UEON018h3Y4)aBDgA`fWH)85W&VU
`}R^kKApb%9m2_4v+dAl!5{*3mJDw2EGI#Q;pAvC4X}a@Mmc!n-U?1ss-0UCRG`P4>qhOMmxXO4Hfb-puC^lTAe78(_MQjxYocNW=Be9Oz7X#H_GQ}E&)g21
ny(a2N@=lll<uF+jeXo7dy_G6?o{^Jsu$h>fOb4QKnjaBt|Cc6ZU4;%+CrJ!4X3yUO6>W_noUJLbcR~uYGn3hF$`)r*;g!uZoxm*|G=tB>y&wS35GR5Lw7Uj
41^1k9AjE5tV%>J-5_cEk(or`6~kTB<$8!&-?|5J_tx3}LmA5>hf?z87aspLRAhXt1WpwOg@N~mSccB~t^{9p$J#%beNwJ*+(1YV_Utbyg8`j>wSVloO|{dD
q{pL30;d7@j}`pII5?5yZ(_xT_OS_(z@Gi)bTo`+P_FEgmQYI~k%7<()3*9><W_i|&Ug^F`@k4t!*L-`F4_Xt9>S_bA8Vm!(%3q<1wyBNTxINMAj{Pnb$ki8
tgr2Ca)$JW!)6+e(;O3blda;nzyOcAZNp-%<K%#88uOibls9{y%qXh8<K>9<<dhSnF#x*yY2MnoO<9L)OEdyw22f@4l)B>L*Fx5Dk6Nd-(vBws>QuoBxhvaB
JPkTQasgo&_bIE)1<&~#@~~48VaV0uK^iR)T9rbPpwO?BNkMoOnU95Q{x6CTc2hcs2J(rbd4VK`v!et*ykI_5{1h#I;{H)CvKfyFR7dWHUP8Faw>~H^Kr#R2
7<m4!3^;5>VkCub1BdrPMp&j?VQ0KMRFxi&kUA#9WHJBMU)eK(I|N-k$sJQ1u@$(Q5PJSd)%7TaqH*-yv>Vr@tIt*wj?{)M7@b@?g?tdVt6tXq$}LQlE8zL5
WeJ7^WFe~B81t~_$^#~TG|eo+Rw<ztf@TzIho9lpjka`2Tas}kC_Rao!dfy1b%}FE)o;R5u(XEaX4B3NF2kU7dYY@8!p$)5rtkXDDL5UBcjR_d{Pdxr*VmX|
r(>O66D02@O4A?-fde?(R8G^apwX#eQ-45|hJAj*z+e#id7|1K>o)yDe@L(R**3A@2J_cBH#hO#j3!j$q6lq3!T>@5d%hU6R-NH9B2CrJm0A<)S2}r+;{NJ)
L&V)iZvunAi0+AE=|41Wnh8zt8e#?O+Uscy7(AJM^3(5p5M96cgE2xpfIiK&2M7IB@eZJ#*fH^ty+K-mDh`eV#pIQ;6=P^W44SMrAVL58c+r5AQd$~aM?anH
dBn>ARmCm>P#CCYTh<7mAvOyt*d4iSF%cU;V)Q^bpH#KbnM##S<O%bPP=2&#ul-G7JhsrM@knfiGAP{%b~X#QpG8v|G2OCF<y7}rN|#CU*hSLzBut{ECWYOJ
G#$(j)Np5RRtt$mcR<w3;Z7>gMO*HMAp6ltEWJiM5wGO+GBu;IhVtNMd!7p<7b+C|8jP8Cs8kP{R0m_7`ztH-G_Tuq5|jOj@J}xwSZjMv(!%;t2NA~afF`F|
O+XJQ-iG&}pAgRf;tM!$A$5f=4u*TB+d4PD%ZJxVA=F?q*r9x4_vHSroI(0(5r-g=3r@w*(}NPov+X_QL2@p7>aFe9{%q6&?Uh>2S`!CX+y{w4Py|}dS3QDw
-x+@zB{XBzJZJnX>#khA$k&;7QeRBosQ7MlVJP?0Qq68XRe$cza|6>v9w)4A?sUk-KsLJK)eK7JQzTJp;b;eL3MUQQ#&(KCfJybT-qUDbWpMm;6pVr@mU0aK
@XiL&@0Mp50X^L!hfwG6(5Tblj?7Z??=@&Gt0r?|t7S<s$e~k@)=J$vuv|XfYAh>2u-k)l@cypO$9THq`;A}k2-)C?iRz*teu7FRY~Qu=k7B>Ky>vl$Ve2&B
wP-mk$(){jN<5WDUjut7x?{B~WISNFTx1+y=vv&>#W?01y6eR%*1PL`|8Mw_s?&ZmnYcnP-IqTwMQwy=sbOf$oBhktC-@sp+QDYjLPbzO(~g`0>msm;kw-W1
04Ag7y1Us>K5WH=oLM&-JOg6EYK<?{#EC8bbiL(Rc9Xl41j#ix;POsKyA)%Bs+*W<@X<8Bw`^%~!>dsd|I!PItoHcrFfb<z0dcEX4k!t92Gkx_xPz@cE}}+O
V<Ay167v`Pv!at7yb*_}i$2Q|KuqNJ2QRt;6aRddTfP3_^klqJE#$&Ivb5@*!)!FCMGyfAN}*f=V^EQH2D>J2Lmg)&HvJ->U9Nkru=tyQvh81%!v!oygOeF#
;Ya%3<=PJeAZ>$qKr<wMDX&xD)Iqk6T6Ot))M@ii<6acfALxX{50Z)q%rkGY|1?!(QT|j7Aho}%bP-m@!;-Bupku7*8~})y@Lu$P%TktZfy?mjkfPNJ5oe4!
sh|Z<c}A#GrK!Qby}Ec23gQ@^3GX3Sj8>@R4-ss)1)fyv9n%s^)$ldWJ>i-Gp9#XmjO%s}Rf}?u;*suOZcYYGh86ghKQIH3FGv+FgdcC??btBUsK(Q0FL}B}
n`1TVo~7$p{D6o{@oNg`!Z8&xTt;)_7EUXi29$J}ei;mEM2jm7G}QOYTrPtkJ;s3BwKg8O(ty;}&pCX0=?5=%n?J@`s9Vstle}@1|EUoa__M(bPg)J)1V#|>
^u-Okr53i=0JCT)h(MTc5-uXUlhrKW%}=t>6PLxsp&(f@sdP0nQ!0#=+8SV`y}UtRlk%~h$Hg%SQkoK%{}y|Q@H}}@FNeKr#!a$TMUz@JI1t@ScUFa10!`LW
!iW{-9GUv{LE5B>)yU28i!V^)qP9vD;B~2V6RSug4R8()5)k*AbQ6ZtMy_$rgiPUYu*%h%UNwDtRricT5f#JbN<DZ>|09Gx)oUdPNa4gEqAhwc*a${_;jN_c
Qx<9PxHceso?}0K&qpu@muhM<lFr+*ko^d31SP%_Z1tt&F$s%Lz#3^tJswy{pJw`X1V>RqF!p3z1+|qnJ{ol_Jx9xG0~^zTs74Rj@5k!pJe%TE<R<0wQ6iL{
|9aPE#&;plOb8%yRxu}>&@gxO$XPH7)&G7+NcCtBt5PR<>S0KPy?kuN6-3XCCr!nb)31}mChuCsm(0~`tW^P4e?pBO9zYFF!`NxR-%=}etM9@ApCYf_f7A^*
Vfl({L5t${OPt1f0hRfGA|MZdn|)L2DsRhdq+0}zgL+W9UJ(>cs<j1?K(!XF-U2#~wA@Mkjv7Ng7n9L9ICMLF`{*Q<M@xFB_)XIQdBIULPuimCz7g`gk@%G`
-f=V>CG6FDjE8-72I>O&#s?L|T@u>_id{Gl?0y<)(_)A_Ob#9e%;Yja<O6DIpLwP=DnRL5VhXFfX<vGD)rih<y5Xzd*&s_*s1Ih8pbh``OUSAsMF=3`JhS{s
8s|)d5R$!u49f?h;aZ4YbncN_fmKw#p5P+<8U<YnY%IxmMN-Q>wkb^1buX&|sw!gthgE+?=8|QFs~gP|O#w=iGc#Q}&-QUj(bK$71NO#=<+hk6%!R4=b6<rJ
5YpG0Z%*t4;sBreD=MKCr0IoxoEaAWhEZ9nM}_@GZ>x<;Fex~p`fQN>8TZ`X6#-1A4zf;k*3~j=g1b52Z?FR{zGFH(M8FgkRoN`G{jFkA&P}h#SC%O4<;%l+
)2OMFRo&>ekD89U{tPQ%xgCEfn+yt%(@gU1T=}Nk2xk8bbeS>W))=P^D)+iQ!Oy&vc#c6pa4Jwnu%3S3+k@$%)q8FwJ<l!^X#Jd!$qB)&HBvqn$r`ra@E^&N
DPoUBonIKW9#`dd&v7gMFSrxL5BE3>0ak&Q+nU8$a3Vuv>+78b*}^ZnA?qudEdLri;zbeKwLW{`)W*&3NiD5yIcfY2*O=<$#rV9V5Qhs(#XhA=n`iq6{8Va0
Oi1F@@AMzrJh39f;cd<2k34+Y;k^ms$c{ha7xkpX0Fa|hWlHilm(|Ub3}H3KD=mx;^%6zW>Vs|O@)O*qY8eOAgAU(`IPT5oZ0UazxO$XW;%Vg}<Jc~qh(7q@
CN_iU4e@^%y0+@Nl^y)RDhE!ECNv8BTM2_i59>$~2LjeFCga4jMk0Va7Ta78nYnkKm?c2WjU7YI81tQ`8#>*|Es%V2Gu#$!ca%ZP1R`=b_~WfY?@e?a9um#b
A|Y$4nmM}q$@syV>7y!~#=?`~1Zx?+X-%pzm+Yfh$C&Bx7hF{FkztT0*=+l1=xJ=#6JOp)#0U{G{;$^i9+XrDRmn8~ZZf}p)f|d&C?uKR(+*hi2d4DCTS&J+
{s<j7^Z`<!G0$OHAgVoF6-YGU!gwAF^LIpnnZ<wiHm!cemnn6;X-+uhy9Trt75#d1r($N<XHM*0QuvqSKU~uQ*(JisPCHYM$QjYY7^2=YOs&)*Xm{EP$XSor
69j-l{7(q%ImGUXE*TFrg^6~gN6j0DzHy7E-#r`Typ{Dz0tqq<+^dfmwLEk|ZW}_P!<qS)vP4GBwoj@v$US7xf6gpMgD-p3HHF|Nv{C^1eKA5cT(WLLM=mjM
pI$8hlvgZm4D$J-SknRe_<vW9tQ(FM$9c&k{*oXHnLLasg)5gb)4;U?f6GqQDu~<F8B3Z-+=NUhAABKyl-Tm-IhqLj5T3T(ptj<e@r%&kBC9k13Br<IK8rBf
y;dTzJA8G|rWLU+DHYjWlN<!~Vg!dJCY4CRhJ*t6yvNdx(sQyD0dA1caq$o=|J&Aw`Vqk%o(kJj{WS|`5seRGV(&tdpZu^aU+jJ-{3@c|AeR`ap4Y1Oe{q1)
5R{tHIS{|ZTo~+PxFNUR9c);s(*a!=?Mfaz6#%Esi>>MNkLJC&eF-<-fCgT0^s)p%`d@w>d7vnuF_Z#Mv2j7rexl`F|9<4=2bz<9yQ!>05N0B_n*))V0X<An
UGid2F(Bvz`b1}UP%M?6k%yM$ZdqwnEW{o0Nf}Z>_zRRo>t{!8N=1UDrKD(q#n}(HmG4zMSL@-&A|W}S64R+Lgo(EGwF4LR?fIAi=;RCl^h5N3p-6Qe>*39l
%Em+DMXMcAabr%D%LSjiWDc<6pjM@)Eo=Fw)@Lubx8x1U1g`y6_^IXB0N*h`7BAiD1cUKrXp`{!e@f6)Fad!|;anm)<*T^&h^Jt0n<2YdqHqxdRi<6=zQ#Bg
F?p`WES|ZeSL%AhakOPq*)XW<P>twB>dI~rcNnYXO*ehRkrDc|QSZLUodgVMtSfK6U|6G;;KC?R11x4;m-<c7hFR4!Z?Nwqe*phIU_T4mBv0dhNTTtow(iwY
Nm~FhIRnAZ(hwC&stfFG+<x^g!bKGJxKvgE(I{V!H^j7)Qu!aps|~x1ka6N}+8|NJux*wrBcLEWIie<uYJ&m6bRjlSOzo)o7pGMBMJ5b$&GdzBO*iM?vrW;>
e0gzlLd=%EV+&wNCC2dH{GFr&5%fqFjCeo|F}lac{xr(0RCmJ3zUUr7+tCqT`7C&)OxPSFQmw7o$`DR&3!a9bjN}tSfff-p$4d;>{9vTzx5Qf}GdTGG@HrFv
77!o0m%Ylt!SLU3YrAIsU~Z!-@_$p)by?cbH0V_7iEtuRMi}2lJz>T=|7Fbqt_g>Br_#T$w1X^=Ls7OX^N`|~1uWa~AL3eNpv&QFrCBo73ydHJ61R0N!Oo*L
6E3&kXHeYMz}JER)iJe)6rT7Re`L5*y7w=DK9}F_qk!g10zkE?!J(l*%k<uKl?r-P!e6=yA7%Kx680xkK8v&BaTtSKL8Ep!IcuTLfZu@aeKm|d6aBE)-t(w~
IiUM0i~C5Ekr1mRK|jCDg&PaS^q2kYUUlzZq{+c%*MVvfep6*W{4`4AuDuW9D=qE?9<SvpO1hv{T*s4-Ww;J`l`d_~4UrnO$1tqdueMwtQf*8CVDYYiyVjKl
Ix%Oi9AT+8+p_tu19$2N`_QD@x70#e_V@&EU<X+bReMhNB>Ano6|%BbgCuj9Vt#u5LgDw+9K(X+?h%;0f?&QFTu})QrQ_{I15&?|5GGHfVUuQMx(lv+5Wdu|
>+#fNt|zCgHLT}`sf4@s0}v(SM|453Wy(fVpgT~MN;%5|UV%=c`7gXl&1tR8mv7wboj}gz_gGFh65I&@MHfe+jFocS(4aD-j&78yeT?Y;_&TQx`KXf{{9RHF
R6@h}>haSj&R48?8epHXwk*sqljJkoj=c~>`q=LkqXAH@{?iYkzR&t!g`-N!@A2!2o1IKY|2=QyH7wVts$Nj0>DG|HYXV>(je7(5EGTMRZGu>g)C3Vx__B;Y
a1El>=S9Sn<8rzdQQ#QCQ=2jSo~RmSJiM4(?s6E?hXV!P7WECdN=~0AbDXl1F&=;pcFc4JyjPiSV-gn`>=mc)MBlc`dWv1;`FTlz7^sd6(l+%{RB7m7_t2yp
he_3>GSYqn&NFinC*+?X3Ddq;u;51$sXGxOD^;>~(iQu>1v~Nf<Dzyr?K}&O(@jU9BU!)cS4Y~G740eLtl;2Y%t8~6upW)OI1o|anuFf}Mo;FlC6>TMw~^6?
S(Ka_0Qyq(&sXc1UnBv*hu>2u(*Bi)*b{Wp5~OMg>8aV##=p|gwbsx2RA9{qvZc1h%0$8XZ<8abh<BX)RHPOfw-4{0o0jxaDG<Fxm!$~$Ged##eJZL%y28Qg
q~7n`tFFiKBTt1i^XKqh4OQg$_}5C#>@uf(76V1ht}!X-gJODK(zGB}O<`+cASDv@<9>s-<<o_ha*4GB&6V`^8FFuT-g{8dC@H_!&`@m_JyPehV&2bl=FL!z
+^)UxMY;@QeKy$Un|&qgj$Cm9ux&y+7-EV~dVAlstM=;4<x&YM&JU^0{k4{57;$+yz=}Ggh@$Lov7=X3&Au-Y+jf*Y;9*x6EQ<85xoY=?tTMd?%p?21GFMiY
h`1MV4iNlGFgq{v#nZ)hixLZ%$2->$;WgL20MU9PyNAeW-q^+;JMOX~Ctgc|l>M!)7!Ykksh6!9g=FOkQ2Io|gi#~)*MwW3EgkCXiwmIb(2!j%F4eRouzaJe
LnZfBiwJU1ZlKV$Ke})5@j&sh<wOmdi~qbmkXzO%&XpRe(D{d2;{IR0d{F;ST3xC&l{XiV?ZldrW#^0wVfE){;9usiwVCEDFoI9u`3+&BTh$kHS{(bNqPHGX
dk0d*Qi~m9p|yKY1-SCst-HEHy;Jx@X|~rOQx4uV@<W9$?g`QwBGPWCv&6o-ji@~@w4JiKZ&$hrYYf}znV#)YKmXz&)(fCZy`Z~6eBU0zyxiam!9sl>`*5`a
+Xj;2UGDV-)hcxV2z<T!)oH8eTmSAq%?O*3K|od<hY0wL?*4qjg{wHL2W3r6!<@)i1e?|ciS4#gcNVj}t(~^Xw%~`La`(40lpQ&13bx8=4!Hd}{^vt8s&zU`
FoAeJ!WqE(m;i-j(%-E0n2UV)fb?zZQ^k<iOGmkCjjJdqg$`)#AUXGkznc2rfLn45G~5)k9g2`_Ky|D41iL;l_LKHB(ym|dAh%2F(qNH(h4|t9Wkmkx^QKIC
3)?sa*jYg_g;H_X?BLSMVdnLD#_HIEWL_|2cxS{$a~Md)nTw_f-uSv8D>8UaFglF`6h}=GpA3}~&8rK>IuC@mId0JjIg6FLInY`3qZFgNBeZ^#$aHi~9f_2v
x^zb_>{G>H@#{5RPnHYHno!>b7Fp5!_L`|KZSfsN`|76pw2i7)XzO?bHc<w?-1>u9It|hf|BXRw=(X!8-)$EF7KwjrU7$K-3xZT*OC#$jKqp$~qqutWJvX`U
$T&YuW<3e7gq3jZ<WcKHnEFCo8h9e|ij)z-c{@D}z(rD@K})$tU}ab!)p8a39jrUVm}ywtr>X9mSWBXhKleZsF<j)SLxw~HyeX>WDtY>9bY&pe3x;66`9PH?
fnGb^<V>~c|D_YW<3_qxfW8-~E1~k3-{sD`z$Aint6Uc!o$De@m<#F$g+y3q6VH{(FtVhXn6dB(-U^)oFQsBVHiD={y62{9A&HGs@UVR|7-e&qSwP@w3QiG~
1{prW>zgxOd&Kp7nTe`BRgfmWdsqG&sMfGes<p5oeqp>)IqWb7(^;xvaMJo98k~4kv)P+SOQ?h|7b-%jVU<?FLlL~XvQJY8K!VT!`w+LFxb*X)OC5;vd32;q
!SvF?)bS&^#!e>(2DC-rpqI<)K<MOT%1AQ=$h03r!1$p5S|i4Jh`h-^)o$x7(O<k!p>y9^LAQBXkzck|0WQMjhbk!Pb}Fh(d^C7J3|*>}iGvwyi;Wq1!sysh
n0cxu3?B3GOnu`}BY#rvHCuyefdO=&sb#(Euyo;1QWXG!n;E}yq%w*n8j!=B%@C`@Y!sZwZ-{k6gZWh|V!WA8WA=A6@YzY4c3H?%Db2#=FY^D-vwHh&!j+F9
;v9xY100`WareL89YHUXoBsj7QWn$HPcv(6yGtv0>`_c1#;Cr}ssm^;jp^Yx1^NNnOU-lqGXMI+ELWH{yR12sfh)=!PJ6P_N_i?>v`wIC^;No#N@2%W_nt_5
b({H8;~Y=)STKv1{^_84(-VIqfns-KKG`i)-vd{`cD+s!GX<`SIH`%w3#t#F(mcO9zrEU(s59wR4_fIx_1BD!1cnVte3eYk!v9ALN)=;yWJ>%bkHoms%_=6c
&N`rql|*9`mg#AU(QQ(d#7+PWYs~4Pf;P2Cz8<^TntFQqY6$(GU|LUp+Lw?r7*CYZWnQwL`WIJSq71<>TDUYW@}iNbV~TU>R3VS(bMsEUwRd=Cn|Fd60{caJ
C9jxiKlW)Ijxvaan4B%U5v&s_h^z<>?{#1|?T2-U)vNV8!9=W7ABKgzWd)GXrA;Hn56TC5-MCGLG4$=mtRAdWPa-en>yoZ)1&gkPQ9nEpCt=Qbp}rBA(C&^*
3GS~E*mvRueL^v9%|f|C1!q3f$k7d?%r9#6@uLOKRw8J~)8C1_&`16MTW5N(%E=+c*##PG9?f}_e`0TpGhz8kqtGQDEpRqyvsJ<dge8JgL(laTnLfW7DVG_D
X-y7vx^eL*ta=hN>v2X*qPeQqAzP0rPc`RC+9`kDz+@Rls1#@_jcdTfO_7PDXH%6vp3iOdtU)z_@Hs%SHMBg;&do~4l*QUY>AVM0bqGV?0M2L&pzRH*qagCh
7rz%o+QvuCL>XHcbD)N?>$k4)wG;exhuX=v06dgBts{M^*KsiaCLSnJm)+RYzLkB6?jtqRx9QbfDzVufOQVOf|D{L4)2gLz_gs=;<>+=ycm5ua)Ix+%-Ka&_
;q%6ornWt`O3CvqihQny`A;E4dbmQ0Y-5C*-D;u%LnPWQJOkYE8t+_h-n#6zRUQ{k*0JiBPs<ZHQK5XxyywBr*im{4Rkmb-_tgM}S)%V^R6Ito0G^}}`o>y;
;VM@KA)YUOMT6yq=0hB<6PznwT-49bHQL1o{1cI(vfc*e*V+f&l5@_!0po28S1~+8sx4#1+13TiW(KH=`uSy2EF&XnE5~J&K?ReuaSg7Wham{o7LH#E2ZR%0
Y&t}#e;mn-H83g)YTF!uUoQ&~Uq1Dd6<kxKc|meu6*c4MrvmF}1y>RXfFNL3UzSCLZGSAW+5M2z!_;O3XCTy?8+5CNZE?SB6&a^!9Cgo|+nYIhk7Yx70%}5i
Yxx&+RN8mS&iNS7Z%IgdTd}Xa(r|COZTaqN;CSD(8ta6|R{hj<E0UL?N`EY&434e{pXix^Gt}PO26Bg7nbVD*=dl%Q&Q{n?gGkD@o`kq4;5r_cC1Yt7XTSn5
JRn*~OamX=OM=9#gKQhJ;yfD=#di9@{<Y|nF55GS=GP<b?whh%H$B=hW5Fc;@o5*;k`Uj={#gwaL?^yu-u95%O{nvf`Ey81QwN%3C|AZB;DrnDp_`PXzF8_s
@m@!8X-Ui+WtxJUSCO~I3p1m?C12xKgd(m+;=Afr-Ii!R=T(zO8Rt?C6%X|_G6rk=LavMn5#N(2H}(V_AF@*yl-3#6MrBCtsa(jS)<K0XJ1?k3^O&&Ebp-wM
7>tT<V~IB&FR?OxIoR$`mhm3M6a~W*hXK3&VSF~3kfUy&XH#Sc{haL=@Tx(vMZU%87$w1n5jpNm`cX~|u&eLT3FzqMQhp-vC-Guhc~D`~Py`pqG#(=#mvOnR
)l~hlWLmUVsH6+0W30k0x!d!~G2dswcb`uws%(4>TabDFl=lYX3!Y5v4FCMOI-w8am@l_b1(HrWMS_B3*@QFKoJ%4(wRA=u1{Qx4t6m{(ZEYRg8gLyqGI$xn
VF;=(wRj$(WdByC&Wg!2blYPL*2H}lP|X=?sl=JV2Cp9#gFo200(qGIe(~YaI*BURY>INhABmhXErm(&X)!HsIAJ?0D=u2a2^YB<O#J1zfG$kmnI#*t1Ziyw
+FvQrqqQ!tYn;=PX!aup@sF}e)I9!?=x~-VwYApglZ?o6U@`xiqXGs*b!>mG3y9i>wbzYINd92%ugmzN$}<fL?jaU+{jKnIkDxnzo1k)7sbEz!g?(F(?dT>1
eL(HX6;<5K#nWl~M}Ym{Q;sENn_`4<5{#H`GoWyu7+esGg9Pn6CaPN@q^G@(6GuraEa3i;+ZptACKKI@hqs9&Z;xnaX6JcRGm;mEC~8o7dMk8?Ou~iF%)`o$
kRKr`Se@~}M)NXe3bq5J%lSB~_-y{S+k$p{8B1Awi*>jxESgofXQX~2rVS&k77VT!rfqF@Rj(0|KHhkORsL}g3;k;YLO#~<hipY2!E_FR{&{0=8CCW<GGUd?
Oj>saIl&vUU7$OPTUCt{$BhY<qlXC^`Q+E+5<fL~(_Ez(5<7UEhp7A;UjYB2e7~yodD>mxnAql}8CLx5<Jau<9ODZ+Ki<@_DBv&4AAf5d(2>{ffnZ7bd`4XN
kiGMo1$2i0SZ#tzY*ol2J3F=cR!Zv3gd>~qX(Sd-qj5rMTb6GG(YE@0&Ir<igP_ltSuNJqamqAZUKSljv5RnOK0TeLz(;J>aYWg5`(kZ`I80GXE*|OTWOH0*
cw#6hD>yK+W3XxCPWzkDyakto%q4lL(Yuj^NtR9jcj7b(wX!smGTD}u+y`S|I|j1YarCV{c;n=liW^`vw{1JZ$GHVCxel95P-3@6wX@k(^=<_vi&SB8I#^=f
zg+%}j6Vk)ZHM6qgnUatM_-atoEqscB34@-g?c4MT29jG?d<T_08iJobe;_vtHlU*0Wop9gzcO|O6RBAt!MBYl%};oaLny2hdBS^X95}rCLLkCvFe>8YE<9C
yEbQ-P$MN~RdDWgDc4xR2FhI#Lo-cg55lsHOHTs;@O(=9T_R6x=&#Q~JRS>a>ziF0zMKNN4wy(XB?Eh51=K;@_gtC^xfYn<dz^3eca@+PG&3DNPHFdfcZC^;
C`XC=d5k$>J^_CS_<!t7nmxE)mmIm4C*PIU^<l3js4WdVaX6-@KtEW@d7cwGK<k1aULtJJDng9yDsvYL2uGDS{j4e+_ST#oo{KFHdH?Fc1owqY0TK{RyuP`L
d`kXjWdV^847?cvOpG~U%A(EBOhVxpCV!Q-^Rv?L6l|cVc@^l)TpvvXM%NKngOU0WJ<U#o_Fa##`91?ouiNnyc^HR#j+}-P`uD$1*6v0!cmdfoq`ocmk)h$T
d;cmIg$LbTGSG#xfv9KHe5;B*85_@img}4LviZ(B^RTf+Zx1XIl_5~UiFe=q74Ju#Uvj~q?8P~j{-A#k`D!1*B_7C+$aZ)X8w%4e3s?vx8p8ya(MpJP|HW({
Xw{jSZL@LVt6QicotNLDJFb*WbRjHlg!5GZQr!bwT*obgoU~}MKTHzlvX>ZZ^aipl+qX|$3%_=0oR|OOl1})rA!MCsqkaSH=qeXogLgZrq(ToJ+{UY7nbu3L
J)J?|Plm6lhUlZ$UJw=g1UueZdP#xN4znMW^jD71PFHAaG+UOCXe}fkC}q&LWnyyec5VKEdExqf<M9iHTC=V{CZ{6d&N2ciGrYbCgo0(+pCR((f6R9n2oX<}
oN9$Wpt0b{t6%#|og}7PTiLx1pAj*oM5XI`U8vm%<EOgwCk$D$`WYuS+jO#IF1KCU<ed-}QtpEc!r!A2Qx#Dh!FX_mnQx*eDkhqXDX*6vAcDcl5~wGt#-svo
>91N+L8$vvKqi1u4BeK2e5_0~JF~<kZ%S30_hI`3k_E9+2o_H9d4d0V!3VI$TK1IFNiV#3CWqkRW>;_~Eulqt&6)-65g2iLAJ3p($!l}?Gig1pbm{{M1rU`=
`@WmLY2y;TFbUDmcu`jaq?qK@l+8AfI?iC4uu&fq{Lz`Wgz0n63jAMoAQ5;(AKMq~+h+E`GM2tuz5Zombjvs5lWfQ*ChN=dXR!~Tv?$fN7}KWf-+#}!U4usc
2GEAb{sPwX*arC0M1~6@b>RcOVxi`U@X+~q((;*c$vOqm+P%QPT0#8f7SF#7x>;<G7xt_s*LvwG^hHjZ$6)mMz%0;JPZr%eUUQtCsQ)RY@zLlBA=|vC+Ph}h
?Wy*@%!=`^X~4_^Brrwql2qWk&^@<9GjYub&<17qGV9#kQRnYucQFwO7VaBUipe{$Z4B>hXZ|f_xlL1-2SXC!Fd^m?v9Qt!S$VzS|D(LfSmd3}2X+<3$*NuH
3NZ%jcknr%<)Gm1<aiO(cO_nV;v!r>Nn?<>RdH>4z?%*-I}Kc3!SP#GNd_V^8_573up1fSbMqL4FDg)oA;`{`pRwqGWhOA0-e%#2JdVw`5wQKTMsXCcp<xsE
#W?x9O(a@tHU1Am3w)R51TPE2RZwOLU@te;T@uEZV{}VRL|G)_U5AtgBkHsqyDR^<Hlk;uUp0Yr6C2>_9YL$kMfJ=Kn<SN>NNFPGWe?RB+(IR$kI`o1@+I%(
)pFk4$5Jz;8HzdL!80HkV%j@v{1}U~Im!ESPnybrO{A%)I51|qDp8E?=fiT}?aGQ$P-nI{h%iwF;#BU(yid|8$f9l*taT2w0QU>!G2$tmkZ(uecMyzNTr6dr
vBLNOsRw7vu;nSGR3`a+t{Ld-1QlKnO0H;Bs<dFY^2*jvh{$`N7MASe3!wN6aV(0)iZ#P75Dfp$r+JmwJH)Q5Is<<R5^1%f8aQSW?Xm9kBAqC_oDpD_jQq)7
r;6o=FL1zLqdJZAbz1m-TXYvx>F$N!3eEI5`;3ZelC)h`QFsST#ER5N5=LQz_AC}zKR~6YH8&Mio3ZFL|68=O#FI1uP!eRtk9!z`n*J#9D1Ax`mY|6K0jGyh
s{(9FP>BU!PTj;G5T(8p^1wfO_Oo_@x&ymU<Nb$C;dLYSooa8F3oda-OP?D#)U__1$@r5fG_?aHg%P}f4FLkMAJhRY27$2~e6Mfejq(a+@aJT4^CW*V-tX0K
ngygLT>yZWB&f%!mLh8~ZZ|D~uVJXBh|~M}Ce7^MF8WUA2>(CKqpESL%U>E$cm;;NlPrTs{|?Hyyh<0;q@-fSlK;FEzf=5X$&IJIspQZ9hh#qMi#5ZG^x_L|
B)tgmF3kP;D51?ildmqYD=p$xlcs{^GvHT8@3{77$K0|PKT$Ryim!)yD|W+w;u%aq5LbwJ^B%S}ce;^~+aS(`Bs(JI{0vL1`zvC&DK>Iu?O&;@lx2{G&ZqLS
d2r6tLB3w2qims*w%#WY&N34Xu^aXW(8f#bu%efZ-(jS&7SZN-NOox~8SF{3w)B!L)~=%IKF_G@L>%+a1OU*<9~Q9SFB%6RVEG?`ihck9Bi8MR*Xg~_Dw92=
0FaD{V{1+5-Po~kCi3x{wAvH9J>246$%9adZe0eKc6uqyTUvPnLKUQytA^<ReP#h+abNzpeyUz7Afml{u`e*T(BS%B6?8uYv&?Nb5xxDR4X@b-zK>o$+>=|G
&8yXo0zrbSqaG%F7%k*NCf1GR446(kgvX<r@2+kw%JBQpB3~lpT1muDpZyB?tH==vX(Q5~;3P29U<oWk%d=$TnwtY6Co1I(2lMVso2^L}982S_VcSYU-kJc!
K&)t{H~^It4f9}}LdkyO0Vct1)(>k-07*;_!peA01wrs9=)&<}e;I7Zp?!3!T4iRrf{*5$=2W}0tYU}V_NS&*Tvw3wILD)J&!(DGYWUi-Pez!%wFMjw*>U(D
feBkYbzTv<)8#vYOnTR+V4+JoVSe$Lsz$4}L}CM&TU<SBRW!Z{0KgAy<aXxvj|U7@)lkD!E-$n73s6~#zXH%u^my^qQ`PRY#1p8ypKC6q*iDb61MIrW9nNo^
a3-p`IcT~eijkQM^Qf?Z$V&^#J`FWlVDm4)12|XwAmfWZY(ygYVTp6V!U3)%r{}Ulq4)j~*3fhelZTi8HNA-~4P8ZDY}&DR`e8Koei6Lj46FD(jUWGePJA3=
nO|ddgK}OUFAS!ZZmw_7S?8)ROT(1B-$%MD5iR!9OpoC1lXwduwlvH9m|GRc%h_)p9jnYBQ786fRk8dD$vjQ_;u@U>gZU$8GFr2;T;_iQB)M}IWIVos=1*{K
`6aw1O<81n9tz$^gcADGeb(8Bew;(%-D5REu63>$^yR?|D04ikq&G#a@vH?(GtwB7Crf5V^JPhgPovjO^K3L4&I_qo2MaBHFS?duI_J!+OD<&Z#<35!6*`_6
0<4`Hs<IgIu_Tg+411;0)K{FPy35W`XL)SF5|(EA<HK_Q2VkBZ3sD!nHH6$QN(HDNBXc~yUy6=pw9hV3FX;uo+to@>Z#<irF-hl^G^ZbsVTftggS0gd>G@cq
ED!T{mzFj~?}`Ql|L%HBn3Dfup4rF)`~0iXoE`h;^cc=pw-{h=r^b0uPOisI4puYSX~YT$!OFQRyDi7n1yar9)?L9)vgR!%N@_vI=Yjh_ZsP^l?0424UedC{
rZ$Y3J{q2e3TR$Al+ZnanbOR#O9jzZrNiwLY{2O^Pduan=)s0*^qN2UTa7xOQw!6Mpz71#_p1(3fM{6*%C8e^?NWE@lIc|aUig<Vk3)OD@LV-Mk@0Zb(DQkB
?Z+=1r$dYwK(m>r<oM`*2O=L$LiV{RlB?l!GVgJN+j?J3HeHB3#4MCYns6XZ4_y5{SUjO4%bFM*sto6+p;4oWMu!)TYR7L%RgO6pIoQ=?l-A1ATOe*Ycgk3W
lHN)%tWW7)XBCVvGs>^l8B;+?tc(43p~3rdi?b^i=)d(kUBEey?2_0G#e^5zoxB}SJ+{9t*P~Q@JV#bzpnt)U2_AcOnogs6AMB|lx~q6!8ww6E;W*KpTs?-8
_fUu_b^R1di>)PL{B+<WGkjLYXa;?J07IAXfvuc7F%eMd9_02!#xEeANlH&xjH_S^d9b}r&K_whHmF2mQr{+GkONMu_OuA6v|@V1?C?@lL$DrdqonMiZB`O|
DYP!LvH=9tKatkAlCeFgR_i&WkjB4~&bLJmV&R&r#a-_?s@$q9QnHmJa2G&vyp7n+q~V+O$kJi0yDM|<aUfx(?tF)cO^2mmm64`V25f>jkXUT7uztCZIP*0P
j8eUlm#{HIL0(2o=d^;_xf-HU^6_8>W+kq6zN3|oSp)~KT5`C~)^#tZEr(5mz?L*43n4k=7t=zr9!JRni2#dfIB^K)oUo7h9TT<e!`$WTi|ApA+!YB1UK^~D
(H~h*of>2+!L@ujE`aIJP`Hk|+0;ODnN`Hm9>=a<OWlhJq2%cmzWHd)7OF2mgTRg2lwI^jHVETRm&wf(j{X%bRz7bO8eSTha34XRb2v(Q_@;z=$<4eWdjFq_
a^e8jb^kbHG^GR(5_MgM;J-n}Nam;tB|B4*r+kYy#f7BGpmkw>c=o4-RJS1<5h^IuHL~{bhu#`}jarD5&r$L46ndpaYR7l|VBg;#YdyHY6=d^|o--bKN^=WD
d)4FMH^n3O0d+2XU#VP<QIcI*lxmlhSa`4?sL~kpsSdc~ksx_z#L?uPUii7_)KuqK;~vkWEca{4CW)Di?kQ33h+Gb|Rn8%9RBM17lbw!gMsNmG7?^3C8)-!}
AP=G5u<OxRm_%SL$ylrP6J4kwOR&B%#Z4IQApeVAkSiVHk&xnb+CV*(LCo-d0S}9EhWJ#Vh83t}RqE58y2MTpcb5p;?x$IaEFC21?HQFmoui+xlLHP7I3ghk
)a5Z!*kdsN`@5>5DXQl~{tI@7Yw@<?br@<R@4}Mdw;K%v_=;kXhDB@El@Nr?nDO_K;aEsHFah+Rk~^l(PR+Fia5>33g%#!>^}u=H-|WYo@T>_!d8G<_HTb4Y
BDsXnHccVBm^bxpsCrT|GzL|Ah~*`2Q$Bf`*nk+Quv1dzTGxsg^;7LQT`9OnIIw*34_>PpI2K(5`@H9@p7D{XF*spW+j1VPT$S6|Pk^M_>#aq{za`vp%1&wn
`sqE+IdpktGH8^s;(C6XT+`UP$$E3VY_brfSrFkHkWx2g0RdSUh2v~=rw6B#*XIlVFrcBtEfF1^BJFVpMBndVO#SaPDb{I@Avu(PNc1YY`6W=cWZe1<7xFnS
S@u>+G_uxmd9W|0_2*O6cD-8#zGYEPH_^+8R)p{~9j<jG-sLJR_=&lvnCHl&EojKWba^TH1`<0%?B9B<%_m(aa2BtENA!I%D_2Q6)5+%2*nETX)CV^j)d>ie
(#D&oy&fu-69}GXUojVY8sj4S#q1+y{d0kow*pA{Bu(kq$D4DE%thKgnZ-AM$F`We;f6HWD-w#QCxP2}`isfDr`HNJELv;idjc{MG=FEq;0fa5P3)5oco3Zu
D68acPyWwGK3dQR)>#iUgM6@D{}QSNfy9~N2I_#Ht|`$%e(~bc<KZ&?MfdC+0c?7l)|R%7bE<NDuB^RBYkQ+7167}=i~8*j1Lt~PpCZd}+D+aGrTekJyLNgv
V=sY9?IrR%IB)p-DJX>qqczJ<{JAwlZ=2{WwPU8)^AJx%z2Y8F^i#?MYeOHM3<b9z%*&(0R3bCEGnD*9gs<o1>})Y4HT<doHO0%}S7m!pt{<IaB{gV|@Ef;k
c-CJmlXz3T7gp}_+-xtW3{&>m@A6+@;EO}=R3w{LzUHu3`l;xEaJ}mWXjhnP23Kzhg$*q7UknyQlFx{H;bh`I)80VgD#`L&Js-^3QLyULmyiV&*CV7T0<hQW
THSX$c+tsy0M;SkxQ;;gn=jj*Z!Q5!!vGr!4#+-~tgco4)67XV6iMMsr()i!NXLU>>Cc%@YoZZOZF2cPD2!su$$(u1h@tcDy(u)kcOT708O~)XV5S-srXD0(
J_;9;uv8-;pIl}7M133(YJwv=CP=l*<A_rRaX(|iA=8m5!!KGP`#)zil7yNHLQLE##If#=p7tiezs1EXLFR1{y;n(8-rb4M$L7fTw^>JQ#)F{PzsdBo8b6T0
dfFp7TCA38eMg{GK@U~-jkB`K02PYi=*7||?ReGfOZ>f1Hsw<D;cFBabR!2>^$AzyR$o2eFOZM|QQ_pgQs&ve0N(*rSbd(;5hMd23^}ou?a4iqk5o3~!;SwP
Wkwuk0(r|Es(M62OSP#;E&6mwfYIs)wC=vPnN#WqeD^FcgE8qI9+UR=$LsN?8z^#@E=(d1Mx#E*e{4g&?#5Xmx*Pwgs*tNXgU%89_7C7eP%e04n%j0qQH}4^
`i?jrJ*|rNk;ZZzxfT3RD*=J{ha75q{E%fV<Dsx(yIu$ye<CXKpk^@nlEjo5M<U_b`|PbajSeOj@tx-2o83pJMTPGO-|IR)Ykx{}os#J1=9}7-D=|Fv!4aM=
Y`4BpapcfxSO+r^fLLEBqP;+XhzuuQFu0IuJs4U|gtRzTkxAu_3Epyh&N4hFm2^$42Ap|bap$9m{NJ?B92;@H?W%Z9Mt`$fE3{y^{tlXY)80A)ZAK_rGut~-
=BR`BP7o}l3E8;u+)^#~u*L=vh;KhzK3>T1@a+;SF$l1Y`H^W^{u2`ZQv%8fCj6Wtk09vgg~MV0Fj@bM?O#M>xN=Uxy1qvoazn4-_13|Xv5`iJ5)r2DkI7EJ
?`ZrjPvUyJXUSTiA#0!{Eud}DC!y=3kuC5-zP}XC0M#|cOcA9&C_a|3E!*H2YzQlu2KBrY!;m@4tjXX*#ZH&C4drWOf+cB_Hlgm$9n8<S9xbyK2+dLpQ4oFo
eD;bD=9M$y%wa#s?gK<$=X@yh;Rh;vHw9qa<9p*7kYEj=F{u9_TSdzS&{J`T<#Ku*&DwQkynIo_fxw#@kgsYP+@eiCTHI}|`tl{zhhh9sqo0V*^2%ZJSD=(m
9Lnbrpg`G|fp;$SXkOlJ6J#UO0{|e%A7*cZRWF@S@SV@NN9&uSx#eIHX#AWr5((@XqBJs$*V2T$%;+>4r?xg2u0x(H*8_iiE+g0^>X<#}m*ut}rX<7c&KZ1*
wpzE&V!qKu3B>0Ng|gb9ynL<;V(|<!)iW1Bs~Q$9$2r5=F`}c>YMnC!RF9gg0Icc{BjM%cr9gh4giNQPZw4>MHeP&yOq?C5J8!;t2_(S6rs1HIh7ZnUtR7v0
pFP^Y)+(aJ{%gxJuO=EOwQC3{n)!O#N`VH)W=aoGgD0dHysqOWAk<<Wil(&>iMjoeLnNG3Hs<7E*oXd;2!X79UZ46Y`-N4n{w~8LH2U)LUVr|Zk?-ni{J~H*
)0#vZQh}l8#E|b#HrrPep6YQhoC|oz=MBwMvdU~fSEeMlr3LS8K36?2jJ04Xqj*P?4AizC^?Bb_Kwru+e!eKo8j)}j$y;s8^zc`m?GvUmzOC(m?RKhEhKNPA
03tUoA3%o9S8wLQcx%uN7m85fw1%+Pm)J)&;FBUzT^j3<3!88>=xyZKZ}X~4I<*-hefE;nk(u$li^t)QSom(LC0JM2z$$ksmlvT6`LQ}n8)bLYJyE5Luy8)<
OJ<6oe;!g{f<itK#PSaLCQcL@2Y**Gsz)^R)tJ@zDz?}^#RW=P%DDV6`*fQ9)C5l8G^ANbhkiN0c%Rz+yBbCAVOXJQfcB`kV|w{ST;IWcPeAe`=f*t#h(nYD
;A3qprt7Roo9;PKs(fR#HpnGoR9m`eaj%TgBE3~~^hnTqQqBiTL?debW@S036!w|%hPSfG!-Aqco}7%&Zx5@|*^c6p8L2;b>7m?49<?+uPCAC=4A30yhG1pt
Kbc@Bq4B8u*mXv4p=`XsA7vnHY_&x5$nOE)gONfBKP<$ZhLc^-;PS-{&eBYv&x3*e27Zz2`!b6R?RFO16*Qs04rqy~@`5t8{mT2*2nbfbi9h8Od3l(Q@y+Gb
#^H2x)=UyCysX>liGQ~@V1%mzZHbu|8M`S(16<Bv*?V;2AP^gp3|RD59kKOm3M&omg~c}m0AB*2yS0}zYuuoe&;X(uFb=7P321A&g%Q;N4?&Nu4M}r3Mr(wF
YQ<CYR}t+ut!0A&eQMlQ3h)z@?`P{l9fW*I;vA{$p1ghoRr=Ofg0t+ONYcqU=AI{pu+o+#jch1PMW!zzu}fQ+BjWXd?;(=zit!4@8@E&7c1ik)CeejFw}Cg0
roJaYe;M2njw93JAcUBDW<;Ic5xgr9Of4s6_spDYQWW!5Mf<l6q?V(W&qfaU;4K!$WVMb?hv9ydiWYRrllI>B;xB-+2~^U5*z8syvmpLfBf^}@&qalwDB$6j
J!;O0ra_NJUFSb@A{L#}lY?!HE|`W5vTk~rx~K<A#M9n)*)hf`cb`7A)I+b!)T)24p;x9Mzun8Q9qxM(xr@Qyu0KG)EC{dv{&bZRJ)4%Qr;oVL@P;4MB@ZD?
O6=Zs7Tz$?W!pOirijfwA-i7n+H%3kKZt)*$5ajTu+T*GaxT|b>9paj(>{N?eMRr$J@bj+uK@AXg-*`6MZjPqb$3<rFQb<_xU=`ec_V@9?JfEK+b7^yoigp%
dyQtjnh9&TJ-MssMoI(pVp~N|JP6rq0K;)?8Qyeeuwj5Nzy4nvPZ(TQQnD{-iot(n>ZXU#Mc>u*gC?gabEK*krj$!nH>yn8-%-G~vA5&0`T@4}x8&@UwhHte
9Fem_WbeKVncFSnYQa>$9yGuS*{Mf<1^K>RXqaaP3da=<yOZi~`KMfezIa>NLD=!29lAfQRAN1IRxU9ZgmqeT@)v596zJ84KrNmauoN*CSMNWH%F0IGqIOwL
rXF3|Hk@hDt)d1mTJ-})wRnf#?F77^?=EFD)Nkq%*rm^;6VqcW4(-7ogqf&XCCS<$Gg|oZIZ9#jQ+r&y|AO7=<u>~5cY3?1L7gX^<vBF_VXgJMCS;?_G|TV1
EZMGiBMqNwDgT<egeh|@2X1BMe28b1Pc=nGf&CQVy+EGz**$xIQ&62<TAYSF;OZUJf&=Z*edfKW^TJ6l#ge1=UZ}+0IUZOlC(gxq$~a=HiBT_{SX9i1@}|=1
d=lbgxK~f@eOi>=a$IS4@}NhO11CAnYD^At3>_TY@A2|eA8J9;aI%+#=sl8OUlOm8dm+Ed^HYb=;3RE3*N^}Ma7F*ieR#dM7I^*?e{aOE-l-BG$B?Dn138u_
A~nVD%;zm!5=$eJZ(%NhY(}e^GsR!f8|Rt)m8v`KS+_%0tP~fqJ|*O$=CJhn)Px1XxTdL+Lop|lV#)of2}xuk>VWG3zcc4o5i#_-7kha^O+%&vfkv3K3cTx?
)kWjWKUpqD=aN$RR8-}ZSHkXKRBO|={-EFP!!%Jv+&wme(1=d*WU=_fE8pnJ9@In*fS{Ox%VTIw@iAe3tgMDepNbJbPE3*g4E4j{(to!X+e~bm&RntLwuG$L
X~EE0L!%&pU%-%?u$G&@pQL;UK#U;sexHs{J9E*}!;lMa%R3JGn6LpFKW`@TAMoQG7rXIGAk~-*t(SwdEyNDT@n#aa0|cQ_ywa6kFnz>=?_(!rDNTe^@-Gg$
*W*(Heh>w+micCkty;U7!fE((%)j}%25T(-FCYfW<WY2^tfAA;=mWG!u-xq&Kvn*2GDw8#rfKini7?-Mm=kz0a4A=|Od60j?^-o<yxowtH<ADK_wh0OxPg$p
q-*CW2Q;@!8|2HHY>ZMv`KZY%Y&a}uur_eqUI&MgHvDyXgrF}53Gg1O#g3?dZ`z3Cb!+#p*c5Qc7+spN_(Y=8a=E_Y%R9bZ;m$7tfnARBPawa%HS)~Se~C@V
E)}9jmYi^P`MXmrE$S}EP`|@iY>70uk53|RAJJwu!SONu;!(TKtTf}fbs3+~S3Y(ei^G!oWW^5N(Vr(H7r~6)z~P8`e<fWnbBw899cVz&Z#MMLDB?tfSma*4
MZcLZZ9FT<U5+VxSuBnch2PXm;B~5GODiSS`R4<xKL9^V|9n1U!&N1)qgDLIPH-NME{6`wPeY?0TRXwFw>aZWUeYo_O?z2wF8bZ<od5tV$NnHq@xGWTnGh8H
ydD`Zg$<dRcujd4x$b-z9y*K|IF<?sP<$a-#}J}YasJOQiIPkuwj*va`uj7Zx+AGN;ii+vF*CIf80klmiG+}j4i_>q?=>CwOk62)I`~Sw2AcmjB*+!mo!Aky
*Z;*XRSEQ(x*7py$?VM>F-T3*iBgKgMuf)94O7MPMdFcgwC*asyR_Vb&#v}8uY4-Q>iZh1@<?aEzF2SY5DEiwOC1r)vzoUDqynfy@v4NgjYvb{fEG9RW<U&P
rjPCj%9MUrOW4apYLSt-o1?=TylChC>lDy|v0<h<MzB-uI|@}Uz5bF)&s}z><z|z^WJo|;O~rtm{9wYaljP1BpO%b4CxxhUqdu#miR5eN?#imsHserR_>87%
m|#PMv=!s;9=fQ<a{x5R7E3o~%O4VP>VsD51qf<iLgG`#kmR_9!_~Wvw><`?7<7v#3`}b%0ap5c?q6}|{Ctttx;vaN?c80+vUmkiP-vd4DyrPmv(T_?M|Zf6
k0XR!4+>*j_GQqWn2ZaNWMRo3B4&@eW@OA>03YVZO?Qyv;cJ_RaDw+3KqpRsN34%<b_5?zeSa5h3}A;r(~LlABk?uTX|-{s^r0Ntp=j?Yk-D)nFzuPg8dnqF
>DBj#PbwNxi2wL`bI#+}0v~?H-}a{!bK#WokW8Sl-ZU<UJxobe53aJ>nlA3SKu<P6{w#uImYM{5XM0IvQGe_$6P;^nV)XQKn|%QuI$FRF)6_ZeZgUINuMqUm
_Ha?vJq7P(;R4xcG3ho;0n1ANapx+ua-aXIDyhZ(_uMVsS@;hEA>Yxm7o^>zZp^K_>5%D353)%jnGcR_&tQ$~n-UVlx*!v+t{%>veMZjn0+rJxOm>JKwZ~#_
a_%s3G~itC6Vv^uZy>>!WXi7g*<{aY31r2%qg1zhoGpmCECbVJiP?MmNkRShN@^8DEHt(g!J&sxuzc`kn(*zs07f0~Dc?;Cp<&aKM3A@Fr_VSTl3r7fU{4_U
A{_dTuPN{%6VN~-^-euO+c+VIJ)_5SqH42mw1;&&n0*^gOm*Gq>t+*F=X<2TfKw}mausW4#=tJ``*@5D=!t!_?{1=i0vcSf;)Z*JgUZx~n0#ptoM4P7(vI-A
auup9GhVGTrND`2kMN5u8x)}QB=kJEE1?^v<Q0%BbODSp6czy7JgY$^w;r>)ACT{UHAl=Mq_KX!h$sc^R&NT=Blu#)bW58^`f8uR7apyu@u>k3XV-YN5sy=c
WJ3$<=01e_wlQ;8wROl?-lR20^$BxUk7K-KO&bq1MM<>)eisv-Lazu-r(iJQPlG&77p{0oue*}GTa=M-B)Zyufm`lr$ju*!6~WV*M?X!=k+Z1)6s{mcKbF^)
&%{->h62OI^R9zatoQ6vrO-9Nf@8Phkj&OVp21lp*<{Tt9#ot%cIEM`3)xh}ZZUU_wsH|7XNp@=LuK0b#5IJMW8D*rFB@EIblwwi#fBgDCemHSX`~*h;(nuV
+bojrY+NLOP^>-Ke88-w!fFI}ex&PwjdTG@aH7G&N0S`#E9bG(Has}s2x(GlTtHP5Vz!vMWf!B}cp1d4<@mB}W<+v|CIvr^^MwN(6X+J{5ArkN#7Q9-Ga=UP
f<%R-dGW_Pq1SRo)jt}^pu*x3+Zd}X6p_Z7%JT_BO^QzP@(x6TP&dt7QG}`QgZQC$R3lg0NZfcqh{{7nI*+&qM^nCXW8TXMG}E2I#1X58hl%ORK6M8%E+Vxm
R*J-~8snuo%)MT{PR;*RQ${72Od-qAB15Jjgnaq-vk=lK(QWJ1EH&x7a(Y;3)O=rls!i7ePc(?UaY5E6I|q3J)|EOw*W~R-^^3~hJoewa?43FHI9-pn;T(~k
gpwR<bQZ#+p+LyY`&bNQA$Y|qtV40O&{LUnC8p=2v4LmqUwF;Y?uULm%5sbU5{<jOeI}L`dAi4lf3bJ9KL>7#+U|3Z?7F=~Qh>PO>c~WOxg7`%Ga}>*8-5Bw
w*&MD$k!>mQO1{$9N@3GH0A&eZln=eL=Db9Bpe@hyq}NcdAw4@BV!f`)YS5v0*Zj=Ns|ZNdH;N?+Gkx#Ndbr^<quT$$%dZo;cL>qfywp?4kKUu^}NVYuW~FQ
>|TcUS8+Yd;Do3!$+$Afcw<L1dW*O}CH%{AHWJTh)F^PDv9mDiS(rd+Oe^v{ZbDGQ^)07_MyEpGdnpM46DUH5QFzm##~;u;e;m5}Lla*lF1md8u>QbO23@{o
4@Msgz41t217<{|lVWj1G_U203qR_R{=QYwxFi;^N9I+if1buQl!E8vzFK~y-Rv=D3ncQ)(Gl3rAbFdyQuQBG6|EAdvGV_*r0>XhCI8;t@8)1y#p7G3#jfCX
@nw@ShA-0lVpsocX%w9!Bvx?$?_pPS#|)GEkPX)a(;k6T{$2}QH?enS*jV^NMx7ga(32fh(1_PtaGX!DFF$P;LZ|jf-C)rX{)@fI_Ah(@1{_S+Q%8ASc0A=$
EV_b%VAm+7t(UbwX6koWbqDJkK*|BCbZ&>sI)G?h*DJsoU&7jPAW^@2-Pkq;5Ctx9@p|j7j$o@qz)h7{iTTU-ycQ*Gw&^5L0LA#;Jmp_H(S1H3^vc#J;pOiF
(!gXlvDj>V>$Sr#*&X$YwZ98Sv=YfsIKwq@l%DjSuzm$~2kMJt?q&;Lr9s_#<l}7HS<L`6dcaYsPr>8vqw;sofL2$HVr_X$-d3QJGQ9Say2e?Sgtl<<;rN*~
<qbN}1R%r|TuY*~tGB@947Nf8oq6&16o+f{kDl5xXGq?`htc9yVIIqvo2dN{!f0XWN6i9(m@n==Ihjh^LZ!6iZ`M~!+jC;rvaI4SQh5)ihyQ^3KeaMWLx~>k
!tYC+l9Gj<P#+_!ditU#(i=RT&aKhv&AcuFIibw$Zd$rpEr^SL0+86nHwVeo1sBY{y287ET2=0pbevD66YDBxj%EbR<pL}X7`&2V;uSn)E9vMhT{N2-^>l>G
{PO?%LpPyb%r$O0VCS(QJ7OOiZz+P@WWI=s^`bR<;AtAkk)jX|#vCLQSXX$*e7}v^yH$Mpp{keli;;7M_*(am!MD0dfgK(5+>x58c;-*rWLhzVifk=~)4q_8
Ci06`b=!|7vNccsxP*m!BHXFM;i&!|wl0Nt;)w<n3uj|uCEz7Zdug8ggb0gbbd12GV)NcT&ISSJ#b~|I09R7lIetS2t}dwwinN9?QOcThKJxhM%|7qojB2Ja
7mw+hF{w3(V`suaH9f=)2b%{_JhcD^*D48dEt*)g>th~Vcgl;GFug#wKUH&U3&8__Tp>OXy{Lvbna#T#t7&24Q~>VNzhae0Qt+DsxLA83vNvMd^$L-_tp;y;
e$Y@<UBg_C)lYOwS>(Y8qGqBh@9wXLNV`D3XpO|?M(jqX1sGc9fuW;a(hoj0b7-}_b+Q3=4SpU4(jGGPbCJ9yVMYo0+gST(pyRG3?k=~S&B#zC@2Jv2O4I8a
0SfbxCohP12*h!Qmfog$pG~Plqxka%a)rMJm=Vs(FZd9pWOJyRU1_11<DI++02vaiDgwOGHYyo*I=KTU6Migecm-2!a5gA^AGm|-4HwY!va3l$?pMvLai&(c
?+~~-Bp#@pCsRBy{ebp;`$uahmekG+hq2c`&R-78d12)~Y8J(*&*<}Lf+ZCBxR%q(z6)&9wL3y%Du(+OcAWf*ya*Nz!74=qN^t;6C0aAU$2v`S{@`iE_%%T0
WXp~nCpXlMyj%l*Yb=)WYA>?<xlJZwJ7YDhS;=z>z_r?qJFj{yPCidy)7e&-R#}`V$GyuRC?ZbFL7C+KTG@Jdm>o?e*BvjJll}(vZX8R!jC7634Ks-5Gn2{+
2i*#H?9aasbtw<Ya<2GQbMTNJnxc1*!kSuP_(3@?&G`I1_S^5KmGee(tNA%SxH|`*2ok1zCZVAsR+V1iMoA#$@8OiLt&@-6OG!uoNlJYD&Sr6Juj%}~wiHpb
d+5KLw6fQ>5Qy{Lt2<tGDcrb?hO|w?7>`>j8(lSUkCv1q0p4fbH!L)nWA4_FqMS4n!h5@$&37Sah$X*OF!;Hm{0Yaj)Ab4BJw7m`yJ+47*#BNu%!CL37DZk`
98&t$tI>@$ZHG$~t?9VaL>fgf$~FyrSj-(K*?$)u6F5o_3^>H=Y!JDuHWptxbER=L5VWdZo+rxti3@}nPB&!<VpBe_XGXLT#Qb_=FxW=FSYSdN=82CN<^A+1
G<k|9-Poo9)&vIPYAnJ4GaB>jEyj&2#KGKm?a81de4wA+_M&Nz9lRkpmxj~rWYA?5RIRQ0+GL*cNZgt%G7qZzfG|+F7%RzEvrQ^|L<qS&D@7xb0c&H+z$5Kj
Sjz;Y2lTUa#G^JSh;Eq)wM)VfLpNwU%J<*?vgBn~a#~P0JTU;p{tzaDdjd7h)e|Uu<v3h@qZD&$=;UaaYajLAT(KPj#s`yLI6}T_M9)wFq-+!jP$cp=GG(_2
kZSkkjQliU?6g7s{E_(6R&I0?<@cNF7CO1{R7<hm`x+88ijKov)HFSX&H@{#)YTRe!T4@_z%O6l$U#>quLI<Y+Exo;Dd!qTVd|QV_kCM2z3}K#3d^<_sGDR?
U|xlpnF9$?5NYJb!~`)IVBZ*}LF*%?s&#71RR>UIjM2VYL>ds!iW*7lRgE7K+t#!*qm&0qNMd6Frm*}}Vz6ld`c0R|3&jTaVWk~R&Ue@By9CCcB?FXn!QZr>
v-YMI|6pq&@R1vXWz?P9zM$-Cm;GVsfhrX59%A3UCVtsQrU90DB?={g)?Gx%7C^<orUzoiEo=nhuq&oQ8cIb9v;YmBrg|X0g`vV=6=bPAfdhj1(LZBqy-#Va
Y&j@_cF!pDmVl&1e)_Nju`(8zT#hjHfCob!=tyuh?3rRa>X_W;$etKv#u=$=?JMKw);K^*jSfgqWY@MVW5m&XsXEfbI=)VhWAe7X>$lmf!7&#FA(j{Btsikc
^i}CP32gO`pbzE+$yDHHo?V>$MEicv@JnL3{5VR1K1h=D|0nXRC=)e9Z$A&49+pXF1#oq32jza#yK$GU%qbNTpciY#(b&OXm2C)fmaug=^V8nN{+28n>CzqX
bZ%MR(k}OcBq`<TECBZ@^foRytW8LtjfJHveYKYenW?BFwi2;cnMte!Io_OnAWL*S#DIHz(Db>Obb0;Ujy2+_B1BGrpd#C%3g|>U9&Ba<W4oVEOB~R&sG)V7
RQ21}0)?w{&~UOq?#pa_y~M47qM@(ULccWQp6suyYno-7;tUso_d^4h2y9U}Z-sY93efL7U_)#TVaz!e+-L-dz}!JYQ6uV1_N2e{`qQ3K1Vq15_M4Nm6KVm>
>^q#NQ$>j<c<;n8nkq_l%u0By{m$|((C!lz^c=nxV|!<e+BmJmeiKo^<pge)P5M8c!!}m5fG0bXP|+4P=C8n4Q-ZUh5#l$i#~M@=6nIwy!#olBiG%ksaK}?7
8ElfT*ioL#rmp?@h`q-kdaR8zQP-|~QikO#MD;4&uetlY)QqCsXv;(}j1K=cNCI1~zKKlc_(xKir`zsywadq0Q)8sHE*Z9`s9qmNVhb}BskK^R^DS)M$gI`L
%JJ{+rT?>}vF;}S&_c$z2d}{q4L4SJm-o>g)fPNZbDu17+Dfs5R}g9DDX0;r95;w~C)cODy(@qR7wdkJ;Y08)D0N!YlfG2sT7oW;du}Zhc|gI5I5dZP<kzk}
sWvbn5V&V*c<e#2;{u;zEDqe}<Vy{_E48XtSLYay%F$ublXyk(O&9iC0QR^K;c7LF_|9p5_j(ZxM3ION(n(H$iG%e&s$C&_ZY%e9HBhDu@ZX6zQS(=(jiB!s
rY$;(8YM{Sl%FP;O1Ib9-uqNhcheY87x=%{SqKXxbwPfR?Iv0+Zxj%v;?rCx{LLDL15IBk_%h{`iX|0kBTJkM{oPk$qH+t>azu6Ae;?g*N_jUlTFq*jC7o@d
Q#d>h(#16M7C;UYKt8eLLie+D&T7$&g}UZkArlBR4h|sx9=Z{C^V%*$QkpPVKbE&OZev1MIj^2f((=`gVc+(RJ4<q)<%{WS@|eCo1H@l>!OxXSM9x7=9gX!Y
ct)y(j0Vq?<l*To$02}eN<uNkZ-k1oJ{w>e7IBosvM<}bV5w6cxu^!G{(l^?_S{3R*GrQ%kys4UmK`|p$P0G4K;91?vn}QClU(;H+n!BlrWrnwn>(`IT$(+X
=qsIRzevKR8HPIxfavyq#htTG%^Q(c1pxpYGRs<<f6LLYIPe;@h-556PeCQZ)wvn0f!wcB2;O$Z@kIPMKAyS<@2H2jBc@+M@#FS_I~Cdi$IwGzPEan9YZ?Ev
GGq6uvU2ixEZS&@io5#NY4UZ=%R;`@5mSy0;MN{Qmvap_Kyq{Bb`0XbbQ(oiAW&B}CV+HX-YVva7_YtTk1<l9xJ7|o+A!9{ch8V52x?Q#dT%u4KFGr2ytH+R
c6L3!V<$8O`p7teH#G9q=mob6C~z4rNl-RH`~gYoByR3=n76@pzavLw1`E}$zo{MVHP62}`)a}yjlxAh-)56b6kOQGKpEKU3okTzlUjST%hcMhan|(bRutg!
f%od9f&Vjy<E3lO@yS41vjn1LAdE3xLbtU$kDea!5*kq4lq#Ui*3J6|zq3+$C53N#64KlI&*~BZ6$iM{5Fz$57h3<SFqo9W<O!mswBJ@L=;Qz}Qfb#+;7HUO
zb*{MyzO1%tBo9NX^qlIA_7Hj^g*#+_5=#%*ZA~@uLBvR=@V}u8cNH3eaBW66A?7Z3|jEu`F+-PH@&^ofx~5fgZmz1!J}z{7wgWm!b_%X@y6=Kq9}Kyo<h;#
;yTqn!l2>_qRF4=#r*j~Mn&k%dos((8+$g6rc1#(tT}jdr%n|xJF`SW)13T)r@oXiRP{>gYwineSLjS0SGUmXN}qx{_MV|qnQ-w|M5u!9c;DJDiN-Jh#iF;3
mM$mOHD<RKWCNXYOv<bqT2mS}ZdfS7g=F^nJzsQRTWp(E0XQ_2b?qsPg(%Tv#=2^*g&fAcI@`QrhY~O8gU40Y3a9~>bBgc}wT?s2WHUApdRfDp(AK8EzHUCU
D*V69->iYh!8|t`U8l5^jhQhx>18I>aY2S)HWQ|ew=p2abR?)GEMeu!$?~QuWALkWA%o*f%fr1@+F*$^@4Ue0>hJ&FL-=_8D0OA7(uKJN^$27!%`OSeEEI*-
vp(YvMLI12{2zu1$SrVrat%j5%{B^eCWeiYBUWw*+`(;X#fGgxa0T9oOYSR|<-w+lNQC?=cYErR>0^1+5IQ8YuTPIjp^lM-$i@`dB~(PT!34){M!Ie1P|>pF
2ijc&+OC36rMF%VJ8%qL6A0j-9bdPaP<dz!wFW+%$dr{FAl?2UD@@XwIz9RSs1&Mh$T)QTPXVPjLls3dQji#?m53A30#EskS>L)c4MtP%r%h5m`z~r}^3ziJ
qfz)`q}W%i$GR64rtTMRr*uc3d7Ftj30gMaHbQAdH?vDEwb#j7?M?O!AJE0>@vfwD`1kT*=_Z+N+@pzJyxrSz(PyEkZFytD`RoJ8^Y$8%?B<vC6>2$gXd^?^
_x&jvT*b%NPDA|e#|(DqOoNt@%tNTUnxT7sxp5Qi)QfHVnP7p^^K^}z*KNpxync#b@IZQ!0ie+S!7;Vu-oIkhKQrs$B$E2}*7ogA;BnYPUMNRHpnJG16S&M>
G|>O=wlZ>$Y!h>i6VQ$A9cvmB3p<nmwlpi!f@$J^!*z0pmJlBI?Ut->x4erWt0eq6Qtr9ds63J$IwwACV`TT-)FdN=guo(~%#NjlYaub=BUm>}R^zqyr6o3$
INwO*x9#jTd{DK1pX1{=hCuS=e`ulF#=eKsG*Gul)`RlnDHv2Fa%eKrU}ufEYdSMt-TeGAFrJZ`|7!K;AQc>gnlMPsZJ)jzOV~xFJ$f>%SJ9(P=9z+r{g9Z~
_tfnv%wxKpR!|u?UcCGu<P~1^E&nWUleeq>6?r;1w-c+28Nh*5iUWT0$?;`cp>|Dmqw6>~HG^adqVP4&S`QK<EskL|L$j}3nsamldf;dq4)tcGbqsCKUy+P>
eTMf0Z21)MUoy;Vja2U3uqPEm+KSHVJ+ch@SnN}XExRDv-AOYmKx8C<t=-A@1jqt@`c#ycw{sR(MlfqDWu%2c$Nkm_IJicaM8O*nI`vTRDAJaMJ<4V(E>!BG
+T+s)=Qo0ahFXnd{sy*Xe4NCosWZZ~1u9iM6)C{v@>x`i+av)Yu9He?Ljv0Xu|`2QF%+jUjWL)vRX-hK0SMnd40G8QVfzsxKc(&POy;oc!B)Sj(_GfuDCBju
Q99Lyu>jk&&Wg1uk)-LcEn*#pa79*-@VP1L!2U?~7W5dbq~Nk1PIs>ArzMryp1K7*-LVR;7yVM*Z|=;~D^ceVBGcW_(Pbq;=LTwF?Y4bk+LPF`iJJAs=d98p
=S1G-l0M&Sec5jKv>tf}V!l&uvG20-cDl#9v(W()m!Vs@E6@=l8OK*@m}$`Dc%U5{tfex^ZveZCs&%}9j8Bx>8Jb2rqpan`k+aoyj%$*Qhf@RSmJvj=sTb0|
3a-k(`_1WlO)YBu9PpbJ93bja1la;zX{YH=8Q+dQN4)22V(0WW7ES!R5pK|Eh)bDtEckX*u+dB~ffGJzrxUwlO%GAf9s*?|uXx(SM&$x1^qQg1#`n+m6=GDe
F}?`8+GzE{Oo=ljhL?xMIX+&8WHc_?+>CRi<0++5L@>w1Sd`7MsExTlap@{Q$8O-QS19?wB{q<vZ&$`%h|D?LG32eZd<clGDZvD%G<s~H#@YyJL@UnG-VPv2
k<w_KsVq80gJi7^6#$)bzh+NQ;sxD);oRa4_DqHDvY{+6+C7E+eSISf3K(ij1hnhtWAR!2beVS)IRTybZtL3ZkFTcmCgSLOv>0$iQ!e%*7cjkJvikfQio>{!
v9maKvv0M;o4a7fpc?$|CrIUQx%=2S!jV@zM$jDn2r=DIw+6CNr)jr(+!SS}(O|)}cHpM83U&y7&*Z}791<xB>W26YGr-FB+}?@{ClFM4L1))W^ae-t6%5NN
d_4xnbKnh{;aI8}dqNdfK}QMnjrFxjnKf9C*RP1x-`ko2C(-GXR4Pn4+XAWL2Q=;#0!m#;UY@#GmdY+Q14O9p$EE^XcFZ6K5{oS76=S|d&3UVwkf;gpyB+$^
uqnu#*;81+ARtF|lLNe<fj6XP19l+K=78s&137mD^}wvM9P!(}FEOhg5F5qU(`S0HXG@d=H~CMK)8?%zUhOv_*saP0mcsX^6u#>hGU_(^{wb%0KAtvQNmKSd
0{`EcByOCIpNN!c8=KQVe0?wOL5qL5$6!fZ5Z~D2r9Iq3_s|#B!(HXtE^HStuQmj%(%IVZ#|a&#nLk$*;~saoKD>O&d}25E7M#V>PcDGSZv`Jk1qjKsU~o5d
#pUB~{1+V9^`zgtKfpe72)_G2oWl;!pZ8ZU#Hj|1HshI^7=F*vx3tkGYRYlS62P3kH)B{1@5=l8KZ2bSI&3d-E~LoU2IIT<fc=cU2+@+m^lO)E{*VKuyG<I0
#;Q$Kz46@nJe>D0X6?~c5x8#1i_tBC9mC!K5ZZW<<ed^7gyq_1QfG#-OvNj^6Up}t6eI+ld$)Q|Wq*6^*(uakIdHO-8}pBxo&#?5*IeR!RiD<G!0Q5?a5x_r
$4N>Tny2!?h;6g!=HdUT8rcSnPj5MDJ7m=jy>te4O7*ku?UWVYTGs@fPQJjWip6<xc_i0~1Q0OzlO97qEV1okY7mfJ^=o+?Rh{4Fs&i7`Li;E7L?4deBS(w1
#w3Ve+bOuk4Bh2qA2(fK>tNG`tohH<LA3T*KVsm0Y1Wf;hxAl&Ao3lLgS^771K&hV`z(k^W$HXP6+wWeA3~3;Osv5I+^yH<oDWOY`iZO89ne!fjL!h?MGF0_
*-191ZDjyy#5f)jLELIThp#lSLMS%21{XBo-sH8Rs~TO{tM8O<Sa?MU+F&YXG8s+oKUr(XtX9)Wl|_NY(*4~)=4=t+6^(DXj|d$k!pOr*<W^glKM<M83$n88
njT9qzhJ`L%m-cEMS88{_k<6NVclwpJHa4**QQhfXVo##k87i^X%p_=z4AcrITa5Vu9Uh5rVApP9l)0R2vooAvAU+OZ0VFmm+uLq*8Dk+(BYp1#B7i03mN>S
rkzgGM&620G;BK8Y3aw)5<cr6`EM8pJJyk3=N{inH}T}zc2sE3lkV#Q5A2e9>#$QM5O1zy2rxGr<TRYB3nqaCyIdGQ;*8z61(k}a{h(bC;1gX%^H80py#Hht
KngiTTxK%}SRptB^cah<WZ_8Xe7IOtpv78pFlGV#gs4IaCT9>vFpxaFq=XxHhg?RB?`nsL<L&uuIT7YP_Pwk)fqR@Qsk=!QdD?f5XCQP44>dfv-HsZ<kq^Cl
w4L9vdMI6MkRg1<O0S0MV_X~l9AP46`&m#27#$0DeeuySlX395+Wmv)-02EBX<yD+cek_T#T|K(R&T@0#WE&~IUJho#QXNpt1cC{(oiJQObE0(&>2bddj;Ya
`AkA#!NZ~~pZTkAGth2$@$b~38c<vxHmw`D8=5oemGBZXRXn5Sx{^;U2>thaE=)a2e_mdB!a~EsyJ`cU7ecnTw7r6LPE%<U6e)+Srg^fbez^(e+Eid{ou?x7
;VrG=7inCxmX#!-LBpRp3HhR~kY_qRrv>b}90R#U1c!-z#d2i^N8L$=UpS764!5MUJjP~1{+_8UlN!wHm@!NE5Lez-x9c=0yUt18?fGOpe%IdnAf0LkbG6<J
GR{Hy>AaVh11+QA2-Ye=;-Frey2@)m#u=^1c@LC1h-7>BB5jYy&)Ih7I5GNj4}Ji;pQ#@)q8iiZRiXKe-Pu>ns=dRT0%v94YwN`*Q|TuzPJ$>-VlRM%?qTKu
V(F%gNHfv!dt%h5Z3&LO;VQQe#7C!W_guWWfz1a##<LfpOmizO0=g#|J`QK+>+Drroiwj6YRe|1h}D2%Am6+Z{d#wG1BBys9GQi_DyZt35-dwihFOA|jlk0L
&l?p<@`w>esV*fi#M0NCqxz`au@-L1(5TtUK3Yu{C++0PfQ|#WH+u3H9vD4+=Z4HRSyM0oq|wmo0cx)>00FeZ2F0Wt&=9`8vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/treino_gpu.py')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/teste_r8.py', '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())

In [ ]:
import json, os
m = os.path.join(SAIDA, 'manifesto_r8.json')
if os.path.exists(m):
    m = json.load(open(m))
    print({k: m.get(k) for k in ('teste', 'download', 'tempos_seg', 'duracao_min')})
    print('detecção:', {k: v for k, v in (m.get('deteccao') or {}).items() if k != 'por_periodo'})
    print('remontagem confere:', (m.get('remontagem') or {}).get('confere'))
print(sorted(os.listdir(SAIDA)))